In [ ]:
from google.colab import auth
auth.authenticate_user()

In [1]:

from google.colab import drive
import shutil
import zipfile
import os

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/reinterhand_5114_dataset.zip'
dest_dir = '/content/dataset'

os.makedirs(dest_dir, exist_ok=True)
print("Extracting dataset to local Colab SSD...")
!unzip -q {zip_path} -d {dest_dir}
print("Extraction complete!")

Mounted at /content/drive
Extracting dataset to local Colab SSD...
Extraction complete!


In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

# 1. Environment & Device Setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Accelerator      : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found at {MANIFEST_PATH}. Check extraction output.")

# 2. File Path Resolution Helper
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

# 3. Read Manifest
records = []
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(row)

print(f"Loaded manifest records: {len(records):,} samples")

# 4. In-Memory Image Pre-caching (256x192 uint8 RGB)
TARGET_W, TARGET_H = 256, 192

cached_rgb = []
cached_targets = []
valid_records = []

print("\nPre-caching 5,114 resized images and 3D targets into RAM...")
start_time = time.time()

for row in tqdm(records, desc="Ingesting Samples"):
    try:
        img_p = resolve_file(DATA_DIR, row["image_path"], "images")
        kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

        # Read & resize RGB image
        img_bgr = cv2.imread(str(img_p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        # Read 3D landmarks (24 joints x 3, in millimeters)
        with open(kp_p, "r", encoding="utf-8") as f:
            kps = np.array(json.load(f), dtype=np.float32)

        # Compute root-relative coordinate target (wrist = joint 0)
        kps_rel = (kps - kps[0:1]).flatten()

        cached_rgb.append(img_resized)
        cached_targets.append(kps_rel)
        valid_records.append(row)
    except Exception as e:
        continue

cached_rgb = np.stack(cached_rgb, axis=0)                     # (N, 192, 256, 3) uint8
cached_targets = torch.tensor(np.stack(cached_targets), dtype=torch.float32)  # (N, 72) float32

elapsed = time.time() - start_time
ram_usage_mb = (cached_rgb.nbytes + cached_targets.untyped_storage().nbytes()) / (1024 * 1024)
print(f"Cached {len(cached_rgb):,} samples in {elapsed:.1f}s (RAM Footprint: {ram_usage_mb:.1f} MB)")

# 5. Segment-Stratified Split (80% Train, 20% Val per interaction)
segment_indices = defaultdict(list)
for idx, r in enumerate(valid_records):
    segment_indices[r["segment"]].append(idx)

train_indices, val_indices = [], []
for seg, idxs in segment_indices.items():
    n = len(idxs)
    n_train = max(1, int(0.8 * n))
    train_indices.extend(idxs[:n_train])
    val_indices.extend(idxs[n_train:])

train_indices = np.array(train_indices)
val_indices = np.array(val_indices)
print(f"Train samples: {len(train_indices):,} | Validation samples: {len(val_indices):,} across {len(segment_indices)} segments")

# 6. Representation Generator
def compute_representation(img_rgb: np.ndarray, rep_key: str) -> np.ndarray:
    img_f = img_rgb.astype(np.float32)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)

    if rep_key == "rgb":
        return (img_f / 255.0).transpose(2, 0, 1)

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3) / 1020.0
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3) / 1020.0
    g_mag = np.sqrt(gx**2 + gy**2)

    if rep_key == "sobel_mag":
        return g_mag[np.newaxis, ...]
    elif rep_key == "sobel_dir":
        return np.stack([gx, gy, g_mag], axis=0)

    # Multi-spectral Di Zenzo structure tensor
    rx = cv2.Sobel(img_f[:, :, 0], cv2.CV_32F, 1, 0, ksize=3)
    ry = cv2.Sobel(img_f[:, :, 0], cv2.CV_32F, 0, 1, ksize=3)
    gx_c = cv2.Sobel(img_f[:, :, 1], cv2.CV_32F, 1, 0, ksize=3)
    gy_c = cv2.Sobel(img_f[:, :, 1], cv2.CV_32F, 0, 1, ksize=3)
    bx = cv2.Sobel(img_f[:, :, 2], cv2.CV_32F, 1, 0, ksize=3)
    by = cv2.Sobel(img_f[:, :, 2], cv2.CV_32F, 0, 1, ksize=3)

    gxx = rx**2 + gx_c**2 + bx**2
    gyy = ry**2 + gy_c**2 + by**2
    gxy = rx * ry + gx_c * gy_c + bx * by

    diff = gxx - gyy
    disc = np.sqrt(np.maximum(diff**2 + 4.0 * (gxy**2), 0.0))
    lambda_max = 0.5 * ((gxx + gyy) + disc)
    dizenzo_mag = np.sqrt(np.maximum(lambda_max, 0.0)) / 1766.7

    if rep_key == "dizenzo":
        return dizenzo_mag[np.newaxis, ...]
    elif rep_key == "hybrid":
        return np.stack([gx, gy, g_mag, dizenzo_mag], axis=0)

# 7. Lightweight Pose Regression Network
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int, out_dim: int = 72):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# 8. Training & Benchmarking Loop
VARIANTS = [
    ("Scalar Sobel (|G|)", "sobel_mag", 1),
    ("Directional Sobel (Gx, Gy, |G|)", "sobel_dir", 3),
    ("Di Zenzo Tensor (λ_max^0.5)", "dizenzo", 1),
    ("Hybrid Vector Field (Sobel + DiZenzo)", "hybrid", 4),
    ("RGB Reference Baseline", "rgb", 3)
]

benchmark_results = {}
EPOCHS = 15
BATCH_SIZE = 64

print("\n" + "=" * 80)
print(" RUNNING 5-REPRESENTATION BENCHMARK (5,114 SAMPLES)")
print("=" * 80)

for name, rep_key, channels in VARIANTS:
    print(f"\n--- Model: {name} ({channels} Ch) ---")

    # Generate representation on the fly for cached images
    rep_tensors = np.stack([compute_representation(img, rep_key) for img in cached_rgb], axis=0)
    rep_tensors = torch.from_numpy(rep_tensors).float()

    train_ds = TensorDataset(rep_tensors[train_indices], cached_targets[train_indices])
    val_ds = TensorDataset(rep_tensors[val_indices], cached_targets[val_indices])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available())

    model = PoseBackbone(in_channels=channels, out_dim=72).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()

    best_val_mpjpe = float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation MPJPE computation
        model.eval()
        total_error = 0.0
        total_joints = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                pred = model(bx).view(-1, 24, 3)
                gt = by.view(-1, 24, 3)
                dist = torch.norm(pred - gt, dim=-1)  # Euclidean error per joint (mm)
                total_error += dist.sum().item()
                total_joints += dist.numel()

        val_mpjpe = total_error / total_joints
        if val_mpjpe < best_val_mpjpe:
            best_val_mpjpe = val_mpjpe

        if epoch % 5 == 0 or epoch == EPOCHS:
            avg_train = train_loss / len(train_loader)
            print(f"  Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {avg_train:8.2f} | Val MPJPE: {val_mpjpe:6.2f} mm")

    benchmark_results[name] = best_val_mpjpe

    # Free intermediate representation tensors before next iteration
    del rep_tensors, train_ds, val_ds, train_loader, val_loader, model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 9. Final Benchmark Summary
print("\n" + "=" * 80)
print(" BENCHMARK RESULTS SUMMARY: 3D POSE RECONSTRUCTION ERROR")
print("=" * 80)
print(f"{'Input Representation':<40} | {'Best Validation MPJPE':<22}")
print("-" * 80)
for name, mpjpe in sorted(benchmark_results.items(), key=lambda x: x[1]):
    print(f"{name:<40} | {mpjpe:6.2f} mm")
print("=" * 80)

Active Compute Device: cuda
GPU Accelerator      : Tesla T4
Loaded manifest records: 5,114 samples

Pre-caching 5,114 resized images and 3D targets into RAM...


Ingesting Samples:   0%|          | 0/5114 [00:00<?, ?it/s]

Cached 5,114 samples in 130.3s (RAM Footprint: 720.6 MB)
Train samples: 4,071 | Validation samples: 1,043 across 56 segments

 RUNNING 5-REPRESENTATION BENCHMARK (5,114 SAMPLES)

--- Model: Scalar Sobel (|G|) (1 Ch) ---
  Epoch [05/15] | Train Loss:   135.08 | Val MPJPE:  15.26 mm
  Epoch [10/15] | Train Loss:    61.53 | Val MPJPE:  11.99 mm
  Epoch [15/15] | Train Loss:    41.76 | Val MPJPE:   9.51 mm

--- Model: Directional Sobel (Gx, Gy, |G|) (3 Ch) ---
  Epoch [05/15] | Train Loss:   117.30 | Val MPJPE:  13.79 mm
  Epoch [10/15] | Train Loss:    54.70 | Val MPJPE:  11.26 mm
  Epoch [15/15] | Train Loss:    35.16 | Val MPJPE:   8.75 mm

--- Model: Di Zenzo Tensor (λ_max^0.5) (1 Ch) ---
  Epoch [05/15] | Train Loss:   121.92 | Val MPJPE:  14.30 mm
  Epoch [10/15] | Train Loss:    59.60 | Val MPJPE:  10.90 mm
  Epoch [15/15] | Train Loss:    41.62 | Val MPJPE:  10.12 mm

--- Model: Hybrid Vector Field (Sobel + DiZenzo) (4 Ch) ---
  Epoch [05/15] | Train Loss:   121.46 | Val MPJPE:  16

In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# -------------------------------------------------------------------------
# 1. HARDWARE VERIFICATION & VRAM ALLOCATION
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Hardware Accelerator : {gpu_name}")
print(f"Available VRAM       : {total_vram_gb:.2f} GB")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

# Official Re:InterHand 24 -> Kinematic 21 Joint Index Mapping
# Dropping Thumb_0 (1), Pinky_0 (18), and Forearm_Stub (23)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

# -------------------------------------------------------------------------
# 2. HIGH-PERFORMANCE GPU EDGE / STRUCTURE TENSOR EXTRACTOR
# -------------------------------------------------------------------------
class GPURepresentationEngine(nn.Module):
    """Executes Sobel and Di Zenzo tensor fields entirely in CUDA VRAM."""
    def __init__(self, device: torch.device):
        super().__init__()
        # Sobel spatial differentiation kernels
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)

        self.kx_1 = kx.view(1, 1, 3, 3)
        self.ky_1 = ky.view(1, 1, 3, 3)
        self.kx_3 = kx.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.ky_3 = ky.view(1, 1, 3, 3).repeat(3, 1, 1, 1)

    @torch.no_grad()
    def rgb_to_gray(self, rgb_f: torch.Tensor) -> torch.Tensor:
        # Standard luminance formula (B, 1, H, W)
        return 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, rep_key: str) -> torch.Tensor:
        """
        Input: rgb_f in range [0.0, 255.0] on CUDA (B, 3, H, W)
        Output: Normalized feature tensor on CUDA
        """
        if rep_key == "rgb":
            return rgb_f / 255.0

        gray = self.rgb_to_gray(rgb_f)

        if rep_key in ("sobel_mag", "sobel_dir", "hybrid"):
            gx = F.conv2d(gray, self.kx_1, padding=1) / 1020.0
            gy = F.conv2d(gray, self.ky_1, padding=1) / 1020.0
            g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)

            if rep_key == "sobel_mag":
                return g_mag
            elif rep_key == "sobel_dir":
                return torch.cat([gx, gy, g_mag], dim=1)

        if rep_key in ("dizenzo", "hybrid"):
            # Multi-channel color gradient via grouped convolution
            grad_x = F.conv2d(rgb_f, self.kx_3, padding=1, groups=3)
            grad_y = F.conv2d(rgb_f, self.ky_3, padding=1, groups=3)
            rx, gx_c, bx = grad_x[:, 0:1], grad_x[:, 1:2], grad_x[:, 2:3]
            ry, gy_c, by = grad_y[:, 0:1], grad_y[:, 1:2], grad_y[:, 2:3]

            gxx = rx**2 + gx_c**2 + bx**2
            gyy = ry**2 + gy_c**2 + by**2
            gxy = rx * ry + gx_c * gy_c + bx * by

            diff = gxx - gyy
            disc = torch.sqrt(torch.clamp(diff**2 + 4.0 * (gxy**2), min=0.0))
            lambda_max = 0.5 * ((gxx + gyy) + disc)
            dizenzo_mag = torch.sqrt(torch.clamp(lambda_max, min=0.0)) / 1766.7

            if rep_key == "dizenzo":
                return dizenzo_mag
            elif rep_key == "hybrid":
                return torch.cat([gx, gy, g_mag, dizenzo_mag], dim=1)

gpu_extractor = GPURepresentationEngine(DEVICE)

# -------------------------------------------------------------------------
# 3. FAST VRAM DATASET INGESTION (UINT8 COMPACT STORAGE)
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path}")

print("\n" + "=" * 75)
print(" INGESTING 5,114 SAMPLES DIRECTLY INTO GPU VRAM (AS UINT8)")
print("=" * 75)

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    records = list(csv.DictReader(f))

raw_images_list = []
raw_targets_list = []
sample_segments = []

for row in tqdm(records, desc="Decoding Images to RAM"):
    try:
        img_p = resolve_file(DATA_DIR, row["image_path"], "images")
        kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

        bgr = cv2.imread(str(img_p))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        with open(kp_p, "r", encoding="utf-8") as kf:
            full_kps = np.array(json.load(kf), dtype=np.float32)

        kps_21 = full_kps[KEEP_21_INDICES]
        kps_rel = (kps_21 - kps_21[0:1]).flatten()

        # Channel-first uint8 array: (3, 192, 256)
        raw_images_list.append(rgb_resized.transpose(2, 0, 1))
        raw_targets_list.append(kps_rel)
        sample_segments.append(row["segment"])
    except Exception:
        continue

# Convert directly to GPU resident tensors
print("Transferring dataset into Tesla T4 VRAM...")
vram_images = torch.tensor(np.stack(raw_images_list), dtype=torch.uint8, device=DEVICE)
vram_targets = torch.tensor(np.stack(raw_targets_list), dtype=torch.float32, device=DEVICE)
sample_segments = np.array(sample_segments)
unique_segments = np.unique(sample_segments)

# Explicitly free CPU memory
del raw_images_list, raw_targets_list
gc.collect()

vram_used_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
print(f"Successfully staged {len(vram_images):,} frames on GPU!")
print(f"Dataset VRAM Footprint: {vram_used_mb:.1f} MB (Only {vram_used_mb / (total_vram_gb * 1024) * 100:.1f}% of total VRAM)")
print(f"Host System RAM is safe: Zero Risk of Crash.\n")

# -------------------------------------------------------------------------
# 4. LIGHTWEIGHT POSE REGRESSION ARCHITECTURE
# -------------------------------------------------------------------------
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 5. EXECUTE FULL 3-SEED SEGMENT-DISJOINT BENCHMARK (100% ON GPU)
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

VARIANTS = [
    ("Scalar Sobel (|G|)", 1, "sobel_mag"),
    ("Directional Sobel (Gx, Gy, |G|)", 3, "sobel_dir"),
    ("Di Zenzo (λ_max^0.5)", 1, "dizenzo"),
    ("Hybrid Vector Field", 4, "hybrid"),
    ("RGB Baseline", 3, "rgb")
]

experiment_matrix = {name: [] for name, _, _ in VARIANTS}

print("=" * 80)
print(f" RUNNING SEGMENT-LEVEL BENCHMARK ({len(SEEDS)} SEEDS, 5 VARIANTS)")
print("=" * 80)

for name, in_ch, rep_key in VARIANTS:
    print(f"\n>>> Representation: {name} ({in_ch} Channels) <<<")

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        # Segment-disjoint partition (80% train segments, 20% validation segments)
        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=in_ch, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        criterion = nn.MSELoss()

        best_val_mpjpe = float("inf")

        for epoch in range(1, EPOCHS + 1):
            # Train step (On-the-fly GPU edge computation)
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb_batch = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx]

                x_batch = gpu_extractor(raw_rgb_batch, rep_key)

                optimizer.zero_grad()
                pred = model(x_batch)
                loss = criterion(pred, y_batch)
                loss.backward()
                optimizer.step()

            # Validation step (All inside VRAM)
            model.eval()
            total_err, total_pts = 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb_batch = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx]

                    x_batch = gpu_extractor(raw_rgb_batch, rep_key)

                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)
                    gt = y_batch.view(-1, NUM_JOINTS, 3)
                    dist = torch.norm(pred - gt, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

            val_mpjpe = total_err / total_pts
            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe

        experiment_matrix[name].append(best_val_mpjpe)
        print(f"  [Seed {seed:<3}] Train Segs: {len(train_segs):2d} | Val Segs: {len(val_segs):2d} | Best Val MPJPE: {best_val_mpjpe:6.2f} mm")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. FINAL RESULTS
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print(" RIGOROUS BENCHMARK RESULTS: SEGMENT-HELD-OUT GENERALIZATION (21 JOINTS)")
print("=" * 85)
print(f"{'Input Representation':<32} | {'Seed Runs (mm)':<22} | {'Mean ± Std MPJPE':<18}")
print("-" * 85)

for name, _, _ in VARIANTS:
    runs = experiment_matrix[name]
    mean_val = np.mean(runs)
    std_val = np.std(runs)
    runs_str = ", ".join([f"{v:.2f}" for v in runs])
    print(f"{name:<32} | {runs_str:<22} | {mean_val:6.2f} ± {std_val:4.2f} mm")

print("=" * 85)

Hardware Accelerator : Tesla T4
Available VRAM       : 14.56 GB

 INGESTING 5,114 SAMPLES DIRECTLY INTO GPU VRAM (AS UINT8)


Decoding Images to RAM:   0%|          | 0/5114 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE VERIFICATION & VRAM PREPARATION
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Go to Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Hardware Accelerator : {gpu_name}")
print(f"Available VRAM       : {total_vram_gb:.2f} GB")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure the dataset is extracted.")

# Official Re:InterHand 24 -> Kinematic 21 Joint Index Mapping
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

# -------------------------------------------------------------------------
# 2. GPU SOBEL EXTRACTOR
# -------------------------------------------------------------------------
class GPURepresentationEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx_1 = kx.view(1, 1, 3, 3)
        self.ky_1 = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        # Standard luminance (B, 1, H, W)
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx_1, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky_1, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPURepresentationEngine(DEVICE)

# -------------------------------------------------------------------------
# 3. SELF-HEALING VRAM INGESTION (SAFE UINT8 CACHING)
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("\nPreloading dataset into Tesla T4 VRAM (uint8 compact storage)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_images_list, raw_targets_list, sample_segments_list = [], [], []

    for row in records:
        try:
            img_p = resolve_file(DATA_DIR, row["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_images_list.append(rgb_resized.transpose(2, 0, 1))
            raw_targets_list.append(kps_rel)
            sample_segments_list.append(row["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_images_list), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targets_list), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(sample_segments_list)
    unique_segments = np.unique(sample_segments)

    del raw_images_list, raw_targets_list, sample_segments_list
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} existing frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 4. KINEMATIC TOPOLOGY & STRUCTURAL LOSS DEFINITIONS
# -------------------------------------------------------------------------
# 20 directed bone parent-child pairs in 21-joint space
BONE_PAIRS = [
    # Thumb: Wrist -> CMC -> MCP -> IP -> Tip
    (0, 1), (1, 2), (2, 3), (3, 4),
    # Index: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 5), (5, 6), (6, 7), (7, 8),
    # Middle: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 9), (9, 10), (10, 11), (11, 12),
    # Ring: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 13), (13, 14), (14, 15), (15, 16),
    # Pinky: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 17), (17, 18), (18, 19), (19, 20)
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

class KinematicStructureLoss(nn.Module):
    def __init__(self, lambda_len: float = 0.5, lambda_dir: float = 10.0):
        super().__init__()
        self.lambda_len = lambda_len
        self.lambda_dir = lambda_dir

    def forward(self, pred: torch.Tensor, target: torch.Tensor):
        # Base Coordinate Position Loss (MSE)
        l_pos = F.mse_loss(pred, target)

        # Bone vectors (B, 20, 3)
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # Bone length consistency (L1 error in mm)
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # Bone direction angular alignment (Cosine distance)
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        l_dir = (1.0 - (u_pred * u_gt).sum(dim=-1)).mean()

        total = l_pos + self.lambda_len * l_len + self.lambda_dir * l_dir
        return total

# -------------------------------------------------------------------------
# 5. POSE BACKBONE (1-CHANNEL INPUT)
# -------------------------------------------------------------------------
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int = 1, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS: UNCONSTRAINED MSE VS. KINEMATIC LOSS
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

EXPERIMENTS = [
    ("Unconstrained MSE Loss", False),
    ("Kinematic Structural Loss", True)
]

comparison_matrix = {name: [] for name, _ in EXPERIMENTS}
bone_error_matrix = {name: [] for name, _ in EXPERIMENTS}

print("\n" + "=" * 80)
print(" BENCHMARKING LOSS OBJECTIVES ACROSS 3 SEGMENT SPLITS")
print("=" * 80)

for exp_name, use_kinematic in EXPERIMENTS:
    print(f"\n>>> Running: {exp_name} <<<")
    kin_loss_fn = KinematicStructureLoss()

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        # Strictly identical segment splits across both runs
        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=1, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_bone_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb_batch = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb_batch)

                optimizer.zero_grad()
                pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                loss = kin_loss_fn(pred, y_batch) if use_kinematic else F.mse_loss(pred, y_batch)
                loss.backward()
                optimizer.step()

            # Validation evaluation
            model.eval()
            total_err, total_bone_err, total_pts = 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb_batch = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb_batch)
                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                    # MPJPE (mm)
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone length deformation error (mm)
                    b_pred = torch.norm(pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES], dim=-1)
                    b_gt = torch.norm(y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES], dim=-1)
                    total_bone_err += torch.abs(b_pred - b_gt).sum().item()

            val_mpjpe = total_err / total_pts
            val_bone_err = total_bone_err / (len(val_indices) * len(BONE_PAIRS))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_bone_err = val_bone_err

        comparison_matrix[exp_name].append(best_val_mpjpe)
        bone_error_matrix[exp_name].append(best_val_bone_err)
        print(f"  [Seed {seed:<3}] Val MPJPE: {best_val_mpjpe:6.2f} mm | Bone Length Error: {best_val_bone_err:5.2f} mm")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 7. RESULTS SUMMARY
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" LOSS OBJECTIVE COMPARISON: 3-SPLIT SEGMENT-HELD-OUT EVALUATION")
print("=" * 80)
print(f"{'Objective Formulation':<30} | {'Mean MPJPE':<18} | {'Bone Length Error':<18}")
print("-" * 80)
for exp_name, _ in EXPERIMENTS:
    m_pjpe = np.mean(comparison_matrix[exp_name])
    s_pjpe = np.std(comparison_matrix[exp_name])
    m_bone = np.mean(bone_error_matrix[exp_name])
    s_bone = np.std(bone_error_matrix[exp_name])
    print(f"{exp_name:<30} | {m_pjpe:6.2f} ± {s_pjpe:4.2f} mm | {m_bone:5.2f} ± {s_bone:4.2f} mm")
print("=" * 80)

Hardware Accelerator : Tesla T4
Available VRAM       : 14.56 GB

Preloading dataset into Tesla T4 VRAM (uint8 compact storage)...
Staged 5,114 frames into VRAM (720.4 MB used).

 BENCHMARKING LOSS OBJECTIVES ACROSS 3 SEGMENT SPLITS

>>> Running: Unconstrained MSE Loss <<<
  [Seed 42 ] Val MPJPE:  34.29 mm | Bone Length Error:  5.52 mm
  [Seed 101] Val MPJPE:  36.43 mm | Bone Length Error:  5.70 mm
  [Seed 777] Val MPJPE:  31.36 mm | Bone Length Error:  5.77 mm

>>> Running: Kinematic Structural Loss <<<
  [Seed 42 ] Val MPJPE:  33.80 mm | Bone Length Error:  5.40 mm
  [Seed 101] Val MPJPE:  36.47 mm | Bone Length Error:  5.58 mm
  [Seed 777] Val MPJPE:  30.71 mm | Bone Length Error:  5.17 mm

 LOSS OBJECTIVE COMPARISON: 3-SPLIT SEGMENT-HELD-OUT EVALUATION
Objective Formulation          | Mean MPJPE         | Bone Length Error 
--------------------------------------------------------------------------------
Unconstrained MSE Loss         |  34.03 ± 2.08 mm |  5.66 ± 0.10 mm
Kinematic St

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

# 21-Joint Topology & 20 Directed Bones
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

# -------------------------------------------------------------------------
# 1. GPU EDGE EXTRACTOR & VRAM INGESTION
# -------------------------------------------------------------------------
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into VRAM...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()
    print(f"Staged {len(vram_images):,} frames into VRAM.")
else:
    print(f"Reusing {len(vram_images):,} frames in VRAM.")

# -------------------------------------------------------------------------
# 2. CALIBRATED LOSS MODULE & NETWORK ARCHITECTURE
# -------------------------------------------------------------------------
class LossEvaluator(nn.Module):
    def __init__(self, lambda_len: float = 0.0, lambda_dir: float = 0.0):
        super().__init__()
        self.lambda_len = lambda_len
        self.lambda_dir = lambda_dir

    def forward(self, pred: torch.Tensor, target: torch.Tensor):
        # 1. Coordinate Euclidean distance in mm
        l_pos = torch.norm(pred - target, dim=-1).mean()

        # 2. Bone vectors
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # 3. Bone length L1 in mm
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # 4. Bone direction cosine distance
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        l_dir = (1.0 - (u_pred * u_gt).sum(dim=-1)).mean()

        total = l_pos + self.lambda_len * l_len + self.lambda_dir * l_dir
        return total, l_pos.item(), l_len.item(), l_dir.item()

class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int = 1, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 3. CONTROLLED ABLATION EXPERIMENTS
# -------------------------------------------------------------------------
ABLATION_CONFIGS = [
    ("Pos Only (mm L1)",            0.0,  0.0),
    ("Pos + Len (λ=0.2)",           0.2,  0.0),
    ("Pos + Len (λ=0.5)",           0.5,  0.0),
    ("Pos + Dir (λ=20.0)",          0.0, 20.0),
    ("Full Structural (0.5L + 20D)", 0.5, 20.0)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _ in ABLATION_CONFIGS}
results_bone = {name: [] for name, _, _ in ABLATION_CONFIGS}

print("\n" + "=" * 80)
print(" EXECUTING STRUCTURAL LOSS ABLATION (3 SEEDS, DISJOINT SEGMENTS)")
print("=" * 80)

for exp_name, l_len_wt, l_dir_wt in ABLATION_CONFIGS:
    print(f"\n>>> Configuration: {exp_name} <<<")
    loss_module = LossEvaluator(lambda_len=l_len_wt, lambda_dir=l_dir_wt)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=1, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_bone_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            total_pos_term, total_len_term, total_dir_term = 0.0, 0.0, 0.0
            num_batches = 0

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                loss, pos_val, len_val, dir_val = loss_module(pred, y_batch)
                loss.backward()
                optimizer.step()

                total_pos_term += pos_val
                total_len_term += len_val
                total_dir_term += dir_val
                num_batches += 1

            # Validation evaluation
            model.eval()
            total_err, total_bone_err, total_pts = 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

                    b_pred = torch.norm(pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES], dim=-1)
                    b_gt = torch.norm(y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES], dim=-1)
                    total_bone_err += torch.abs(b_pred - b_gt).sum().item()

            val_mpjpe = total_err / total_pts
            val_bone_err = total_bone_err / (len(val_indices) * len(BONE_PAIRS))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_bone_err = val_bone_err

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone[exp_name].append(best_val_bone_err)
        print(f"  [Seed {seed:<3}] Val MPJPE: {best_val_mpjpe:6.2f} mm | Bone Error: {best_val_bone_err:5.2f} mm | Final Train Terms: [Pos: {total_pos_term/num_batches:.1f}mm, Len: {total_len_term/num_batches:.1f}mm, Dir: {total_dir_term/num_batches:.4f}]")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 4. FINAL COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" STRUCTURAL LOSS ABLATION RESULTS (3-SPLIT SEGMENT DISJOINT)")
print("=" * 80)
print(f"{'Loss Formulation':<30} | {'Mean MPJPE (mm)':<18} | {'Bone Length Error (mm)':<22}")
print("-" * 80)
for exp_name, _, _ in ABLATION_CONFIGS:
    m_p = np.mean(results_mpjpe[exp_name])
    s_p = np.std(results_mpjpe[exp_name])
    m_b = np.mean(results_bone[exp_name])
    s_b = np.std(results_bone[exp_name])
    print(f"{exp_name:<30} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_b:5.2f} ± {s_b:4.2f} mm")
print("=" * 80)

Staging 5,114 frames into VRAM...
Staged 5,114 frames into VRAM.

 EXECUTING STRUCTURAL LOSS ABLATION (3 SEEDS, DISJOINT SEGMENTS)

>>> Configuration: Pos Only (mm L1) <<<
  [Seed 42 ] Val MPJPE:  33.73 mm | Bone Error:  5.53 mm | Final Train Terms: [Pos: 9.3mm, Len: 2.6mm, Dir: 0.0290]
  [Seed 101] Val MPJPE:  35.78 mm | Bone Error:  6.13 mm | Final Train Terms: [Pos: 9.4mm, Len: 2.5mm, Dir: 0.0288]
  [Seed 777] Val MPJPE:  29.68 mm | Bone Error:  5.09 mm | Final Train Terms: [Pos: 9.7mm, Len: 2.6mm, Dir: 0.0303]

>>> Configuration: Pos + Len (λ=0.2) <<<
  [Seed 42 ] Val MPJPE:  34.55 mm | Bone Error:  5.38 mm | Final Train Terms: [Pos: 9.4mm, Len: 2.3mm, Dir: 0.0298]
  [Seed 101] Val MPJPE:  36.62 mm | Bone Error:  6.00 mm | Final Train Terms: [Pos: 9.5mm, Len: 2.3mm, Dir: 0.0283]
  [Seed 777] Val MPJPE:  30.01 mm | Bone Error:  4.92 mm | Final Train Terms: [Pos: 9.6mm, Len: 2.3mm, Dir: 0.0304]

>>> Configuration: Pos + Len (λ=0.5) <<<
  [Seed 42 ] Val MPJPE:  35.45 mm | Bone Error: 

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

# 21-Joint Topology & 20 Directed Bones
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
TARGET_W, TARGET_H = 256, 192

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

# GPU Sobel Extractor
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

# Data Verification / In-Memory Staging
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into VRAM...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()
    print(f"Staged {len(vram_images):,} frames into VRAM.")
else:
    print(f"Reusing {len(vram_images):,} frames in VRAM.")

# -------------------------------------------------------------------------
# KINEMATIC TREE RECONSTRUCTION LAYER
# -------------------------------------------------------------------------
def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    """
    Assembles 21 joint positions from 20 directed bone vectors.
    bone_vectors: (B, 20, 3)
    Output: (B, 21, 3) with root joint 0 = (0, 0, 0)
    """
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)

    # Recursive propagation following BONE_PAIRS order
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]

    return joints

# -------------------------------------------------------------------------
# BACKBONE & PARAMETERIZED HEADS
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Head A: Cartesian Coordinates (Baseline)
class CartesianHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.head = nn.Linear(512, 63)

    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat).view(-1, 21, 3)

# Head B: Directed Bone Vectors
class BoneVectorHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.head = nn.Linear(512, NUM_BONES * 3)

    def forward(self, x):
        feat = self.backbone(x)
        bone_vecs = self.head(feat).view(-1, NUM_BONES, 3)
        return reconstruct_joints_from_bones(bone_vecs)

# Head C: Decoupled Direction + Explicit Length
class DecoupledKinematicHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.len_head = nn.Linear(512, NUM_BONES)

    def forward(self, x):
        feat = self.backbone(x)

        # Directions on S² unit sphere
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = dir_raw / (torch.norm(dir_raw, dim=-1, keepdim=True) + 1e-7)

        # Scalar lengths guaranteed positive (Softplus initialized near ~30mm)
        lengths = F.softplus(self.len_head(feat)).unsqueeze(-1)

        bone_vecs = u_dirs * lengths
        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# THREE-METRIC EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        # 1. MPJPE (mm)
        l_pos = torch.norm(pred - target, dim=-1).mean()

        # Bone representations
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # 2. Bone Length Error (mm)
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # 3. Bone Directional Cosine Loss (for backprop)
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_pred * u_gt).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        # Bone Direction Angular Error in Degrees (for evaluation)
        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)

        # Total Structural Loss (0.5 L_len + 20.0 L_dir)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir

        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# EXPERIMENTAL MATRIX: 3 HEAD ARCHITECTURES ACROSS 3 SPLITS
# -------------------------------------------------------------------------
HEAD_CONFIGS = [
    ("Cartesian XYZ (Baseline)",   CartesianHeadModel),
    ("Directed Bone Vectors",      BoneVectorHeadModel),
    ("Decoupled Dir + Length",     DecoupledKinematicHeadModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in HEAD_CONFIGS}
results_bone_len = {name: [] for name, _ in HEAD_CONFIGS}
results_bone_dir = {name: [] for name, _ in HEAD_CONFIGS}

print("\n" + "=" * 80)
print(" BENCHMARKING KINEMATIC DECODER HEADS (3 SEEDS, SEGMENT-DISJOINT)")
print("=" * 80)

for exp_name, ModelCls in HEAD_CONFIGS:
    print(f"\n>>> Model Architecture: {exp_name} <<<")

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_len_err = 0.0
        best_val_deg_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation across 3 Metrics
            model.eval()
            total_pjpe, total_len_err, total_deg_err, total_pts = 0.0, 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch)

                    b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
                    b_gt = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Length Error (mm)
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    total_len_err += torch.abs(len_p - len_g).sum().item()

                    # Bone Direction Error (Degrees)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    total_deg_err += torch.acos(cos_s).sum().item()

            val_mpjpe = total_pjpe / total_pts
            val_len_err = total_len_err / (len(val_indices) * NUM_BONES)
            val_deg_err = (total_deg_err / (len(val_indices) * NUM_BONES)) * (180.0 / np.pi)

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_len_err = val_len_err
                best_val_deg_err = val_deg_err

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_len[exp_name].append(best_val_len_err)
        results_bone_dir[exp_name].append(best_val_deg_err)
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Length Error: {best_val_len_err:5.2f} mm | Direction Error: {best_val_deg_err:5.2f}°")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# FINAL MULTI-METRIC PERFORMANCE AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" KINEMATIC DECODER BENCHMARK: 3-SPLIT SEGMENT-DISJOINT GENERALIZATION")
print("=" * 90)
print(f"{'Head Parameterization':<28} | {'MPJPE (mm)':<18} | {'Bone Length (mm)':<18} | {'Bone Direction (deg)':<20}")
print("-" * 90)
for exp_name, _ in HEAD_CONFIGS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_l, s_l = np.mean(results_bone_len[exp_name]), np.std(results_bone_len[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<28} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_l:5.2f} ± {s_l:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

Reusing 5,114 frames in VRAM.

 BENCHMARKING KINEMATIC DECODER HEADS (3 SEEDS, SEGMENT-DISJOINT)

>>> Model Architecture: Cartesian XYZ (Baseline) <<<
  [Seed 42 ] MPJPE:  33.64 mm | Length Error:  5.59 mm | Direction Error: 31.23°
  [Seed 101] MPJPE:  35.23 mm | Length Error:  4.62 mm | Direction Error: 35.83°
  [Seed 777] MPJPE:  29.31 mm | Length Error:  4.29 mm | Direction Error: 24.98°

>>> Model Architecture: Directed Bone Vectors <<<
  [Seed 42 ] MPJPE:  34.04 mm | Length Error:  4.35 mm | Direction Error: 31.68°
  [Seed 101] MPJPE:  35.16 mm | Length Error:  5.19 mm | Direction Error: 37.73°
  [Seed 777] MPJPE:  28.34 mm | Length Error:  3.79 mm | Direction Error: 23.74°

>>> Model Architecture: Decoupled Dir + Length <<<
  [Seed 42 ] MPJPE:  33.04 mm | Length Error:  1.81 mm | Direction Error: 30.86°
  [Seed 101] MPJPE:  35.33 mm | Length Error:  3.67 mm | Direction Error: 38.40°
  [Seed 777] MPJPE:  27.82 mm | Length Error:  2.77 mm | Direction Error: 23.51°

 KINEMATIC DECOD

In [ ]:
# @title
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert 'vram_images' in globals(), "Dataset tensors missing from memory."

# Canonical 20-Bone Names
BONE_NAMES = [
    # Thumb
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    # Index
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    # Middle
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    # Ring
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    # Pinky
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# 1. Compute Dataset-Level Empirical Bone Statistics
with torch.no_grad():
    all_targets = vram_targets.view(-1, 21, 3)
    b_all = all_targets[:, CHILD_INDICES] - all_targets[:, PARENT_INDICES]
    lens_all = torch.norm(b_all, dim=-1) # (N, 20)

    gt_mean_lens = lens_all.mean(dim=0).cpu().numpy()
    gt_std_lens = lens_all.std(dim=0).cpu().numpy()

# 2. Calibrated Decoupled Model with Analytical Inverse-Softplus Bias
class CalibratedDecoupledModel(nn.Module):
    def __init__(self, init_lengths: np.ndarray):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.len_head = nn.Linear(512, NUM_BONES)

        # Analytical Inverse-Softplus initialization: bias = ln(exp(mean) - 1)
        inv_softplus = np.log(np.exp(init_lengths) - 1.0)
        self.len_head.bias.data = torch.tensor(inv_softplus, dtype=torch.float32)
        self.len_head.weight.data.zero_()  # Zero-init weights: start exactly at mean

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = dir_raw / (torch.norm(dir_raw, dim=-1, keepdim=True) + 1e-7)
        lengths = F.softplus(self.len_head(feat))

        bone_vecs = u_dirs * lengths.unsqueeze(-1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, lengths

# 3. Train on Seed 42 Split to Inspect Learned Length Behavior
torch.manual_seed(42)
np.random.seed(42)
shuffled_segs = np.random.permutation(unique_segments)
split_idx = int(0.8 * len(shuffled_segs))
train_segs, val_segs = set(shuffled_segs[:split_idx]), set(shuffled_segs[split_idx:])
train_idx = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
val_idx = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

model = CalibratedDecoupledModel(gt_mean_lens).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(1, 16):
    model.train()
    perm = train_idx[torch.randperm(len(train_idx))]
    for i in range(0, len(perm), 64):
        b = perm[i:i + 64]
        raw_rgb = vram_images[b].float()
        y_batch = vram_targets[b].view(-1, 21, 3)

        x_batch = gpu_extractor(raw_rgb)
        optimizer.zero_grad()
        pred_joints, _ = model(x_batch)
        loss, _, _, _ = eval_engine(pred_joints, y_batch)
        loss.backward()
        optimizer.step()

# 4. Bone-by-Bone Audit on Held-out Validation Segments
model.eval()
val_pred_lens = []
val_gt_lens = []

with torch.no_grad():
    for i in range(0, len(val_idx), 64):
        b = val_idx[i:i + 64]
        raw_rgb = vram_images[b].float()
        y_batch = vram_targets[b].view(-1, 21, 3)

        x_batch = gpu_extractor(raw_rgb)
        _, pred_l = model(x_batch)

        gt_b = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]
        gt_l = torch.norm(gt_b, dim=-1)

        val_pred_lens.append(pred_l.cpu())
        val_gt_lens.append(gt_l.cpu())

val_pred_lens = torch.cat(val_pred_lens, dim=0).numpy()
val_gt_lens = torch.cat(val_gt_lens, dim=0).numpy()

# 5. Output Tabular Breakdown
print("\n" + "=" * 90)
print(f"{'Bone Identifier':<24} | {'GT μ':<7} | {'Pred μ':<7} | {'GT σ':<6} | {'Pred σ':<6} | {'MAE (mm)':<8}")
print("-" * 90)
for k in range(20):
    gt_u = val_gt_lens[:, k].mean()
    pr_u = val_pred_lens[:, k].mean()
    gt_s = val_gt_lens[:, k].std()
    pr_s = val_pred_lens[:, k].std()
    mae = np.abs(val_pred_lens[:, k] - val_gt_lens[:, k]).mean()
    print(f"{BONE_NAMES[k]:<24} | {gt_u:6.2f} | {pr_u:6.2f} | {gt_s:5.2f} | {pr_s:5.2f} | {mae:6.2f}")
print("=" * 90)


Bone Identifier          | GT μ    | Pred μ  | GT σ   | Pred σ | MAE (mm)
------------------------------------------------------------------------------------------
Wrist -> Thumb CMC       |  38.52 |  38.58 |  1.26 |  0.44 |   0.93
Thumb CMC -> MCP         |  30.32 |  30.37 |  0.00 |  0.05 |   0.06
Thumb MCP -> IP          |  28.92 |  28.92 |  0.00 |  0.02 |   0.02
Thumb IP -> Tip          |  14.78 |  14.77 |  0.00 |  0.01 |   0.01
Wrist -> Index MCP       |  82.04 |  81.95 |  0.00 |  0.13 |   0.13
Index MCP -> PIP         |  36.16 |  36.05 |  0.00 |  0.07 |   0.11
Index PIP -> DIP         |  23.81 |  23.75 |  0.00 |  0.05 |   0.06
Index DIP -> Tip         |  13.39 |  13.39 |  0.00 |  0.02 |   0.02
Wrist -> Middle MCP      |  78.78 |  78.78 |  0.00 |  0.06 |   0.05
Middle MCP -> PIP        |  40.80 |  40.74 |  0.00 |  0.08 |   0.08
Middle PIP -> DIP        |  28.15 |  28.01 |  0.00 |  0.05 |   0.14
Middle DIP -> Tip        |  13.27 |  13.28 |  0.00 |  0.02 |   0.01
Wrist -> Ring MCP 

In [ ]:
# @title
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert 'vram_images' in globals(), "Dataset tensors missing from memory."

# -------------------------------------------------------------------------
# 1. TOPOLOGY & EMPIRICAL CANONICAL BONE LENGTHS
# -------------------------------------------------------------------------
BONE_NAMES = [
    # Thumb
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    # Index
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    # Middle
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    # Ring
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    # Pinky
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

CANONICAL_LENGTHS = torch.tensor([
    38.52, 30.32, 28.92, 14.78,  # Thumb
    82.04, 36.16, 23.81, 13.39,  # Index
    78.78, 40.80, 28.15, 13.27,  # Middle
    75.69, 36.32, 26.51, 13.30,  # Ring
    74.02, 26.61, 18.70, 13.72   # Pinky
], dtype=torch.float32, device=DEVICE)

# -------------------------------------------------------------------------
# 2. MODEL DEFINITIONS
# -------------------------------------------------------------------------
def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1 = d6[:, :3]
    a2 = d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

# Model A: Uncoupled Global Unit Directions on S²
class UncoupledFixedLengthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", CANONICAL_LENGTHS.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Model B: Hierarchical 6D Chaining
FINGER_CHAINS = [
    (0,  [1, 2, 3]),     # Thumb
    (4,  [5, 6, 7]),     # Index
    (8,  [9, 10, 11]),   # Middle
    (12, [13, 14, 15]),  # Ring
    (16, [17, 18, 19])   # Pinky
]

class HierarchicalFKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.metacarpal_head = nn.Linear(512, 5 * 6)
        self.relative_joint_head = nn.Linear(512, 15 * 6)
        self.register_buffer("bone_lens", CANONICAL_LENGTHS)
        self.register_buffer("rest_dir", torch.tensor([0., 0., 1.], device=DEVICE).view(1, 3, 1))

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        meta_6d = self.metacarpal_head(feat).view(-1, 5, 6)
        meta_R = rotation_6d_to_matrix(meta_6d.view(-1, 6)).view(B, 5, 3, 3)

        rel_6d = self.relative_joint_head(feat).view(-1, 15, 6)
        rel_R = rotation_6d_to_matrix(rel_6d.view(-1, 6)).view(B, 15, 3, 3)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        rel_idx = 0

        for f_idx, (root_bone_idx, child_bone_indices) in enumerate(FINGER_CHAINS):
            R_parent = meta_R[:, f_idx]
            u_root = torch.bmm(R_parent, self.rest_dir.expand(B, -1, -1)).squeeze(-1)
            bone_vecs[:, root_bone_idx] = u_root * self.bone_lens[root_bone_idx]

            for c_bone_idx in child_bone_indices:
                delta_R = rel_R[:, rel_idx]
                rel_idx += 1

                R_child = torch.bmm(R_parent, delta_R)
                u_child = torch.bmm(R_child, self.rest_dir.expand(B, -1, -1)).squeeze(-1)
                bone_vecs[:, c_bone_idx] = u_child * self.bone_lens[c_bone_idx]
                R_parent = R_child

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 3. BENCHMARK EXECUTION WITH PER-BONE DIRECTION PROFILING
# -------------------------------------------------------------------------
MODELS = [
    ("Uncoupled S² Directions", UncoupledFixedLengthModel),
    ("Hierarchical 6D Chaining", HierarchicalFKModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in MODELS}
results_bone_len = {name: [] for name, _ in MODELS}
results_bone_dir = {name: [] for name, _ in MODELS}
per_bone_dir_matrix = {name: np.zeros(20) for name, _ in MODELS}

print("=" * 85)
print(" RUNNING COMPARATIVE BENCHMARK: UNCOUPLED S² VS HIERARCHICAL 6D")
print("=" * 85)

for exp_name, ModelCls in MODELS:
    print(f"\n>>> Evaluating Architecture: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_len_err = 0.0
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_len_err, total_deg_err, total_pts = 0.0, 0.0, 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch)

                    b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
                    b_gt = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Residual Bone Length Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    total_len_err += torch.abs(len_p - len_g).sum().item()

                    # Bone Direction Error
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)

                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_len_err = total_len_err / (len(val_indices) * NUM_BONES)
            val_deg_err = (cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_len_err = val_len_err
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_len[exp_name].append(best_val_len_err)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Length Residual: {best_val_len_err:5.2f} mm | Direction Error: {best_val_deg_err:5.2f}°")

    per_bone_dir_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 4. REPORT: AGGREGATE SUMMARY & PER-BONE BREAKDOWN
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" AGGREGATE GENERALIZATION RESULTS: UNCOUPLED S² VS HIERARCHICAL 6D")
print("=" * 90)
print(f"{'Architecture':<28} | {'MPJPE (mm)':<18} | {'Length Residual':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _ in MODELS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_l, s_l = np.mean(results_bone_len[exp_name]), np.std(results_bone_len[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<28} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_l:5.2f} ± {s_l:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 90)
print(" PER-BONE DIRECTION ERROR COMPARISON (MEAN OVER 3 SPLITS in DEGREES)")
print("=" * 90)
print(f"{'Bone Identifier':<24} | {'Uncoupled S²':<15} | {'Hierarchical 6D':<17} | {'Delta':<10}")
print("-" * 90)
uncoupled_deg = per_bone_dir_matrix["Uncoupled S² Directions"]
hierarch_deg = per_bone_dir_matrix["Hierarchical 6D Chaining"]

for k in range(20):
    u_val = uncoupled_deg[k]
    h_val = hierarch_deg[k]
    delta = h_val - u_val
    sign = "+" if delta > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {u_val:6.2f}°         | {h_val:6.2f}°           | {sign}{delta:5.2f}°")
print("=" * 90)

AssertionError: Dataset tensors missing from memory.

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE & TOPOLOGY SETUP
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure the dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Bone Parent-Child Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 2. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 3. EMPIRICAL CALIBRATION OF CANONICAL SKELETON
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)  # (20,)

    # Compute empirical palm rest geometry from participant0 targets:
    # Palm frame: Z = Wrist -> Middle MCP, X = lateral towards Index, Y = Palm Normal
    p_wrist = all_kps[:, 0]
    p_idx_mcp = all_kps[:, 5]
    p_mid_mcp = all_kps[:, 9]
    p_pky_mcp = all_kps[:, 17]

    z_palm = F.normalize(p_mid_mcp - p_wrist, dim=-1)
    trans_palm = F.normalize(p_pky_mcp - p_idx_mcp, dim=-1)
    y_palm = F.normalize(torch.cross(z_palm, trans_palm, dim=-1), dim=-1)
    x_palm = F.normalize(torch.cross(y_palm, z_palm, dim=-1), dim=-1)
    R_palm_all = torch.stack([x_palm, y_palm, z_palm], dim=-1)  # (N, 3, 3)

    # Project each metacarpal into the local palm frame
    meta_kps = [all_kps[:, 1], all_kps[:, 5], all_kps[:, 9], all_kps[:, 13], all_kps[:, 17]]
    meta_dirs_local = []
    for mkp in meta_kps:
        v_meta = F.normalize(mkp - p_wrist, dim=-1)  # (N, 3)
        v_local = torch.bmm(R_palm_all.transpose(1, 2), v_meta.unsqueeze(-1)).squeeze(-1)
        meta_dirs_local.append(F.normalize(v_local.mean(dim=0), dim=-1))

    EMPIRICAL_META_DIRS = torch.stack(meta_dirs_local, dim=0)  # (5, 3)
    print("Empirical Metacarpal Rest Directions Calibrated Successfully.")

# -------------------------------------------------------------------------
# 4. GPU SOBEL ENGINE & ROTATION MODULES
# -------------------------------------------------------------------------
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1 = d6[:, :3]
    a2 = d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 5. BACKBONE & HEAD ARCHITECTURES
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Architecture 1: Uncoupled S² Directions (Canonical Baseline)
class UncoupledFixedLengthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Architecture 2: Calibrated Anatomical Forward Kinematics
class CalibratedAnatomicalFKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        # 6 (Global Palm 6D) + 15 * 3 (15 local phalangeal direction updates in palm frame)
        self.palm_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)

        self.register_buffer("bone_lens", L_CANON)
        self.register_buffer("meta_dirs_local", EMPIRICAL_META_DIRS)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 1. Global Palm Rotation R_palm ∈ SO(3)
        R_palm = rotation_6d_to_matrix(self.palm_head(feat))

        # 2. Local Phalangeal Directions (in calibrated Palm Frame)
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)

        # Rotate local directions into camera frame
        phalanx_cam_u = torch.bmm(R_palm, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Metacarpals rigidly anchored to calibrated Palm Frame
        meta_cam_u = torch.matmul(R_palm, self.meta_dirs_local.transpose(0, 1)).transpose(1, 2)  # (B, 5, 3)

        # Thumb Chain
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Finger Chains (Index, Middle, Ring, Pinky)
        finger_bone_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_offset = finger_bone_offsets[f_idx]
            p_offset = 3 + f_idx * 3
            bone_vecs[:, b_offset] = meta_cam_u[:, f_idx + 1] * self.bone_lens[b_offset]
            bone_vecs[:, b_offset + 1] = phalanx_cam_u[:, p_offset] * self.bone_lens[b_offset + 1]
            bone_vecs[:, b_offset + 2] = phalanx_cam_u[:, p_offset + 1] * self.bone_lens[b_offset + 2]
            bone_vecs[:, b_offset + 3] = phalanx_cam_u[:, p_offset + 2] * self.bone_lens[b_offset + 3]

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 6. EVALUATION MODULE & LOSS
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()

        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_pred * u_gt).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir

        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 7. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
MODELS = [
    ("Uncoupled S² (Baseline)", UncoupledFixedLengthModel),
    ("Calibrated Anatomical FK", CalibratedAnatomicalFKModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in MODELS}
results_bone_dir = {name: [] for name, _ in MODELS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in MODELS}

print("\n" + "=" * 85)
print(" BENCHMARKING CALIBRATED ANATOMICAL FORWARD KINEMATICS")
print("=" * 85)

for exp_name, ModelCls in MODELS:
    print(f"\n>>> Model Configuration: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch)

                    b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
                    b_gt = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)

                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = (cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Direction Error: {best_val_deg_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 8. AUDIT & DIAGNOSTIC TABLE
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" BENCHMARK SUMMARY: UNCOUPLED S² VS CALIBRATED ANATOMICAL FK")
print("=" * 90)
print(f"{'Architecture':<32} | {'MPJPE (mm)':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _ in MODELS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<32} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 90)
print(" PER-BONE DIRECTION ERROR: UNCOUPLED S² VS CALIBRATED ANATOMICAL FK")
print("=" * 90)
print(f"{'Bone Identifier':<24} | {'Uncoupled S²':<15} | {'Calibrated FK':<17} | {'Delta':<10}")
print("-" * 90)
u_deg = per_bone_deg_matrix["Uncoupled S² (Baseline)"]
a_deg = per_bone_deg_matrix["Calibrated Anatomical FK"]

for k in range(20):
    u_val = u_deg[k]
    a_val = a_deg[k]
    delta = a_val - u_val
    sign = "+" if delta > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {u_val:6.2f}°         | {a_val:6.2f}°           | {sign}{delta:5.2f}°")
print("=" * 90)

Hardware Accelerator : Tesla T4
Reusing 5,114 frames already resident in VRAM.
Empirical Metacarpal Rest Directions Calibrated Successfully.

 BENCHMARKING CALIBRATED ANATOMICAL FORWARD KINEMATICS

>>> Model Configuration: Uncoupled S² (Baseline) <<<
  [Seed 42 ] MPJPE:  33.21 mm | Direction Error: 29.95°
  [Seed 101] MPJPE:  32.51 mm | Direction Error: 34.98°
  [Seed 777] MPJPE:  28.27 mm | Direction Error: 24.56°

>>> Model Configuration: Calibrated Anatomical FK <<<
  [Seed 42 ] MPJPE:  35.39 mm | Direction Error: 33.10°
  [Seed 101] MPJPE:  32.51 mm | Direction Error: 33.36°
  [Seed 777] MPJPE:  30.12 mm | Direction Error: 25.88°

 BENCHMARK SUMMARY: UNCOUPLED S² VS CALIBRATED ANATOMICAL FK
Architecture                     | MPJPE (mm)         | Direction Error (deg)
------------------------------------------------------------------------------------------
Uncoupled S² (Baseline)          |  31.33 ± 2.18 mm | 29.83° ± 4.26°
Calibrated Anatomical FK         |  32.67 ± 2.15 mm | 30.7

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. GEOMETRIC UTILITIES & PALM FRAME EXTRACTION
# -------------------------------------------------------------------------
def construct_palm_frame(joints_21: torch.Tensor) -> torch.Tensor:
    """
    Computes an orthonormal coordinate system from joint positions:
    joints_21: (B, 21, 3)
    Output: R_palm (B, 3, 3) with columns [x_palm, y_palm, z_palm]
    """
    p_wrist   = joints_21[:, 0]
    p_idx_mcp = joints_21[:, 5]
    p_mid_mcp = joints_21[:, 9]
    p_pky_mcp = joints_21[:, 17]

    z_palm = F.normalize(p_mid_mcp - p_wrist, dim=-1)
    trans  = F.normalize(p_pky_mcp - p_idx_mcp, dim=-1)
    y_palm = F.normalize(torch.cross(z_palm, trans, dim=-1), dim=-1)
    x_palm = F.normalize(torch.cross(y_palm, z_palm, dim=-1), dim=-1)
    return torch.stack([x_palm, y_palm, z_palm], dim=-1)

def so3_geodesic_error(R_pred: torch.Tensor, R_gt: torch.Tensor) -> torch.Tensor:
    """Computes angular distance between two SO(3) matrices in degrees."""
    R_err = torch.bmm(R_pred.transpose(1, 2), R_gt)
    tr = R_err[:, 0, 0] + R_err[:, 1, 1] + R_err[:, 2, 2]
    cos_theta = torch.clamp((tr - 1.0) / 2.0, -1.0, 1.0)
    return torch.acos(cos_theta) * (180.0 / np.pi)

# Calibrate empirical constants
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

    R_palm_gt_all = construct_palm_frame(all_kps)
    meta_kps = [all_kps[:, 1], all_kps[:, 5], all_kps[:, 9], all_kps[:, 13], all_kps[:, 17]]
    meta_dirs_local = []
    for mkp in meta_kps:
        v_meta = F.normalize(mkp - all_kps[:, 0], dim=-1)
        v_loc = torch.bmm(R_palm_gt_all.transpose(1, 2), v_meta.unsqueeze(-1)).squeeze(-1)
        meta_dirs_local.append(F.normalize(v_loc.mean(dim=0), dim=-1))
    EMPIRICAL_META_DIRS = torch.stack(meta_dirs_local, dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. MODEL ARCHITECTURES
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Model A: Uncoupled S² Directions (Baseline)
class UncoupledFixedLengthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        joints = reconstruct_joints_from_bones(bone_vecs)
        R_palm_implied = construct_palm_frame(joints)
        return joints, R_palm_implied

# Model B: Calibrated Anatomical FK
class CalibratedAnatomicalFKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.palm_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)
        self.register_buffer("meta_dirs_local", EMPIRICAL_META_DIRS)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        R_palm = rotation_6d_to_matrix(self.palm_head(feat))
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_palm, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        meta_cam_u = torch.matmul(R_palm, self.meta_dirs_local.transpose(0, 1)).transpose(1, 2)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs), R_palm

# Model C: Shared Palm Frame + Independent Metacarpals (Ablation D)
class FreeMetaPalmModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.meta_head = nn.Linear(512, 5 * 3)       # 5 independent metacarpal directions (camera S²)
        self.palm_head = nn.Linear(512, 6)           # 1 global palm orientation SO(3)
        self.phalanx_head = nn.Linear(512, 15 * 3)   # 15 phalanx directions (palm local S²)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 5 unconstrained metacarpal directions in camera space
        meta_raw = self.meta_head(feat).view(B, 5, 3)
        meta_cam_u = F.normalize(meta_raw, dim=-1, eps=1e-7)

        # Explicit palm frame
        R_palm = rotation_6d_to_matrix(self.palm_head(feat))

        # 15 phalanx directions defined locally in the palm frame
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_palm, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs), R_palm

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
MODELS = [
    ("Uncoupled S² (Baseline)",       UncoupledFixedLengthModel),
    ("Calibrated Anatomical FK",      CalibratedAnatomicalFKModel),
    ("Free Metacarpals + Palm Frame", FreeMetaPalmModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in MODELS}
results_bone_dir = {name: [] for name, _ in MODELS}
results_palm_err = {name: [] for name, _ in MODELS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in MODELS}

print("\n" + "=" * 95)
print(" RUNNING SYSTEMATIC FACTORIZATION ABLATION & PALM ERROR AUDIT")
print("=" * 95)

for exp_name, ModelCls in MODELS:
    print(f"\n>>> Model Configuration: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_val_palm_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred_joints, _ = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            total_palm_deg, val_frames_count = 0.0, 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred_joints, pred_R_palm = model(x_batch)

                    b_pred = pred_joints[:, CHILD_INDICES] - pred_joints[:, PARENT_INDICES]
                    b_gt   = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred_joints - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)
                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)

                    # Palm SO(3) Geodesic Angular Error
                    R_palm_gt = construct_palm_frame(y_batch)
                    palm_deg = so3_geodesic_error(pred_R_palm, R_palm_gt)
                    total_palm_deg += palm_deg.sum().item()
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES)
            val_palm_err = total_palm_deg / val_frames_count

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_val_palm_err = val_palm_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        results_palm_err[exp_name].append(best_val_palm_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Dir Error: {best_val_deg_err:5.2f}° | Palm Error: {best_val_palm_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. RESULTS & DIAGNOSTIC AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 95)
print(" BENCHMARK SUMMARY: FACTORIZATION & PALM FRAME ACCURACY")
print("=" * 95)
print(f"{'Architecture':<32} | {'MPJPE (mm)':<18} | {'Dir Error (deg)':<18} | {'Palm Error (deg)':<18}")
print("-" * 95)
for exp_name, _ in MODELS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    m_pl, s_pl = np.mean(results_palm_err[exp_name]), np.std(results_palm_err[exp_name])
    print(f"{exp_name:<32} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}° | {m_pl:5.2f}° ± {s_pl:4.2f}°")
print("=" * 95)

print("\n" + "=" * 95)
print(" PER-BONE DIRECTION ERROR: UNCOUPLED S² VS FREE METACARPALS VS CALIBRATED FK")
print("=" * 95)
print(f"{'Bone Identifier':<24} | {'Uncoupled S²':<14} | {'Free Meta+Palm':<16} | {'Calibrated FK':<15} | {'Delta (Free-S²)':<16}")
print("-" * 95)
u_deg = per_bone_deg_matrix["Uncoupled S² (Baseline)"]
f_deg = per_bone_deg_matrix["Free Metacarpals + Palm Frame"]
c_deg = per_bone_deg_matrix["Calibrated Anatomical FK"]

for k in range(20):
    diff = f_deg[k] - u_deg[k]
    sign = "+" if diff > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {u_deg[k]:5.2f}°        | {f_deg[k]:5.2f}°          | {c_deg[k]:5.2f}°         | {sign}{diff:5.2f}°")
print("=" * 95)

Hardware Accelerator : Tesla T4
Reusing 5,114 frames already resident in VRAM.

 RUNNING SYSTEMATIC FACTORIZATION ABLATION & PALM ERROR AUDIT

>>> Model Configuration: Uncoupled S² (Baseline) <<<
  [Seed 42 ] MPJPE:  33.00 mm | Dir Error: 30.29° | Palm Error: 30.46°
  [Seed 101] MPJPE:  32.80 mm | Dir Error: 35.30° | Palm Error: 29.29°
  [Seed 777] MPJPE:  28.18 mm | Dir Error: 24.20° | Palm Error: 30.14°

>>> Model Configuration: Calibrated Anatomical FK <<<
  [Seed 42 ] MPJPE:  35.82 mm | Dir Error: 33.30° | Palm Error: 34.20°
  [Seed 101] MPJPE:  32.37 mm | Dir Error: 33.49° | Palm Error: 30.27°
  [Seed 777] MPJPE:  30.06 mm | Dir Error: 25.76° | Palm Error: 32.50°

>>> Model Configuration: Free Metacarpals + Palm Frame <<<
  [Seed 42 ] MPJPE:  32.92 mm | Dir Error: 29.98° | Palm Error: 133.83°
  [Seed 101] MPJPE:  32.05 mm | Dir Error: 34.49° | Palm Error: 142.46°
  [Seed 777] MPJPE:  27.80 mm | Dir Error: 24.14° | Palm Error: 150.37°

 BENCHMARK SUMMARY: FACTORIZATION & PALM FRAME

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, mode: str = "scalar") -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)
        if mode == "scalar":
            return g_mag
        elif mode == "directional":
            return torch.cat([gx, gy, g_mag], dim=1)
        raise ValueError(f"Unknown mode: {mode}")

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. BACKBONES & HEADS
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Head Formulation 1: Uncoupled S² Directions
class UncoupledHeadModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Head Formulation 2: Latent Coordinate Factorization (LCF)
class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)       # 5 independent metacarpal directions
        self.latent_frame_head = nn.Linear(512, 6)   # 1 latent coordinate frame SO(3)
        self.phalanx_head = nn.Linear(512, 15 * 3)   # 15 phalanx directions in latent frame
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 5 unconstrained metacarpal directions in camera space
        meta_raw = self.meta_head(feat).view(B, 5, 3)
        meta_cam_u = F.normalize(meta_raw, dim=-1, eps=1e-7)

        # Latent reference frame
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))

        # 15 phalanx directions rotated out of latent frame into camera space
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. 2x2 FACTORIAL BENCHMARK HARNESS
# -------------------------------------------------------------------------
CONFIGS = [
    ("Uncoupled S² + Scalar Sobel",      UncoupledHeadModel,       1, "scalar"),
    ("LCF + Scalar Sobel",               LatentFactorizationModel, 1, "scalar"),
    ("Uncoupled S² + Directional Sobel", UncoupledHeadModel,       3, "directional"),
    ("LCF + Directional Sobel",          LatentFactorizationModel, 3, "directional")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _, _ in CONFIGS}
results_bone_dir = {name: [] for name, _, _, _ in CONFIGS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _, _, _ in CONFIGS}

print("\n" + "=" * 95)
print(" RUNNING 2x2 FACTORIAL BENCHMARK: GRADIENT DIRECTION VS LATENT FACTORIZATION")
print("=" * 95)

for exp_name, ModelCls, in_ch, grad_mode in CONFIGS:
    print(f"\n>>> Running: {exp_name} ({in_ch} Channels) <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb, mode=grad_mode)

                optimizer.zero_grad()
                pred_joints = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb, mode=grad_mode)
                    pred_joints = model(x_batch)

                    b_pred = pred_joints[:, CHILD_INDICES] - pred_joints[:, PARENT_INDICES]
                    b_gt   = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred_joints - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)
                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES)

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Dir Error: {best_val_deg_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. RESULTS & PER-BONE STAIRCASE AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 2x2 FACTORIAL BENCHMARK SUMMARY")
print("=" * 90)
print(f"{'Configuration':<36} | {'MPJPE (mm)':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _, _, _ in CONFIGS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<36} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: DISTAL STAIRCASE AUDIT (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'Unc (Scalar)':<14} | {'LCF (Scalar)':<14} | {'Unc (Direct)':<14} | {'LCF (Direct)':<14}"
print(header)
print("-" * 105)

u_sc = per_bone_deg_matrix["Uncoupled S² + Scalar Sobel"]
l_sc = per_bone_deg_matrix["LCF + Scalar Sobel"]
u_di = per_bone_deg_matrix["Uncoupled S² + Directional Sobel"]
l_di = per_bone_deg_matrix["LCF + Directional Sobel"]

for k in range(20):
    print(f"{BONE_NAMES[k]:<24} | {u_sc[k]:5.2f}°        | {l_sc[k]:5.2f}°        | {u_di[k]:5.2f}°        | {l_di[k]:5.2f}°")
print("=" * 105)

Hardware Accelerator : Tesla T4
Reusing 5,114 frames already resident in VRAM.

 RUNNING 2x2 FACTORIAL BENCHMARK: GRADIENT DIRECTION VS LATENT FACTORIZATION

>>> Running: Uncoupled S² + Scalar Sobel (1 Channels) <<<
  [Seed 42 ] MPJPE:  32.26 mm | Dir Error: 30.28°
  [Seed 101] MPJPE:  32.80 mm | Dir Error: 34.85°
  [Seed 777] MPJPE:  27.87 mm | Dir Error: 24.46°

>>> Running: LCF + Scalar Sobel (1 Channels) <<<
  [Seed 42 ] MPJPE:  33.02 mm | Dir Error: 30.61°
  [Seed 101] MPJPE:  32.25 mm | Dir Error: 34.91°
  [Seed 777] MPJPE:  27.97 mm | Dir Error: 24.02°

>>> Running: Uncoupled S² + Directional Sobel (3 Channels) <<<
  [Seed 42 ] MPJPE:  33.75 mm | Dir Error: 30.99°
  [Seed 101] MPJPE:  32.76 mm | Dir Error: 34.26°
  [Seed 777] MPJPE:  29.91 mm | Dir Error: 24.77°

>>> Running: LCF + Directional Sobel (3 Channels) <<<
  [Seed 42 ] MPJPE:  33.84 mm | Dir Error: 31.22°
  [Seed 101] MPJPE:  34.09 mm | Dir Error: 35.47°
  [Seed 777] MPJPE:  29.60 mm | Dir Error: 25.24°

 2x2 FACTORIAL

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, mode: str = "scalar") -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)
        if mode == "scalar":
            return g_mag
        elif mode == "directional":
            return torch.cat([gx, gy, g_mag], dim=1)
        raise ValueError(f"Unknown mode: {mode}")

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. BACKBONES & HEADS
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Head Formulation 1: Uncoupled S² Directions
class UncoupledHeadModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Head Formulation 2: Latent Coordinate Factorization (LCF)
class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)       # 5 independent metacarpal directions
        self.latent_frame_head = nn.Linear(512, 6)   # 1 latent coordinate frame SO(3)
        self.phalanx_head = nn.Linear(512, 15 * 3)   # 15 phalanx directions in latent frame
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 5 unconstrained metacarpal directions in camera space
        meta_raw = self.meta_head(feat).view(B, 5, 3)
        meta_cam_u = F.normalize(meta_raw, dim=-1, eps=1e-7)

        # Latent reference frame
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))

        # 15 phalanx directions rotated out of latent frame into camera space
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. 2x2 FACTORIAL BENCHMARK HARNESS
# -------------------------------------------------------------------------
CONFIGS = [
    ("Uncoupled S² + Scalar Sobel",      UncoupledHeadModel,       1, "scalar"),
    ("LCF + Scalar Sobel",               LatentFactorizationModel, 1, "scalar"),
    ("Uncoupled S² + Directional Sobel", UncoupledHeadModel,       3, "directional"),
    ("LCF + Directional Sobel",          LatentFactorizationModel, 3, "directional")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _, _ in CONFIGS}
results_bone_dir = {name: [] for name, _, _, _ in CONFIGS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _, _, _ in CONFIGS}

print("\n" + "=" * 95)
print(" RUNNING 2x2 FACTORIAL BENCHMARK: GRADIENT DIRECTION VS LATENT FACTORIZATION")
print("=" * 95)

for exp_name, ModelCls, in_ch, grad_mode in CONFIGS:
    print(f"\n>>> Running: {exp_name} ({in_ch} Channels) <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb, mode=grad_mode)

                optimizer.zero_grad()
                pred_joints = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb, mode=grad_mode)
                    pred_joints = model(x_batch)

                    b_pred = pred_joints[:, CHILD_INDICES] - pred_joints[:, PARENT_INDICES]
                    b_gt   = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred_joints - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)
                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES)

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Dir Error: {best_val_deg_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. RESULTS & PER-BONE STAIRCASE AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 2x2 FACTORIAL BENCHMARK SUMMARY")
print("=" * 90)
print(f"{'Configuration':<36} | {'MPJPE (mm)':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _, _, _ in CONFIGS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<36} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: DISTAL STAIRCASE AUDIT (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'Unc (Scalar)':<14} | {'LCF (Scalar)':<14} | {'Unc (Direct)':<14} | {'LCF (Direct)':<14}"
print(header)
print("-" * 105)

u_sc = per_bone_deg_matrix["Uncoupled S² + Scalar Sobel"]
l_sc = per_bone_deg_matrix["LCF + Scalar Sobel"]
u_di = per_bone_deg_matrix["Uncoupled S² + Directional Sobel"]
l_di = per_bone_deg_matrix["LCF + Directional Sobel"]

for k in range(20):
    print(f"{BONE_NAMES[k]:<24} | {u_sc[k]:5.2f}°        | {l_sc[k]:5.2f}°        | {u_di[k]:5.2f}°        | {l_di[k]:5.2f}°")
print("=" * 105)

Hardware Accelerator : Tesla T4
Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...
Staged 5,114 frames into VRAM (720.4 MB used).

 RUNNING 2x2 FACTORIAL BENCHMARK: GRADIENT DIRECTION VS LATENT FACTORIZATION

>>> Running: Uncoupled S² + Scalar Sobel (1 Channels) <<<
  [Seed 42 ] MPJPE:  33.33 mm | Dir Error: 31.16°
  [Seed 101] MPJPE:  32.62 mm | Dir Error: 35.02°
  [Seed 777] MPJPE:  27.78 mm | Dir Error: 24.41°

>>> Running: LCF + Scalar Sobel (1 Channels) <<<
  [Seed 42 ] MPJPE:  32.57 mm | Dir Error: 29.98°
  [Seed 101] MPJPE:  32.24 mm | Dir Error: 35.02°
  [Seed 777] MPJPE:  27.84 mm | Dir Error: 24.05°

>>> Running: Uncoupled S² + Directional Sobel (3 Channels) <<<
  [Seed 42 ] MPJPE:  33.69 mm | Dir Error: 30.36°
  [Seed 101] MPJPE:  32.41 mm | Dir Error: 34.27°
  [Seed 777] MPJPE:  29.79 mm | Dir Error: 24.97°

>>> Running: LCF + Directional Sobel (3 Channels) <<<
  [Seed 42 ] MPJPE:  33.85 mm | Dir Error: 31.17°
  [Seed 101] MPJPE:  33.69 mm | Dir Error: 35.21°
  [See

In [ ]:
# @title
import os
import re
import csv
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

assert MANIFEST_PATH.exists(), f"Missing manifest at {MANIFEST_PATH}"

# Extract frame number from path (e.g., 'frame_000123.jpg' or '.../000123.jpg')
def extract_frame_id(path_str: str) -> int:
    stem = Path(path_str).stem
    nums = re.findall(r"\d+", stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Cannot extract numeric frame ID from: {path_str}")

# Group records by segment
segments_data = defaultdict(list)
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        seg = row["segment"]
        frame_id = extract_frame_id(row["image_path"])
        segments_data[seg].append({
            "frame_id": frame_id,
            "image_path": row["image_path"],
            "kp_path": row["orig_kp_right"]
        })

print("=" * 100)
print(f"{'Segment Name':<35} | {'Frames':<7} | {'ID Range':<15} | {'Gaps > 1':<9} | {'Max Gap':<8} | {'Contig. Pairs':<14}")
print("-" * 100)

total_segments = len(segments_data)
total_frames = 0
total_contiguous_pairs = 0
total_gaps = 0
segment_continuity = {}

for seg, rows in sorted(segments_data.items()):
    # Sort strictly by frame_id
    rows.sort(key=lambda r: r["frame_id"])
    frame_ids = [r["frame_id"] for r in rows]

    diffs = np.diff(frame_ids)
    gaps = np.where(diffs > 1)[0]
    num_gaps = len(gaps)
    max_gap = int(np.max(diffs)) if len(diffs) > 0 else 0
    contiguous_pairs = int(np.sum(diffs == 1))

    total_frames += len(rows)
    total_contiguous_pairs += contiguous_pairs
    total_gaps += num_gaps

    segment_continuity[seg] = {
        "rows": rows,
        "diffs": diffs,
        "contiguous_pairs": contiguous_pairs
    }

    id_range_str = f"[{frame_ids[0]}..{frame_ids[-1]}]"
    print(f"{seg:<35} | {len(rows):<7} | {id_range_str:<15} | {num_gaps:<9} | {max_gap:<8} | {contiguous_pairs:<14}")

print("=" * 100)
print(f"Summary: {total_frames:,} frames across {total_segments} segments | {total_contiguous_pairs:,} contiguous step-1 pairs | {total_gaps:,} gaps")
print("=" * 100)

# -------------------------------------------------------------------------
# Empirical Motion Dynamics (Inter-frame Ground Truth Joint Velocities)
# -------------------------------------------------------------------------
KEEP_21 = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]

def resolve_file(base_dir: Path, rel_path: str, fallback: str) -> Path:
    p = base_dir / rel_path
    if p.exists(): return p
    p2 = base_dir / fallback / Path(rel_path).name
    if p2.exists(): return p2
    raise FileNotFoundError(rel_path)

step1_velocities_root_rel = []
step1_velocities_distal = []

for seg, meta in segment_continuity.items():
    rows = meta["rows"]
    for i in range(len(rows) - 1):
        if rows[i+1]["frame_id"] - rows[i]["frame_id"] == 1:
            p1 = resolve_file(DATA_DIR, rows[i]["kp_path"], "keypoints_right")
            p2 = resolve_file(DATA_DIR, rows[i+1]["kp_path"], "keypoints_right")

            with open(p1) as f: kps1 = np.array(json.load(f), dtype=np.float32)[KEEP_21]
            with open(p2) as f: kps2 = np.array(json.load(f), dtype=np.float32)[KEEP_21]

            # Root-relative joint displacement in mm
            rel1 = kps1 - kps1[0:1]
            rel2 = kps2 - kps2[0:1]
            disp = np.linalg.norm(rel2 - rel1, axis=-1)  # (21,)

            step1_velocities_root_rel.append(disp)
            step1_velocities_distal.append(disp[[4, 8, 12, 16, 20]])  # 5 Fingertips

vel_all = np.array(step1_velocities_root_rel)
vel_distal = np.array(step1_velocities_distal)

print("\nEMPIRICAL INTER-FRAME DISPLACEMENT PROFILE (Adjacent Pairs, Step=1):")
print(f"  All 21 Joints Mean Displacement : {np.mean(vel_all):5.2f} mm/frame  (Std: {np.std(vel_all):4.2f} mm/frame, Max: {np.max(vel_all):5.2f} mm/frame)")
print(f"  Fingertip Mean Displacement     : {np.mean(vel_distal):5.2f} mm/frame  (Std: {np.std(vel_distal):4.2f} mm/frame, Max: {np.max(vel_distal):5.2f} mm/frame)")
print(f"  95th Percentile Fingertip Speed : {np.percentile(vel_distal, 95):5.2f} mm/frame")

Segment Name                        | Frames  | ID Range        | Gaps > 1  | Max Gap  | Contig. Pairs 
----------------------------------------------------------------------------------------------------
Rom0029_point_left_palm_features    | 249     | [275750..278525] | 248       | 84       | 0             
Rom0030_R_index_rubbing_back_of_L_hand | 112     | [278781..280179] | 111       | 54       | 0             
Rom0031_L_index_rubbing_back_of_R_hand | 139     | [280473..282063] | 138       | 57       | 0             
Rom0032_R_hand_rubbing_back_of_L_hand | 201     | [282237..284442] | 200       | 72       | 0             
Rom0033_L_hand_rubbing_back_of_R_hand | 207     | [284613..286785] | 206       | 48       | 0             
Rom0034_R_hand_massages_L_palm      | 211     | [286878..289325] | 210       | 96       | 0             
Rom0035_L_hand_massages_R_palm      | 224     | [289367..292064] | 223       | 57       | 0             
Rom0036_hand_washing_easy           | 203     | [3

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
# @title
import os
import re
import csv
import json
from pathlib import Path
from collections import defaultdict
import numpy as np

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

def extract_frame_id(path_str: str) -> int:
    stem = Path(path_str).stem
    nums = re.findall(r"\d+", stem)
    return int(nums[-1])

def resolve_file(base_dir: Path, rel_path: str, fallback: str) -> Path:
    p = base_dir / rel_path
    if p.exists(): return p
    p2 = base_dir / fallback / Path(rel_path).name
    if p2.exists(): return p2
    raise FileNotFoundError(rel_path)

segments_data = defaultdict(list)
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        segments_data[row["segment"]].append({
            "frame_id": extract_frame_id(row["image_path"]),
            "kp_path": row["orig_kp_right"]
        })

KEEP_21 = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]

all_diffs = []
all_displacements = []
distal_displacements = []

for seg, rows in segments_data.items():
    rows.sort(key=lambda r: r["frame_id"])
    for i in range(len(rows) - 1):
        step = rows[i+1]["frame_id"] - rows[i]["frame_id"]
        all_diffs.append(step)

        p1 = resolve_file(DATA_DIR, rows[i]["kp_path"], "keypoints_right")
        p2 = resolve_file(DATA_DIR, rows[i+1]["kp_path"], "keypoints_right")
        with open(p1) as f: k1 = np.array(json.load(f), dtype=np.float32)[KEEP_21]
        with open(p2) as f: k2 = np.array(json.load(f), dtype=np.float32)[KEEP_21]

        rel1 = k1 - k1[0:1]
        rel2 = k2 - k2[0:1]
        disp = np.linalg.norm(rel2 - rel1, axis=-1)
        all_displacements.append(disp.mean())
        distal_displacements.append(disp[[4, 8, 12, 16, 20]].mean())

diffs_arr = np.array(all_diffs)
disp_arr = np.array(all_displacements)
dist_arr = np.array(distal_displacements)

print("=" * 80)
print(" EMPIRICAL TEMPORAL SAMPLING PROFILE")
print("=" * 80)
print(f"Total Sequence Transitions: {len(diffs_arr):,}")
print(f"Frame Delta (Δframe)      : Mean={np.mean(diffs_arr):.1f} | Median={np.median(diffs_arr):.1f} | Min={np.min(diffs_arr)} | Max={np.max(diffs_arr)}")
print(f"Δframe Percentiles        : 25%={np.percentile(diffs_arr, 25):.0f} | 50%={np.percentile(diffs_arr, 50):.0f} | 75%={np.percentile(diffs_arr, 75):.0f} | 95%={np.percentile(diffs_arr, 95):.0f}")

print("-" * 80)
print(" GROUND TRUTH JOINT MOTION ACROSS EXTRACTED STEPS")
print("-" * 80)
print(f"Mean Joint Displacement   : {np.mean(disp_arr):.2f} mm/sample (Std: {np.std(disp_arr):.2f} mm)")
print(f"Distal Fingertip Motion   : {np.mean(dist_arr):.2f} mm/sample (Std: {np.std(dist_arr):.2f} mm)")
print(f"Max Fingertip Excursion   : {np.max(dist_arr):.2f} mm/sample")
print("=" * 80)

 EMPIRICAL TEMPORAL SAMPLING PROFILE
Total Sequence Transitions: 5,058
Frame Delta (Δframe)      : Mean=12.1 | Median=9.0 | Min=2 | Max=441
Δframe Percentiles        : 25%=3 | 50%=9 | 75%=15 | 95%=33
--------------------------------------------------------------------------------
 GROUND TRUTH JOINT MOTION ACROSS EXTRACTED STEPS
--------------------------------------------------------------------------------
Mean Joint Displacement   : 4.99 mm/sample (Std: 7.34 mm)
Distal Fingertip Motion   : 6.87 mm/sample (Std: 10.21 mm)
Max Fingertip Excursion   : 114.12 mm/sample


In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert 'vram_images' in globals(), "Dataset tensors missing from memory."

# Nominal camera capture rate (Re:InterHand source video is 30 FPS)
CAPTURE_FPS = 30.0

# -------------------------------------------------------------------------
# 1. TEMPORAL METADATA EXTRACTION & SEGMENT ORDERING
# -------------------------------------------------------------------------
def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

frame_ids = np.array([extract_frame_id(r["image_path"]) for r in manifest_rows])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

for seg in seg_to_indices:
    seg_to_indices[seg].sort(key=lambda i: frame_ids[i])

# -------------------------------------------------------------------------
# 2. FROZEN PERCEPTION BACKBONE (LCF ON SCALAR SOBEL)
# -------------------------------------------------------------------------
class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=1)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 3. MANIFOLD-CONSTRAINED DIRECTIONAL FILTER ON S²
# -------------------------------------------------------------------------
def slerp(u0: np.ndarray, u1: np.ndarray, t: float) -> np.ndarray:
    """Geodesic spherical interpolation on S²."""
    dot = np.clip(np.dot(u0, u1), -1.0, 1.0)
    theta = np.arccos(dot)
    if theta < 1e-4:
        blend = (1.0 - t) * u0 + t * u1
        return blend / (np.linalg.norm(blend) + 1e-7)
    sin_theta = np.sin(theta)
    return (np.sin((1.0 - t) * theta) / sin_theta) * u0 + (np.sin(t * theta) / sin_theta) * u1

class AdaptiveDirectionalFilter:
    def __init__(self, fps: float = 30.0):
        self.fps = fps
        # Maximum physiological velocity: 180 deg/s
        self.omega_max = 180.0 * (np.pi / 180.0)

        # Empirical measurement standard deviations per bone in radians
        sigma_degs = np.array([
            22.5, 27.7, 27.3, 33.6,  # Thumb
            14.3, 22.8, 32.8, 34.6,  # Index
            12.7, 22.6, 37.6, 41.1,  # Middle
            13.5, 25.1, 38.9, 43.6,  # Ring
            15.9, 33.0, 44.8, 48.2   # Pinky
        ])
        self.sigma_R = sigma_degs * (np.pi / 180.0)
        self.R_var = self.sigma_R ** 2

        # Continuous process drift rate: rad²/s
        self.q_rate = np.array([
            0.15, 0.20, 0.25, 0.30,  # Thumb
            0.10, 0.20, 0.35, 0.45,  # Index
            0.08, 0.20, 0.35, 0.45,  # Middle
            0.10, 0.20, 0.35, 0.45,  # Ring
            0.12, 0.25, 0.40, 0.50   # Pinky
        ])

    def filter_sequence(self, raw_dirs: np.ndarray, f_ids: np.ndarray):
        T = len(f_ids)
        filtered_dirs = np.zeros_like(raw_dirs)

        u_state = raw_dirs[0].copy()
        P_cov = self.R_var.copy()
        filtered_dirs[0] = u_state

        total_gates = 0
        total_evals = (T - 1) * 20

        for t in range(1, T):
            delta_frame = max(1, f_ids[t] - f_ids[t - 1])
            delta_t = delta_frame / self.fps

            # Continuous process variance update
            P_pred = P_cov + self.q_rate * delta_t
            z_meas = raw_dirs[t]

            u_next = np.zeros((20, 3))
            for b in range(20):
                u_p = u_state[b]
                u_m = z_meas[b]

                # Angular innovation
                dot = np.clip(np.dot(u_p, u_m), -1.0, 1.0)
                theta_innov = np.arccos(dot)

                # Continuous physiological innovation gate
                theta_gate = self.omega_max * delta_t + 3.0 * self.sigma_R[b]

                if theta_innov > theta_gate:
                    total_gates += 1
                    # Clamp measurement along geodesic to gate boundary
                    clamp_ratio = theta_gate / (theta_innov + 1e-7)
                    u_m = slerp(u_p, u_m, clamp_ratio)
                    theta_innov = theta_gate

                # Slerp state update on S²
                K = P_pred[b] / (P_pred[b] + self.R_var[b])
                u_next[b] = slerp(u_p, u_m, K)
                P_cov[b] = (1.0 - K) * P_pred[b]

            u_state = u_next
            filtered_dirs[t] = u_state

        gate_rate = (total_gates / max(1, total_evals)) * 100.0
        return filtered_dirs, gate_rate

# -------------------------------------------------------------------------
# 4. BENCHMARK HARNESS ACROSS 3 HELD-OUT SPLITS
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_raw_mpjpe = []
results_filt_mpjpe = []
results_raw_distal = []
results_filt_distal = []
results_raw_fast = []
results_filt_fast = []
results_gating = []
results_tau_star = []

filter_engine = AdaptiveDirectionalFilter(fps=CAPTURE_FPS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

print("=" * 95)
print(" BENCHMARKING PHYSICAL-TIME DIRECTIONAL FILTER (3 HELD-OUT SPLITS)")
print("=" * 95)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
    model = LatentFactorizationModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)

            x_batch = gpu_extractor(raw_rgb, mode="scalar")
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss, _, _, _ = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    # Chronological validation evaluation per held-out segment
    model.eval()
    val_raw_joints, val_filt_joints, val_gt_joints = [], [], []
    val_displacements = []
    split_gate_rates = []

    with torch.no_grad():
        for seg in val_segs:
            s_indices = seg_to_indices[seg]
            f_ids = frame_ids[s_indices]

            s_raw_dirs, s_gt = [], []
            for i in range(0, len(s_indices), BATCH_SIZE):
                b_idx = s_indices[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                x_batch = gpu_extractor(raw_rgb, mode="scalar")
                _, b_dirs = model(x_batch)
                s_raw_dirs.append(b_dirs.cpu().numpy())
                s_gt.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

            raw_dirs_np = np.concatenate(s_raw_dirs, axis=0)
            gt_joints_np = np.concatenate(s_gt, axis=0)

            filt_dirs_np, gate_pct = filter_engine.filter_sequence(raw_dirs_np, f_ids)
            split_gate_rates.append(gate_pct)

            raw_vecs = torch.tensor(raw_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)
            filt_vecs = torch.tensor(filt_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)

            raw_j = reconstruct_joints_from_bones(raw_vecs).cpu().numpy()
            filt_j = reconstruct_joints_from_bones(filt_vecs).cpu().numpy()

            val_raw_joints.append(raw_j)
            val_filt_joints.append(filt_j)
            val_gt_joints.append(gt_joints_np)

            # Measure GT inter-frame displacement (mm) for fast-motion slicing
            disp_seg = [0.0]
            for t in range(len(s_indices) - 1):
                d = np.linalg.norm(gt_joints_np[t + 1] - gt_joints_np[t], axis=-1).mean()
                disp_seg.append(d)
            val_displacements.append(np.array(disp_seg))

    all_raw = np.concatenate(val_raw_joints, axis=0)
    all_filt = np.concatenate(val_filt_joints, axis=0)
    all_gt = np.concatenate(val_gt_joints, axis=0)
    all_disp = np.concatenate(val_displacements, axis=0)

    # 1. Global MPJPE
    raw_mpjpe = np.linalg.norm(all_raw - all_gt, axis=-1).mean()
    filt_mpjpe = np.linalg.norm(all_filt - all_gt, axis=-1).mean()

    # 2. Distal Direction Error
    b_gt = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]
    u_gt = b_gt / (np.linalg.norm(b_gt, axis=-1, keepdims=True) + 1e-7)

    raw_b = all_raw[:, CHILD_INDICES.cpu().numpy()] - all_raw[:, PARENT_INDICES.cpu().numpy()]
    u_raw = raw_b / (np.linalg.norm(raw_b, axis=-1, keepdims=True) + 1e-7)

    filt_b = all_filt[:, CHILD_INDICES.cpu().numpy()] - all_filt[:, PARENT_INDICES.cpu().numpy()]
    u_filt = filt_b / (np.linalg.norm(filt_b, axis=-1, keepdims=True) + 1e-7)

    deg_raw = np.arccos(np.clip(np.sum(u_raw[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)
    deg_filt = np.arccos(np.clip(np.sum(u_filt[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)

    # 3. Fast-Motion Slice (>95th percentile velocity)
    threshold_95 = np.percentile(all_disp, 95)
    fast_mask = all_disp >= threshold_95
    fast_raw_mpjpe = np.linalg.norm(all_raw[fast_mask] - all_gt[fast_mask], axis=-1).mean()
    fast_filt_mpjpe = np.linalg.norm(all_filt[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    # 4. Empirical Lag Offset Search: tau* = argmin_tau E[P_filt(t), P_gt(t + tau)]
    tau_shifts = [-2, -1, 0, 1, 2]
    tau_errors = []
    for tau in tau_shifts:
        if tau < 0:
            err = np.linalg.norm(all_filt[:tau] - all_gt[-tau:], axis=-1).mean()
        elif tau > 0:
            err = np.linalg.norm(all_filt[tau:] - all_gt[:-tau], axis=-1).mean()
        else:
            err = filt_mpjpe
        tau_errors.append(err)
    best_tau = tau_shifts[int(np.argmin(tau_errors))]

    results_raw_mpjpe.append(raw_mpjpe)
    results_filt_mpjpe.append(filt_mpjpe)
    results_raw_distal.append(deg_raw.mean())
    results_filt_distal.append(deg_filt.mean())
    results_raw_fast.append(fast_raw_mpjpe)
    results_filt_fast.append(fast_filt_mpjpe)
    results_gating.append(np.mean(split_gate_rates))
    results_tau_star.append(best_tau)

    print(f"  [Seed {seed:<3}] MPJPE: {raw_mpjpe:5.2f}->{filt_mpjpe:5.2f} mm | Distal: {deg_raw.mean():4.1f}°->{deg_filt.mean():4.1f}° | Fast(>95%): {fast_raw_mpjpe:5.2f}->{fast_filt_mpjpe:5.2f} mm | Gated: {np.mean(split_gate_rates):4.1f}% | tau*: {best_tau}")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 5. FINAL REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" ADAPTIVE DIRECTIONAL FILTER: MULTI-METRIC PERFORMANCE AUDIT")
print("=" * 105)
print(f"{'Evaluation Metric':<32} | {'Raw Static LCF':<20} | {'Filtered Kinematic':<20} | {'Delta':<12}")
print("-" * 105)

m_raw, s_raw = np.mean(results_raw_mpjpe), np.std(results_raw_mpjpe)
m_flt, s_flt = np.mean(results_filt_mpjpe), np.std(results_filt_mpjpe)
print(f"{'Overall MPJPE (mm)':<32} | {m_raw:5.2f} ± {s_raw:4.2f} mm    | {m_flt:5.2f} ± {s_flt:4.2f} mm    | {m_flt - m_raw:+5.2f} mm")

m_dr, s_dr = np.mean(results_raw_distal), np.std(results_raw_distal)
m_df, s_df = np.mean(results_filt_distal), np.std(results_filt_distal)
print(f"{'Distal Tip Direction Error':<32} | {m_dr:5.2f}° ± {s_dr:4.2f}°     | {m_df:5.2f}° ± {s_df:4.2f}°     | {m_df - m_dr:+5.2f}°")

m_fr, s_fr = np.mean(results_raw_fast), np.std(results_raw_fast)
m_ff, s_ff = np.mean(results_filt_fast), np.std(results_filt_fast)
print(f"{'Fast-Motion MPJPE (>95th %)':<32} | {m_fr:5.2f} ± {s_fr:4.2f} mm    | {m_ff:5.2f} ± {s_ff:4.2f} mm    | {m_ff - m_fr:+5.2f} mm")

print(f"{'Mean Innovation Gating Rate':<32} | {'—':<20} | {np.mean(results_gating):5.2f} %            | —")
print(f"{'Optimal Lag Shift (tau*)':<32} | {'—':<20} | {int(np.median(results_tau_star)):<2} frames            | —")
print("=" * 105)

AssertionError: Dataset tensors missing from memory.

In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE VERIFICATION & VRAM PREPARATION
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Go to Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Hardware Accelerator : {gpu_name}")
print(f"Available VRAM       : {total_vram_gb:.2f} GB")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure the dataset is extracted.")

# Official Re:InterHand 24 -> Kinematic 21 Joint Index Mapping
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

# -------------------------------------------------------------------------
# 2. GPU SOBEL EXTRACTOR
# -------------------------------------------------------------------------
class GPURepresentationEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx_1 = kx.view(1, 1, 3, 3)
        self.ky_1 = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        # Standard luminance (B, 1, H, W)
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx_1, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky_1, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPURepresentationEngine(DEVICE)

# -------------------------------------------------------------------------
# 3. SELF-HEALING VRAM INGESTION (SAFE UINT8 CACHING)
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("\nPreloading dataset into Tesla T4 VRAM (uint8 compact storage)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_images_list, raw_targets_list, sample_segments_list = [], [], []

    for row in records:
        try:
            img_p = resolve_file(DATA_DIR, row["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_images_list.append(rgb_resized.transpose(2, 0, 1))
            raw_targets_list.append(kps_rel)
            sample_segments_list.append(row["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_images_list), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targets_list), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(sample_segments_list)
    unique_segments = np.unique(sample_segments)

    del raw_images_list, raw_targets_list, sample_segments_list
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} existing frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 4. KINEMATIC TOPOLOGY & STRUCTURAL LOSS DEFINITIONS
# -------------------------------------------------------------------------
# 20 directed bone parent-child pairs in 21-joint space
BONE_PAIRS = [
    # Thumb: Wrist -> CMC -> MCP -> IP -> Tip
    (0, 1), (1, 2), (2, 3), (3, 4),
    # Index: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 5), (5, 6), (6, 7), (7, 8),
    # Middle: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 9), (9, 10), (10, 11), (11, 12),
    # Ring: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 13), (13, 14), (14, 15), (15, 16),
    # Pinky: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 17), (17, 18), (18, 19), (19, 20)
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

class KinematicStructureLoss(nn.Module):
    def __init__(self, lambda_len: float = 0.5, lambda_dir: float = 10.0):
        super().__init__()
        self.lambda_len = lambda_len
        self.lambda_dir = lambda_dir

    def forward(self, pred: torch.Tensor, target: torch.Tensor):
        # Base Coordinate Position Loss (MSE)
        l_pos = F.mse_loss(pred, target)

        # Bone vectors (B, 20, 3)
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # Bone length consistency (L1 error in mm)
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # Bone direction angular alignment (Cosine distance)
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        l_dir = (1.0 - (u_pred * u_gt).sum(dim=-1)).mean()

        total = l_pos + self.lambda_len * l_len + self.lambda_dir * l_dir
        return total

# -------------------------------------------------------------------------
# 5. POSE BACKBONE (1-CHANNEL INPUT)
# -------------------------------------------------------------------------
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int = 1, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS: UNCONSTRAINED MSE VS. KINEMATIC LOSS
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

EXPERIMENTS = [
    ("Unconstrained MSE Loss", False),
    ("Kinematic Structural Loss", True)
]

comparison_matrix = {name: [] for name, _ in EXPERIMENTS}
bone_error_matrix = {name: [] for name, _ in EXPERIMENTS}

print("\n" + "=" * 80)
print(" BENCHMARKING LOSS OBJECTIVES ACROSS 3 SEGMENT SPLITS")
print("=" * 80)

for exp_name, use_kinematic in EXPERIMENTS:
    print(f"\n>>> Running: {exp_name} <<<")
    kin_loss_fn = KinematicStructureLoss()

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        # Strictly identical segment splits across both runs
        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=1, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_bone_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb_batch = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb_batch)

                optimizer.zero_grad()
                pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                loss = kin_loss_fn(pred, y_batch) if use_kinematic else F.mse_loss(pred, y_batch)
                loss.backward()
                optimizer.step()

            # Validation evaluation
            model.eval()
            total_err, total_bone_err, total_pts = 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb_batch = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb_batch)
                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                    # MPJPE (mm)
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone length deformation error (mm)
                    b_pred = torch.norm(pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES], dim=-1)
                    b_gt = torch.norm(y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES], dim=-1)
                    total_bone_err += torch.abs(b_pred - b_gt).sum().item()

            val_mpjpe = total_err / total_pts
            val_bone_err = total_bone_err / (len(val_indices) * len(BONE_PAIRS))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_bone_err = val_bone_err

        comparison_matrix[exp_name].append(best_val_mpjpe)
        bone_error_matrix[exp_name].append(best_val_bone_err)
        print(f"  [Seed {seed:<3}] Val MPJPE: {best_val_mpjpe:6.2f} mm | Bone Length Error: {best_val_bone_err:5.2f} mm")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 7. RESULTS SUMMARY
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" LOSS OBJECTIVE COMPARISON: 3-SPLIT SEGMENT-HELD-OUT EVALUATION")
print("=" * 80)
print(f"{'Objective Formulation':<30} | {'Mean MPJPE':<18} | {'Bone Length Error':<18}")
print("-" * 80)
for exp_name, _ in EXPERIMENTS:
    m_pjpe = np.mean(comparison_matrix[exp_name])
    s_pjpe = np.std(comparison_matrix[exp_name])
    m_bone = np.mean(bone_error_matrix[exp_name])
    s_bone = np.std(bone_error_matrix[exp_name])
    print(f"{exp_name:<30} | {m_pjpe:6.2f} ± {s_pjpe:4.2f} mm | {m_bone:5.2f} ± {s_bone:4.2f} mm")
print("=" * 80)

Hardware Accelerator : Tesla T4
Available VRAM       : 14.56 GB

Preloading dataset into Tesla T4 VRAM (uint8 compact storage)...
Staged 5,114 frames into VRAM (720.4 MB used).

 BENCHMARKING LOSS OBJECTIVES ACROSS 3 SEGMENT SPLITS

>>> Running: Unconstrained MSE Loss <<<
  [Seed 42 ] Val MPJPE:  33.69 mm | Bone Length Error:  5.20 mm
  [Seed 101] Val MPJPE:  35.87 mm | Bone Length Error:  5.77 mm
  [Seed 777] Val MPJPE:  31.32 mm | Bone Length Error:  5.03 mm

>>> Running: Kinematic Structural Loss <<<
  [Seed 42 ] Val MPJPE:  33.89 mm | Bone Length Error:  4.83 mm
  [Seed 101] Val MPJPE:  36.35 mm | Bone Length Error:  5.54 mm
  [Seed 777] Val MPJPE:  30.79 mm | Bone Length Error:  4.79 mm

 LOSS OBJECTIVE COMPARISON: 3-SPLIT SEGMENT-HELD-OUT EVALUATION
Objective Formulation          | Mean MPJPE         | Bone Length Error 
--------------------------------------------------------------------------------
Unconstrained MSE Loss         |  33.63 ± 1.86 mm |  5.33 ± 0.32 mm
Kinematic St

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192
CAPTURE_FPS = 30.0

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Parent-Child Bone Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

frame_ids = np.array([extract_frame_id(r["image_path"]) for r in manifest_rows])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

for seg in seg_to_indices:
    seg_to_indices[seg].sort(key=lambda i: frame_ids[i])

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, mode: str = "scalar") -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)
        if mode == "scalar":
            return g_mag
        elif mode == "directional":
            return torch.cat([gx, gy, g_mag], dim=1)
        raise ValueError(f"Unknown mode: {mode}")

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. BACKBONE & LCF HEAD
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 4. LOSS EVALUATOR
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. MANIFOLD-CONSTRAINED DIRECTIONAL FILTER ON S²
# -------------------------------------------------------------------------
def slerp(u0: np.ndarray, u1: np.ndarray, t: float) -> np.ndarray:
    dot = np.clip(np.dot(u0, u1), -1.0, 1.0)
    theta = np.arccos(dot)
    if theta < 1e-4:
        blend = (1.0 - t) * u0 + t * u1
        return blend / (np.linalg.norm(blend) + 1e-7)
    sin_theta = np.sin(theta)
    return (np.sin((1.0 - t) * theta) / sin_theta) * u0 + (np.sin(t * theta) / sin_theta) * u1

class AdaptiveDirectionalFilter:
    def __init__(self, fps: float = 30.0):
        self.fps = fps
        self.omega_max = 180.0 * (np.pi / 180.0)  # Maximum physiological velocity: 180 deg/s

        sigma_degs = np.array([
            22.5, 27.7, 27.3, 33.6,  # Thumb
            14.3, 22.8, 32.8, 34.6,  # Index
            12.7, 22.6, 37.6, 41.1,  # Middle
            13.5, 25.1, 38.9, 43.6,  # Ring
            15.9, 33.0, 44.8, 48.2   # Pinky
        ])
        self.sigma_R = sigma_degs * (np.pi / 180.0)
        self.R_var = self.sigma_R ** 2

        self.q_rate = np.array([
            0.15, 0.20, 0.25, 0.30,  # Thumb
            0.10, 0.20, 0.35, 0.45,  # Index
            0.08, 0.20, 0.35, 0.45,  # Middle
            0.10, 0.20, 0.35, 0.45,  # Ring
            0.12, 0.25, 0.40, 0.50   # Pinky
        ])

    def filter_sequence(self, raw_dirs: np.ndarray, f_ids: np.ndarray):
        T = len(f_ids)
        filtered_dirs = np.zeros_like(raw_dirs)

        u_state = raw_dirs[0].copy()
        P_cov = self.R_var.copy()
        filtered_dirs[0] = u_state

        total_gates = 0
        total_evals = (T - 1) * 20

        for t in range(1, T):
            delta_frame = max(1, f_ids[t] - f_ids[t - 1])
            delta_t = delta_frame / self.fps

            P_pred = P_cov + self.q_rate * delta_t
            z_meas = raw_dirs[t]

            u_next = np.zeros((20, 3))
            for b in range(20):
                u_p = u_state[b]
                u_m = z_meas[b]

                dot = np.clip(np.dot(u_p, u_m), -1.0, 1.0)
                theta_innov = np.arccos(dot)
                theta_gate = self.omega_max * delta_t + 3.0 * self.sigma_R[b]

                if theta_innov > theta_gate:
                    total_gates += 1
                    clamp_ratio = theta_gate / (theta_innov + 1e-7)
                    u_m = slerp(u_p, u_m, clamp_ratio)

                K = P_pred[b] / (P_pred[b] + self.R_var[b])
                u_next[b] = slerp(u_p, u_m, K)
                P_cov[b] = (1.0 - K) * P_pred[b]

            u_state = u_next
            filtered_dirs[t] = u_state

        gate_rate = (total_gates / max(1, total_evals)) * 100.0
        return filtered_dirs, gate_rate

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS (3 HELD-OUT SPLITS)
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_raw_mpjpe = []
results_filt_mpjpe = []
results_raw_distal = []
results_filt_distal = []
results_raw_fast = []
results_filt_fast = []
results_gating = []
results_tau_star = []

filter_engine = AdaptiveDirectionalFilter(fps=CAPTURE_FPS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

print("=" * 95)
print(f" BENCHMARKING PHYSICAL-TIME DIRECTIONAL FILTER (FPS={CAPTURE_FPS}, 3 HELD-OUT SPLITS)")
print("=" * 95)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
    model = LatentFactorizationModel(in_channels=1).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)

            x_batch = gpu_extractor(raw_rgb, mode="scalar")
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss, _, _, _ = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    val_raw_joints, val_filt_joints, val_gt_joints = [], [], []
    val_displacements = []
    split_gate_rates = []

    with torch.no_grad():
        for seg in val_segs:
            s_indices = seg_to_indices[seg]
            f_ids = frame_ids[s_indices]

            s_raw_dirs, s_gt = [], []
            for i in range(0, len(s_indices), BATCH_SIZE):
                b_idx = s_indices[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                x_batch = gpu_extractor(raw_rgb, mode="scalar")
                _, b_dirs = model(x_batch)
                s_raw_dirs.append(b_dirs.cpu().numpy())
                s_gt.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

            raw_dirs_np = np.concatenate(s_raw_dirs, axis=0)
            gt_joints_np = np.concatenate(s_gt, axis=0)

            filt_dirs_np, gate_pct = filter_engine.filter_sequence(raw_dirs_np, f_ids)
            split_gate_rates.append(gate_pct)

            raw_vecs = torch.tensor(raw_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)
            filt_vecs = torch.tensor(filt_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)

            raw_j = reconstruct_joints_from_bones(raw_vecs).cpu().numpy()
            filt_j = reconstruct_joints_from_bones(filt_vecs).cpu().numpy()

            val_raw_joints.append(raw_j)
            val_filt_joints.append(filt_j)
            val_gt_joints.append(gt_joints_np)

            disp_seg = [0.0]
            for t in range(len(s_indices) - 1):
                d = np.linalg.norm(gt_joints_np[t + 1] - gt_joints_np[t], axis=-1).mean()
                disp_seg.append(d)
            val_displacements.append(np.array(disp_seg))

    all_raw = np.concatenate(val_raw_joints, axis=0)
    all_filt = np.concatenate(val_filt_joints, axis=0)
    all_gt = np.concatenate(val_gt_joints, axis=0)
    all_disp = np.concatenate(val_displacements, axis=0)

    # 1. Global MPJPE
    raw_mpjpe = np.linalg.norm(all_raw - all_gt, axis=-1).mean()
    filt_mpjpe = np.linalg.norm(all_filt - all_gt, axis=-1).mean()

    # 2. Distal Direction Error
    b_gt = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]
    u_gt = b_gt / (np.linalg.norm(b_gt, axis=-1, keepdims=True) + 1e-7)

    raw_b = all_raw[:, CHILD_INDICES.cpu().numpy()] - all_raw[:, PARENT_INDICES.cpu().numpy()]
    u_raw = raw_b / (np.linalg.norm(raw_b, axis=-1, keepdims=True) + 1e-7)

    filt_b = all_filt[:, CHILD_INDICES.cpu().numpy()] - all_filt[:, PARENT_INDICES.cpu().numpy()]
    u_filt = filt_b / (np.linalg.norm(filt_b, axis=-1, keepdims=True) + 1e-7)

    deg_raw = np.arccos(np.clip(np.sum(u_raw[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)
    deg_filt = np.arccos(np.clip(np.sum(u_filt[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)

    # 3. Fast-Motion Slice (>95th percentile displacement)
    threshold_95 = np.percentile(all_disp, 95)
    fast_mask = all_disp >= threshold_95
    fast_raw_mpjpe = np.linalg.norm(all_raw[fast_mask] - all_gt[fast_mask], axis=-1).mean()
    fast_filt_mpjpe = np.linalg.norm(all_filt[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    # 4. Lag Shift Search: tau* = argmin_tau E[P_filt(t), P_gt(t + tau)]
    tau_shifts = [-2, -1, 0, 1, 2]
    tau_errors = []
    for tau in tau_shifts:
        if tau < 0:
            err = np.linalg.norm(all_filt[:tau] - all_gt[-tau:], axis=-1).mean()
        elif tau > 0:
            err = np.linalg.norm(all_filt[tau:] - all_gt[:-tau], axis=-1).mean()
        else:
            err = filt_mpjpe
        tau_errors.append(err)
    best_tau = tau_shifts[int(np.argmin(tau_errors))]

    results_raw_mpjpe.append(raw_mpjpe)
    results_filt_mpjpe.append(filt_mpjpe)
    results_raw_distal.append(deg_raw.mean())
    results_filt_distal.append(deg_filt.mean())
    results_raw_fast.append(fast_raw_mpjpe)
    results_filt_fast.append(fast_filt_mpjpe)
    results_gating.append(np.mean(split_gate_rates))
    results_tau_star.append(best_tau)

    print(f"  [Seed {seed:<3}] MPJPE: {raw_mpjpe:5.2f}->{filt_mpjpe:5.2f} mm | Distal: {deg_raw.mean():4.1f}°->{deg_filt.mean():4.1f}° | Fast(>95%): {fast_raw_mpjpe:5.2f}->{fast_filt_mpjpe:5.2f} mm | Gated: {np.mean(split_gate_rates):4.1f}% | tau*: {best_tau}")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 7. FINAL COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" ADAPTIVE DIRECTIONAL FILTER: MULTI-METRIC PERFORMANCE AUDIT")
print("=" * 105)
print(f"{'Evaluation Metric':<32} | {'Raw Static LCF':<20} | {'Filtered Kinematic':<20} | {'Delta':<12}")
print("-" * 105)

m_raw, s_raw = np.mean(results_raw_mpjpe), np.std(results_raw_mpjpe)
m_flt, s_flt = np.mean(results_filt_mpjpe), np.std(results_filt_mpjpe)
print(f"{'Overall MPJPE (mm)':<32} | {m_raw:5.2f} ± {s_raw:4.2f} mm    | {m_flt:5.2f} ± {s_flt:4.2f} mm    | {m_flt - m_raw:+5.2f} mm")

m_dr, s_dr = np.mean(results_raw_distal), np.std(results_raw_distal)
m_df, s_df = np.mean(results_filt_distal), np.std(results_filt_distal)
print(f"{'Distal Tip Direction Error':<32} | {m_dr:5.2f}° ± {s_dr:4.2f}°     | {m_df:5.2f}° ± {s_df:4.2f}°     | {m_df - m_dr:+5.2f}°")

m_fr, s_fr = np.mean(results_raw_fast), np.std(results_raw_fast)
m_ff, s_ff = np.mean(results_filt_fast), np.std(results_filt_fast)
print(f"{'Fast-Motion MPJPE (>95th %)':<32} | {m_fr:5.2f} ± {s_fr:4.2f} mm    | {m_ff:5.2f} ± {s_ff:4.2f} mm    | {m_ff - m_fr:+5.2f} mm")

print(f"{'Mean Innovation Gating Rate':<32} | {'—':<20} | {np.mean(results_gating):5.2f} %            | —")
print(f"{'Optimal Lag Shift (tau*)':<32} | {'—':<20} | {int(np.median(results_tau_star)):<2} frames            | —")
print("=" * 105)

Hardware Accelerator : Tesla T4
Reusing 5,114 frames already resident in VRAM.
 BENCHMARKING PHYSICAL-TIME DIRECTIONAL FILTER (FPS=30.0, 3 HELD-OUT SPLITS)
  [Seed 42 ] MPJPE: 32.79->33.00 mm | Distal: 39.7°->39.9° | Fast(>95%): 34.23->36.04 mm | Gated:  0.0% | tau*: 0
  [Seed 101] MPJPE: 32.43->32.37 mm | Distal: 51.0°->51.0° | Fast(>95%): 37.14->38.53 mm | Gated:  0.0% | tau*: 0
  [Seed 777] MPJPE: 28.08->28.26 mm | Distal: 30.0°->29.9° | Fast(>95%): 33.28->34.67 mm | Gated:  0.0% | tau*: 0

 ADAPTIVE DIRECTIONAL FILTER: MULTI-METRIC PERFORMANCE AUDIT
Evaluation Metric                | Raw Static LCF       | Filtered Kinematic   | Delta       
---------------------------------------------------------------------------------------------------------
Overall MPJPE (mm)               | 31.10 ± 2.14 mm    | 31.21 ± 2.10 mm    | +0.11 mm
Distal Tip Direction Error       | 40.20° ± 8.57°     | 40.28° ± 8.62°     | +0.08°
Fast-Motion MPJPE (>95th %)      | 34.88 ± 1.64 mm    | 36.42 ± 1.60 m

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Parent-Child Bone Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

# -------------------------------------------------------------------------
# 2. FILE MAP & VRAM INGESTION
# -------------------------------------------------------------------------
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))

    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

print("Building file map across filesystem...")
FILE_MAP = build_file_map(DATA_DIR)
print(f"Indexed {len(FILE_MAP):,} candidate files.")

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p

    p1 = base_dir / rel_clean
    if p1.exists(): return p1

    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2

    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3

    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5

    if fname in FILE_MAP:
        return FILE_MAP[fname]

    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []

    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            fid = extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError(f"Failed to load any images! Sample errors encountered:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 3. CAUSAL TEMPORAL LOOKUP TABLE
# -------------------------------------------------------------------------
seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

prev_frame_indices = np.zeros(len(sample_segments), dtype=np.int64)
for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            prev_frame_indices[i] = i  # Causal boundary condition: repeat frame at t=0
        else:
            prev_frame_indices[i] = indices[step_idx - 1]

vram_prev_indices = torch.tensor(prev_frame_indices, dtype=torch.long, device=DEVICE)

# -------------------------------------------------------------------------
# 4. CANONICAL CONSTANTS & GPU SOBEL ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 5. BACKBONE & LCF HEAD (SUPPORTS VARIABLE INPUT CHANNELS)
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class TemporalLCFModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 6. EVALUATION MODULE & INPUT BUILDER
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

def build_input_tensor(b_idx: torch.Tensor, mode: str) -> torch.Tensor:
    raw_rgb_t = vram_images[b_idx].float()
    g_t = gpu_extractor(raw_rgb_t)

    if mode == "static_1ch":
        return g_t

    prev_b_idx = vram_prev_indices[b_idx]
    raw_rgb_prev = vram_images[prev_b_idx].float()
    g_prev = gpu_extractor(raw_rgb_prev)

    if mode == "context_2ch":
        return torch.cat([g_t, g_prev], dim=1)
    elif mode == "motion_3ch":
        delta_g = g_t - g_prev
        return torch.cat([g_t, g_prev, delta_g], dim=1)
    raise ValueError(f"Unknown mode: {mode}")

# -------------------------------------------------------------------------
# 7. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
CONFIGURATIONS = [
    ("Static Baseline (|G_t|)",               1, "static_1ch"),
    ("Dual-Frame Context ([|G_t|, |G_t-1|])",  2, "context_2ch"),
    ("Differential Motion ([|G_t|, |G_t-1|, Δ|G|])", 3, "motion_3ch")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _ in CONFIGURATIONS}
results_distal_deg = {name: [] for name, _, _ in CONFIGURATIONS}
results_fast_motion = {name: [] for name, _, _ in CONFIGURATIONS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _, _ in CONFIGURATIONS}

print("\n" + "=" * 105)
print(" BENCHMARKING TEMPORAL PERCEPTION (EARLY FUSION) ACROSS 3 HELD-OUT SPLITS")
print("=" * 105)

for exp_name, in_ch, mode in CONFIGURATIONS:
    print(f"\n>>> Running Configuration: {exp_name} ({in_ch} Channels) <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = TemporalLCFModel(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)
                x_batch = build_input_tensor(b_idx, mode)

                optimizer.zero_grad()
                pred_joints, _ = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

        # Chronological validation evaluation
        model.eval()
        val_preds = []
        val_targets = []
        val_displacements = []

        with torch.no_grad():
            for seg in val_segs:
                s_indices = seg_to_indices[seg]
                s_idx_tensor = torch.tensor(s_indices, device=DEVICE)

                s_preds = []
                s_gt = []
                for i in range(0, len(s_idx_tensor), BATCH_SIZE):
                    b_idx = s_idx_tensor[i:i + BATCH_SIZE]
                    x_batch = build_input_tensor(b_idx, mode)
                    pred_j, _ = model(x_batch)
                    s_preds.append(pred_j.cpu().numpy())
                    s_gt.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

                s_pred_arr = np.concatenate(s_preds, axis=0)
                s_gt_arr = np.concatenate(s_gt, axis=0)

                val_preds.append(s_pred_arr)
                val_targets.append(s_gt_arr)

                disp_seg = [0.0]
                for t in range(len(s_gt_arr) - 1):
                    d = np.linalg.norm(s_gt_arr[t + 1] - s_gt_arr[t], axis=-1).mean()
                    disp_seg.append(d)
                val_displacements.append(np.array(disp_seg))

        all_pred = np.concatenate(val_preds, axis=0)
        all_gt = np.concatenate(val_targets, axis=0)
        all_disp = np.concatenate(val_displacements, axis=0)

        # 1. MPJPE (mm)
        val_mpjpe = np.linalg.norm(all_pred - all_gt, axis=-1).mean()

        # 2. Per-bone and Distal Direction Error (degrees)
        b_pred = all_pred[:, CHILD_INDICES.cpu().numpy()] - all_pred[:, PARENT_INDICES.cpu().numpy()]
        b_gt = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]

        u_pred = b_pred / (np.linalg.norm(b_pred, axis=-1, keepdims=True) + 1e-7)
        u_gt = b_gt / (np.linalg.norm(b_gt, axis=-1, keepdims=True) + 1e-7)

        cos_sim = np.clip(np.sum(u_pred * u_gt, axis=-1), -1.0, 1.0)
        deg_per_bone = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_per_bone[:, DISTAL_BONE_INDICES].mean()

        # 3. Fast-Motion Slice (>95th percentile displacement)
        threshold_95 = np.percentile(all_disp, 95)
        fast_mask = all_disp >= threshold_95
        fast_mpjpe = np.linalg.norm(all_pred[fast_mask] - all_gt[fast_mask], axis=-1).mean()

        results_mpjpe[exp_name].append(val_mpjpe)
        results_distal_deg[exp_name].append(distal_deg)
        results_fast_motion[exp_name].append(fast_mpjpe)
        accum_bone_deg += deg_per_bone.mean(axis=0)

        print(f"  [Seed {seed:<3}] MPJPE: {val_mpjpe:5.2f} mm | Distal Dir: {distal_deg:4.1f}° | Fast-Motion (>95%): {fast_mpjpe:5.2f} mm")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 8. COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" TEMPORAL PERCEPTION BENCHMARK: 3-SPLIT SEGMENT-DISJOINT EVALUATION")
print("=" * 105)
print(f"{'Input Formulation':<38} | {'MPJPE (mm)':<18} | {'Distal Dir (deg)':<18} | {'Fast-Motion MPJPE':<18}")
print("-" * 105)
for exp_name, _, _ in CONFIGURATIONS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_distal_deg[exp_name]), np.std(results_distal_deg[exp_name])
    m_f, s_f = np.mean(results_fast_motion[exp_name]), np.std(results_fast_motion[exp_name])
    print(f"{exp_name:<38} | {m_p:5.2f} ± {s_p:4.2f} mm    | {m_d:4.1f}° ± {s_d:3.1f}°     | {m_f:5.2f} ± {s_f:4.2f} mm")
print("=" * 105)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: DISTAL STAIRCASE AUDIT (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'Static (|G_t|)':<16} | {'Dual-Frame':<16} | {'Motion Stack':<16} | {'Delta (Mot - Stat)':<18}"
print(header)
print("-" * 105)

deg_stat = per_bone_deg_matrix["Static Baseline (|G_t|)"]
deg_dual = per_bone_deg_matrix["Dual-Frame Context ([|G_t|, |G_t-1|])"]
deg_mot  = per_bone_deg_matrix["Differential Motion ([|G_t|, |G_t-1|, Δ|G|])"]

for k in range(20):
    diff = deg_mot[k] - deg_stat[k]
    sign = "+" if diff > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {deg_stat[k]:5.2f}°          | {deg_dual[k]:5.2f}°          | {deg_mot[k]:5.2f}°          | {sign}{diff:5.2f}°")
print("=" * 105)

Hardware Accelerator : Tesla T4
Building file map across filesystem...
Indexed 20,467 candidate files.
Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...
Successfully staged 5,114 frames into VRAM (720.4 MB used).

 BENCHMARKING TEMPORAL PERCEPTION (EARLY FUSION) ACROSS 3 HELD-OUT SPLITS

>>> Running Configuration: Static Baseline (|G_t|) (1 Channels) <<<
  [Seed 42 ] MPJPE: 32.88 mm | Distal Dir: 39.7° | Fast-Motion (>95%): 34.38 mm
  [Seed 101] MPJPE: 32.75 mm | Distal Dir: 51.6° | Fast-Motion (>95%): 37.75 mm
  [Seed 777] MPJPE: 28.08 mm | Distal Dir: 29.9° | Fast-Motion (>95%): 32.98 mm

>>> Running Configuration: Dual-Frame Context ([|G_t|, |G_t-1|]) (2 Channels) <<<
  [Seed 42 ] MPJPE: 34.83 mm | Distal Dir: 43.6° | Fast-Motion (>95%): 36.24 mm
  [Seed 101] MPJPE: 31.56 mm | Distal Dir: 51.4° | Fast-Motion (>95%): 36.23 mm
  [Seed 777] MPJPE: 29.22 mm | Distal Dir: 32.1° | Fast-Motion (>95%): 34.10 mm

>>> Running Configuration: Differential Motion ([|G_t|, |G_t-1|, Δ|G|]

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. FILE MAP & VRAM INGESTION
# -------------------------------------------------------------------------
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

print("Building file map across filesystem...")
FILE_MAP = build_file_map(DATA_DIR)
print(f"Indexed {len(FILE_MAP):,} candidate files.")

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p
    p1 = base_dir / rel_clean
    if p1.exists(): return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5
    if fname in FILE_MAP:
        return FILE_MAP[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None: continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. FROZEN PERCEPTION BACKBONE (STATIC LCF)
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=1)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. KINEMATIC MANIFOLD PROJECTOR (02_STATE)
# -------------------------------------------------------------------------
class KinematicManifoldProjector:
    """
    Solves: P* = argmin_{P in M} (P - P_meas)^T W (P - P_meas)
    subject to P in M_anatomy.
    """
    def __init__(self, canonical_lens: torch.Tensor):
        self.bone_lens = canonical_lens  # (20,)

        # Empirical measurement standard deviations per bone in degrees
        sigma_degs = torch.tensor([
            22.5, 27.7, 27.3, 33.6,  # Thumb
            14.3, 22.8, 32.8, 34.6,  # Index
            12.7, 22.6, 37.6, 41.1,  # Middle
            13.5, 25.1, 38.9, 43.6,  # Ring
            15.9, 33.0, 44.8, 48.2   # Pinky
        ], device=DEVICE)
        # Confidence weights: w_k = 1 / sigma_k^2 normalized
        self.w_bone = (1.0 / (sigma_degs * (np.pi / 180.0))**2)
        self.w_bone = self.w_bone / self.w_bone.max()  # [0.08 .. 1.0]

    def project_batch(self, raw_dirs: torch.Tensor, stage: str = "raw") -> torch.Tensor:
        """
        raw_dirs: (B, 20, 3) unit bone vectors from LCF
        stage: 'raw' | 'hinge' | 'mcp_bounds' | 'full_coupling'
        Returns: (B, 20, 3) projected unit bone vectors
        """
        if stage == "raw":
            return raw_dirs

        B = raw_dirs.shape[0]
        p_dirs = raw_dirs.clone()

        # Palm Frame from Metacarpals (Middle MCP as longitudinal axis, Index/Pinky span lateral)
        u_mid_meta = p_dirs[:, 8]   # Wrist -> Middle MCP
        u_idx_meta = p_dirs[:, 4]   # Wrist -> Index MCP
        u_pky_meta = p_dirs[:, 16]  # Wrist -> Pinky MCP

        z_palm = u_mid_meta
        trans_palm = F.normalize(u_pky_meta - u_idx_meta, dim=-1)
        y_palm = F.normalize(torch.cross(z_palm, trans_palm, dim=-1), dim=-1) # Palm normal (dorsal/palmar)
        x_palm = F.normalize(torch.cross(y_palm, z_palm, dim=-1), dim=-1)     # Radial/Ulnar lateral

        finger_roots = [4, 8, 12, 16]  # Index, Middle, Ring, Pinky base offsets

        for f_idx, base in enumerate(finger_roots):
            u_meta = p_dirs[:, base]
            u_mcp  = p_dirs[:, base + 1]
            u_pip  = p_dirs[:, base + 2]
            u_dip  = p_dirs[:, base + 3]

            # -------------------------------------------------------------
            # Stage 2+: MCP Abduction & Flexion Bounds
            # -------------------------------------------------------------
            if stage in ["mcp_bounds", "full_coupling"]:
                # Lateral abduction axis: cross(palm_normal, metacarpal_dir)
                flex_axis = F.normalize(torch.cross(y_palm, u_meta, dim=-1), dim=-1)
                abd_axis  = F.normalize(torch.cross(flex_axis, u_meta, dim=-1), dim=-1)

                dot_long = torch.clamp((u_mcp * u_meta).sum(dim=-1, keepdim=True), -1.0, 1.0)
                dot_abd  = (u_mcp * flex_axis).sum(dim=-1, keepdim=True)
                dot_flex = (u_mcp * abd_axis).sum(dim=-1, keepdim=True)

                theta_mcp_flex = torch.atan2(dot_flex, dot_long) # [-10 deg .. 90 deg]
                theta_mcp_abd  = torch.asin(torch.clamp(dot_abd, -0.99, 0.99)) # [-15 deg .. 15 deg]

                # Clamp MCP bounds
                theta_mcp_flex = torch.clamp(theta_mcp_flex, -10.0 * (np.pi / 180.0), 90.0 * (np.pi / 180.0))
                theta_mcp_abd  = torch.clamp(theta_mcp_abd,  -15.0 * (np.pi / 180.0), 15.0 * (np.pi / 180.0))

                # Reconstruct bounded MCP direction
                u_mcp_clamped = (
                    torch.cos(theta_mcp_flex) * torch.cos(theta_mcp_abd) * u_meta +
                    torch.sin(theta_mcp_abd) * flex_axis +
                    torch.sin(theta_mcp_flex) * torch.cos(theta_mcp_abd) * abd_axis
                )
                u_mcp = F.normalize(u_mcp_clamped, dim=-1)
                p_dirs[:, base + 1] = u_mcp

            # -------------------------------------------------------------
            # Stage 1+: Planar Hinge on PIP & DIP (No Hyperextension)
            # -------------------------------------------------------------
            # True anatomical hinge axis of the finger: normal to MCP-PIP plane
            hinge_norm = F.normalize(torch.cross(u_meta, u_mcp, dim=-1), dim=-1)
            # Degeneracy guard: fallback to palm lateral if collinear
            collinear = (torch.norm(torch.cross(u_meta, u_mcp, dim=-1), dim=-1, keepdim=True) < 1e-3)
            hinge_norm = torch.where(collinear, x_palm, hinge_norm)

            # Project PIP into flexion plane perpendicular to hinge normal
            u_pip_planar = u_pip - (u_pip * hinge_norm).sum(dim=-1, keepdim=True) * hinge_norm
            u_pip_planar = F.normalize(u_pip_planar, dim=-1)

            # Compute PIP Flexion Angle
            dot_pip = torch.clamp((u_pip_planar * u_mcp).sum(dim=-1, keepdim=True), -1.0, 1.0)
            cross_pip = (torch.cross(u_mcp, u_pip_planar, dim=-1) * hinge_norm).sum(dim=-1, keepdim=True)
            theta_pip = torch.atan2(cross_pip, dot_pip)

            # Hinge limit: PIP flexion in [0 deg, 105 deg] (no hyperextension)
            theta_pip_clamped = torch.clamp(theta_pip, 0.0, 105.0 * (np.pi / 180.0))

            # Reconstruct PIP vector
            perp_pip = F.normalize(torch.cross(hinge_norm, u_mcp, dim=-1), dim=-1)
            u_pip_proj = torch.cos(theta_pip_clamped) * u_mcp + torch.sin(theta_pip_clamped) * perp_pip
            u_pip_final = F.normalize(u_pip_proj, dim=-1)
            p_dirs[:, base + 2] = u_pip_final

            # Project DIP into same planar flexion hinge
            u_dip_planar = u_dip - (u_dip * hinge_norm).sum(dim=-1, keepdim=True) * hinge_norm
            u_dip_planar = F.normalize(u_dip_planar, dim=-1)

            dot_dip = torch.clamp((u_dip_planar * u_pip_final).sum(dim=-1, keepdim=True), -1.0, 1.0)
            cross_dip = (torch.cross(u_pip_final, u_dip_planar, dim=-1) * hinge_norm).sum(dim=-1, keepdim=True)
            theta_dip = torch.atan2(cross_dip, dot_dip)

            # -------------------------------------------------------------
            # Stage 3+: Anatomical DIP-PIP Tendon Coupling (Landsmeer)
            # -------------------------------------------------------------
            if stage == "full_coupling":
                # Extensor coupling: theta_DIP in [0.5 * theta_PIP - 10 deg, 0.85 * theta_PIP + 10 deg]
                min_dip = torch.clamp(0.50 * theta_pip_clamped - 10.0 * (np.pi / 180.0), min=0.0)
                max_dip = torch.clamp(0.85 * theta_pip_clamped + 10.0 * (np.pi / 180.0), max=85.0 * (np.pi / 180.0))
                theta_dip_clamped = torch.clamp(theta_dip, min_dip, max_dip)
            else:
                # Stage 1 & 2 Hinge limit: DIP flexion in [0 deg, 85 deg]
                theta_dip_clamped = torch.clamp(theta_dip, 0.0, 85.0 * (np.pi / 180.0))

            perp_dip = F.normalize(torch.cross(hinge_norm, u_pip_final, dim=-1), dim=-1)
            u_dip_proj = torch.cos(theta_dip_clamped) * u_pip_final + torch.sin(theta_dip_clamped) * perp_dip
            p_dirs[:, base + 3] = F.normalize(u_dip_proj, dim=-1)

        # Thumb Articulation: Hinge on IP, bounded CMC/MCP
        if stage in ["hinge", "mcp_bounds", "full_coupling"]:
            u_th_mcp = p_dirs[:, 1]
            u_th_ip  = p_dirs[:, 2]
            u_th_tip = p_dirs[:, 3]

            th_hinge = F.normalize(torch.cross(u_th_mcp, u_th_ip, dim=-1), dim=-1)
            u_tip_planar = u_th_tip - (u_th_tip * th_hinge).sum(dim=-1, keepdim=True) * th_hinge
            u_tip_planar = F.normalize(u_tip_planar, dim=-1)

            dot_th = torch.clamp((u_tip_planar * u_th_ip).sum(dim=-1, keepdim=True), -1.0, 1.0)
            cross_th = (torch.cross(u_th_ip, u_tip_planar, dim=-1) * th_hinge).sum(dim=-1, keepdim=True)
            theta_th = torch.atan2(cross_th, dot_th)
            theta_th_clamped = torch.clamp(theta_th, 0.0, 80.0 * (np.pi / 180.0)) # Non-negative IP flexion

            perp_th = F.normalize(torch.cross(th_hinge, u_th_ip, dim=-1), dim=-1)
            p_dirs[:, 3] = F.normalize(torch.cos(theta_th_clamped) * u_th_ip + torch.sin(theta_th_clamped) * perp_th, dim=-1)

        return p_dirs

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
STAGES = [
    ("Stage 0: Raw Static LCF",          "raw"),
    ("Stage 1: Projection A (+ Hinge)",  "hinge"),
    ("Stage 2: Projection B (+ MCP)",    "mcp_bounds"),
    ("Stage 3: Projection C (+ Coupling)", "full_coupling")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in STAGES}
results_distal_deg = {name: [] for name, _ in STAGES}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in STAGES}

projector = KinematicManifoldProjector(L_CANON)

print("\n" + "=" * 105)
print(" BENCHMARKING KINEMATIC MANIFOLD PROJECTION (02_STATE) ACROSS 3 HELD-OUT SPLITS")
print("=" * 105)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
    val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

    model = LatentFactorizationModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # Train static LCF backbone
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)

            x_batch = gpu_extractor(raw_rgb)
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss, _, _, _ = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    # Cache raw model predictions on validation split
    model.eval()
    cached_raw_dirs = []
    cached_gt_joints = []

    with torch.no_grad():
        for i in range(0, len(val_indices), BATCH_SIZE):
            b_idx = val_indices[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            x_batch = gpu_extractor(raw_rgb)
            _, b_dirs = model(x_batch)
            cached_raw_dirs.append(b_dirs)
            cached_gt_joints.append(vram_targets[b_idx].view(-1, 21, 3))

    all_raw_dirs = torch.cat(cached_raw_dirs, dim=0)
    all_gt_joints = torch.cat(cached_gt_joints, dim=0)

    # Evaluate each projection stage
    for stage_name, stage_key in STAGES:
        proj_dirs = projector.project_batch(all_raw_dirs, stage=stage_key)
        proj_vecs = proj_dirs * L_CANON.view(1, 20, 1)
        proj_joints = reconstruct_joints_from_bones(proj_vecs)

        # 1. MPJPE (mm)
        mpjpe = torch.norm(proj_joints - all_gt_joints, dim=-1).mean().item()

        # 2. Per-Bone Angular Error (degrees)
        b_pred = proj_joints[:, CHILD_INDICES] - proj_joints[:, PARENT_INDICES]
        b_gt   = all_gt_joints[:, CHILD_INDICES] - all_gt_joints[:, PARENT_INDICES]

        u_p = b_pred / (torch.norm(b_pred, dim=-1, keepdim=True) + 1e-7)
        u_g = b_gt / (torch.norm(b_gt, dim=-1, keepdim=True) + 1e-7)

        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        deg_bones = torch.acos(cos_sim) * (180.0 / np.pi)

        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean().item()

        results_mpjpe[stage_name].append(mpjpe)
        results_distal_deg[stage_name].append(distal_deg)
        per_bone_deg_matrix[stage_name] += deg_bones.mean(dim=0).cpu().numpy()

        print(f"  [Seed {seed:<3}] {stage_name:<34} | MPJPE: {mpjpe:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

    del model, optimizer
    torch.cuda.empty_cache()

for name, _ in STAGES:
    per_bone_deg_matrix[name] /= len(SEEDS)

# -------------------------------------------------------------------------
# 7. COMPARATIVE AUDIT REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" KINEMATIC MANIFOLD PROJECTION BENCHMARK: 3-SPLIT SUMMARY")
print("=" * 90)
print(f"{'Projection Stage':<38} | {'MPJPE (mm)':<18} | {'Distal Dir (deg)':<18}")
print("-" * 90)
for stage_name, _ in STAGES:
    m_p, s_p = np.mean(results_mpjpe[stage_name]), np.std(results_mpjpe[stage_name])
    m_d, s_d = np.mean(results_distal_deg[stage_name]), np.std(results_distal_deg[stage_name])
    print(f"{stage_name:<38} | {m_p:5.2f} ± {s_p:4.2f} mm    | {m_d:4.1f}° ± {s_d:3.1f}°")
print("=" * 90)

print("\n" + "=" * 115)
print(" PER-BONE DIRECTION ERROR: MANIFOLD ABLATION BREAKDOWN (DEGREES)")
print("=" * 115)
header = f"{'Bone Identifier':<24} | {'Raw Static':<12} | {'+ Hinge':<12} | {'+ MCP':<12} | {'+ Coupling':<12} | {'Net Delta':<10}"
print(header)
print("-" * 115)

d_raw = per_bone_deg_matrix["Stage 0: Raw Static LCF"]
d_hng = per_bone_deg_matrix["Stage 1: Projection A (+ Hinge)"]
d_mcp = per_bone_deg_matrix["Stage 2: Projection B (+ MCP)"]
d_cpl = per_bone_deg_matrix["Stage 3: Projection C (+ Coupling)"]

for k in range(20):
    delta = d_cpl[k] - d_raw[k]
    sign = "+" if delta > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {d_raw[k]:5.2f}°      | {d_hng[k]:5.2f}°      | {d_mcp[k]:5.2f}°      | {d_cpl[k]:5.2f}°      | {sign}{delta:5.2f}°")
print("=" * 115)

Hardware Accelerator : Tesla T4
Building file map across filesystem...
Indexed 20,467 candidate files.
Reusing 5,114 frames already resident in VRAM.

 BENCHMARKING KINEMATIC MANIFOLD PROJECTION (02_STATE) ACROSS 3 HELD-OUT SPLITS
  [Seed 42 ] Stage 0: Raw Static LCF            | MPJPE: 32.89 mm | Distal Dir: 39.5°
  [Seed 42 ] Stage 1: Projection A (+ Hinge)    | MPJPE: 33.15 mm | Distal Dir: 41.0°
  [Seed 42 ] Stage 2: Projection B (+ MCP)      | MPJPE: 34.28 mm | Distal Dir: 42.1°
  [Seed 42 ] Stage 3: Projection C (+ Coupling) | MPJPE: 34.25 mm | Distal Dir: 41.6°
  [Seed 101] Stage 0: Raw Static LCF            | MPJPE: 33.66 mm | Distal Dir: 52.2°
  [Seed 101] Stage 1: Projection A (+ Hinge)    | MPJPE: 33.88 mm | Distal Dir: 54.1°
  [Seed 101] Stage 2: Projection B (+ MCP)      | MPJPE: 33.83 mm | Distal Dir: 54.7°
  [Seed 101] Stage 3: Projection C (+ Coupling) | MPJPE: 33.81 mm | Distal Dir: 54.4°
  [Seed 777] Stage 0: Raw Static LCF            | MPJPE: 28.08 mm | Distal Dir: 2

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Parent-Child Bone Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 2. BULLETPROOF FILE MAP & VRAM INGESTION
# -------------------------------------------------------------------------
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

print("Building file map across filesystem...")
FILE_MAP = build_file_map(DATA_DIR)
print(f"Indexed {len(FILE_MAP):,} candidate files.")

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p
    p1 = base_dir / rel_clean
    if p1.exists(): return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5
    if fname in FILE_MAP:
        return FILE_MAP[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None: continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 3. CANONICAL BONE LENGTHS & GPU SOBEL ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 4. LCF PERCEPTION BACKBONE
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=1)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 5. DIFFERENTIABLE SOFT ANATOMICAL REGULARIZER
# -------------------------------------------------------------------------
def compute_anatomy_loss(p_dirs: torch.Tensor) -> torch.Tensor:
    """
    Computes smooth soft penalties on anatomical invalidity:
    - Planar hinge deviation on PIP & DIP
    - Hyperextension violations on PIP, DIP, & Thumb IP (with natural tolerance)
    - Excessive MCP knuckle abduction
    - Landsmeer DIP-PIP extensor coupling deviations
    """
    # Palm frame derived from predicted metacarpals
    u_mid_meta = p_dirs[:, 8]
    u_idx_meta = p_dirs[:, 4]
    u_pky_meta = p_dirs[:, 16]

    z_palm = u_mid_meta
    trans_palm = F.normalize(u_pky_meta - u_idx_meta, dim=-1, eps=1e-7)
    y_palm = F.normalize(torch.cross(z_palm, trans_palm, dim=-1), dim=-1) # Palmar normal

    finger_roots = [4, 8, 12, 16] # Index, Middle, Ring, Pinky
    loss_planar = 0.0
    loss_hyperext = 0.0
    loss_coupling = 0.0
    loss_mcp = 0.0

    # Margins and bounds (radians)
    hyperext_margin = 5.0 * (np.pi / 180.0) # 5 deg natural play allowed
    max_pip = 105.0 * (np.pi / 180.0)
    max_dip = 85.0 * (np.pi / 180.0)
    max_mcp_abd = np.sin(18.0 * (np.pi / 180.0))

    for base in finger_roots:
        u_meta = p_dirs[:, base]
        u_mcp  = p_dirs[:, base + 1]
        u_pip  = p_dirs[:, base + 2]
        u_dip  = p_dirs[:, base + 3]

        # Lateral hinge axis running across the finger knuckle
        x_finger = F.normalize(torch.cross(u_meta, y_palm, dim=-1), dim=-1)
        v_flex_mcp = F.normalize(torch.cross(x_finger, u_mcp, dim=-1), dim=-1)
        v_flex_pip = F.normalize(torch.cross(x_finger, u_pip, dim=-1), dim=-1)

        # 1. Hinge Planarity: Lateral deviation out of the flexion plane
        lat_pip = (u_pip * x_finger).sum(dim=-1)
        lat_dip = (u_dip * x_finger).sum(dim=-1)
        loss_planar += (lat_pip.pow(2) + lat_dip.pow(2)).mean()

        # 2. MCP Abduction Bound
        abd_mcp = (u_mcp * x_finger).sum(dim=-1).abs()
        loss_mcp += F.relu(abd_mcp - max_mcp_abd).pow(2).mean()

        # 3. Flexion Angle & Hyperextension Penalties
        sin_pip = (u_pip * v_flex_mcp).sum(dim=-1)
        cos_pip = (u_pip * u_mcp).sum(dim=-1)
        theta_pip = torch.atan2(sin_pip, cos_pip + 1e-7)

        sin_dip = (u_dip * v_flex_pip).sum(dim=-1)
        cos_dip = (u_dip * u_pip).sum(dim=-1)
        theta_dip = torch.atan2(sin_dip, cos_dip + 1e-7)

        loss_hyperext += F.relu(-theta_pip - hyperext_margin).pow(2).mean()
        loss_hyperext += F.relu(theta_pip - max_pip).pow(2).mean()
        loss_hyperext += F.relu(-theta_dip - hyperext_margin).pow(2).mean()
        loss_hyperext += F.relu(theta_dip - max_dip).pow(2).mean()

        # 4. Tendon Extensor Coupling (Landsmeer)
        min_dip = 0.50 * F.relu(theta_pip) - 0.20
        max_dip_cpl = 0.85 * F.relu(theta_pip) + 0.25
        loss_coupling += F.relu(min_dip - theta_dip).pow(2).mean()
        loss_coupling += F.relu(theta_dip - max_dip_cpl).pow(2).mean()

    # Thumb IP hyperextension penalty
    u_th_mcp = p_dirs[:, 1]
    u_th_ip  = p_dirs[:, 2]
    u_th_tip = p_dirs[:, 3]
    th_hinge = F.normalize(torch.cross(u_th_mcp, u_th_ip, dim=-1), dim=-1)
    v_th_flex = F.normalize(torch.cross(th_hinge, u_th_ip, dim=-1), dim=-1)
    sin_th = (u_th_tip * v_th_flex).sum(dim=-1)
    cos_th = (u_th_tip * u_th_ip).sum(dim=-1)
    theta_th = torch.atan2(sin_th, cos_th + 1e-7)
    loss_hyperext += F.relu(-theta_th - hyperext_margin).pow(2).mean()

    total_anat = (loss_planar + loss_hyperext + 0.5 * loss_mcp + 0.5 * loss_coupling) / (len(finger_roots) + 1)
    return total_anat

# -------------------------------------------------------------------------
# 6. LOSS EVALUATOR
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        base_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return base_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 7. 3-WAY BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
CONFIGURATIONS = [
    ("LCF Baseline (λ_anat = 0.0)",        0.0),
    ("LCF + Weak Anatomy (λ_anat = 5.0)",   5.0),
    ("LCF + Moderate Anatomy (λ_anat = 20.0)", 20.0)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in CONFIGURATIONS}
results_distal_deg = {name: [] for name, _ in CONFIGURATIONS}
results_anat_score = {name: [] for name, _ in CONFIGURATIONS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in CONFIGURATIONS}

print("\n" + "=" * 105)
print(" BENCHMARKING SOFT ANATOMICAL LATENT REGULARIZATION (3 SEEDS, SEGMENT-DISJOINT)")
print("=" * 105)

for exp_name, lambda_anat in CONFIGURATIONS:
    print(f"\n>>> Running Configuration: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = LatentFactorizationModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)
                optimizer.zero_grad()
                pred_joints, pred_dirs = model(x_batch)

                base_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                if lambda_anat > 0.0:
                    l_anat = compute_anatomy_loss(pred_dirs)
                    total_loss = base_loss + lambda_anat * l_anat
                else:
                    total_loss = base_loss

                total_loss.backward()
                optimizer.step()

        # Validation evaluation
        model.eval()
        cached_preds = []
        cached_targets = []
        val_anat_scores = []

        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)
                pred_joints, pred_dirs = model(x_batch)

                cached_preds.append(pred_joints)
                cached_targets.append(y_batch)
                val_anat_scores.append(compute_anatomy_loss(pred_dirs).item())

        all_pred = torch.cat(cached_preds, dim=0)
        all_gt = torch.cat(cached_targets, dim=0)

        # 1. MPJPE (mm)
        val_mpjpe = torch.norm(all_pred - all_gt, dim=-1).mean().item()

        # 2. Per-Bone Direction Error (degrees)
        b_pred = all_pred[:, CHILD_INDICES] - all_pred[:, PARENT_INDICES]
        b_gt   = all_gt[:, CHILD_INDICES] - all_gt[:, PARENT_INDICES]

        u_pred = b_pred / (torch.norm(b_pred, dim=-1, keepdim=True) + 1e-7)
        u_gt   = b_gt / (torch.norm(b_gt, dim=-1, keepdim=True) + 1e-7)

        cos_sim = torch.clamp((u_pred * u_gt).sum(dim=-1), -1.0, 1.0)
        deg_per_bone = torch.acos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_per_bone[:, DISTAL_BONE_INDICES].mean().item()
        mean_anat_score = np.mean(val_anat_scores)

        results_mpjpe[exp_name].append(val_mpjpe)
        results_distal_deg[exp_name].append(distal_deg)
        results_anat_score[exp_name].append(mean_anat_score)
        accum_bone_deg += deg_per_bone.mean(dim=0).cpu().numpy()

        print(f"  [Seed {seed:<3}] MPJPE: {val_mpjpe:5.2f} mm | Distal Dir: {distal_deg:4.1f}° | Anat Violation Score: {mean_anat_score:6.4f}")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 8. COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" SOFT ANATOMICAL REGULARIZATION: 3-SPLIT SEGMENT-DISJOINT SUMMARY")
print("=" * 105)
print(f"{'Configuration':<38} | {'MPJPE (mm)':<18} | {'Distal Dir (deg)':<18} | {'Anat Score':<12}")
print("-" * 105)
for exp_name, _ in CONFIGURATIONS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_distal_deg[exp_name]), np.std(results_distal_deg[exp_name])
    m_a, s_a = np.mean(results_anat_score[exp_name]), np.std(results_anat_score[exp_name])
    print(f"{exp_name:<38} | {m_p:5.2f} ± {s_p:4.2f} mm    | {m_d:4.1f}° ± {s_d:3.1f}°     | {m_a:6.4f}")
print("=" * 105)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: SOFT REGULARIZATION BREAKDOWN (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'LCF Base (λ=0)':<16} | {'Weak (λ=5)':<16} | {'Mod (λ=20)':<16} | {'Delta (Weak-Base)':<18}"
print(header)
print("-" * 105)

d_base = per_bone_deg_matrix["LCF Baseline (λ_anat = 0.0)"]
d_weak = per_bone_deg_matrix["LCF + Weak Anatomy (λ_anat = 5.0)"]
d_mod  = per_bone_deg_matrix["LCF + Moderate Anatomy (λ_anat = 20.0)"]

for k in range(20):
    diff = d_weak[k] - d_base[k]
    sign = "+" if diff > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {d_base[k]:5.2f}°          | {d_weak[k]:5.2f}°          | {d_mod[k]:5.2f}°          | {sign}{diff:5.2f}°")
print("=" * 105)

Hardware Accelerator : Tesla T4
Building file map across filesystem...
Indexed 20,467 candidate files.
Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...
Staged 5,114 frames into VRAM (720.4 MB used).

 BENCHMARKING SOFT ANATOMICAL LATENT REGULARIZATION (3 SEEDS, SEGMENT-DISJOINT)

>>> Running Configuration: LCF Baseline (λ_anat = 0.0) <<<
  [Seed 42 ] MPJPE: 32.93 mm | Distal Dir: 39.7° | Anat Violation Score: 0.0779
  [Seed 101] MPJPE: 33.11 mm | Distal Dir: 51.9° | Anat Violation Score: 0.0862
  [Seed 777] MPJPE: 27.93 mm | Distal Dir: 30.1° | Anat Violation Score: 0.0701

>>> Running Configuration: LCF + Weak Anatomy (λ_anat = 5.0) <<<
  [Seed 42 ] MPJPE: 34.90 mm | Distal Dir: 42.3° | Anat Violation Score: 0.0338
  [Seed 101] MPJPE: 32.68 mm | Distal Dir: 52.9° | Anat Violation Score: 0.0330
  [Seed 777] MPJPE: 29.64 mm | Distal Dir: 31.8° | Anat Violation Score: 0.0225

>>> Running Configuration: LCF + Moderate Anatomy (λ_anat = 20.0) <<<
  [Seed 42 ] MPJPE: 35.99 mm | Di

In [ ]:
# @title
import csv
import re
from pathlib import Path
from collections import defaultdict
import numpy as np

# Auto-detect Colab vs. Local hm_ex environment
CANDIDATE_DIRS = [
    Path("/content/dataset"),
    Path("/content"),
    Path("datasets/reinterhand/captures/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands"),
    Path("datasets/reinterhand/captures/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands/derived/mugsy_subset"),
]

def find_file(filename: str):
    for d in CANDIDATE_DIRS:
        if d.exists():
            p = d / filename
            if p.exists():
                return p
            matches = list(d.rglob(filename))
            if matches:
                return matches[0]
    return None

def extract_fid(s: str) -> int:
    nums = re.findall(r"\d+", str(s))
    return int(nums[-1])

manifest_p = find_file("manifest.csv")
flist_p = find_file("frame_list.txt")
flist_orig_p = find_file("frame_list_orig.txt")

print("=" * 95)
print(" EXPERIMENT A: TIMESTAMP, FRAME-RATE & CONTIGUITY AUDIT")
print("=" * 95)

# 1. Inspect Native Capture Lists (if available locally)
for label, path in [("Raw Capture (frame_list_orig.txt)", flist_orig_p),
                    ("Curated Segments (frame_list.txt)", flist_p)]:
    if path is not None:
        seg_frames = defaultdict(list)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    seg_frames[parts[0]].append(extract_fid(parts[1]))
                elif len(parts) == 1 and parts[0]:
                    seg_frames["__all__"].append(extract_fid(parts[0]))
        deltas = []
        for seg, fids in seg_frames.items():
            fids = sorted(set(fids))
            if len(fids) > 1:
                deltas.extend(np.diff(fids))
        deltas = np.array(deltas)
        unique_d, counts_d = np.unique(deltas, return_counts=True)
        top_strides = ", ".join([f"Δf={d}: {c:,} ({100*c/len(deltas):.1f}%)"
                                 for d, c in zip(unique_d[:5], counts_d[:5])])
        print(f"\n[{label}] ({path})")
        print(f"  Total Segments: {len(seg_frames):,} | Total Transitions: {len(deltas):,}")
        print(f"  Min Δf: {deltas.min()} | Median Δf: {np.median(deltas):.1f} | Mean Δf: {deltas.mean():.2f} | Max Δf: {deltas.max()}")
        print(f"  Dominant Native Strides: {top_strides}")

# 2. Inspect Extracted 5,114-Frame Subset (manifest.csv)
if manifest_p is None:
    raise FileNotFoundError("Could not locate manifest.csv in Colab or local directory.")

seg_to_fids = defaultdict(list)
with open(manifest_p, "r", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        fid = int(r["frame_id"]) if "frame_id" in r else extract_fid(r["image_path"])
        seg_to_fids[r["segment"]].append(fid)

sub_deltas = []
for seg, fids in seg_to_fids.items():
    fids.sort()
    if len(fids) > 1:
        sub_deltas.extend(np.diff(fids))
sub_deltas = np.array(sub_deltas)

print(f"\n[Extracted Visual Subset (manifest.csv)] ({manifest_p})")
print(f"  Total Frames: {sum(len(v) for v in seg_to_fids.values()):,} | Segments: {len(seg_to_fids)} | Transitions: {len(sub_deltas):,}")
print(f"  Min Δf: {sub_deltas.min()} | 25%: {np.percentile(sub_deltas, 25):.0f} | Median: {np.median(sub_deltas):.0f} | 75%: {np.percentile(sub_deltas, 75):.0f} | 95%: {np.percentile(sub_deltas, 95):.0f} | Max: {sub_deltas.max()}")

print("\n  Stride Distribution in manifest.csv:")
for max_df in [1, 2, 3, 6, 9, 15, 30]:
    cnt = int(np.sum(sub_deltas <= max_df))
    print(f"    Transitions with Δf <= {max_df:<2d} : {cnt:4,d} / {len(sub_deltas):,d} ({100.0 * cnt / len(sub_deltas):5.1f}%)")

# 3. Contiguous Run-Length Analysis for Sequence Models (Experiment C)
print("\n  Contiguous Sequence Runs Available (at native stride thresholds):")
for max_gap in [3, 6]:
    run_lengths = []
    for seg, fids in seg_to_fids.items():
        fids = sorted(set(fids))
        cur_len = 1
        for i in range(1, len(fids)):
            if (fids[i] - fids[i - 1]) <= max_gap:
                cur_len += 1
            else:
                run_lengths.append(cur_len)
                cur_len = 1
        run_lengths.append(cur_len)
    runs = np.array(run_lengths)
    w3 = int(np.sum(np.maximum(0, runs - 2)))
    w5 = int(np.sum(np.maximum(0, runs - 4)))
    w8 = int(np.sum(np.maximum(0, runs - 7)))
    print(f"    Max Step Δf <= {max_gap}: Max Run = {runs.max():2d} frames | "
          f"Valid 3-Frame Windows: {w3:4,d} | 5-Frame Windows: {w5:4,d} | 8-Frame Windows: {w8:4,d}")
print("=" * 95)

 EXPERIMENT A: TIMESTAMP, FRAME-RATE & CONTIGUITY AUDIT

[Extracted Visual Subset (manifest.csv)] (/content/dataset/manifest.csv)
  Total Frames: 5,114 | Segments: 56 | Transitions: 5,058
  Min Δf: 2 | 25%: 3 | Median: 9 | 75%: 15 | 95%: 33 | Max: 441

  Stride Distribution in manifest.csv:
    Transitions with Δf <= 1  :    0 / 5,058 (  0.0%)
    Transitions with Δf <= 2  :    2 / 5,058 (  0.0%)
    Transitions with Δf <= 3  : 1,327 / 5,058 ( 26.2%)
    Transitions with Δf <= 6  : 2,269 / 5,058 ( 44.9%)
    Transitions with Δf <= 9  : 3,011 / 5,058 ( 59.5%)
    Transitions with Δf <= 15 : 3,924 / 5,058 ( 77.6%)
    Transitions with Δf <= 30 : 4,772 / 5,058 ( 94.3%)

  Contiguous Sequence Runs Available (at native stride thresholds):
    Max Step Δf <= 3: Max Run =  7 frames | Valid 3-Frame Windows:  350 | 5-Frame Windows:   25 | 8-Frame Windows:    0
    Max Step Δf <= 6: Max Run = 10 frames | Valid 3-Frame Windows:  983 | 5-Frame Windows:  173 | 8-Frame Windows:    9


In [ ]:
# @title
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "Enable GPU in Runtime -> Change runtime type -> T4 GPU"
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

# =============================================================================
# 1. DIFFERENTIABLE / Z-BUFFERED PERSPECTIVE SILHOUETTE & SOBEL RENDERER
# =============================================================================
IMG_H, IMG_W = 128, 128
FOCAL = 380.0
CX, CY = IMG_W / 2.0, IMG_H / 2.0
Z_ROOT = 420.0  # mm from camera pinhole

class GPUSobel(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, img: torch.Tensor) -> torch.Tensor:
        # img: (B, T, H, W) -> returns (B, T, H, W) scalar Sobel magnitude |G|
        B, T, H, W = img.shape
        x = img.view(B * T, 1, H, W)
        gx = F.conv2d(x, self.kx, padding=1) / 4.0
        gy = F.conv2d(x, self.ky, padding=1) / 4.0
        mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)
        return mag.view(B, T, H, W)

sobel_engine = GPUSobel(DEVICE)

# Precompute pixel grid (1, 1, H, W)
yy, xx = torch.meshgrid(
    torch.arange(IMG_H, dtype=torch.float32, device=DEVICE),
    torch.arange(IMG_W, dtype=torch.float32, device=DEVICE),
    indexing="ij"
)
GRID_X = xx.view(1, 1, IMG_H, IMG_W)
GRID_Y = yy.view(1, 1, IMG_H, IMG_W)

@torch.no_grad()
def render_zbuffered_capsules(seg_p0: torch.Tensor, seg_p1: torch.Tensor,
                              radii_mm: torch.Tensor, intensities: torch.Tensor) -> torch.Tensor:
    """
    Renders 3D capsules with perspective projection and Z-buffer occlusion.
    seg_p0, seg_p1: (N, T, S, 3) in camera coordinates (mm, +Z away from camera)
    radii_mm: (S,) physical capsule radii in mm
    intensities: (S,) surface luminance per segment (allows internal occlusion boundaries)
    Returns: (N, T, H, W) rendered grayscale image in [0, 255]
    """
    N, T, S, _ = seg_p0.shape
    canvas = torch.zeros((N, T, IMG_H, IMG_W), dtype=torch.float32, device=DEVICE)
    z_buffer = torch.full((N, T, IMG_H, IMG_W), 1e6, dtype=torch.float32, device=DEVICE)

    for s in range(S):
        p0 = seg_p0[:, :, s, :]  # (N, T, 3)
        p1 = seg_p1[:, :, s, :]  # (N, T, 3)

        z0 = torch.clamp(p0[..., 2:3], min=50.0)
        z1 = torch.clamp(p1[..., 2:3], min=50.0)

        u0 = FOCAL * (p0[..., 0:1] / z0) + CX
        v0 = FOCAL * (p0[..., 1:2] / z0) + CY
        u1 = FOCAL * (p1[..., 0:1] / z1) + CX
        v1 = FOCAL * (p1[..., 1:2] / z1) + CY

        u0 = u0.unsqueeze(-1)
        v0 = v0.unsqueeze(-1)
        u1 = u1.unsqueeze(-1)
        v1 = v1.unsqueeze(-1)
        z0_map = z0.unsqueeze(-1)
        z1_map = z1.unsqueeze(-1)

        du = u1 - u0
        dv = v1 - v0
        len_sq = du ** 2 + dv ** 2 + 1e-6

        t_param = torch.clamp(((GRID_X - u0) * du + (GRID_Y - v0) * dv) / len_sq, 0.0, 1.0)
        proj_u = u0 + t_param * du
        proj_v = v0 + t_param * dv
        dist_px = torch.sqrt((GRID_X - proj_u) ** 2 + (GRID_Y - proj_v) ** 2 + 1e-6)

        z_seg = z0_map + t_param * (z1_map - z0_map)
        rad_px = FOCAL * radii_mm[s] / z_seg

        # Smooth anti-aliased silhouette edge (1.2 px ramp)
        alpha = torch.clamp((rad_px - dist_px) / 1.2 + 0.5, 0.0, 1.0)
        mask = alpha > 0.05

        # Z-buffer occlusion update: closer segments overwrite farther segments
        closer = mask & (z_seg < (z_buffer - 1.5))
        z_buffer = torch.where(closer, z_seg, z_buffer)
        canvas = torch.where(closer, alpha * intensities[s], canvas)

    return canvas

# =============================================================================
# 2. CONJUGATE NECKER-AMBIGUOUS FINGER GENERATOR (T = 5 CONSECUTIVE FRAMES)
# =============================================================================
L_META, L_PROX, L_MID, L_DIST = 48.0, 32.0, 22.0, 17.0
RADII = torch.tensor([8.5, 7.2, 6.2, 5.4, 7.0], device=DEVICE)       # 4 finger bones + 1 occluder digit
INTENSITIES = torch.tensor([190., 205., 220., 235., 165.], device=DEVICE)

@torch.no_grad()
def generate_conjugate_sequences(num_pairs: int, regime: str, seed: int = 42):
    """
    Generates N conjugate pairs (Branch 0: -Z toward camera, Branch 1: +Z away from camera)
    that project to matched 2D coordinates at t=0, and evolves them over T=5 frames.
    """
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    N = num_pairs
    T = 5

    base_x = (torch.rand((N, 1), generator=g, device=DEVICE) - 0.5) * 16.0
    base_y = (torch.rand((N, 1), generator=g, device=DEVICE) - 0.5) * 16.0 + 28.0
    in_plane_roll = (torch.rand((N, 1), generator=g, device=DEVICE) - 0.5) * 0.45  # rad

    # Ambiguous out-of-plane distal tilt magnitude at t=0: [24 deg .. 42 deg]
    theta0 = (24.0 + 18.0 * torch.rand((N, 1), generator=g, device=DEVICE)) * (np.pi / 180.0)

    # Create 2N samples: first N are Branch A (sign = -1, toward camera), next N are Branch B (sign = +1, away)
    signs = torch.cat([-torch.ones((N, 1), device=DEVICE), torch.ones((N, 1), device=DEVICE)], dim=0)
    base_x = base_x.repeat(2, 1)
    base_y = base_y.repeat(2, 1)
    in_plane_roll = in_plane_roll.repeat(2, 1)
    theta0 = theta0.repeat(2, 1)

    t_steps = torch.arange(T, dtype=torch.float32, device=DEVICE).view(1, T)  # [0, 1, 2, 3, 4]

    if regime == "regime1_static":
        theta_t = theta0 + 0.0 * t_steps
        wrist_yaw = torch.zeros((2 * N, T), device=DEVICE)
        occ_offset_x = torch.full((2 * N, T), 999.0, device=DEVICE)
    elif regime == "regime2_articulation":
        omega = (4.5 * np.pi / 180.0)
        theta_t = theta0 + omega * t_steps
        wrist_yaw = torch.zeros((2 * N, T), device=DEVICE)
        occ_offset_x = torch.full((2 * N, T), 999.0, device=DEVICE)
    elif regime == "regime3_occlusion":
        theta_t = theta0 + (1.5 * np.pi / 180.0) * t_steps
        wrist_yaw = torch.zeros((2 * N, T), device=DEVICE)
        occ_offset_x = -18.0 + 9.0 * t_steps.expand(2 * N, T)
    elif regime == "regime4_wrist_rot":
        theta_t = theta0 + (1.0 * np.pi / 180.0) * t_steps
        wrist_yaw = (3.0 * np.pi / 180.0) * t_steps.expand(2 * N, T)
        occ_offset_x = torch.full((2 * N, T), 999.0, device=DEVICE)
    else:
        raise ValueError(f"Unknown regime: {regime}")

    j0 = torch.stack([base_x.expand(-1, T), base_y.expand(-1, T), torch.zeros((2 * N, T), device=DEVICE)], dim=-1)

    dir_x = torch.sin(in_plane_roll).expand(-1, T)
    dir_y = -torch.cos(in_plane_roll).expand(-1, T)

    j1 = j0 + torch.stack([L_META * dir_x, L_META * dir_y, torch.zeros_like(dir_x)], dim=-1)

    ang_prox = signs * (0.35 * theta_t)
    ang_mid = signs * (0.85 * theta_t)
    ang_dist = signs * (1.35 * theta_t)

    j2_local = j1 + torch.stack([
        L_PROX * torch.cos(ang_prox) * dir_x,
        L_PROX * torch.cos(ang_prox) * dir_y,
        L_PROX * torch.sin(ang_prox)
    ], dim=-1)

    j3_local = j2_local + torch.stack([
        L_MID * torch.cos(ang_mid) * dir_x,
        L_MID * torch.cos(ang_mid) * dir_y,
        L_MID * torch.sin(ang_mid)
    ], dim=-1)

    j4_local = j3_local + torch.stack([
        L_DIST * torch.cos(ang_dist) * dir_x,
        L_DIST * torch.cos(ang_dist) * dir_y,
        L_DIST * torch.sin(ang_dist)
    ], dim=-1)

    # Match perspective projection at t=0 so frame 0 has matched 2D coordinates
    for j_tensor in [j2_local, j3_local, j4_local]:
        z_scale_t0 = (Z_ROOT + j_tensor[:, 0:1, 2]) / Z_ROOT
        j_tensor[:, :, 0] = j_tensor[:, :, 0] * z_scale_t0
        j_tensor[:, :, 1] = j_tensor[:, :, 1] * z_scale_t0

    joints = torch.stack([j0, j1, j2_local, j3_local, j4_local], dim=2)  # (2N, T, 5, 3)

    # Apply global wrist yaw rotation around J1 (MCP pivot) if active
    cy = torch.cos(wrist_yaw).unsqueeze(-1)
    sy = torch.sin(wrist_yaw).unsqueeze(-1)
    rel = joints - j1.unsqueeze(2)
    rot_x = rel[..., 0] * cy + rel[..., 2] * sy
    rot_y = rel[..., 1]
    rot_z = -rel[..., 0] * sy + rel[..., 2] * cy
    joints_cam = torch.stack([rot_x, rot_y, rot_z], dim=-1) + j1.unsqueeze(2)
    joints_cam[..., 2] += Z_ROOT

    seg_p0_finger = joints_cam[:, :, 0:4, :]
    seg_p1_finger = joints_cam[:, :, 1:5, :]

    tip_t0_xy = joints_cam[:, 0:1, 4, 0:2].expand(-1, T, -1)
    occ_p0 = torch.stack([tip_t0_xy[..., 0] + occ_offset_x - 12.0, tip_t0_xy[..., 1] - 6.0,
                          torch.full_like(occ_offset_x, Z_ROOT)], dim=-1).unsqueeze(2)
    occ_p1 = torch.stack([tip_t0_xy[..., 0] + occ_offset_x + 12.0, tip_t0_xy[..., 1] + 4.0,
                          torch.full_like(occ_offset_x, Z_ROOT)], dim=-1).unsqueeze(2)

    seg_p0 = torch.cat([seg_p0_finger, occ_p0], dim=2)
    seg_p1 = torch.cat([seg_p1_finger, occ_p1], dim=2)

    imgs = render_zbuffered_capsules(seg_p0, seg_p1, RADII, INTENSITIES)
    sobel_seq = sobel_engine(imgs)  # (2N, T, H, W)

    distal_vec = joints_cam[:, -1, 4, :] - joints_cam[:, -1, 3, :]
    distal_u = F.normalize(distal_vec, dim=-1)
    tip_rel_mm = joints_cam[:, -1, 4, :] - joints_cam[:, -1, 1, :]
    labels = (signs.squeeze(-1) > 0).long()

    return sobel_seq, distal_u, tip_rel_mm, labels

# =============================================================================
# 3. LIGHTWEIGHT TEMPORAL OBSERVATION ENCODER vs. SINGLE-FRAME CNN
# =============================================================================
class TemporalObservationEncoder(nn.Module):
    def __init__(self, use_temporal: bool = True):
        super().__init__()
        self.use_temporal = use_temporal
        self.spatial_cnn = nn.Sequential(
            nn.Conv2d(1, 24, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(24),
            nn.ReLU(inplace=True),
            nn.Conv2d(24, 48, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            nn.Conv2d(48, 96, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(96 * 16, 128),
            nn.ReLU(inplace=True),
        )
        if use_temporal:
            self.gru = nn.GRU(input_size=128, hidden_size=128, num_layers=1, batch_first=True)
        self.dir_head = nn.Linear(128, 3)
        self.tip_head = nn.Linear(128, 3)
        self.branch_head = nn.Linear(128, 2)

    def forward(self, seq_g: torch.Tensor):
        B, T, H, W = seq_g.shape
        if not self.use_temporal:
            feat = self.spatial_cnn(seq_g[:, -1:, :, :])
        else:
            feats = self.spatial_cnn(seq_g.reshape(B * T, 1, H, W)).view(B, T, 128)
            out, _ = self.gru(feats)
            feat = out[:, -1, :]

        u_pred = F.normalize(self.dir_head(feat), dim=-1, eps=1e-7)
        tip_pred = self.tip_head(feat)
        logits = self.branch_head(feat)
        return u_pred, tip_pred, logits

# =============================================================================
# 4. RUN EXPERIMENT B BENCHMARK ACROSS ALL 4 TEMPORAL REGIMES
# =============================================================================
REGIMES = [
    ("1. Static Ambiguous Persistence (No Motion)", "regime1_static"),
    ("2. Active Finger Articulation (Perspective)", "regime2_articulation"),
    ("3. Adjacent Finger Occlusion Sweep (Z-Order)", "regime3_occlusion"),
    ("4. Continuous Wrist Rotation (3 deg/frame)",  "regime4_wrist_rot"),
]

print("\n" + "=" * 108)
print(" EXPERIMENT B: SYNTHETIC AMBIGUOUS FINGER OBSERVABILITY BENCHMARK (CONJUGATE +/-Z BRANCHES)")
print("=" * 108)
print(f"{'Temporal Regime (T=5 Consecutive Frames)':<44} | {'t=0 Diff':<9} | {'Seq Diff':<9} | {'Model':<12} | {'Sign Acc':<9} | {'Distal Err':<11} | {'Tip MPJPE'}")
print("-" * 108)

for title, reg_key in REGIMES:
    X_tr, U_tr, Tip_tr, Y_tr = generate_conjugate_sequences(400, reg_key, seed=42)
    X_val, U_val, Tip_val, Y_val = generate_conjugate_sequences(150, reg_key, seed=999)

    N_val = 150
    branch_A = X_val[:N_val]
    branch_B = X_val[N_val:]
    diff_t0 = torch.norm(branch_A[:, 0] - branch_B[:, 0], dim=(-2, -1)).mean().item()
    diff_seq = torch.norm(branch_A - branch_B, dim=(-2, -1)).mean().item()

    for model_label, is_temp in [("1-Frame CNN", False), ("5-Frame GRU", True)]:
        torch.manual_seed(42)
        model = TemporalObservationEncoder(use_temporal=is_temp).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)

        BATCH = 64
        for epoch in range(20):
            model.train()
            perm = torch.randperm(len(X_tr), device=DEVICE)
            for i in range(0, len(X_tr), BATCH):
                idx = perm[i:i + BATCH]
                opt.zero_grad()
                u_p, tip_p, logits = model(X_tr[idx])
                loss_dir = (1.0 - (u_p * U_tr[idx]).sum(dim=-1)).mean()
                loss_tip = F.l1_loss(tip_p, Tip_tr[idx])
                loss_cls = F.cross_entropy(logits, Y_tr[idx])
                loss = loss_tip + 25.0 * loss_dir + 2.0 * loss_cls
                loss.backward()
                opt.step()

        model.eval()
        with torch.no_grad():
            u_p, tip_p, logits = model(X_val)
            sign_acc = (logits.argmax(dim=-1) == Y_val).float().mean().item() * 100.0
            cos_sim = torch.clamp((u_p * U_val).sum(dim=-1), -1.0, 1.0)
            deg_err = (torch.acos(cos_sim) * (180.0 / np.pi)).mean().item()
            tip_err = torch.norm(tip_p - Tip_val, dim=-1).mean().item()

        row_title = title if not is_temp else ""
        d0_str = f"{diff_t0:6.2f}" if not is_temp else ""
        ds_str = f"{diff_seq:6.2f}" if not is_temp else ""
        print(f"{row_title:<44} | {d0_str:<9} | {ds_str:<9} | {model_label:<12} | {sign_acc:6.1f}%   | {deg_err:6.2f} deg  | {tip_err:6.2f} mm")
    print("-" * 108)

Hardware Accelerator: Tesla T4

 EXPERIMENT B: SYNTHETIC AMBIGUOUS FINGER OBSERVABILITY BENCHMARK (CONJUGATE +/-Z BRANCHES)
Temporal Regime (T=5 Consecutive Frames)     | t=0 Diff  | Seq Diff  | Model        | Sign Acc  | Distal Err  | Tip MPJPE
------------------------------------------------------------------------------------------------------------
1. Static Ambiguous Persistence (No Motion)  | 1019.43   | 1019.43   | 1-Frame CNN  |   89.7%   |  18.29 deg  |  24.47 mm
                                             |           |           | 5-Frame GRU  |   88.3%   |  18.01 deg  |  32.87 mm
------------------------------------------------------------------------------------------------------------
2. Active Finger Articulation (Perspective)  | 1019.43   | 1274.74   | 1-Frame CNN  |   99.3%   |   8.18 deg  |  13.14 mm
                                             |           |           | 5-Frame GRU  |  100.0%   |   5.28 deg  |  33.06 mm
------------------------------------------------

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# =============================================================================
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]

# =============================================================================
# 2. BULLETPROOF FILE MAP & SELF-HEALING VRAM INGESTION
# =============================================================================
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists():
        search_dirs.append(base_dir.parent)
    if Path("/content").exists():
        search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists():
        return p
    p1 = base_dir / rel_clean
    if p1.exists():
        return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists():
        return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists():
            return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists():
            return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists():
            return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")
    if 'sample_fids' not in globals() or len(sample_fids) != len(vram_images):
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest_rows = list(csv.DictReader(f))
        sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                                for r in manifest_rows[:len(vram_images)]])

# =============================================================================
# 3. CHRONOLOGICAL CONTIGUITY INDEXING (df <= 6 GATED)
# =============================================================================
seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

N_SAMPLES = len(vram_images)
idx_t_minus_1 = np.zeros(N_SAMPLES, dtype=np.int64)
idx_t_minus_2 = np.zeros(N_SAMPLES, dtype=np.int64)
delta_f_1 = np.zeros(N_SAMPLES, dtype=np.float32)
delta_f_2 = np.zeros(N_SAMPLES, dtype=np.float32)

MAX_CONTIGUOUS_DF = 6  # Native 1-step (df=3) or 2-step (df=6)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            idx_t_minus_1[i] = i
            idx_t_minus_2[i] = i
            delta_f_1[i] = 0.0
            delta_f_2[i] = 0.0
        else:
            prev1 = indices[step_idx - 1]
            df1 = float(sample_fids[i] - sample_fids[prev1])
            if df1 <= MAX_CONTIGUOUS_DF:
                idx_t_minus_1[i] = prev1
                delta_f_1[i] = df1
                if step_idx >= 2:
                    prev2 = indices[step_idx - 2]
                    df2 = float(sample_fids[prev1] - sample_fids[prev2])
                    if df2 <= MAX_CONTIGUOUS_DF:
                        idx_t_minus_2[i] = prev2
                        delta_f_2[i] = df2
                    else:
                        idx_t_minus_2[i] = prev1
                        delta_f_2[i] = 0.0
                else:
                    idx_t_minus_2[i] = prev1
                    delta_f_2[i] = 0.0
            else:
                # Non-contiguous extraction jump: clamp history to self so stale frames never pollute
                idx_t_minus_1[i] = i
                idx_t_minus_2[i] = i
                delta_f_1[i] = 999.0
                delta_f_2[i] = 999.0

vram_prev1 = torch.tensor(idx_t_minus_1, dtype=torch.long, device=DEVICE)
vram_prev2 = torch.tensor(idx_t_minus_2, dtype=torch.long, device=DEVICE)

dt1_norm = np.where(delta_f_1 <= MAX_CONTIGUOUS_DF, delta_f_1 / 3.0, 0.0).astype(np.float32)
dt2_norm = np.where(delta_f_2 <= MAX_CONTIGUOUS_DF, delta_f_2 / 3.0, 0.0).astype(np.float32)
vram_dt1 = torch.tensor(dt1_norm, dtype=torch.float32, device=DEVICE)
vram_dt2 = torch.tensor(dt2_norm, dtype=torch.float32, device=DEVICE)

# =============================================================================
# 4. CANONICAL LENGTHS & GPU SOBEL ENGINE
# =============================================================================
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# =============================================================================
# 5. STATIC LCF BACKBONE (04_NETWORK) & TEMPORAL OBSERVATION ENCODER
# =============================================================================
class SpatialEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward_spatial(self, x: torch.Tensor):
        fmap = self.encoder(x)                         # (B, 256, 6, 8)
        vec = self.shared_fc(torch.flatten(fmap, 1))   # (B, 512)
        return fmap, vec

class LCFHead(nn.Module):
    def __init__(self, in_dim: int = 512):
        super().__init__()
        self.meta_head = nn.Linear(in_dim, 5 * 3)
        self.latent_frame_head = nn.Linear(in_dim, 6)
        self.phalanx_head = nn.Linear(in_dim, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, feat: torch.Tensor):
        B = feat.shape[0]
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=feat.dtype, device=feat.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

class StaticLCFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SpatialEncoder()
        self.lcf_head = LCFHead(512)

    def forward(self, g_t: torch.Tensor):
        _, vec_t = self.backbone.forward_spatial(g_t)
        return self.lcf_head(vec_t)

class TemporalCorrespondenceEncoder(nn.Module):
    """
    Takes frozen 04_network spatial feature maps & static LCF embedding across
    consecutive window (t-2, t-1, t) + physical dt conditioning, and predicts
    a residual latent refinement before decoding through LCF + L_CANON.
    """
    def __init__(self, frozen_static: StaticLCFModel):
        super().__init__()
        self.backbone = frozen_static.backbone
        self.lcf_head = frozen_static.lcf_head

        for p in self.backbone.parameters():
            p.requires_grad = False

        self.spatial_corr = nn.Sequential(
            nn.Conv2d(256 * 3, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(64 * 6 * 8, 256),
            nn.ReLU(inplace=True),
        )

        self.gru = nn.GRU(input_size=512 + 256 + 1, hidden_size=256, num_layers=1, batch_first=True)

        self.refine_fc = nn.Sequential(
            nn.Linear(512 + 256, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512)
        )
        nn.init.zeros_(self.refine_fc[-1].weight)
        nn.init.zeros_(self.refine_fc[-1].bias)

    def forward(self, g_t2: torch.Tensor, g_t1: torch.Tensor, g_t: torch.Tensor,
                dt2: torch.Tensor, dt1: torch.Tensor):
        with torch.no_grad():
            fmap_2, vec_2 = self.backbone.forward_spatial(g_t2)
            fmap_1, vec_1 = self.backbone.forward_spatial(g_t1)
            fmap_0, vec_0 = self.backbone.forward_spatial(g_t)

        c_2 = self.spatial_corr(torch.cat([fmap_2, fmap_2, torch.zeros_like(fmap_2)], dim=1))
        c_1 = self.spatial_corr(torch.cat([fmap_1, fmap_2, fmap_1 - fmap_2], dim=1))
        c_0 = self.spatial_corr(torch.cat([fmap_0, fmap_1, fmap_0 - fmap_1], dim=1))

        z_2 = torch.cat([vec_2, c_2, torch.zeros_like(dt2)], dim=-1)
        z_1 = torch.cat([vec_1, c_1, dt2], dim=-1)
        z_0 = torch.cat([vec_0, c_0, dt1], dim=-1)

        seq = torch.stack([z_2, z_1, z_0], dim=1)
        gru_out, _ = self.gru(seq)
        h_t = gru_out[:, -1, :]

        has_contig = (dt1 > 0.0).float()
        delta_vec = self.refine_fc(torch.cat([vec_0, h_t], dim=-1)) * has_contig
        vec_refined = vec_0 + delta_vec

        return self.lcf_head(vec_refined)

# =============================================================================
# 6. LOSS & MULTI-METRIC EVALUATOR
# =============================================================================
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss

eval_engine = MetricLossEvaluator()

def get_window_batch(b_idx: torch.Tensor):
    i_t = b_idx
    i_1 = vram_prev1[b_idx]
    i_2 = vram_prev2[b_idx]
    g_t = gpu_extractor(vram_images[i_t].float())
    g_1 = gpu_extractor(vram_images[i_1].float())
    g_2 = gpu_extractor(vram_images[i_2].float())
    dt1 = vram_dt1[b_idx].unsqueeze(-1)
    dt2 = vram_dt2[b_idx].unsqueeze(-1)
    return g_2, g_1, g_t, dt2, dt1

def compute_audit_metrics(all_pred: np.ndarray, all_gt: np.ndarray,
                          all_df: np.ndarray, all_disp: np.ndarray):
    mpjpe_all = np.linalg.norm(all_pred - all_gt, axis=-1).mean()
    contig_mask = (all_df <= 3.5) & (all_df > 0.5)
    mpjpe_df3 = np.linalg.norm(all_pred[contig_mask] - all_gt[contig_mask], axis=-1).mean()
    short_mask = (all_df <= 6.5) & (all_df > 0.5)
    mpjpe_df6 = np.linalg.norm(all_pred[short_mask] - all_gt[short_mask], axis=-1).mean()
    tip_mpjpe = np.linalg.norm(all_pred[:, FINGERTIP_JOINTS] - all_gt[:, FINGERTIP_JOINTS], axis=-1).mean()

    b_p = all_pred[:, CHILD_INDICES.cpu().numpy()] - all_pred[:, PARENT_INDICES.cpu().numpy()]
    b_g = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]
    u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
    u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
    cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
    deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
    distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()
    distal_deg_df3 = deg_bones[contig_mask][:, DISTAL_BONE_INDICES].mean()

    if np.sum(short_mask) > 20:
        thresh_95 = np.percentile(all_disp[short_mask], 95)
        fast_mask = short_mask & (all_disp >= thresh_95)
    else:
        thresh_95 = np.percentile(all_disp, 95)
        fast_mask = all_disp >= thresh_95
    fast_mpjpe = np.linalg.norm(all_pred[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    return {
        "mpjpe_all": mpjpe_all,
        "mpjpe_df3": mpjpe_df3,
        "mpjpe_df6": mpjpe_df6,
        "tip_mpjpe": tip_mpjpe,
        "distal_deg": distal_deg,
        "distal_deg_df3": distal_deg_df3,
        "fast_mpjpe": fast_mpjpe,
    }

# =============================================================================
# 7. RUN EXPERIMENT C ACROSS 3 SEGMENT-DISJOINT SPLITS
# =============================================================================
SEEDS = [42, 101, 777]
STATIC_EPOCHS = 15
TEMPORAL_EPOCHS = 10
BATCH_SIZE = 64

MODELS = ["Static 04_network (Frozen LCF)", "Temporal Correspondence GRU (df<=6 Gated)"]
metrics_log = {m: defaultdict(list) for m in MODELS}

print("\n" + "=" * 112)
print(" EXPERIMENT C: CONSECUTIVE-FRAME TEMPORAL CORRESPONDENCE BENCHMARK (3 HELD-OUT SPLITS)")
print("=" * 112)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)

    # 1. Train Frozen Static LCF Baseline
    static_model = StaticLCFModel().to(DEVICE)
    opt_static = torch.optim.AdamW(static_model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, STATIC_EPOCHS + 1):
        static_model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            g_t = gpu_extractor(vram_images[b_idx].float())
            y_b = vram_targets[b_idx].view(-1, 21, 3)
            opt_static.zero_grad()
            pred_j, _ = static_model(g_t)
            loss = eval_engine(pred_j, y_b)
            loss.backward()
            opt_static.step()

    static_model.eval()

    # 2. Train Contiguity-Gated Temporal Correspondence Encoder on Contiguous Windows
    temp_model = TemporalCorrespondenceEncoder(static_model).to(DEVICE)
    trainable_params = [p for p in temp_model.parameters() if p.requires_grad]
    opt_temp = torch.optim.AdamW(trainable_params, lr=8e-4, weight_decay=1e-4)

    contig_train_mask = vram_dt1[train_indices] > 0.0
    contig_train_indices = train_indices[contig_train_mask]

    for epoch in range(1, TEMPORAL_EPOCHS + 1):
        temp_model.train()
        perm = contig_train_indices[torch.randperm(len(contig_train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            g_2, g_1, g_t, dt2, dt1 = get_window_batch(b_idx)
            y_b = vram_targets[b_idx].view(-1, 21, 3)
            opt_temp.zero_grad()
            pred_j, _ = temp_model(g_2, g_1, g_t, dt2, dt1)
            loss = eval_engine(pred_j, y_b)
            loss.backward()
            opt_temp.step()

    temp_model.eval()

    # 3. Evaluate Both Models Chronologically on Held-Out Validation Segments
    for m_name, is_temp in [(MODELS[0], False), (MODELS[1], True)]:
        val_preds, val_gts, val_dfs, val_disps = [], [], [], []
        with torch.no_grad():
            for seg in val_segs:
                s_indices = seg_to_indices[seg]
                s_tensor = torch.tensor(s_indices, device=DEVICE)
                s_p, s_g = [], []
                for i in range(0, len(s_tensor), BATCH_SIZE):
                    b_idx = s_tensor[i:i + BATCH_SIZE]
                    if not is_temp:
                        g_t = gpu_extractor(vram_images[b_idx].float())
                        pj, _ = static_model(g_t)
                    else:
                        g_2, g_1, g_t, dt2, dt1 = get_window_batch(b_idx)
                        pj, _ = temp_model(g_2, g_1, g_t, dt2, dt1)
                    s_p.append(pj.cpu().numpy())
                    s_g.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

                s_p_arr = np.concatenate(s_p, axis=0)
                s_g_arr = np.concatenate(s_g, axis=0)
                val_preds.append(s_p_arr)
                val_gts.append(s_g_arr)
                val_dfs.append(delta_f_1[s_indices])

                disp_seg = [0.0]
                for t in range(len(s_g_arr) - 1):
                    disp_seg.append(np.linalg.norm(s_g_arr[t + 1] - s_g_arr[t], axis=-1).mean())
                val_disps.append(np.array(disp_seg))

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)
        all_df = np.concatenate(val_dfs, axis=0)
        all_dp = np.concatenate(val_disps, axis=0)

        res = compute_audit_metrics(all_p, all_g, all_df, all_dp)
        for k, v in res.items():
            metrics_log[m_name][k].append(v)

        print(f"  [Seed {seed:<3}] {m_name:<42} | All: {res['mpjpe_all']:5.2f} mm | "
              f"Adj(df=3): {res['mpjpe_df3']:5.2f} mm | Tips: {res['tip_mpjpe']:5.2f} mm | "
              f"Distal(df=3): {res['distal_deg_df3']:4.1f}° | Fast: {res['fast_mpjpe']:5.2f} mm")

    del static_model, temp_model, opt_static, opt_temp
    torch.cuda.empty_cache()

# =============================================================================
# 8. SUMMARY AUDIT TABLE
# =============================================================================
print("\n" + "=" * 118)
print(" EXPERIMENT C SUMMARY: CONTIGUOUS TEMPORAL CORRESPONDENCE vs. FROZEN STATIC LCF")
print("=" * 118)
print(f"{'Model Architecture':<42} | {'All MPJPE':<14} | {'Adj (df=3) MPJPE':<16} | {'Fingertips':<14} | {'Distal (df=3)':<14} | {'Fast-Motion'}")
print("-" * 118)
for m_name in MODELS:
    d = metrics_log[m_name]
    print(f"{m_name:<42} | "
          f"{np.mean(d['mpjpe_all']):5.2f}±{np.std(d['mpjpe_all']):4.2f} mm | "
          f"{np.mean(d['mpjpe_df3']):5.2f}±{np.std(d['mpjpe_df3']):4.2f} mm   | "
          f"{np.mean(d['tip_mpjpe']):5.2f}±{np.std(d['tip_mpjpe']):4.2f} mm | "
          f"{np.mean(d['distal_deg_df3']):4.1f}°±{np.std(d['distal_deg_df3']):3.1f}°    | "
          f"{np.mean(d['fast_mpjpe']):5.2f}±{np.std(d['fast_mpjpe']):4.2f} mm")
print("=" * 118)

Hardware Accelerator: Tesla T4
Building file map across filesystem...
Indexed 20,467 candidate files.
Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...
Successfully staged 5,114 frames into VRAM (720.4 MB used).

 EXPERIMENT C: CONSECUTIVE-FRAME TEMPORAL CORRESPONDENCE BENCHMARK (3 HELD-OUT SPLITS)
  [Seed 42 ] Static 04_network (Frozen LCF)             | All: 32.38 mm | Adj(df=3): 32.72 mm | Tips: 46.81 mm | Distal(df=3): 38.4° | Fast: 36.06 mm
  [Seed 42 ] Temporal Correspondence GRU (df<=6 Gated)  | All: 32.38 mm | Adj(df=3): 32.73 mm | Tips: 46.82 mm | Distal(df=3): 38.4° | Fast: 36.08 mm
  [Seed 101] Static 04_network (Frozen LCF)             | All: 32.36 mm | Adj(df=3): 31.19 mm | Tips: 50.09 mm | Distal(df=3): 47.9° | Fast: 40.71 mm
  [Seed 101] Temporal Correspondence GRU (df<=6 Gated)  | All: 32.38 mm | Adj(df=3): 31.23 mm | Tips: 50.11 mm | Distal(df=3): 47.9° | Fast: 40.65 mm
  [Seed 777] Static 04_network (Frozen LCF)             | All: 27.65 mm | Adj(df=3): 28.79 

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. HARDWARE ACCELERATOR & PATHS
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

# 2. FILE MAP & VRAM INGESTION (SELF-HEALING)
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p
    p1 = base_dir / rel_clean
    if p1.exists(): return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None: continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3: errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()
    print(f"Successfully staged {len(vram_images):,} frames into VRAM.")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

print("Ready for training and evaluation pipeline.")

Hardware Accelerator: Tesla T4
Reusing 5,114 frames already resident in VRAM.
Ready for training and evaluation pipeline.


In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    raise RuntimeError("VRAM tensors not found! Please run the VRAM ingestion cell first.")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))
sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                        for r in manifest_rows[:len(vram_images)]])

# Build chronological lookup and delta frame intervals
seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

prev_frame_indices = np.zeros(len(sample_segments), dtype=np.int64)
delta_frames = np.zeros(len(sample_segments), dtype=np.float32)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            prev_frame_indices[i] = i
            delta_frames[i] = 3.0
        else:
            prev_idx = indices[step_idx - 1]
            prev_frame_indices[i] = prev_idx
            df = float(sample_fids[i] - sample_fids[prev_idx])
            delta_frames[i] = max(df, 1.0)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).view(-1, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(-1, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((x.shape[0], 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

eval_engine = MetricLossEvaluator()

# =============================================================================
# PHYSICAL-TIME S^2 DIRECTIONAL FILTER (02_STATE) - CONTIGUITY & UNCERTAINTY AWARE
# =============================================================================
class PhysicalTimeS2Filter:
    def __init__(self, canonical_lens: torch.Tensor):
        self.bone_lens = canonical_lens
        # Heterogeneous directional uncertainty per bone (radians)
        self.sigma_R = torch.tensor([
            0.39, 0.48, 0.48, 0.58,  # Thumb
            0.25, 0.40, 0.57, 0.60,  # Index
            0.22, 0.40, 0.66, 0.72,  # Middle
            0.24, 0.44, 0.68, 0.76,  # Ring
            0.28, 0.58, 0.78, 0.84   # Pinky
        ], device=DEVICE)
        self.omega_max = 180.0 * (np.pi / 180.0)  # rad/s

    def filter_segment_sequence(self, raw_dirs_seq: torch.Tensor, df_seq: torch.Tensor):
        """
        raw_dirs_seq: (T, 20, 3) unconstrained LCF bone directions along segment
        df_seq: (T,) hardware frame-ID differences (native consecutive step = 3.0)
        """
        T_len = raw_dirs_seq.shape[0]
        filtered_dirs = []
        gates_fired = 0
        total_evals = 0

        prev_u = raw_dirs_seq[0]
        filtered_dirs.append(prev_u)

        for t in range(1, T_len):
            df_val = float(df_seq[t].item())
            curr_u = raw_dirs_seq[t]

            # If extraction gap > 6 frame ticks (>2 native steps), reset state prior to avoid ghost lag
            if df_val > 6.5:
                filtered_dirs.append(curr_u)
                prev_u = curr_u
                continue

            # Physical elapsed time at 30 FPS where native frame-ID stride is 3 ticks (90 ticks/sec)
            dt_t = torch.clamp(df_seq[t] / 90.0, min=1.0 / 30.0)

            # Tighter innovation gate (1.5 * sigma_R + omega_max * dt) to catch Necker flips on S^2
            theta_gate = self.omega_max * dt_t + 1.5 * self.sigma_R  # (20,)

            dot = torch.clamp((prev_u * curr_u).sum(dim=-1), -1.0, 1.0)
            dist = torch.acos(dot)  # (20,)

            gated = dist > theta_gate
            gates_fired += int(gated.sum().item())
            total_evals += 20

            # Bone-dependent Kalman gain: proximal bones (low sigma_R) trust measurement more;
            # noisy distal bones smooth across consecutive steps (df <= 6), unless gated by fast motion
            base_alpha = torch.clamp(dt_t / (dt_t + 0.04 * (self.sigma_R / 0.40)), min=0.35, max=0.92)
            alpha = torch.where(gated, torch.full_like(base_alpha, 0.95), base_alpha).unsqueeze(-1)  # (20, 1)

            # Spherical linear interpolation (SLERP) on S^2
            dist_col = dist.unsqueeze(-1)
            sin_dist = torch.sin(dist_col)
            small_ang = (sin_dist.abs() < 1e-5)

            w1 = torch.where(small_ang, 1.0 - alpha, torch.sin((1.0 - alpha) * dist_col) / (sin_dist + 1e-7))
            w2 = torch.where(small_ang, alpha, torch.sin(alpha * dist_col) / (sin_dist + 1e-7))

            updated_u = F.normalize(w1 * prev_u + w2 * curr_u, dim=-1, eps=1e-7)
            filtered_dirs.append(updated_u)
            prev_u = updated_u

        return torch.stack(filtered_dirs, dim=0), (gates_fired / max(total_evals, 1)) * 100.0

# =============================================================================
# BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# =============================================================================
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

print("\n" + "=" * 108)
print(" EXPERIMENT D: UNCERTAINTY & CONTIGUITY-AWARE S^2 STATE FILTER ACROSS 3 HELD-OUT SPLITS")
print("=" * 108)

raw_mpjpe_list, filt_mpjpe_list = [], []
raw_df3_list, filt_df3_list = [], []
raw_distal_list, filt_distal_list = [], []
fast_raw_list, fast_filt_list = [], []
gating_rates = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)

    model = LatentFactorizationModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)
            x_batch = gpu_extractor(raw_rgb)
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    filter_engine = PhysicalTimeS2Filter(L_CANON)

    val_raw_preds, val_filt_preds, val_gts, val_disps, val_dfs = [], [], [], [], []
    segment_gating_rates = []

    with torch.no_grad():
        for seg in val_segs:
            s_indices = seg_to_indices[seg]
            s_tensor = torch.tensor(s_indices, device=DEVICE)

            s_preds_j, s_preds_d, s_gts = [], [], []
            for i in range(0, len(s_tensor), BATCH_SIZE):
                b_idx = s_tensor[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                x_batch = gpu_extractor(raw_rgb)
                pj, pd = model(x_batch)
                s_preds_j.append(pj)
                s_preds_d.append(pd)
                s_gts.append(vram_targets[b_idx].view(-1, 21, 3))

            seg_j = torch.cat(s_preds_j, dim=0)
            seg_d = torch.cat(s_preds_d, dim=0)
            seg_gt = torch.cat(s_gts, dim=0)

            seg_df = delta_frames[s_indices]
            seg_df_t = torch.tensor(seg_df, dtype=torch.float32, device=DEVICE)

            filt_d, gate_rate = filter_engine.filter_segment_sequence(seg_d, seg_df_t)
            filt_vecs = filt_d * L_CANON.view(1, 20, 1)
            filt_j = reconstruct_joints_from_bones(filt_vecs)

            val_raw_preds.append(seg_j.cpu().numpy())
            val_filt_preds.append(filt_j.cpu().numpy())
            val_gts.append(seg_gt.cpu().numpy())
            val_dfs.append(seg_df)
            segment_gating_rates.append(gate_rate)

            disp_seg = [0.0]
            s_gt_np = seg_gt.cpu().numpy()
            for t in range(len(s_gt_np) - 1):
                disp_seg.append(np.linalg.norm(s_gt_np[t + 1] - s_gt_np[t], axis=-1).mean())
            val_disps.append(np.array(disp_seg))

    all_raw = np.concatenate(val_raw_preds, axis=0)
    all_filt = np.concatenate(val_filt_preds, axis=0)
    all_gt = np.concatenate(val_gts, axis=0)
    all_disp = np.concatenate(val_disps, axis=0)
    all_df = np.concatenate(val_dfs, axis=0)

    mpjpe_raw = np.linalg.norm(all_raw - all_gt, axis=-1).mean()
    mpjpe_filt = np.linalg.norm(all_filt - all_gt, axis=-1).mean()

    df3_mask = (all_df <= 3.5)
    df3_raw = np.linalg.norm(all_raw[df3_mask] - all_gt[df3_mask], axis=-1).mean()
    df3_filt = np.linalg.norm(all_filt[df3_mask] - all_gt[df3_mask], axis=-1).mean()

    def get_distal_deg(pred_j, gt_j):
        b_p = pred_j[:, CHILD_INDICES.cpu().numpy()] - pred_j[:, PARENT_INDICES.cpu().numpy()]
        b_g = gt_j[:, CHILD_INDICES.cpu().numpy()] - gt_j[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        return (np.arccos(cos_sim) * (180.0 / np.pi))[:, DISTAL_BONE_INDICES].mean()

    dist_raw = get_distal_deg(all_raw, all_gt)
    dist_filt = get_distal_deg(all_filt, all_gt)

    thresh_95 = np.percentile(all_disp, 95)
    fast_mask = all_disp >= thresh_95
    fast_raw = np.linalg.norm(all_raw[fast_mask] - all_gt[fast_mask], axis=-1).mean()
    fast_filt = np.linalg.norm(all_filt[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    raw_mpjpe_list.append(mpjpe_raw)
    filt_mpjpe_list.append(mpjpe_filt)
    raw_df3_list.append(df3_raw)
    filt_df3_list.append(df3_filt)
    raw_distal_list.append(dist_raw)
    filt_distal_list.append(dist_filt)
    fast_raw_list.append(fast_raw)
    fast_filt_list.append(fast_filt)
    gating_rates.append(np.mean(segment_gating_rates))

    print(f" [Seed {seed:<3}] All MPJPE: {mpjpe_raw:5.2f} -> {mpjpe_filt:5.2f} mm | "
          f"Adj(df=3): {df3_raw:5.2f} -> {df3_filt:5.2f} mm | "
          f"Fast (>95%): {fast_raw:5.2f} -> {fast_filt:5.2f} mm | Gated: {gating_rates[-1]:5.2f}%")

    del model, optimizer
    torch.cuda.empty_cache()

print("\n" + "=" * 108)
print(" EXPERIMENT D SUMMARY: CONTIGUITY-GATED S^2 FILTER vs. RAW STATIC LCF")
print("=" * 108)
print(f"{'Evaluation Metric':<34} | {'Raw Static LCF':<18} | {'Filtered Kinematic':<18} | {'Delta'}")
print("-" * 108)
print(f"{'Overall MPJPE (mm)':<34} | {np.mean(raw_mpjpe_list):5.2f} ± {np.std(raw_mpjpe_list):4.2f} mm | {np.mean(filt_mpjpe_list):5.2f} ± {np.std(filt_mpjpe_list):4.2f} mm | {np.mean(filt_mpjpe_list) - np.mean(raw_mpjpe_list):+5.2f} mm")
print(f"{'Consecutive (df=3) MPJPE (mm)':<34} | {np.mean(raw_df3_list):5.2f} ± {np.std(raw_df3_list):4.2f} mm | {np.mean(filt_df3_list):5.2f} ± {np.std(filt_df3_list):4.2f} mm | {np.mean(filt_df3_list) - np.mean(raw_df3_list):+5.2f} mm")
print(f"{'Distal Tip Direction Error':<34} | {np.mean(raw_distal_list):4.1f}° ± {np.std(raw_distal_list):3.1f}° | {np.mean(filt_distal_list):4.1f}° ± {np.std(filt_distal_list):3.1f}° | {np.mean(filt_distal_list) - np.mean(raw_distal_list):+4.1f}°")
print(f"{'Fast-Motion MPJPE (>95th %)':<34} | {np.mean(fast_raw_list):5.2f} ± {np.std(fast_raw_list):4.2f} mm | {np.mean(fast_filt_list):5.2f} ± {np.std(fast_filt_list):4.2f} mm | {np.mean(fast_filt_list) - np.mean(fast_raw_list):+5.2f} mm")
print(f"{'Mean Innovation Gating Rate':<34} | {'-':<18} | {np.mean(gating_rates):5.2f}%             | -")
print("=" * 108)

Hardware Accelerator: Tesla T4
Reusing 5,114 frames already resident in VRAM.

 EXPERIMENT D: UNCERTAINTY & CONTIGUITY-AWARE S^2 STATE FILTER ACROSS 3 HELD-OUT SPLITS
 [Seed 42 ] All MPJPE: 33.53 -> 33.51 mm | Adj(df=3): 33.06 -> 32.95 mm | Fast (>95%): 34.75 -> 34.82 mm | Gated:  0.00%
 [Seed 101] All MPJPE: 32.30 -> 32.30 mm | Adj(df=3): 30.25 -> 30.22 mm | Fast (>95%): 37.25 -> 37.31 mm | Gated:  0.00%
 [Seed 777] All MPJPE: 27.86 -> 27.82 mm | Adj(df=3): 27.88 -> 27.87 mm | Fast (>95%): 32.88 -> 32.88 mm | Gated:  0.00%

 EXPERIMENT D SUMMARY: CONTIGUITY-GATED S^2 FILTER vs. RAW STATIC LCF
Evaluation Metric                  | Raw Static LCF     | Filtered Kinematic | Delta
------------------------------------------------------------------------------------------------------------
Overall MPJPE (mm)                 | 31.23 ± 2.44 mm | 31.21 ± 2.44 mm | -0.02 mm
Consecutive (df=3) MPJPE (mm)      | 30.39 ± 2.12 mm | 30.35 ± 2.08 mm | -0.05 mm
Distal Tip Direction Error         | 40.3

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    raise RuntimeError("VRAM tensors not found! Please run the VRAM ingestion cell first.")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))
sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                        for r in manifest_rows[:len(vram_images)]])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

# Build contiguity-gated previous frame index (only pair if df <= 6, else self)
contig_prev_indices = np.zeros(len(sample_segments), dtype=np.int64)
delta_frames = np.zeros(len(sample_segments), dtype=np.float32)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            contig_prev_indices[i] = i
            delta_frames[i] = 999.0
        else:
            prev_i = indices[step_idx - 1]
            df = float(sample_fids[i] - sample_fids[prev_i])
            delta_frames[i] = df
            if df <= 6.5:
                contig_prev_indices[i] = prev_i
            else:
                contig_prev_indices[i] = i  # Gate out non-adjacent keyframes

vram_contig_prev = torch.tensor(contig_prev_indices, dtype=torch.long, device=DEVICE)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

# =============================================================================
# 1. MULTI-CUE GPU VISUAL FEATURE EXTRACTOR (EXPERIMENT E)
# =============================================================================
class GPUVisualCueExtractor(nn.Module):
    """
    Extracts illumination-invariant interior appearance and crease polarity cues
    alongside Scalar Sobel magnitude |G_t|:
      - |G_t|    : Scalar Sobel edge magnitude (1 Ch)
      - I_norm   : Instance-standardized foreground luminance (shading/depth cue, 1 Ch)
      - Lap(I)   : Signed Laplacian valley/ridge detector (palmar flexion creases vs dorsal surface, 1 Ch)
      - Chrom_RG : Normalized Red-Green chrominance ratio (palm vs dorsal skin tone cue, 1 Ch)
    """
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        # Signed 3x3 Laplacian kernel (valleys/creases vs ridges)
        klap = torch.tensor([[0.,  1., 0.],
                             [1., -4., 1.],
                             [0.,  1., 0.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)
        self.register_buffer("klap", klap)

    @torch.no_grad()
    def extract(self, b_idx: torch.Tensor, mode: str) -> torch.Tensor:
        rgb = vram_images[b_idx].float()  # (B, 3, H, W) in [0, 255]
        r, g, b = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]
        gray = 0.299 * r + 0.587 * g + 0.114 * b  # (B, 1, H, W)

        # 1. Scalar Sobel |G_t|
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        sobel_mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

        if mode == "1_sobel_only":
            return sobel_mag

        # Per-frame foreground-aware luminance normalization (eliminates global exposure drift)
        mean_g = gray.mean(dim=(-2, -1), keepdim=True)
        std_g = gray.std(dim=(-2, -1), keepdim=True) + 1.0
        i_norm = torch.clamp((gray - mean_g) / (3.0 * std_g), -1.0, 1.0) * 0.5 + 0.5

        if mode == "2_sobel_luminance":
            return torch.cat([sobel_mag, i_norm], dim=1)

        # Signed Crease Laplacian (highlights dark palmar creases & inter-finger occlusion valleys)
        lap = F.conv2d(gray, self.klap, padding=1) / 255.0
        lap_signed = torch.tanh(lap * 3.0)

        if mode == "3_sobel_crease_lap":
            return torch.cat([sobel_mag, lap_signed], dim=1)

        if mode == "4_full_appearance_contig":
            # Include contiguity-gated previous Sobel |G_{t-1}| (equals |G_t| when df > 6)
            prev_idx = vram_contig_prev[b_idx]
            rgb_prev = vram_images[prev_idx].float()
            gray_prev = 0.299 * rgb_prev[:, 0:1] + 0.587 * rgb_prev[:, 1:2] + 0.114 * rgb_prev[:, 2:3]
            gx_p = F.conv2d(gray_prev, self.kx, padding=1) / 1020.0
            gy_p = F.conv2d(gray_prev, self.ky, padding=1) / 1020.0
            sobel_prev = torch.sqrt(gx_p ** 2 + gy_p ** 2 + 1e-12)
            return torch.cat([sobel_mag, i_norm, lap_signed, sobel_prev], dim=1)

        raise ValueError(f"Unknown mode: {mode}")

cue_extractor = GPUVisualCueExtractor(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# =============================================================================
# 2. LCF MODEL WITH VARIABLE INPUT CHANNELS
# =============================================================================
class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).view(-1, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(-1, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((x.shape[0], 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

eval_engine = MetricLossEvaluator()

# =============================================================================
# 3. BENCHMARK HARNESS (EXPERIMENT E: 4 VISUAL REPRESENTATIONS x 3 SPLITS)
# =============================================================================
CONFIGS = [
    ("1. Scalar Sobel Baseline (|G_t|)",               1, "1_sobel_only"),
    ("2. Sobel + Norm Luminance ([|G_t|, I_norm])",    2, "2_sobel_luminance"),
    ("3. Sobel + Crease Polarity ([|G_t|, Lap(I)])",   2, "3_sobel_crease_lap"),
    ("4. Full Interior + Contig ([|G|, I, Lap, G_p])", 4, "4_full_appearance_contig"),
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

res_mpjpe = {name: [] for name, _, _ in CONFIGS}
res_tips = {name: [] for name, _, _ in CONFIGS}
res_distal = {name: [] for name, _, _ in CONFIGS}
per_bone_deg = {name: np.zeros(20) for name, _, _ in CONFIGS}

print("\n" + "=" * 110)
print(" EXPERIMENT E: INTERIOR APPEARANCE, CREASE POLARITY & SHADING BENCHMARK (3 HELD-OUT SPLITS)")
print("=" * 110)

for exp_name, in_ch, mode_key in CONFIGS:
    print(f"\n>>> Running {exp_name} ({in_ch} Ch) <<<")
    accum_bones = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = LatentFactorizationModel(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]
            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                x_batch = cue_extractor.extract(b_idx, mode_key)
                y_batch = vram_targets[b_idx].view(-1, 21, 3)

                optimizer.zero_grad()
                pred_joints, _ = model(x_batch)
                loss = eval_engine(pred_joints, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        val_preds, val_gts = [], []
        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                x_batch = cue_extractor.extract(b_idx, mode_key)
                pj, _ = model(x_batch)
                val_preds.append(pj.cpu().numpy())
                val_gts.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)

        mpjpe = np.linalg.norm(all_p - all_g, axis=-1).mean()
        tip_err = np.linalg.norm(all_p[:, FINGERTIP_JOINTS] - all_g[:, FINGERTIP_JOINTS], axis=-1).mean()

        b_p = all_p[:, CHILD_INDICES.cpu().numpy()] - all_p[:, PARENT_INDICES.cpu().numpy()]
        b_g = all_g[:, CHILD_INDICES.cpu().numpy()] - all_g[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()

        res_mpjpe[exp_name].append(mpjpe)
        res_tips[exp_name].append(tip_err)
        res_distal[exp_name].append(distal_deg)
        accum_bones += deg_bones.mean(axis=0)

        print(f"  [Seed {seed:<3}] MPJPE: {mpjpe:5.2f} mm | Fingertips: {tip_err:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

        del model, optimizer
        torch.cuda.empty_cache()

    per_bone_deg[exp_name] = accum_bones / len(SEEDS)

# =============================================================================
# 4. SUMMARY REPORT & PER-BONE DISTAL AUDIT
# =============================================================================
print("\n" + "=" * 110)
print(" EXPERIMENT E SUMMARY: VISUAL CUES BEYOND SCALAR SOBEL MAGNITUDE (3 HELD-OUT SPLITS)")
print("=" * 110)
print(f"{'Visual Input Representation':<48} | {'Overall MPJPE':<18} | {'Fingertip MPJPE':<18} | {'Distal Dir Err'}")
print("-" * 110)
base_mpjpe = np.mean(res_mpjpe[CONFIGS[0][0]])
for exp_name, _, _ in CONFIGS:
    mp, sp = np.mean(res_mpjpe[exp_name]), np.std(res_mpjpe[exp_name])
    mt, st = np.mean(res_tips[exp_name]), np.std(res_tips[exp_name])
    md, sd = np.mean(res_distal[exp_name]), np.std(res_distal[exp_name])
    print(f"{exp_name:<48} | {mp:5.2f} ± {sp:4.2f} mm   | {mt:5.2f} ± {st:4.2f} mm   | {md:4.1f}° ± {sd:3.1f}°")
print("=" * 110)

print("\n" + "=" * 110)
print(" PER-BONE DIRECTION ERROR BREAKDOWN (DEGREES)")
print("=" * 110)
c0, c1, c2, c3 = [c[0] for c in CONFIGS]
print(f"{'Bone Identifier':<24} | {'1. |G_t|':<14} | {'2. +Luminance':<14} | {'3. +CreaseLap':<14} | {'4. FullStack':<14} | {'Best Delta'}")
print("-" * 110)
for k in range(20):
    v0 = per_bone_deg[c0][k]
    v1 = per_bone_deg[c1][k]
    v2 = per_bone_deg[c2][k]
    v3 = per_bone_deg[c3][k]
    best_alt = min(v1, v2, v3)
    delta = best_alt - v0
    print(f"{BONE_NAMES[k]:<24} | {v0:5.2f}°        | {v1:5.2f}°        | {v2:5.2f}°        | {v3:5.2f}°        | {delta:+5.2f}°")
print("=" * 110)

Hardware Accelerator: Tesla T4
Reusing 5,114 frames already resident in VRAM.

 EXPERIMENT E: INTERIOR APPEARANCE, CREASE POLARITY & SHADING BENCHMARK (3 HELD-OUT SPLITS)

>>> Running 1. Scalar Sobel Baseline (|G_t|) (1 Ch) <<<
  [Seed 42 ] MPJPE: 32.68 mm | Fingertips: 47.40 mm | Distal Dir: 39.6°
  [Seed 101] MPJPE: 32.58 mm | Fingertips: 50.22 mm | Distal Dir: 50.5°
  [Seed 777] MPJPE: 28.07 mm | Fingertips: 38.84 mm | Distal Dir: 30.0°

>>> Running 2. Sobel + Norm Luminance ([|G_t|, I_norm]) (2 Ch) <<<
  [Seed 42 ] MPJPE: 43.19 mm | Fingertips: 60.33 mm | Distal Dir: 45.9°
  [Seed 101] MPJPE: 35.13 mm | Fingertips: 51.75 mm | Distal Dir: 51.5°
  [Seed 777] MPJPE: 33.97 mm | Fingertips: 48.24 mm | Distal Dir: 40.9°

>>> Running 3. Sobel + Crease Polarity ([|G_t|, Lap(I)]) (2 Ch) <<<
  [Seed 42 ] MPJPE: 34.45 mm | Fingertips: 50.01 mm | Distal Dir: 42.3°
  [Seed 101] MPJPE: 33.95 mm | Fingertips: 51.88 mm | Distal Dir: 52.6°
  [Seed 777] MPJPE: 30.82 mm | Fingertips: 42.83 mm | Dista

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# =============================================================================
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# =============================================================================
# 2. SELF-HEALING VRAM INGESTION (LOADS IF NOT ALREADY IN VRAM)
# =============================================================================
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists():
        search_dirs.append(base_dir.parent)
    if Path("/content").exists():
        search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists():
        return p
    p1 = base_dir / rel_clean
    if p1.exists():
        return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists():
        return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists():
            return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists():
            return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists():
            return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")
    if 'sample_fids' not in globals() or len(sample_fids) != len(vram_images):
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest_rows = list(csv.DictReader(f))
        sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                                for r in manifest_rows[:len(vram_images)]])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

# =============================================================================
# 3. GPU SOBEL ENGINE & KINEMATIC HELPERS
# =============================================================================
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# =============================================================================
# 4. BASELINE GLOBAL LCF (6x8 POOLING) vs. HIERARCHICAL SPATIAL-ROI LCF (24x32)
# =============================================================================
class GlobalLCFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).view(-1, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(-1, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((x.shape[0], 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs


class HierarchicalSpatialLCFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),   # 96 x 128
            nn.BatchNorm2d(32), nn.ReLU(inplace=True)
        )
        self.stage2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 48 x 64
            nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        self.stage3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 24 x 32
            nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.stage4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),# 12 x 16
            nn.BatchNorm2d(256), nn.ReLU(inplace=True)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((6, 8))
        self.global_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)

        self.fpn_fuse = nn.Sequential(
            nn.Conv2d(64 + 128 + 256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.spatial_attn_conv = nn.Conv2d(128, 15, kernel_size=1)

        self.prox_fc = nn.Sequential(nn.Linear(512 + 130 + 3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))
        self.mid_fc = nn.Sequential(nn.Linear(512 + 130 + 3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))
        self.dist_fc = nn.Sequential(nn.Linear(512 + 130 + 3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))

        self.log_var_head = nn.Linear(512, 20)
        self.register_buffer("bone_lens", L_CANON)

        yy, xx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, 24, device=DEVICE),
            torch.linspace(-1.0, 1.0, 32, device=DEVICE),
            indexing="ij"
        )
        # Fixed: use .reshape() on non-contiguous meshgrid tensors
        self.register_buffer("grid_x", xx.contiguous().reshape(1, 1, 24 * 32))
        self.register_buffer("grid_y", yy.contiguous().reshape(1, 1, 24 * 32))

    def forward(self, x):
        B = x.shape[0]
        f1 = self.stage1(x)
        f2 = self.stage2(f1)  # (B, 64, 48, 64)
        f3 = self.stage3(f2)  # (B, 128, 24, 32)
        f4 = self.stage4(f3)  # (B, 256, 12, 16)

        g_feat = self.global_fc(torch.flatten(self.global_pool(f4), 1))  # (B, 512)
        meta_cam_u = F.normalize(self.meta_head(g_feat).reshape(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(g_feat)) # (B, 3, 3)

        meta_local_u = torch.bmm(R_latent.transpose(1, 2), meta_cam_u.transpose(1, 2)).transpose(1, 2)

        f2_down = F.avg_pool2d(f2, kernel_size=2, stride=2)
        f4_up = F.interpolate(f4, size=(24, 32), mode="bilinear", align_corners=False)
        fpn = self.fpn_fuse(torch.cat([f2_down, f3, f4_up], dim=1))      # (B, 128, 24, 32)

        attn_logits = self.spatial_attn_conv(fpn).reshape(B, 15, 24 * 32)
        attn_weights = F.softmax(attn_logits, dim=-1)                    # (B, 15, 768)

        fpn_flat = fpn.reshape(B, 128, 24 * 32)
        local_feats = torch.bmm(attn_weights, fpn_flat.transpose(1, 2))  # (B, 15, 128)
        exp_x = (attn_weights * self.grid_x).sum(dim=-1, keepdim=True)
        exp_y = (attn_weights * self.grid_y).sum(dim=-1, keepdim=True)
        local_desc = torch.cat([local_feats, exp_x, exp_y], dim=-1)      # (B, 15, 130)

        local_desc_5x3 = local_desc.reshape(B, 5, 3, 130)
        g_exp = g_feat.unsqueeze(1).expand(B, 5, 512)

        prox_in = torch.cat([g_exp, local_desc_5x3[:, :, 0, :], meta_local_u], dim=-1)
        prox_local_u = F.normalize(self.prox_fc(prox_in), dim=-1, eps=1e-7)

        mid_in = torch.cat([g_exp, local_desc_5x3[:, :, 1, :], prox_local_u], dim=-1)
        mid_local_u = F.normalize(self.mid_fc(mid_in), dim=-1, eps=1e-7)

        dist_in = torch.cat([g_exp, local_desc_5x3[:, :, 2, :], mid_local_u], dim=-1)
        dist_local_u = F.normalize(self.dist_fc(dist_in), dim=-1, eps=1e-7)

        phalanx_local_u = torch.stack([prox_local_u, mid_local_u, dist_local_u], dim=2).reshape(B, 15, 3)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        log_var = torch.clamp(self.log_var_head(g_feat), min=-4.0, max=4.0)
        return joints, bone_dirs, log_var

# =============================================================================
# 5. LOSS EVALUATOR & BENCHMARK HARNESS (3 HELD-OUT SPLITS)
# =============================================================================
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor, log_var: torch.Tensor = None):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        bone_dir_err = 1.0 - cos_sim
        l_dir = bone_dir_err.mean()

        base_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        if log_var is not None:
            l_unc = (torch.exp(-log_var) * bone_dir_err.detach() + 0.5 * log_var).mean()
            base_loss = base_loss + 0.5 * l_unc
        return base_loss

eval_engine = MetricLossEvaluator()

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

MODELS = [
    "1. Global LCF Baseline (6x8 Pool)",
    "2. Hierarchical Spatial-FPN LCF (24x32 ROI + Chain)",
]

res_mpjpe = {m: [] for m in MODELS}
res_tips = {m: [] for m in MODELS}
res_distal = {m: [] for m in MODELS}
per_bone_deg = {m: np.zeros(20) for m in MODELS}

print("\n" + "=" * 110)
print(" EXPERIMENT F: GLOBAL LCF (6x8) vs. HIERARCHICAL SPATIAL-ROI FPN LCF (24x32) ACROSS 3 SPLITS")
print("=" * 110)

for m_name in MODELS:
    is_hier = "Hierarchical" in m_name
    print(f"\n>>> Running {m_name} <<<")
    accum_bones = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = HierarchicalSpatialLCFModel().to(DEVICE) if is_hier else GlobalLCFModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]
            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                x_batch = gpu_extractor(vram_images[b_idx].float())
                y_batch = vram_targets[b_idx].view(-1, 21, 3)

                optimizer.zero_grad()
                if is_hier:
                    pred_j, _, log_v = model(x_batch)
                    loss = eval_engine(pred_j, y_batch, log_v)
                else:
                    pred_j, _ = model(x_batch)
                    loss = eval_engine(pred_j, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        val_preds, val_gts = [], []
        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                x_batch = gpu_extractor(vram_images[b_idx].float())
                out = model(x_batch)
                pj = out[0]
                val_preds.append(pj.cpu().numpy())
                val_gts.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)

        mpjpe = np.linalg.norm(all_p - all_g, axis=-1).mean()
        tip_err = np.linalg.norm(all_p[:, FINGERTIP_JOINTS] - all_g[:, FINGERTIP_JOINTS], axis=-1).mean()

        b_p = all_p[:, CHILD_INDICES.cpu().numpy()] - all_p[:, PARENT_INDICES.cpu().numpy()]
        b_g = all_g[:, CHILD_INDICES.cpu().numpy()] - all_g[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()

        res_mpjpe[m_name].append(mpjpe)
        res_tips[m_name].append(tip_err)
        res_distal[m_name].append(distal_deg)
        accum_bones += deg_bones.mean(axis=0)

        print(f"  [Seed {seed:<3}] MPJPE: {mpjpe:5.2f} mm | Fingertips: {tip_err:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

        del model, optimizer
        torch.cuda.empty_cache()

    per_bone_deg[m_name] = accum_bones / len(SEEDS)

print("\n" + "=" * 110)
print(" EXPERIMENT F SUMMARY: SPATIAL RESOLUTION & HIERARCHICAL CHAIN DECODING")
print("=" * 110)
print(f"{'Architecture':<52} | {'Overall MPJPE':<18} | {'Fingertip MPJPE':<18} | {'Distal Dir Err'}")
print("-" * 110)
for m_name in MODELS:
    mp, sp = np.mean(res_mpjpe[m_name]), np.std(res_mpjpe[m_name])
    mt, st = np.mean(res_tips[m_name]), np.std(res_tips[m_name])
    md, sd = np.mean(res_distal[m_name]), np.std(res_distal[m_name])
    print(f"{m_name:<52} | {mp:5.2f} ± {sp:4.2f} mm   | {mt:5.2f} ± {st:4.2f} mm   | {md:4.1f}° ± {sd:3.1f}°")
print("=" * 110)

print("\n" + "=" * 95)
print(" PER-BONE DIRECTION ERROR BREAKDOWN (DEGREES)")
print("=" * 95)
m0, m1 = MODELS[0], MODELS[1]
print(f"{'Bone Identifier':<24} | {'1. Global (6x8)':<18} | {'2. Hier-FPN (24x32)':<20} | {'Delta'}")
print("-" * 95)
for k in range(20):
    v0 = per_bone_deg[m0][k]
    v1 = per_bone_deg[m1][k]
    print(f"{BONE_NAMES[k]:<24} | {v0:6.2f}°            | {v1:6.2f}°              | {v1 - v0:+5.2f}°")
print("=" * 95)

Hardware Accelerator: Tesla T4
Building file map across filesystem...
Indexed 20,467 candidate files.
Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...
Successfully staged 5,114 frames into VRAM (720.4 MB used).

 EXPERIMENT F: GLOBAL LCF (6x8) vs. HIERARCHICAL SPATIAL-ROI FPN LCF (24x32) ACROSS 3 SPLITS

>>> Running 1. Global LCF Baseline (6x8 Pool) <<<
  [Seed 42 ] MPJPE: 33.10 mm | Fingertips: 48.18 mm | Distal Dir: 39.8°
  [Seed 101] MPJPE: 33.43 mm | Fingertips: 51.83 mm | Distal Dir: 52.1°
  [Seed 777] MPJPE: 28.52 mm | Fingertips: 39.59 mm | Distal Dir: 30.4°

>>> Running 2. Hierarchical Spatial-FPN LCF (24x32 ROI + Chain) <<<
  [Seed 42 ] MPJPE: 34.65 mm | Fingertips: 51.73 mm | Distal Dir: 45.0°
  [Seed 101] MPJPE: 33.28 mm | Fingertips: 50.96 mm | Distal Dir: 50.6°
  [Seed 777] MPJPE: 29.01 mm | Fingertips: 40.41 mm | Distal Dir: 33.7°

 EXPERIMENT F SUMMARY: SPATIAL RESOLUTION & HIERARCHICAL CHAIN DECODING
Architecture                                         | Overa

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# =============================================================================
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
MCP_JOINTS = [1, 5, 9, 13, 17]

# =============================================================================
# 2. SELF-HEALING VRAM INGESTION (LOADS IF NOT ALREADY IN VRAM)
# =============================================================================
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists():
        search_dirs.append(base_dir.parent)
    if Path("/content").exists():
        search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists():
        return p
    p1 = base_dir / rel_clean
    if p1.exists():
        return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists():
        return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists():
            return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists():
            return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists():
            return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")
    if 'sample_fids' not in globals() or len(sample_fids) != len(vram_images):
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest_rows = list(csv.DictReader(f))
        sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                                for r in manifest_rows[:len(vram_images)]])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

N_SAMPLES = len(vram_images)
prev1_indices = np.zeros(N_SAMPLES, dtype=np.int64)
delta_f_arr = np.zeros(N_SAMPLES, dtype=np.float32)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            prev1_indices[i] = i
            delta_f_arr[i] = 999.0
        else:
            p_i = indices[step_idx - 1]
            df = float(sample_fids[i] - sample_fids[p_i])
            delta_f_arr[i] = df
            prev1_indices[i] = p_i if df <= 6.5 else i

vram_prev1 = torch.tensor(prev1_indices, dtype=torch.long, device=DEVICE)
vram_contig_flag = torch.tensor((delta_f_arr <= 6.5).astype(np.float32), device=DEVICE)
vram_dt_scale = torch.tensor(np.where(delta_f_arr <= 6.5, delta_f_arr / 3.0, 0.0).astype(np.float32), device=DEVICE)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

# =============================================================================
# 3. GPU SOBEL & DETERMINISTIC GEOMETRIC RELATIONSHIP TENSOR R_t = R(K_t)
# =============================================================================
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

def compute_relationship_tensor_R(joints: torch.Tensor, bone_dirs: torch.Tensor) -> torch.Tensor:
    """
    Deterministic SE(3)-invariant & camera-aware Geometric Relationship Tensor R_t (03_observation/relationship.py):
      1. Pairwise 5x5 fingertip distances (10 unique pairs, normalized by 100mm) -> 10D
      2. Fingertip-to-MCP distances (5 fingers x 5 MCPs = 25D, normalized by 100mm) -> 25D
      3. Intra-finger sequential bone dot products (cos_theta for 5 fingers x 3 joints = 15D) -> 15D
      4. Adjacent-finger parallel phalanx dot products (4 adjacent pairs x 4 bone levels = 16D) -> 16D
      5. Signed palmar elevation of all 5 fingertips relative to palm normal (5D) -> 5D
      Total Relationship State R_t = 71D
    """
    tips = joints[:, FINGERTIP_JOINTS, :]  # (B, 5, 3)
    mcps = joints[:, MCP_JOINTS, :]        # (B, 5, 3)

    # 1. Pairwise fingertip distances (10D)
    tip_diff = tips.unsqueeze(2) - tips.unsqueeze(1)  # (B, 5, 5, 3)
    tip_dists = torch.norm(tip_diff + 1e-7, dim=-1) / 100.0
    tri_u = torch.triu_indices(5, 5, offset=1, device=joints.device)
    r_tip_pairs = tip_dists[:, tri_u[0], tri_u[1]]    # (B, 10)

    # 2. Fingertip to MCP distances (25D)
    tip_mcp_diff = tips.unsqueeze(2) - mcps.unsqueeze(1)
    r_tip_mcp = (torch.norm(tip_mcp_diff + 1e-7, dim=-1) / 100.0).reshape(-1, 25)

    # 3. Intra-finger sequential dot products (15D)
    dirs_5x4 = bone_dirs.reshape(-1, 5, 4, 3)
    r_intra_cos = (dirs_5x4[:, :, 0:3, :] * dirs_5x4[:, :, 1:4, :]).sum(dim=-1).reshape(-1, 15)

    # 4. Cross-finger adjacent bone dot products (16D)
    r_inter_cos = (dirs_5x4[:, 0:4, :, :] * dirs_5x4[:, 1:5, :, :]).sum(dim=-1).reshape(-1, 16)

    # 5. Signed palmar elevation of fingertips relative to palm normal (5D)
    u_idx = bone_dirs[:, 4]
    u_mid = bone_dirs[:, 8]
    u_pky = bone_dirs[:, 16]
    palm_norm = F.normalize(torch.cross(u_mid, u_pky - u_idx, dim=-1), dim=-1, eps=1e-7)  # (B, 3)
    r_palm_elev = ((tips - mcps[:, 2:3, :]) * palm_norm.unsqueeze(1)).sum(dim=-1) / 100.0  # (B, 5)

    return torch.cat([r_tip_pairs, r_tip_mcp, r_intra_cos, r_inter_cos, r_palm_elev], dim=-1)  # (B, 71)

# =============================================================================
# 4. ARCHITECTURES: BASELINE LCF vs. CLOSED-LOOP (K + R) & TEMPORAL STATE REFINER
# =============================================================================
class BaseLCFNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def decode_lcf(self, feat: torch.Tensor):
        B = feat.shape[0]
        meta_cam_u = F.normalize(self.meta_head(feat).reshape(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).reshape(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=feat.dtype, device=feat.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs, R_latent, phalanx_local_u

    def forward(self, g_t: torch.Tensor):
        E_t = self.shared_fc(torch.flatten(self.encoder(g_t), 1))
        joints, bone_dirs, _, _ = self.decode_lcf(E_t)
        return joints, bone_dirs, E_t


class ClosedLoopKRRefiner(nn.Module):
    """
    Implements the hm_ex Definitive Specification (source: 48):
      - Takes Visual Evidence E_t (512D) + Initial Skeleton K_t^0 (63D) + Relationship R_t^0 (71D)
      - Optionally includes contiguous prior state context (K_{t-1}, R_{t-1}, Dual-Edge E_t - E_{t-1})
      - Outputs articulated latent residual updates through the exact L_CANON bottleneck (0.00mm bone stretch).
    """
    def __init__(self, use_temporal_state: bool = False):
        super().__init__()
        self.use_temporal_state = use_temporal_state
        self.base_lcf = BaseLCFNetwork()

        # Input tuple: E_t (512) + K_t (63) + B_t (60) + R_t (71) = 706D
        in_dim = 512 + 63 + 60 + 71
        if use_temporal_state:
            # Add ΔE_t (512) + ΔK_t (63) + ΔR_t (71) + dt_flag (2) = +648D
            in_dim += 512 + 63 + 71 + 2

        self.residual_mlp = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 512)
        )
        # Zero-init final layer so epoch 0 starts cleanly at the proven LCF baseline
        nn.init.zeros_(self.residual_mlp[-1].weight)
        nn.init.zeros_(self.residual_mlp[-1].bias)

    def forward(self, g_t: torch.Tensor, g_prev: torch.Tensor = None,
                contig_flag: torch.Tensor = None, dt_scale: torch.Tensor = None):
        j0, d0, E_t = self.base_lcf(g_t)
        K_0 = j0.reshape(-1, 63) / 100.0
        B_0 = d0.reshape(-1, 60)
        R_0 = compute_relationship_tensor_R(j0, d0)

        if not self.use_temporal_state:
            state_tuple = torch.cat([E_t, K_0, B_0, R_0], dim=-1)
        else:
            with torch.no_grad():
                j_prev, d_prev, E_prev = self.base_lcf(g_prev)
                K_prev = j_prev.reshape(-1, 63) / 100.0
                R_prev = compute_relationship_tensor_R(j_prev, d_prev)

            c_mask = contig_flag.unsqueeze(-1)  # (B, 1): 1 if df <= 6 else 0
            dt_col = dt_scale.unsqueeze(-1)     # (B, 1)

            dE = (E_t - E_prev) * c_mask
            dK = (K_0 - K_prev) * c_mask
            dR = (R_0 - R_prev) * c_mask
            state_tuple = torch.cat([E_t, K_0, B_0, R_0, dE, dK, dR, c_mask, dt_col], dim=-1)

        delta_E = self.residual_mlp(state_tuple)
        j_ref, d_ref, _, _ = self.base_lcf.decode_lcf(E_t + delta_E)
        return j0, j_ref, d_ref

# =============================================================================
# 5. LOSS EVALUATOR & 3-SPLIT BENCHMARK HARNESS
# =============================================================================
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

eval_engine = MetricLossEvaluator()

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

CONFIGS = [
    ("1. Single-Pass LCF Baseline (E_t -> LCF)",                  "base"),
    ("2. Closed-Loop K+R Refiner (E_t + K_t + R_t -> ΔLCF)",      "kr_static"),
    ("3. State + Dual-Edge K+R Tracker (E_t + ΔE + K + R + ΔK)",  "kr_temporal"),
]

res_mpjpe = {c[0]: [] for c in CONFIGS}
res_df3 = {c[0]: [] for c in CONFIGS}
res_tips = {c[0]: [] for c in CONFIGS}
res_distal = {c[0]: [] for c in CONFIGS}

print("\n" + "=" * 112)
print(" EXPERIMENT G: BRIDGING hm_ex (K + R RELATIONSHIP & STATE RESIDUAL) TO REAL RE:INTERHAND")
print("=" * 112)

for exp_name, mode in CONFIGS:
    print(f"\n>>> Running {exp_name} <<<")
    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        if mode == "base":
            model = BaseLCFNetwork().to(DEVICE)
        elif mode == "kr_static":
            model = ClosedLoopKRRefiner(use_temporal_state=False).to(DEVICE)
        else:
            model = ClosedLoopKRRefiner(use_temporal_state=True).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]
            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                g_t = gpu_extractor(vram_images[b_idx].float())
                y_batch = vram_targets[b_idx].view(-1, 21, 3)

                optimizer.zero_grad()
                if mode == "base":
                    pred_j, _, _ = model(g_t)
                    loss = eval_engine(pred_j, y_batch)
                elif mode == "kr_static":
                    j0, j_ref, _ = model(g_t)
                    # Deep supervision on both coarse Stage-0 and refined Stage-1
                    loss = 0.4 * eval_engine(j0, y_batch) + 0.6 * eval_engine(j_ref, y_batch)
                else:
                    p_idx = vram_prev1[b_idx]
                    g_prev = gpu_extractor(vram_images[p_idx].float())
                    c_flag = vram_contig_flag[b_idx]
                    dt_s = vram_dt_scale[b_idx]
                    j0, j_ref, _ = model(g_t, g_prev, c_flag, dt_s)
                    loss = 0.4 * eval_engine(j0, y_batch) + 0.6 * eval_engine(j_ref, y_batch)

                loss.backward()
                optimizer.step()

        model.eval()
        val_preds, val_gts, val_dfs = [], [], []
        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                g_t = gpu_extractor(vram_images[b_idx].float())
                if mode == "base":
                    pj, _, _ = model(g_t)
                elif mode == "kr_static":
                    _, pj, _ = model(g_t)
                else:
                    p_idx = vram_prev1[b_idx]
                    g_prev = gpu_extractor(vram_images[p_idx].float())
                    c_flag = vram_contig_flag[b_idx]
                    dt_s = vram_dt_scale[b_idx]
                    _, pj, _ = model(g_t, g_prev, c_flag, dt_s)

                val_preds.append(pj.cpu().numpy())
                val_gts.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())
                val_dfs.append(delta_f_arr[b_idx.cpu().numpy()])

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)
        all_df = np.concatenate(val_dfs, axis=0)

        mpjpe = np.linalg.norm(all_p - all_g, axis=-1).mean()
        df3_mask = all_df <= 3.5
        mpjpe_df3 = np.linalg.norm(all_p[df3_mask] - all_g[df3_mask], axis=-1).mean()
        tip_err = np.linalg.norm(all_p[:, FINGERTIP_JOINTS] - all_g[:, FINGERTIP_JOINTS], axis=-1).mean()

        b_p = all_p[:, CHILD_INDICES.cpu().numpy()] - all_p[:, PARENT_INDICES.cpu().numpy()]
        b_g = all_g[:, CHILD_INDICES.cpu().numpy()] - all_g[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()

        res_mpjpe[exp_name].append(mpjpe)
        res_df3[exp_name].append(mpjpe_df3)
        res_tips[exp_name].append(tip_err)
        res_distal[exp_name].append(distal_deg)

        print(f"  [Seed {seed:<3}] All MPJPE: {mpjpe:5.2f} mm | Adj(df=3): {mpjpe_df3:5.2f} mm | "
              f"Fingertips: {tip_err:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

        del model, optimizer
        torch.cuda.empty_cache()

print("\n" + "=" * 116)
print(" EXPERIMENT G SUMMARY: CLOSED-LOOP (K + R) RELATIONSHIP & STATE REFINEMENT ACROSS 3 SPLITS")
print("=" * 116)
print(f"{'Model Configuration':<58} | {'All MPJPE':<15} | {'Adj(df=3) MPJPE':<16} | {'Fingertips':<15} | {'Distal Dir'}")
print("-" * 116)
for exp_name, _ in CONFIGS:
    mp, sp = np.mean(res_mpjpe[exp_name]), np.std(res_mpjpe[exp_name])
    m3, s3 = np.mean(res_df3[exp_name]), np.std(res_df3[exp_name])
    mt, st = np.mean(res_tips[exp_name]), np.std(res_tips[exp_name])
    md, sd = np.mean(res_distal[exp_name]), np.std(res_distal[exp_name])
    print(f"{exp_name:<58} | {mp:5.2f}±{sp:4.2f} mm  | {m3:5.2f}±{s3:4.2f} mm   | {mt:5.2f}±{st:4.2f} mm  | {md:4.1f}°±{sd:3.1f}°")
print("=" * 116)

Hardware Accelerator: Tesla T4
Reusing 5,114 frames already resident in VRAM.

 EXPERIMENT G: BRIDGING hm_ex (K + R RELATIONSHIP & STATE RESIDUAL) TO REAL RE:INTERHAND

>>> Running 1. Single-Pass LCF Baseline (E_t -> LCF) <<<
  [Seed 42 ] All MPJPE: 32.88 mm | Adj(df=3): 33.25 mm | Fingertips: 47.72 mm | Distal Dir: 39.5°
  [Seed 101] All MPJPE: 32.41 mm | Adj(df=3): 30.97 mm | Fingertips: 50.34 mm | Distal Dir: 51.6°
  [Seed 777] All MPJPE: 28.45 mm | Adj(df=3): 29.60 mm | Fingertips: 39.39 mm | Distal Dir: 30.5°

>>> Running 2. Closed-Loop K+R Refiner (E_t + K_t + R_t -> ΔLCF) <<<
  [Seed 42 ] All MPJPE: 32.42 mm | Adj(df=3): 32.92 mm | Fingertips: 47.38 mm | Distal Dir: 40.3°
  [Seed 101] All MPJPE: 32.60 mm | Adj(df=3): 31.33 mm | Fingertips: 50.28 mm | Distal Dir: 51.3°
  [Seed 777] All MPJPE: 28.36 mm | Adj(df=3): 29.42 mm | Fingertips: 39.19 mm | Distal Dir: 31.1°

>>> Running 3. State + Dual-Edge K+R Tracker (E_t + ΔE + K + R + ΔK) <<<
  [Seed 42 ] All MPJPE: 34.31 mm | Adj(df=

In [ ]:
# @title
# ==============================================================================
# UNIFIED COLAB RUNNER: AUTO-RESOLVES /content/dataset OR GOOGLE DRIVE
# Runs Phase 0 Sanity + Phase 1 Prior-Conditioned Refiner on GPU or CPU
# ==============================================================================
import csv
import json
import os
import random
import statistics
import sys
import time
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
print(f"Active Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'})")

# ==============================================================================
# 1. BULLETPROOF DATASET & CAMERA PARAMETER RESOLUTION
# ==============================================================================
CAPTURE_ID = "m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands"

def locate_dataset_files():
    search_roots = [Path("/content/dataset"), Path("/content"), Path.cwd()]
    if Path("/content/drive/MyDrive").exists():
        search_roots.append(Path("/content/drive/MyDrive"))

    manifest_path = None
    cam_params_path = None
    file_map = {}

    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            name_low = p.name.lower()
            if name_low == "manifest.csv" and manifest_path is None:
                manifest_path = p
            elif name_low in ("cam_params.json", "cameras.json", "calibration.json") and cam_params_path is None:
                cam_params_path = p
            elif p.suffix.lower() in (".png", ".jpg", ".jpeg", ".json"):
                if p.name not in file_map:
                    file_map[p.name] = p

    if manifest_path is None:
        # Try mounting Google Drive only if manifest wasn't found locally in /content
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            for p in Path("/content/drive/MyDrive").rglob("*"):
                if not p.is_file():
                    continue
                if p.name.lower() == "manifest.csv" and manifest_path is None:
                    manifest_path = p
                elif p.name.lower() == "cam_params.json" and cam_params_path is None:
                    cam_params_path = p
                elif p.suffix.lower() in (".png", ".jpg", ".jpeg", ".json"):
                    if p.name not in file_map:
                        file_map[p.name] = p
        except Exception:
            pass

    if manifest_path is None:
        raise FileNotFoundError("Could not find manifest.csv anywhere under /content or /content/drive!")

    return manifest_path, cam_params_path, file_map

MANIFEST_PATH, CAMERA_PATH, FILE_MAP = locate_dataset_files()
BASE_DIR = MANIFEST_PATH.parent
print(f"Found Manifest : {MANIFEST_PATH}")
print(f"Indexed Files  : {len(FILE_MAP):,} images/JSONs")

def resolve_asset(rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    for cand in [
        Path(rel_clean),
        BASE_DIR / rel_clean,
        BASE_DIR.parent / rel_clean,
        BASE_DIR.parent.parent / rel_clean,
    ]:
        if cand.is_file():
            return cand
    if rel_clean.startswith("dataset/"):
        c = BASE_DIR / rel_clean[len("dataset/"):]
        if c.is_file():
            return c
    fname = Path(rel_clean).name
    if fallback_subdir:
        for cand in [BASE_DIR / fallback_subdir / fname, BASE_DIR.parent / fallback_subdir / fname]:
            if cand.is_file():
                return cand
    if fname in FILE_MAP:
        return FILE_MAP[fname]
    raise FileNotFoundError(f"Cannot resolve asset: {rel_path}")

# ==============================================================================
# 2. PHASE 0 CONSTANTS, CAMERA CALIBRATION & IN-MEMORY DATASET CACHE
# ==============================================================================
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
FINGER_CHAINS = [
    [0, 1, 2, 3, 4], [0, 5, 6, 7, 8], [0, 9, 10, 11, 12],
    [0, 13, 14, 15, 16], [0, 17, 18, 19, 20],
]

IMAGE_W, IMAGE_H = 128, 96
SEED = 20251018
TRAIN_SEGMENTS = 44
VAL_SEGMENTS = 12
OVERFIT_SAMPLES = 64
OVERFIT_MAX_EPOCHS = 250
TRAIN_EPOCHS = 20
PHASE1_EPOCHS = 25
BATCH_SIZE = 64
LR = 1.2e-3
WEIGHT_DECAY = 1e-4

def get_camera_400485_params(sample_world_pts: np.ndarray, img_w: int, img_h: int):
    """Loads cam_params.json if present; otherwise calibrates camera-frame pinhole projection."""
    if CAMERA_PATH is not None and CAMERA_PATH.is_file():
        with CAMERA_PATH.open("r", encoding="utf-8") as f:
            cam_dict = json.load(f)
        cam = cam_dict.get("400485", next(iter(cam_dict.values())))
        R = np.asarray(cam["R"], dtype=np.float32)
        t = np.asarray(cam["t"], dtype=np.float32)
        focal = np.asarray(cam["focal"], dtype=np.float32)
        princpt = np.asarray(cam["princpt"], dtype=np.float32)
        print(f"Loaded official camera 400485 calibration from {CAMERA_PATH}")
        return R, t, focal, princpt

    # Automatic pinhole calibration when running on standalone /content/dataset without cam_params.json:
    # Centers the 3D hand cluster at Z=550mm along the optical axis of the w x h image
    print("[INFO] cam_params.json not in /content/dataset; using auto-centered pinhole calibration for 400485.")
    R = np.eye(3, dtype=np.float32)
    mean_xyz = sample_world_pts.mean(axis=(0, 1)).astype(np.float32)
    t = np.array([-mean_xyz[0], -mean_xyz[1], 550.0 - mean_xyz[2]], dtype=np.float32)
    fx = float(img_w) * 1.8
    fy = float(img_h) * 1.8
    focal = np.array([fx, fy], dtype=np.float32)
    princpt = np.array([img_w * 0.5, img_h * 0.5], dtype=np.float32)
    return R, t, focal, princpt

def crop_affine(uv, width, height, margin=1.4, out_w=None, out_h=None):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    out_w = IMAGE_W if out_w is None else out_w
    out_h = IMAGE_H if out_h is None else out_h
    return np.asarray([
        [out_w / side, 0.0, -left * out_w / side],
        [0.0, out_h / side, -top * out_h / side],
    ], np.float32)

def make_channels(gray_u8: np.ndarray) -> np.ndarray:
    gray = gray_u8.astype(np.float32) / 255.0
    gray_255 = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray_255, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray_255, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.sqrt(gx * gx + gy * gy)
    return np.stack([gray, np.clip(mag, 0.0, 1.0)], axis=0).astype(np.float32)

# Self-healing RAM cache so re-running never re-reads 5,114 images from disk
if "CACHED_RECORDS" not in globals() or len(globals().get("CACHED_RECORDS", [])) == 0:
    t_load = time.perf_counter()
    with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r.get("camera_id", "400485") == "400485"]

    print(f"Ingesting {len(rows):,} frames and 3D keypoints into memory...")
    raw_world_list = []
    valid_rows = []
    first_img_shape = None

    for r in rows:
        try:
            kp_p = resolve_asset(r["orig_kp_right"], "keypoints_right")
            img_p = resolve_asset(r["image_path"], "images")
            with kp_p.open("r", encoding="utf-8") as kf:
                kp_w = np.asarray(json.load(kf), dtype=np.float32)
            if first_img_shape is None:
                im0 = cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE)
                if im0 is None:
                    continue
                first_img_shape = (im0.shape[1], im0.shape[0])
            raw_world_list.append(kp_w)
            valid_rows.append((r, img_p))
        except Exception:
            continue

    raw_world_arr = np.stack(raw_world_list, axis=0)
    IMG_ORIG_W, IMG_ORIG_H = first_img_shape
    CAM_R, CAM_T, CAM_FOCAL, CAM_PRINCPT = get_camera_400485_params(raw_world_arr, IMG_ORIG_W, IMG_ORIG_H)

    CACHED_RECORDS = []
    CACHED_GRAY_BUFFERS = []
    CACHED_BUFFER_AFFINES = []
    CACHED_ROOTS = np.empty((len(valid_rows), 3), dtype=np.float32)

    for idx, (row, img_p) in enumerate(valid_rows):
        gray = cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            continue
        h, w = gray.shape[:2]
        kp_world = raw_world_arr[idx]
        kp_cam_mm = kp_world @ CAM_R.T + CAM_T

        selected_cam = kp_cam_mm[KEEP_21_INDICES]
        z = np.maximum(selected_cam[:, 2], 1.0)
        selected_uv = np.stack([
            CAM_FOCAL[0] * selected_cam[:, 0] / z + CAM_PRINCPT[0],
            CAM_FOCAL[1] * selected_cam[:, 1] / z + CAM_PRINCPT[1],
        ], axis=1).astype(np.float32)

        # If using fallback intrinsics (no cam_params.json), crop from foreground hand silhouette
        if CAMERA_PATH is None:
            ys, xs = np.where(gray > 18)
            if len(xs) > 60:
                umin, umax = float(np.percentile(xs, 2)), float(np.percentile(xs, 98))
                vmin, vmax = float(np.percentile(ys, 2)), float(np.percentile(ys, 98))
                # Map 3D keypoints into the detected foreground bounding box
                s3_min, s3_max = selected_cam[:, :2].min(axis=0), selected_cam[:, :2].max(axis=0)
                s3_span = np.maximum(s3_max - s3_min, 10.0)
                selected_uv[:, 0] = umin + (selected_cam[:, 0] - s3_min[0]) / s3_span[0] * (umax - umin)
                selected_uv[:, 1] = vmin + (selected_cam[:, 1] - s3_min[1]) / s3_span[1] * (vmax - vmin)

        root_relative = selected_cam - selected_cam[0:1]
        affine = crop_affine(selected_uv, w, h, margin=1.4, out_w=IMAGE_W, out_h=IMAGE_H)
        crop = cv2.warpAffine(
            gray, affine, (IMAGE_W, IMAGE_H),
            flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )
        uv_crop = selected_uv @ affine[:, :2].T + affine[:, 2]

        buf_aff = crop_affine(selected_uv, w, h, margin=3.5, out_w=384, out_h=288)
        buf_gray = cv2.warpAffine(
            gray, buf_aff, (384, 288),
            flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )

        CACHED_RECORDS.append({
            "frame_id": row.get("frame_id", str(idx)),
            "segment": row["segment"],
            "uv_image": selected_uv,
            "crop_gray": crop,
            "uv_crop": uv_crop.astype(np.float32),
            "y3d": root_relative.astype(np.float32),
        })
        CACHED_GRAY_BUFFERS.append(buf_gray)
        CACHED_BUFFER_AFFINES.append(buf_aff)
        CACHED_ROOTS[len(CACHED_RECORDS) - 1] = selected_cam[0]

    CACHED_BUFFER_AFFINES = np.stack(CACHED_BUFFER_AFFINES)
    CACHED_ROOTS = CACHED_ROOTS[:len(CACHED_RECORDS)]
    print(f"Staged {len(CACHED_RECORDS):,} frames into memory in {time.perf_counter() - t_load:.1f}s.")
else:
    print(f"Reusing {len(CACHED_RECORDS):,} frames already cached in memory.")

# ==============================================================================
# 3. MODELS & HELPERS (PHASE 0 LCF + PHASE 1 POSE REFINER)
# ==============================================================================
def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    batch = bone_vectors.shape[0]
    joints = torch.zeros((batch, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + bone_vectors[:, idx]
    return joints

class WebcamAlignedLCFTracker(nn.Module):
    def __init__(self, canonical_lens: torch.Tensor):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512), nn.ReLU(inplace=True), nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.proj2d_head = nn.Linear(512, 21 * 2)
        self.register_buffer("bone_lens", canonical_lens.clone())

    def forward(self, x: torch.Tensor):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).reshape(batch, 5, 3), dim=-1, eps=1e-7)
        latent_frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).reshape(batch, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(latent_frame, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((batch, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for finger_idx in range(4):
            bone_offset = finger_offsets[finger_idx]
            phalanx_offset = 3 + finger_idx * 3
            bone_dirs[:, bone_offset] = meta_cam_u[:, finger_idx + 1]
            bone_dirs[:, bone_offset + 1:bone_offset + 4] = phalanx_cam_u[:, phalanx_offset:phalanx_offset + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        kp2d = torch.tanh(self.proj2d_head(feat).reshape(batch, 21, 2))
        return joints, bone_dirs, kp2d

class PoseRefiner(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512), nn.ReLU(inplace=True), nn.Dropout(0.1)
        )
        self.delta_head = nn.Linear(512, 21 * 3)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        return self.delta_head(feat).reshape(-1, 21, 3)

class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        l_pos = torch.norm(pred - target, dim=-1).mean()
        pred_bones = pred[:, [c for _, c in BONE_PAIRS]] - pred[:, [p for p, _ in BONE_PAIRS]]
        target_bones = target[:, [c for _, c in BONE_PAIRS]] - target[:, [p for p, _ in BONE_PAIRS]]
        pred_len = torch.norm(pred_bones, dim=-1)
        target_len = torch.norm(target_bones, dim=-1)
        l_len = F.l1_loss(pred_len, target_len)
        eps = 1e-7
        pred_u = pred_bones / (pred_len.unsqueeze(-1) + eps)
        target_u = target_bones / (target_len.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((pred_u * target_u).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

def rotate_camera_xy(y3d: np.ndarray, angle_rad: float) -> np.ndarray:
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    out = y3d.copy()
    x, y = y3d[:, 0].copy(), y3d[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out

def augment_one(gray_u8: np.ndarray, uv: np.ndarray, y3d: np.ndarray, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    shift_x = float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    shift_y = float(rng.uniform(-0.10, 0.10) * IMAGE_H)
    center = (IMAGE_W * 0.5, IMAGE_H * 0.5)
    matrix = cv2.getRotationMatrix2D(center, angle_deg, scale).astype(np.float32)
    matrix[0, 2] += shift_x
    matrix[1, 2] += shift_y

    warped = cv2.warpAffine(
        gray_u8, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(warped.astype(np.float32) * brightness + offset, 0, 255).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle)
    y2d_aug = np.empty_like(uv_aug, dtype=np.float32)
    y2d_aug[:, 0] = 2.0 * uv_aug[:, 0] / IMAGE_W - 1.0
    y2d_aug[:, 1] = 2.0 * uv_aug[:, 1] / IMAGE_H - 1.0
    return make_channels(warped), y3d_aug, y2d_aug

def normalize_crop_points(uv: np.ndarray) -> np.ndarray:
    out = np.empty_like(uv, dtype=np.float32)
    out[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    out[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return out

def score(pred: np.ndarray, target: np.ndarray) -> tuple[float, list[float]]:
    distance = np.linalg.norm(pred - target, axis=-1)
    return float(distance.mean()), [float(x) for x in distance[:, FINGERTIP_JOINTS].mean(axis=0)]

def predict_metrics(model: nn.Module, x: torch.Tensor, target: np.ndarray):
    preds = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(x), 128):
            pred, _, _ = model(x[start:start + 128].to(DEVICE))
            preds.append(pred.cpu().numpy())
    return score(np.concatenate(preds, axis=0), target)

def train_tracker(model, train_records, train_indices, train_y3d, train_y2d, x_base, epochs, augment, val_x=None, val_y3d=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1), eta_min=1e-4)
    criterion = MetricLossEvaluator()
    rng = np.random.default_rng(SEED + (1 if augment else 2))
    val_history = []
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        order = np.random.permutation(len(train_indices))
        for offset in range(0, len(order), BATCH_SIZE):
            local_batch = order[offset:offset + BATCH_SIZE]
            indices = train_indices[local_batch]
            if augment:
                batch = [
                    augment_one(train_records[int(i)]["crop_gray"], train_records[int(i)]["uv_crop"], train_y3d[local_pos], rng)
                    for local_pos, i in zip(local_batch, indices)
                ]
                bx = torch.from_numpy(np.stack([item[0] for item in batch])).to(DEVICE)
                by3d = torch.from_numpy(np.stack([item[1] for item in batch])).to(DEVICE)
                by2d = torch.from_numpy(np.stack([item[2] for item in batch])).to(DEVICE)
            else:
                bx = x_base[indices].to(DEVICE)
                by3d = torch.from_numpy(train_y3d[local_batch]).to(DEVICE)
                by2d = torch.from_numpy(train_y2d[local_batch]).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred3d, _, pred2d = model(bx)
            loss = criterion(pred3d, by3d) + 15.0 * F.smooth_l1_loss(pred2d, by2d)
            loss.backward()
            optimizer.step()

        scheduler.step()
        if val_x is not None and val_y3d is not None and epoch % 2 == 0:
            mpjpe, tips = predict_metrics(model, val_x, val_y3d)
            val_history.append((epoch, mpjpe, tips))
        if not augment:
            train_mpjpe, _ = predict_metrics(model, x_base[train_indices], train_y3d)
            if train_mpjpe < 3.0:
                break
    return val_history, time.perf_counter() - started

# ==============================================================================
# 4. EXECUTE PHASE 0 SANITY + PHASE 1 PRIOR-CONDITIONED REFINEMENT
# ==============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

segments = sorted({r["segment"] for r in CACHED_RECORDS})
split_rng = np.random.default_rng(SEED)
shuffled = np.asarray(segments, dtype=object)
split_rng.shuffle(shuffled)
train_segments = set(shuffled[:TRAIN_SEGMENTS].tolist())
val_segments = set(shuffled[TRAIN_SEGMENTS:].tolist())

train_idx = np.asarray([i for i, r in enumerate(CACHED_RECORDS) if r["segment"] in train_segments], dtype=np.int64)
val_idx = np.asarray([i for i, r in enumerate(CACHED_RECORDS) if r["segment"] in val_segments], dtype=np.int64)

x_all = torch.from_numpy(np.stack([make_channels(r["crop_gray"]) for r in CACHED_RECORDS]))
y_all = np.stack([r["y3d"] for r in CACHED_RECORDS]).astype(np.float32)
y2d_all = np.stack([normalize_crop_points(r["uv_crop"]) for r in CACHED_RECORDS]).astype(np.float32)

train_y3d, val_y3d = y_all[train_idx], y_all[val_idx]
train_y2d = y2d_all[train_idx]
val_x = x_all[val_idx]

print("\n" + "=" * 95)
print(" PHASE 0: BASELINES, OVERFIT-64 DIAGNOSTIC & 2-CH WEBCAM-ALIGNED LCF")
print("=" * 95)

mean_pose = train_y3d.mean(axis=0, keepdims=True)
mean_mpjpe, mean_tips = score(np.repeat(mean_pose, len(val_idx), axis=0), val_y3d)

with torch.inference_mode():
    low_res = F.adaptive_avg_pool2d(x_all.to(DEVICE), (12, 16)).flatten(1)
    low_train, low_val = low_res[train_idx], low_res[val_idx]
    norm_train = (low_train * low_train).sum(dim=1)
    nn_indices = []
    for start in range(0, len(low_val), 64):
        chunk = low_val[start:start + 64]
        dists = (chunk * chunk).sum(dim=1, keepdim=True) + norm_train.unsqueeze(0) - 2.0 * (chunk @ low_train.T)
        nn_indices.extend(torch.argmin(dists, dim=1).tolist())
nn_mpjpe, nn_tips = score(train_y3d[np.asarray(nn_indices, dtype=np.int64)], val_y3d)

canonical_lens = torch.from_numpy(np.stack([
    np.linalg.norm(train_y3d[:, c] - train_y3d[:, p], axis=1).mean() for p, c in BONE_PAIRS
]).astype(np.float32))

overfit_rng = np.random.default_rng(SEED + 64)
overfit_local = overfit_rng.choice(len(train_idx), size=min(OVERFIT_SAMPLES, len(train_idx)), replace=False)
overfit_indices = train_idx[overfit_local]
overfit_model = WebcamAlignedLCFTracker(canonical_lens).to(DEVICE)
local_records = [CACHED_RECORDS[i] for i in overfit_indices]
_, overfit_sec = train_tracker(
    overfit_model, local_records, np.arange(len(overfit_indices), dtype=np.int64),
    y_all[overfit_indices], y2d_all[overfit_indices], x_all[overfit_indices],
    epochs=OVERFIT_MAX_EPOCHS, augment=False,
)
overfit_mpjpe, overfit_tips = predict_metrics(overfit_model, x_all[overfit_indices], y_all[overfit_indices])
overfit_status = "PASS" if overfit_mpjpe < 3.0 else "FAIL"

direct = WebcamAlignedLCFTracker(canonical_lens).to(DEVICE)
val_history, train_sec = train_tracker(
    direct, CACHED_RECORDS, train_idx, train_y3d, train_y2d, x_all,
    epochs=TRAIN_EPOCHS, augment=True, val_x=val_x, val_y3d=val_y3d,
)

def metric_line(method, split, mpjpe, tips):
    tip_text = " ".join(f"{name}={val:6.2f}" for name, val in zip(FINGER_NAMES, tips))
    return f"{method:<22} {split:<12} {mpjpe:8.2f} mm | {tip_text}"

print(metric_line("Mean pose", "val", mean_mpjpe, mean_tips))
print(metric_line("Nearest neighbor", "val", nn_mpjpe, nn_tips))
print(metric_line(f"Overfit64 ({overfit_status})", "train", overfit_mpjpe, overfit_tips))
for ep, mpjpe, tips in val_history:
    print(metric_line("WebcamAlignedLCF", f"epoch {ep:02d}", mpjpe, tips))

# ==============================================================================
# 5. PHASE 1: PRIOR-CONDITIONED POSE REFINER
# ==============================================================================
print("\n" + "=" * 95)
print(" PHASE 1: PRIOR-CONDITIONED ITERATIVE POSE REFINEMENT")
print("=" * 95)

def project_pts(points_cam, focal, principal):
    z = np.maximum(points_cam[:, 2], 1.0)
    return np.stack((focal[0] * points_cam[:, 0] / z + principal[0],
                     focal[1] * points_cam[:, 1] / z + principal[1]), axis=1).astype(np.float32)

def rotation_matrix(axis, angle):
    axis = np.asarray(axis, np.float64)
    axis /= max(np.linalg.norm(axis), 1e-12)
    x, y, z = axis
    c, s, d = np.cos(angle), np.sin(angle), 1.0 - np.cos(angle)
    return np.asarray([
        [c + x*x*d, x*y*d - z*s, x*z*d + y*s],
        [y*x*d + z*s, c + y*y*d, y*z*d - x*s],
        [z*x*d - y*s, z*y*d + x*s, c + z*z*d],
    ], np.float32)

def synthesize_prior(gt, rng, severity=None):
    s = float(rng.uniform(0.2, 1.0) if severity is None else severity)
    prior = gt.copy()
    for chain in FINGER_CHAINS:
        for hinge_pos in range(len(chain) - 1):
            hinge = chain[hinge_pos]
            subtree = chain[hinge_pos + 1:]
            mat = rotation_matrix(rng.normal(size=3), np.deg2rad(rng.uniform(-25.0, 25.0) * s))
            prior[subtree] = (prior[subtree] - prior[hinge]) @ mat.T + prior[hinge]
    global_rot = rotation_matrix(rng.normal(size=3), np.deg2rad(rng.uniform(-10, 10)))
    prior = prior @ global_rot.T
    prior += rng.normal(0.0, 8.0, size=(1, 3)).astype(np.float32)
    return prior.astype(np.float32)

def channels_from_prior(gray, uv, affine):
    crop = cv2.warpAffine(gray, affine, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    xy = uv @ affine[:, :2].T + affine[:, 2]
    lines = np.zeros((IMAGE_H, IMAGE_W), np.uint8)
    dots = np.zeros_like(lines)
    points = np.rint(xy).astype(int)
    for parent, child in BONE_PAIRS:
        cv2.line(lines, tuple(points[parent]), tuple(points[child]), 255, 2, cv2.LINE_AA)
    for point in points:
        cv2.circle(dots, tuple(point), 3, 255, -1, cv2.LINE_AA)
    base = make_channels(crop)
    return np.concatenate((base, lines[None].astype(np.float32) / 255.0, dots[None].astype(np.float32) / 255.0), axis=0), xy

def pose_input(index, prior):
    uv_img = project_pts(CACHED_ROOTS[index:index+1] + prior, CAM_FOCAL, CAM_PRINCPT)
    affine = crop_affine(uv_img, IMG_ORIG_W, IMG_ORIG_H)
    src = np.eye(3, dtype=np.float32); src[:2] = CACHED_BUFFER_AFFINES[index]
    dst = np.eye(3, dtype=np.float32); dst[:2] = affine
    mapping = dst @ np.linalg.inv(src)
    prior_gray = cv2.warpAffine(CACHED_GRAY_BUFFERS[index], mapping[:2], (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    inp, uv_crop = channels_from_prior(prior_gray, uv_img, affine)
    return inp, uv_crop, affine

def make_sample(index, prior, aug_rng=None):
    inp, uv_crop, affine = pose_input(index, prior)
    gt = CACHED_RECORDS[index]["y3d"]
    if aug_rng is None:
        return inp, prior.astype(np.float32), gt, normalize_crop_points(uv_crop), CACHED_ROOTS[index], affine, np.asarray([[1, 0, 0], [0, 1, 0]], np.float32)
    gray_crop = np.clip(inp[0] * 255.0, 0, 255).astype(np.uint8)
    gt_uv_crop = CACHED_RECORDS[index]["uv_image"] @ affine[:, :2].T + affine[:, 2]
    base, gt_aug, uv_norm = augment_one(gray_crop, gt_uv_crop, gt, aug_rng)
    uv_aug = uv_norm.copy()
    uv_aug[:, 0] = (uv_aug[:, 0] + 1) * IMAGE_W * 0.5
    uv_aug[:, 1] = (uv_aug[:, 1] + 1) * IMAGE_H * 0.5
    design = np.column_stack((gt_uv_crop, np.ones(len(gt_uv_crop))))
    affine_t, *_ = np.linalg.lstsq(design, uv_aug, rcond=None)
    mat = affine_t.T.astype(np.float32)
    angle = float(np.arctan2(mat[0, 1], mat[0, 0]))
    prior_aug = rotate_camera_xy(prior, angle).astype(np.float32)
    lines = cv2.warpAffine(inp[2], mat, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    dots = cv2.warpAffine(inp[3], mat, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    return np.concatenate((base, lines[None], dots[None]), axis=0), prior_aug, gt_aug, uv_norm, CACHED_ROOTS[index], affine, mat

refiner = PoseRefiner().to(DEVICE)
optimizer = torch.optim.AdamW(refiner.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS, eta_min=1e-4)
rng = np.random.default_rng(SEED + 100)
focal_t = torch.from_numpy(CAM_FOCAL).to(DEVICE)
princ_t = torch.from_numpy(CAM_PRINCPT).to(DEVICE)

for epoch in range(1, PHASE1_EPOCHS + 1):
    refiner.train()
    order = np.random.permutation(train_idx)
    for start in range(0, len(order), BATCH_SIZE):
        ids = order[start:start+BATCH_SIZE]
        samples = [make_sample(int(idx), synthesize_prior(y_all[idx], rng), aug_rng=rng) for idx in ids]
        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        pri = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        tgt = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)
        uv_tgt = torch.from_numpy(np.stack([s[3] for s in samples])).to(DEVICE)
        roots_b = torch.from_numpy(np.stack([s[4] for s in samples])).to(DEVICE)
        crop_aff = torch.from_numpy(np.stack([s[5] for s in samples])).to(DEVICE)
        aug_aff = torch.from_numpy(np.stack([s[6] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        corrected = pri + refiner(bx)
        loss3d = F.l1_loss(corrected, tgt)

        angles = torch.atan2(aug_aff[:, 0, 1], aug_aff[:, 0, 0])
        c, s = torch.cos(angles)[:, None], torch.sin(angles)[:, None]
        unrot = corrected.clone()
        x, y = corrected[..., 0], corrected[..., 1]
        unrot[..., 0] = c * x - s * y
        unrot[..., 1] = s * x + c * y
        camera_points = unrot + roots_b[:, None, :]
        z = camera_points[..., 2].clamp_min(1.0)
        uv = torch.stack((focal_t[0] * camera_points[..., 0] / z + princ_t[0],
                          focal_t[1] * camera_points[..., 1] / z + princ_t[1]), dim=-1)
        uv_h = torch.cat((uv, torch.ones_like(uv[..., :1])), dim=-1)
        crop_h = torch.bmm(uv_h, crop_aff.transpose(1, 2))
        aug_h = torch.cat((crop_h, torch.ones_like(crop_h[..., :1])), dim=-1)
        pred_crop = torch.bmm(aug_h, aug_aff.transpose(1, 2))
        pred_norm = pred_crop.clone()
        pred_norm[..., 0] = 2.0 * pred_crop[..., 0] / IMAGE_W - 1.0
        pred_norm[..., 1] = 2.0 * pred_crop[..., 1] / IMAGE_H - 1.0
        loss2d = F.smooth_l1_loss(pred_norm, uv_tgt)
        loss = loss3d + 15.0 * loss2d
        loss.backward()
        optimizer.step()
    scheduler.step()

refiner.eval()
direct.eval()

def run_refiner_step(inp, prior):
    with torch.inference_mode():
        delta = refiner(torch.from_numpy(inp[None]).to(DEVICE))[0].cpu().numpy()
    return (prior + delta).astype(np.float32)

for severity in (0.3, 0.6, 1.0):
    rng_eval = np.random.default_rng(SEED + int(severity * 1000))
    before, after = [], []
    for idx in val_idx:
        idx = int(idx)
        prior = synthesize_prior(y_all[idx], rng_eval, severity)
        inp, _, _ = pose_input(idx, prior)
        before.append(prior)
        after.append(run_refiner_step(inp, prior))
    bscore, _ = score(np.stack(before), val_y3d)
    ascore, tips = score(np.stack(after), val_y3d)
    tip_str = " ".join(f"{n}={v:.2f}" for n, v in zip(FINGER_NAMES, tips))
    print(f"Synthetic s={severity:.1f}          held-out | After: {ascore:6.2f} mm (Before: {bscore:6.2f} mm) | {tip_str}")

with torch.inference_mode():
    priors = np.stack([direct(x_all[int(idx):int(idx)+1].to(DEVICE))[0][0].cpu().numpy() for idx in val_idx])

for iteration in range(4):
    current = priors.copy() if iteration == 0 else refined_stack
    scores, tips = score(current, val_y3d)
    tip_str = " ".join(f"{n}={v:.2f}" for n, v in zip(FINGER_NAMES, tips))
    print(f"Realistic iter {iteration}         held-out | After: {scores:6.2f} mm                     | {tip_str}")
    if iteration < 3:
        refined_stack = np.stack([run_refiner_step(pose_input(int(idx), pr)[0], pr) for idx, pr in zip(val_idx, current)])

# Save combined checkpoint for local live webcam studio
torch.save({
    "direct_state_dict": direct.cpu().state_dict(),
    "refiner_state_dict": refiner.cpu().state_dict(),
    "bone_lens": canonical_lens.cpu(),
}, "/content/phase1_combined_tracker.pt")
print("\nSaved combined Phase 0 + Phase 1 checkpoint to /content/phase1_combined_tracker.pt")

Active Compute Device: cuda (Tesla T4)
Found Manifest : /content/dataset/manifest.csv
Indexed Files  : 20,467 images/JSONs
Ingesting 5,114 frames and 3D keypoints into memory...
Loaded official camera 400485 calibration from /content/dataset/cam_params.json


KeyboardInterrupt: 

In [ ]:
# @title
# =============================================================================
# SELF-CONTAINED COLAB RUNNER
# - Works after a Colab runtime/session reset
# - Resolves flattened keypoint folders such as keypoints_right/
# - Loads and evaluates an older Phase 0/1 checkpoint when available
# - Trains segment-wise out-of-fold direct models
# - Trains and evaluates the new uncertainty-gated refiner
# =============================================================================

import csv
import json
import os
import random
import statistics
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------ Configuration -------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
CHECKPOINT_NAMES = (
    "phase1_combined_tracker.pt",
    "phase0_tracker.pt",
    "phase1_refiner.pt",
)

SEED = 20251018
TRAIN_SEGMENTS, VAL_SEGMENTS = 44, 12
N_FOLDS = 4
TRAIN_EPOCHS = 20
REFINER_EPOCHS = 25
BATCH_SIZE = 64
LR, WEIGHT_DECAY = 1.2e-3, 1e-4
IMAGE_W, IMAGE_H = 128, 96

print(
    f"Active compute device: {DEVICE} "
    f"({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'})"
)

# ----------------------------- Locate the data -------------------------------

def manifests_under(root):
    if not root.exists():
        return []
    return list(root.rglob("manifest.csv"))

def locate_manifest():
    for root in (DATASET_DIR, Path("/content"), Path.cwd()):
        hits = manifests_under(root)
        if hits:
            return next(
                (p for p in hits if "mugsy_subset" in str(p).lower()),
                hits[0],
            )

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped or unavailable: {exc}")

    hits = manifests_under(DRIVE_ROOT)
    if hits:
        return next(
            (p for p in hits if "mugsy_subset" in str(p).lower()),
            hits[0],
        )

    raise FileNotFoundError(
        "Could not find manifest.csv under /content or Google Drive. "
        "Set DATASET_DIR to the extracted dataset folder and rerun."
    )

MANIFEST_PATH = locate_manifest()
SUBSET_DIR = MANIFEST_PATH.parent

def find_capture_root(subset_dir):
    for parent in (subset_dir, *subset_dir.parents):
        if (
            (parent / "Mugsy_cameras" / "cam_params.json").is_file()
            or (parent / "orig_fits").is_dir()
            or (parent / "keypoints_right").is_dir()
        ):
            return parent
    return subset_dir

CAPTURE_DIR = find_capture_root(SUBSET_DIR)
CAMERA_PATH = CAPTURE_DIR / "Mugsy_cameras" / "cam_params.json"

if not CAMERA_PATH.is_file():
    for root in (SUBSET_DIR.parent, SUBSET_DIR.parent.parent, Path("/content")):
        if root.exists():
            hits = list(root.rglob("cam_params.json"))
            if hits:
                CAMERA_PATH = next(
                    (p for p in hits if p.parent.name.lower() == "mugsy_cameras"),
                    hits[0],
                )
                break

if not CAMERA_PATH.is_file():
    # The extracted subset may store the calibration file directly beside the
    # manifest rather than in Mugsy_cameras/.
    nearby = [
        SUBSET_DIR / "cam_params.json",
        SUBSET_DIR.parent / "cam_params.json",
        SUBSET_DIR.parent.parent / "cam_params.json",
    ]
    CAMERA_PATH = next((p for p in nearby if p.is_file()), CAMERA_PATH)

if not CAMERA_PATH.is_file():
    raise FileNotFoundError(
        "Could not locate cam_params.json. Set CAMERA_PATH manually."
    )

print(f"Manifest:     {MANIFEST_PATH}")
print(f"Capture root: {CAPTURE_DIR}")
print(f"Calibration:  {CAMERA_PATH}")

with MANIFEST_PATH.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))

rows = [r for r in all_rows if r.get("camera_id", "400485") == "400485"]
if not rows:
    raise RuntimeError("The manifest has no camera_id=400485 rows.")

# Build a filename index once. It may contain duplicate entries because
# overlapping search roots can index the same path; deduplicate them below.
ASSET_INDEX = {}
for root in dict.fromkeys((CAPTURE_DIR, SUBSET_DIR, SUBSET_DIR.parent)):
    if not root.exists():
        continue
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in {
            ".json", ".png", ".jpg", ".jpeg"
        }:
            ASSET_INDEX.setdefault(path.name.lower(), set()).add(
                path.resolve()
            )

def resolve_asset(rel):
    rel = str(rel).replace("\\", "/").strip()
    rel_path = Path(rel)
    parts = rel_path.parts

    # First try the manifest path literally and relative to likely roots.
    candidates = [
        Path(rel),
        CAPTURE_DIR / rel,
        SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel,
        SUBSET_DIR.parent.parent / rel,
    ]
    if parts and parts[0] == CAPTURE_DIR.name:
        candidates.append(CAPTURE_DIR.joinpath(*parts[1:]))

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    # Try the original relative path against each ancestor of the manifest.
    for ancestor in (SUBSET_DIR, *SUBSET_DIR.parents):
        candidate = ancestor.joinpath(*parts)
        if candidate.is_file():
            return candidate.resolve()

    # Flattened export fallback:
    #   orig_fits/right/Keypoints/foo.json -> keypoints_right/foo.json
    #   orig_fits/left/Keypoints/foo.json  -> keypoints_left/foo.json
    side = None
    lower_parts = [p.lower() for p in parts]
    if "right" in lower_parts:
        side = "right"
    elif "left" in lower_parts:
        side = "left"

    basename = rel_path.name.lower()
    hits = ASSET_INDEX.get(basename, set())

    if side is not None:
        side_folder = f"keypoints_{side}"
        side_hits = {
            p for p in hits
            if side_folder in [part.lower() for part in p.parts]
        }
        if len(side_hits) == 1:
            return next(iter(side_hits))
        if len(side_hits) > 1:
            # Prefer the exact flattened folder directly under the capture root.
            exact = {
                p for p in side_hits
                if p.parent.name.lower() == side_folder
            }
            if len(exact) == 1:
                return next(iter(exact))
            raise FileNotFoundError(
                f"Multiple {side} keypoint matches for {rel}: "
                f"{[str(p) for p in list(side_hits)[:5]]}"
            )

    if len(hits) == 1:
        return next(iter(hits))
    if len(hits) > 1:
        raise FileNotFoundError(
            f"Asset basename is ambiguous: {rel}\n"
            f"Matches: {[str(p) for p in list(hits)[:8]]}"
        )

    raise FileNotFoundError(
        f"Could not resolve manifest asset: {rel}\n"
        f"Capture root searched: {CAPTURE_DIR}"
    )

with CAMERA_PATH.open(encoding="utf-8") as f:
    camera_data = json.load(f)

if "400485" not in camera_data:
    raise KeyError(f"Camera 400485 not found in {CAMERA_PATH}")

camera = camera_data["400485"]
CAM_R = np.asarray(camera["R"], np.float32)
CAM_T = np.asarray(camera["t"], np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], np.float32).reshape(2)

# -------------------------- Shared model/data setup ---------------------------

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
    12, 13, 14, 15, 16, 17, 19, 20, 21, 22,
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]

def project(points):
    points = np.asarray(points, np.float32)
    z = points[..., 2]
    if np.any(z <= 1.0):
        raise ValueError("A projected hand joint has invalid camera depth.")
    return np.stack(
        (
            CAM_FOCAL[0] * points[..., 0] / z + CAM_PRINCPT[0],
            CAM_FOCAL[1] * points[..., 1] / z + CAM_PRINCPT[1],
        ),
        axis=-1,
    ).astype(np.float32)

def crop_affine(uv, margin=1.4, out_w=IMAGE_W, out_h=IMAGE_H):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        np.float32,
    )

def make_channels(gray):
    gray_f = gray.astype(np.float32) / 255.0
    gray_255 = gray.astype(np.float32)
    gx = cv2.Sobel(gray_255, cv2.CV_32F, 1, 0, ksize=3, scale=1 / 1020)
    gy = cv2.Sobel(gray_255, cv2.CV_32F, 0, 1, ksize=3, scale=1 / 1020)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return np.stack((gray_f, mag)).astype(np.float32)

def normalize_uv(uv):
    out = np.empty_like(uv, dtype=np.float32)
    out[..., 0] = 2.0 * uv[..., 0] / IMAGE_W - 1.0
    out[..., 1] = 2.0 * uv[..., 1] / IMAGE_H - 1.0
    return out

def rotate_camera_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x, y = points[..., 0].copy(), points[..., 1].copy()
    out[..., 0] = c * x + s * y
    out[..., 1] = -s * x + c * y
    return out

def augment_one(gray, uv, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    shift_x = float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    shift_y = float(rng.uniform(-0.10, 0.10) * IMAGE_H)

    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += shift_x
    matrix[1, 2] += shift_y

    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset, 0, 255
    ).astype(np.uint8)

    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle)
    return make_channels(warped), y3d_aug, normalize_uv(uv_aug)

class WebcamAlignedLCFTracker(nn.Module):
    def __init__(self, canonical_lens):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer("bone_lens", canonical_lens.clone())

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3), dim=-1, eps=1e-7
        )

        d6 = self.latent_frame_head(feat)
        a1, a2 = d6[:, :3], d6[:, 3:]
        b1 = F.normalize(a1, dim=-1, eps=1e-7)
        b2 = F.normalize(
            a2 - (b1 * a2).sum(-1, keepdim=True) * b1,
            dim=-1,
            eps=1e-7,
        )
        frame = torch.stack(
            (b1, b2, torch.cross(b1, b2, dim=-1)), dim=-1
        )
        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1,
            eps=1e-7,
        )
        phalanx = torch.bmm(frame, phalanx.transpose(1, 2)).transpose(1, 2)

        dirs = torch.zeros((batch, 20, 3), dtype=x.dtype, device=x.device)
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger, offset in enumerate((4, 8, 12, 16)):
            p_offset = 3 + finger * 3
            dirs[:, offset] = meta[:, finger + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[:, p_offset:p_offset + 3]

        joints = torch.zeros((batch, 21, 3), dtype=x.dtype, device=x.device)
        for k, (parent, child) in enumerate(BONE_PAIRS):
            joints[:, child] = (
                joints[:, parent] + dirs[:, k] * self.bone_lens[k]
            )
        uv = torch.tanh(self.proj2d_head(feat).reshape(batch, 21, 2))
        return joints, dirs, uv

def direct_loss(pred3, target3, pred2, target2):
    parents = [p for p, _ in BONE_PAIRS]
    children = [c for _, c in BONE_PAIRS]
    pred_bones = pred3[:, children] - pred3[:, parents]
    target_bones = target3[:, children] - target3[:, parents]
    pred_u = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_u = F.normalize(target_bones, dim=-1, eps=1e-7)
    direction_loss = (1.0 - (pred_u * target_u).sum(-1)).mean()
    return (
        1.5 * F.l1_loss(pred3, target3)
        + 20.0 * direction_loss
        + 15.0 * F.smooth_l1_loss(pred2, target2)
    )

# -------------------------- Read and cache the data --------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Loading {len(rows):,} camera-400485 rows...")
records, roots, gray_buffers, buffer_affines = [], [], [], []
image_width = image_height = None
missing = []

for i, row in enumerate(rows):
    try:
        kp_path = resolve_asset(row["orig_kp_right"])
        image_path = resolve_asset(row["image_path"])
    except FileNotFoundError as exc:
        missing.append((i, str(exc)))
        if len(missing) >= 5:
            break
        continue

    with kp_path.open(encoding="utf-8") as f:
        world = np.asarray(json.load(f), dtype=np.float32)

    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"OpenCV could not read image: {image_path}")

    h, w = gray.shape
    if image_width is None:
        image_width, image_height = w, h
    elif (w, h) != (image_width, image_height):
        raise ValueError("Dataset contains inconsistent image dimensions.")

    camera_pts = world @ CAM_R.T + CAM_T
    selected = camera_pts[KEEP_21_INDICES]
    uv_image = project(selected)
    y3d = selected - selected[0:1]

    affine = crop_affine(uv_image, margin=1.4)
    crop_gray = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    uv_crop = uv_image @ affine[:, :2].T + affine[:, 2]

    buffer_affine = crop_affine(
        uv_image, margin=3.5, out_w=384, out_h=288
    )
    gray_buffer = cv2.warpAffine(
        gray, buffer_affine, (384, 288),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    records.append({
        "frame_id": row.get("frame_id", str(i)),
        "segment": row["segment"],
        "uv_image": uv_image,
        "uv_crop": uv_crop.astype(np.float32),
        "crop_gray": crop_gray,
        "y3d": y3d.astype(np.float32),
    })
    roots.append(selected[0].copy())
    gray_buffers.append(gray_buffer)
    buffer_affines.append(buffer_affine)

if missing:
    details = "\n".join(f"row {i}: {msg}" for i, msg in missing)
    raise FileNotFoundError(
        "Some manifest assets could not be resolved. First failures:\n" + details
    )
if not records:
    raise RuntimeError("No usable records were loaded.")

roots = np.asarray(roots, np.float32)
gray_buffers = np.stack(gray_buffers)
buffer_affines = np.stack(buffer_affines)
y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)
y2d_all = np.stack([normalize_uv(r["uv_crop"]) for r in records])
x_all = torch.from_numpy(
    np.stack([make_channels(r["crop_gray"]) for r in records])
)

print(f"Loaded {len(records):,} frames at {image_width}x{image_height}.")

# ----------------------- Deterministic segment split -------------------------

segments = np.asarray(sorted({r["segment"] for r in records}), dtype=object)
if len(segments) != TRAIN_SEGMENTS + VAL_SEGMENTS:
    raise ValueError(
        f"Expected {TRAIN_SEGMENTS + VAL_SEGMENTS} segments; found {len(segments)}."
    )

split_rng = np.random.default_rng(SEED)
split_rng.shuffle(segments)
train_segments = set(segments[:TRAIN_SEGMENTS].tolist())
val_segments = set(segments[TRAIN_SEGMENTS:].tolist())

train_idx = np.asarray(
    [i for i, r in enumerate(records) if r["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, r in enumerate(records) if r["segment"] in val_segments],
    dtype=np.int64,
)
assert not train_segments.intersection(val_segments)

print(
    f"Split: {len(train_segments)} train / {len(val_segments)} val segments; "
    f"{len(train_idx):,} train / {len(val_idx):,} val frames."
)

def score_pose(pred, target):
    distances = np.linalg.norm(pred - target, axis=-1)
    return float(distances.mean()), [
        float(distances[:, j].mean()) for j in FINGERTIP_JOINTS
    ]

def format_tips(tips):
    return " ".join(
        f"{name}={value:.2f}"
        for name, value in zip(FINGER_NAMES, tips)
    )

def canonical_bone_lengths(ids):
    return torch.from_numpy(np.stack([
        np.linalg.norm(
            y_all[ids, child] - y_all[ids, parent], axis=1
        ).mean()
        for parent, child in BONE_PAIRS
    ]).astype(np.float32))

def predict_direct(model, ids):
    model.eval()
    predictions = []
    with torch.inference_mode():
        for start in range(0, len(ids), 128):
            chunk = ids[start:start + 128]
            pred, _, _ = model(x_all[chunk].to(DEVICE))
            predictions.append(pred.cpu().numpy())
    return np.concatenate(predictions, axis=0)

# ---------------------- Old checkpoint discovery/load ------------------------

def find_old_checkpoint():
    found = []
    for root in (Path("/content"), Path.cwd(), DRIVE_ROOT):
        if not root.exists():
            continue
        for name in CHECKPOINT_NAMES:
            found.extend(root.rglob(name))

    if not found:
        return None

    for name in CHECKPOINT_NAMES:
        match = next((p for p in found if p.name == name), None)
        if match is not None:
            return match
    return found[0]

def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def strip_module_prefix(state):
    if not isinstance(state, dict):
        return state
    return {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in state.items()
    }

class OldPoseRefiner(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.delta_head = nn.Linear(512, 63)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        return self.delta_head(feat).reshape(-1, 21, 3)

old_checkpoint_path = find_old_checkpoint()
old_direct, old_refiner = None, None
old_val_pred = None

if old_checkpoint_path is None:
    print(
        "\nNo old checkpoint found under /content, current folder, or Drive. "
        "The new OOF experiment will train from scratch."
    )
else:
    print(f"\nFound old checkpoint: {old_checkpoint_path}")
    checkpoint = load_checkpoint(old_checkpoint_path)

    if isinstance(checkpoint, dict):
        direct_state = checkpoint.get("direct_state_dict")
        if direct_state is None and "state_dict" in checkpoint:
            direct_state = checkpoint["state_dict"]

        bone_lens = checkpoint.get("bone_lens")
        if bone_lens is None:
            bone_lens = canonical_bone_lengths(train_idx)
        else:
            bone_lens = torch.as_tensor(bone_lens, dtype=torch.float32).cpu()

        if direct_state is not None:
            old_direct = WebcamAlignedLCFTracker(bone_lens).to(DEVICE)
            try:
                old_direct.load_state_dict(
                    strip_module_prefix(direct_state), strict=True
                )
                old_direct.eval()
                print(f"Loaded old direct model onto {DEVICE}.")
            except Exception as exc:
                old_direct = None
                print(f"Old direct weights did not match this architecture: {exc}")

        refiner_state = checkpoint.get("refiner_state_dict")
        if refiner_state is not None:
            old_refiner = OldPoseRefiner().to(DEVICE)
            try:
                old_refiner.load_state_dict(
                    strip_module_prefix(refiner_state), strict=True
                )
                old_refiner.eval()
                print(f"Loaded old Phase 1 refiner onto {DEVICE}.")
            except Exception as exc:
                old_refiner = None
                print(f"Old refiner weights did not match: {exc}")

if old_direct is not None:
    old_val_pred = predict_direct(old_direct, val_idx)
    old_mpjpe, old_tips = score_pose(old_val_pred, y_all[val_idx])
    print(
        f"Old checkpoint direct | val {old_mpjpe:.2f} mm | "
        f"{format_tips(old_tips)}"
    )

# --------------------- Prior crop/render helper functions --------------------

def prior_input(index, prior):
    uv_img = project(roots[index:index + 1] + prior)
    affine = crop_affine(uv_img, margin=1.4)

    src = np.eye(3, dtype=np.float32)
    src[:2] = buffer_affines[index]
    dst = np.eye(3, dtype=np.float32)
    dst[:2] = affine
    mapping = dst @ np.linalg.inv(src)

    gray = cv2.warpAffine(
        gray_buffers[index],
        mapping[:2],
        (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    xy = uv_img @ affine[:, :2].T + affine[:, 2]
    lines = np.zeros((IMAGE_H, IMAGE_W), np.uint8)
    dots = np.zeros_like(lines)
    points = np.rint(xy).astype(int)

    for parent, child in BONE_PAIRS:
        cv2.line(
            lines, tuple(points[parent]), tuple(points[child]),
            255, 2, cv2.LINE_AA,
        )
    for point in points:
        cv2.circle(dots, tuple(point), 3, 255, -1, cv2.LINE_AA)

    inp = np.concatenate((
        make_channels(gray),
        lines[None].astype(np.float32) / 255.0,
        dots[None].astype(np.float32) / 255.0,
    ))
    return inp, affine

# Evaluate old Phase 1 weights, if present, on the old direct model's predictions.
if old_direct is not None and old_refiner is not None:
    print("\nOld checkpoint Phase 1 iterative validation:")
    current = old_val_pred.copy()
    for iteration in range(4):
        mpjpe, tips = score_pose(current, y_all[val_idx])
        print(
            f"  Iteration {iteration}: {mpjpe:.2f} mm | "
            f"{format_tips(tips)}"
        )
        if iteration < 3:
            updated = []
            with torch.inference_mode():
                for idx, prior in zip(val_idx, current):
                    inp, _ = prior_input(int(idx), prior)
                    delta = old_refiner(
                        torch.from_numpy(inp[None]).to(DEVICE)
                    )[0].cpu().numpy()
                    updated.append(prior + delta)
            current = np.asarray(updated, np.float32)

# ----------------------- Train direct models for OOF -------------------------

def train_direct(model, train_ids, seed_offset=0):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4
    )
    rng = np.random.default_rng(SEED + 1 + seed_offset)

    for epoch in range(1, TRAIN_EPOCHS + 1):
        model.train()
        order = np.random.permutation(train_ids)

        for start in range(0, len(order), BATCH_SIZE):
            ids = order[start:start + BATCH_SIZE]
            samples = [
                augment_one(
                    records[int(i)]["crop_gray"],
                    records[int(i)]["uv_crop"],
                    y_all[int(i)],
                    rng,
                )
                for i in ids
            ]
            bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
            by3 = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
            by2 = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred3, _, pred2 = model(bx)
            loss = direct_loss(pred3, by3, pred2, by2)
            loss.backward()
            optimizer.step()

        scheduler.step()

    return model

fold_rng = np.random.default_rng(SEED + 300)
shuffled_train_segments = np.asarray(sorted(train_segments), dtype=object)
fold_rng.shuffle(shuffled_train_segments)
fold_groups = np.array_split(shuffled_train_segments, N_FOLDS)

oof_train = np.full((len(records), 21, 3), np.nan, np.float32)
val_fold_preds = []
fold_states = []
fold_models = []

for fold_id, group in enumerate(fold_groups):
    held_segments = set(group.tolist())
    fold_val = np.asarray(
        [i for i in train_idx if records[int(i)]["segment"] in held_segments],
        dtype=np.int64,
    )
    fold_train = np.asarray(
        [i for i in train_idx if records[int(i)]["segment"] not in held_segments],
        dtype=np.int64,
    )
    assert not held_segments.intersection(
        records[int(i)]["segment"] for i in fold_train
    )

    lens = canonical_bone_lengths(fold_train)
    torch.manual_seed(SEED + 1000 + fold_id)
    model = WebcamAlignedLCFTracker(lens).to(DEVICE)
    train_direct(model, fold_train, seed_offset=fold_id * 100)

    oof_train[fold_val] = predict_direct(model, fold_val)
    val_fold_preds.append(predict_direct(model, val_idx))
    fold_states.append({
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    })
    fold_models.append(model)
    print(
        f"Finished OOF fold {fold_id + 1}/{N_FOLDS}: "
        f"{len(fold_train):,} train, {len(fold_val):,} held-out frames."
    )

assert np.isfinite(oof_train[train_idx]).all()
val_prior = np.mean(np.stack(val_fold_preds), axis=0).astype(np.float32)
oof_mpjpe, oof_tips = score_pose(val_prior, y_all[val_idx])
print(
    f"OOF direct ensemble | val {oof_mpjpe:.2f} mm | "
    f"{format_tips(oof_tips)}"
)

# ---------------------- Augmented prior training samples ---------------------

def make_augmented_prior_sample(index, prior, rng):
    inp, affine = prior_input(index, prior)
    gt = records[index]["y3d"]
    gt_uv_crop = records[index]["uv_image"] @ affine[:, :2].T + affine[:, 2]

    gray_crop = np.clip(inp[0] * 255.0, 0, 255).astype(np.uint8)
    base, gt_aug, uv_target = augment_one(
        gray_crop, gt_uv_crop, gt, rng
    )

    uv_pixels = uv_target.copy()
    uv_pixels[:, 0] = (uv_pixels[:, 0] + 1.0) * IMAGE_W * 0.5
    uv_pixels[:, 1] = (uv_pixels[:, 1] + 1.0) * IMAGE_H * 0.5

    design = np.column_stack((gt_uv_crop, np.ones(len(gt_uv_crop))))
    coeff, *_ = np.linalg.lstsq(design, uv_pixels, rcond=None)
    aug_affine = coeff.T.astype(np.float32)
    angle = float(np.arctan2(aug_affine[0, 1], aug_affine[0, 0]))
    prior_aug = rotate_camera_xy(prior, angle).astype(np.float32)

    lines = cv2.warpAffine(
        inp[2], aug_affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
    )
    dots = cv2.warpAffine(
        inp[3], aug_affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
    )
    x4 = np.concatenate((base, lines[None], dots[None]), axis=0)

    return (
        x4, prior_aug, gt_aug, uv_target,
        roots[index], affine, aug_affine,
    )

# ------------------------- Train gated refiner -------------------------------

class GatedRefiner(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.delta_head = nn.Linear(512, 63)
        self.uncertainty_head = nn.Linear(512, 21)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        delta = 45.0 * torch.tanh(
            self.delta_head(feat).reshape(-1, 21, 3) / 45.0
        )
        sigma = (
            F.softplus(self.uncertainty_head(feat).reshape(-1, 21)) + 1.0
        ).clamp(1.0, 60.0)
        authority = 20.0 / (20.0 + sigma)
        return delta, sigma, authority

gated = GatedRefiner().to(DEVICE)
optimizer = torch.optim.AdamW(
    gated.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=REFINER_EPOCHS, eta_min=1e-4
)
rng = np.random.default_rng(SEED + 1300)
focal_t = torch.as_tensor(CAM_FOCAL, dtype=torch.float32, device=DEVICE)
principal_t = torch.as_tensor(CAM_PRINCPT, dtype=torch.float32, device=DEVICE)

for epoch in range(1, REFINER_EPOCHS + 1):
    gated.train()
    order = np.random.permutation(train_idx)

    for start in range(0, len(order), BATCH_SIZE):
        ids = order[start:start + BATCH_SIZE]
        samples = [
            make_augmented_prior_sample(
                int(i), oof_train[int(i)], rng
            )
            for i in ids
        ]

        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        prior_b = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        target_b = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)
        uv_target = torch.from_numpy(np.stack([s[3] for s in samples])).to(DEVICE)
        roots_b = torch.from_numpy(np.stack([s[4] for s in samples])).to(DEVICE)
        crop_aff = torch.from_numpy(np.stack([s[5] for s in samples])).to(DEVICE)
        aug_aff = torch.from_numpy(np.stack([s[6] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        delta, sigma, authority = gated(bx)
        corrected = prior_b + authority[..., None] * delta

        pose_loss = F.l1_loss(corrected, target_b)
        joint_error = torch.linalg.vector_norm(
            corrected - target_b, dim=-1
        )
        uncertainty_loss = (
            joint_error / sigma + torch.log(sigma)
        ).mean()

        angle = torch.atan2(aug_aff[:, 0, 1], aug_aff[:, 0, 0])
        c, s = torch.cos(angle)[:, None], torch.sin(angle)[:, None]
        unrot = corrected.clone()
        x, y = corrected[..., 0], corrected[..., 1]
        unrot[..., 0] = c * x - s * y
        unrot[..., 1] = s * x + c * y

        camera_pts = unrot + roots_b[:, None, :]
        z = camera_pts[..., 2].clamp_min(1.0)
        uv = torch.stack((
            focal_t[0] * camera_pts[..., 0] / z + principal_t[0],
            focal_t[1] * camera_pts[..., 1] / z + principal_t[1],
        ), dim=-1)

        uv_h = torch.cat((uv, torch.ones_like(uv[..., :1])), dim=-1)
        crop_xy = torch.bmm(uv_h, crop_aff.transpose(1, 2))
        crop_h = torch.cat((crop_xy, torch.ones_like(crop_xy[..., :1])), dim=-1)
        aug_xy = torch.bmm(crop_h, aug_aff.transpose(1, 2))

        uv_pred = aug_xy.clone()
        uv_pred[..., 0] = 2.0 * aug_xy[..., 0] / IMAGE_W - 1.0
        uv_pred[..., 1] = 2.0 * aug_xy[..., 1] / IMAGE_H - 1.0
        reprojection_loss = F.smooth_l1_loss(uv_pred, uv_target)

        loss = pose_loss + 0.25 * uncertainty_loss + 15.0 * reprojection_loss
        loss.backward()
        optimizer.step()

    scheduler.step()
    print(f"Refiner epoch {epoch}/{REFINER_EPOCHS} complete")

# --------------------------- Validation and timing ---------------------------

gated.eval()

def apply_refiner(index, prior):
    inp, _ = prior_input(int(index), prior)
    with torch.inference_mode():
        delta, _, authority = gated(
            torch.from_numpy(inp[None]).to(DEVICE)
        )
    return (
        prior + (authority[0, :, None] * delta[0]).cpu().numpy()
    ).astype(np.float32)

eval_rows = []
current = val_prior.copy()
for iteration in range(4):
    mpjpe, tips = score_pose(current, y_all[val_idx])
    eval_rows.append((iteration, mpjpe, tips))
    if iteration < 3:
        current = np.stack([
            apply_refiner(int(i), pose)
            for i, pose in zip(val_idx, current)
        ])

latency_idx = int(val_idx[0])
latency_prior = val_prior[0].copy()

def one_refiner_step():
    return apply_refiner(latency_idx, latency_prior)

def direct_ensemble_plus_two():
    image = x_all[latency_idx:latency_idx + 1].to(DEVICE)
    with torch.inference_mode():
        poses = [model(image)[0][0] for model in fold_models]
        prior = torch.stack(poses).mean(dim=0).cpu().numpy()
    for _ in range(2):
        prior = apply_refiner(latency_idx, prior)
    return prior

for _ in range(20):
    one_refiner_step()
    direct_ensemble_plus_two()

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

one_ms, combo_ms = [], []
for _ in range(200):
    start = time.perf_counter()
    one_refiner_step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    one_ms.append((time.perf_counter() - start) * 1000.0)

    start = time.perf_counter()
    direct_ensemble_plus_two()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    combo_ms.append((time.perf_counter() - start) * 1000.0)

# --------------------------------- Checkpoint --------------------------------

checkpoint_out = Path("/content/phase2_oof_gated_refiner.pt")
torch.save(
    {
        "fold_direct_state_dicts": fold_states,
        "gated_refiner_state_dict": {
            key: value.detach().cpu().clone()
            for key, value in gated.state_dict().items()
        },
        "fold_segments": [list(group) for group in fold_groups],
        "train_segments": sorted(train_segments),
        "val_segments": sorted(val_segments),
        "seed": SEED,
        "camera_id": "400485",
        "manifest_path": str(MANIFEST_PATH),
    },
    checkpoint_out,
)

print("\nOOF-TRAINED GATED REFINEMENT — HELD-OUT VALIDATION")
print(f"{'Iteration':<12} {'MPJPE mm':>9}  Thumb   Index  Middle    Ring   Pinky")
print("-" * 75)
for iteration, mpjpe, tips in eval_rows:
    print(
        f"{iteration:<12} {mpjpe:9.2f}  "
        + " ".join(f"{value:7.2f}" for value in tips)
    )

print(
    f"\nMedian batch-1 latency on {DEVICE}: "
    f"refiner={statistics.median(one_ms):.3f} ms; "
    f"direct ensemble + 2 refinements="
    f"{statistics.median(combo_ms):.3f} ms"
)
print(f"Checkpoint saved: {checkpoint_out}")

Active compute device: cuda (Tesla T4)
Manifest:     /content/dataset/manifest.csv
Capture root: /content/dataset
Calibration:  /content/dataset/cam_params.json
Loading 5,114 camera-400485 rows...
Loaded 5,114 frames at 667x1024.
Split: 44 train / 12 val segments; 4,246 train / 868 val frames.

No old checkpoint found under /content, current folder, or Drive. The new OOF experiment will train from scratch.
Finished OOF fold 1/4: 3,229 train, 1,017 held-out frames.
Finished OOF fold 2/4: 3,349 train, 897 held-out frames.
Finished OOF fold 3/4: 3,264 train, 982 held-out frames.
Finished OOF fold 4/4: 2,896 train, 1,350 held-out frames.
OOF direct ensemble | val 24.79 mm | Thumb=43.67 Index=36.62 Middle=31.81 Ring=30.74 Pinky=33.78
Refiner epoch 1/25 complete
Refiner epoch 2/25 complete
Refiner epoch 3/25 complete
Refiner epoch 4/25 complete
Refiner epoch 5/25 complete
Refiner epoch 6/25 complete
Refiner epoch 7/25 complete
Refiner epoch 8/25 complete
Refiner epoch 9/25 complete
Refiner e

In [ ]:
# @title
# Damped-refiner diagnostic. Uses variables from the previous runner.

expected_shape = (len(val_idx), 21, 3)
base_prior = np.asarray(val_prior, dtype=np.float32)

# Recover the validation prior if the notebook variable was overwritten.
if base_prior.shape != expected_shape:
    if "val_fold_preds" in globals() and len(val_fold_preds):
        base_prior = np.mean(
            np.stack(val_fold_preds), axis=0
        ).astype(np.float32)
    elif "fold_models" in globals() and len(fold_models):
        base_prior = np.mean(
            np.stack([predict_direct(model, val_idx) for model in fold_models]),
            axis=0,
        ).astype(np.float32)
    else:
        raise ValueError(
            f"Expected val_prior shape {expected_shape}, got "
            f"{base_prior.shape}; cannot rebuild it from available variables."
        )

if base_prior.shape != expected_shape:
    raise ValueError(
        f"Validation prior has shape {base_prior.shape}; expected {expected_shape}."
    )

alphas = [0.0, 0.05, 0.10, 0.20, 0.35, 0.50, 1.0]
diagnostic_rows = []

for alpha in alphas:
    current = base_prior.copy()

    for step in range(4):
        mpjpe, tips = score_pose(current, y_all[val_idx])
        diagnostic_rows.append((alpha, step, mpjpe, tips))

        if step < 3 and alpha > 0:
            # apply_refiner returns a complete (21, 3) pose; do not index [0].
            proposed = np.stack([
                apply_refiner(int(i), pose)
                for i, pose in zip(val_idx, current)
            ]).astype(np.float32)

            if proposed.shape != expected_shape:
                raise ValueError(
                    f"Refiner returned shape {proposed.shape}; "
                    f"expected {expected_shape}."
                )

            current = current + alpha * (proposed - current)

print("DAMPED REFINER DIAGNOSTIC — VALIDATION SEGMENTS")
print(f"{'Alpha':>6} {'Step':>6} {'MPJPE mm':>10}  Thumb   Index  Middle    Ring   Pinky")
print("-" * 78)
for alpha, step, mpjpe, tips in diagnostic_rows:
    print(
        f"{alpha:6.2f} {step:6d} {mpjpe:10.2f}  "
        + " ".join(f"{value:7.2f}" for value in tips)
    )

DAMPED REFINER DIAGNOSTIC — VALIDATION SEGMENTS
 Alpha   Step   MPJPE mm  Thumb   Index  Middle    Ring   Pinky
------------------------------------------------------------------------------
  0.00      0      24.79    43.67   36.62   31.81   30.74   33.78
  0.00      1      24.79    43.67   36.62   31.81   30.74   33.78
  0.00      2      24.79    43.67   36.62   31.81   30.74   33.78
  0.00      3      24.79    43.67   36.62   31.81   30.74   33.78
  0.05      0      24.79    43.67   36.62   31.81   30.74   33.78
  0.05      1      24.77    43.61   36.56   31.76   30.72   33.71
  0.05      2      24.78    43.57   36.53   31.72   30.72   33.68
  0.05      3      24.80    43.55   36.52   31.70   30.76   33.68
  0.10      0      24.79    43.67   36.62   31.81   30.74   33.78
  0.10      1      24.78    43.57   36.53   31.72   30.72   33.67
  0.10      2      24.83    43.54   36.53   31.70   30.82   33.71
  0.10      3      24.95    43.59   36.61   31.74   31.04   33.86
  0.20      0    

In [ ]:
# @title
"""Phase 2 diagnostics: axis errors, oracle depth, reprojection, and input ablation.

Single-file runner for Google Colab or the project checkout. It follows the
Phase 0 data/crop/split/training recipe and trains three fresh 20-epoch direct
models: grayscale only, Sobel only, and grayscale plus Sobel. It selects CUDA
automatically when available and otherwise runs on CPU. It does not download
data or write checkpoints; it prints diagnostic tables and stops.

The Phase 0 module forces CPU with CUDA_VISIBLE_DEVICES at import time, so this
script carries forward its loader/model/loss recipe locally while parameterizing
the first convolution for the requested one- and two-channel comparisons.
"""

from __future__ import annotations

import csv
import json
import os
import random
import time
from pathlib import Path

try:
    import cv2
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise SystemExit(
        "phase2_diagnose.py requires PyTorch, NumPy, and OpenCV (cv2). "
        "Install them in this runtime, then rerun."
    ) from exc


# ------------------------------- Configuration ------------------------------

SEED = 20251018
TRAIN_SEGMENTS = 44
VAL_SEGMENTS = 12
TRAIN_EPOCHS = 20
BATCH_SIZE = 64
LR = 1.2e-3
WEIGHT_DECAY = 1e-4
IMAGE_W, IMAGE_H = 128, 96  # phase0 uses W=128, H=96 (96x128 pixels).
CAMERA_ID = "400485"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
    12, 13, 14, 15, 16, 17, 19, 20, 21, 22,
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
MODES = ("grayscale", "sobel", "both")


# ----------------------------- Dataset discovery ----------------------------

def _find_manifest() -> Path:
    # In a pasted Colab cell, __file__ is not defined. In that case search
    # Colab's usual dataset location and the current working directory first.
    script_file = globals().get("__file__")
    if script_file:
        here = Path(script_file).resolve()
        project_root = here.parents[1] if len(here.parents) > 1 else here.parent
    else:
        project_root = Path.cwd()
    capture_id = (
        "m--20210701--1058--0000000--pilot--relightablehandsy--"
        "participant0--two-hands"
    )
    likely = [
        project_root / "datasets" / "reinterhand" / "captures" / capture_id
        / "derived" / "mugsy_subset" / "manifest.csv",
        Path("/content/dataset/manifest.csv"),
        Path.cwd() / "manifest.csv",
    ]
    for path in likely:
        if path.is_file():
            return path.resolve()

    roots = [Path("/content/dataset"), Path("/content"), Path.cwd()]
    for root in roots:
        if not root.is_dir():
            continue
        found = list(root.rglob("manifest.csv"))
        if found:
            return next(
                (p.resolve() for p in found if "mugsy_subset" in str(p).lower()),
                found[0].resolve(),
            )

    # If a Drive-mounted dataset is in use, mount/search only after local paths.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_root = Path("/content/drive/MyDrive")
        found = list(drive_root.rglob("manifest.csv"))
        if found:
            return next(
                (p.resolve() for p in found if "mugsy_subset" in str(p).lower()),
                found[0].resolve(),
            )
    except Exception as exc:
        print(f"Drive search unavailable: {exc}")

    raise FileNotFoundError(
        "manifest.csv not found. Put the dataset under /content/dataset or "
        "mount it in Google Drive."
    )


MANIFEST_PATH = _find_manifest()
SUBSET_DIR = MANIFEST_PATH.parent


def _find_camera_and_capture_root() -> tuple[Path, Path]:
    ancestors = (SUBSET_DIR, *SUBSET_DIR.parents)
    for parent in ancestors:
        candidate = parent / "Mugsy_cameras" / "cam_params.json"
        if candidate.is_file():
            return candidate.resolve(), parent.resolve()
        candidate = parent / "cam_params.json"
        if candidate.is_file():
            return candidate.resolve(), parent.resolve()

    for root in (SUBSET_DIR.parent, SUBSET_DIR.parent.parent, Path("/content")):
        if root.is_dir():
            found = list(root.rglob("cam_params.json"))
            if found:
                cam = next(
                    (p for p in found if p.parent.name.lower() == "mugsy_cameras"),
                    found[0],
                )
                cap = cam.parent.parent if cam.parent.name.lower() == "mugsy_cameras" else cam.parent
                return cam.resolve(), cap.resolve()

    raise FileNotFoundError("cam_params.json not found near the manifest.")


CAMERA_PATH, CAPTURE_DIR = _find_camera_and_capture_root()


def _build_asset_index() -> dict[str, set[Path]]:
    index: dict[str, set[Path]] = {}
    roots = dict.fromkeys((CAPTURE_DIR, SUBSET_DIR, SUBSET_DIR.parent))
    for root in roots:
        if not root.is_dir():
            continue
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in {
                ".json", ".png", ".jpg", ".jpeg",
            }:
                index.setdefault(path.name.lower(), set()).add(path.resolve())
    return index


ASSET_INDEX = _build_asset_index()


def resolve_asset(rel_path: str) -> Path:
    rel = str(rel_path).replace("\\", "/").strip()
    rel_path_obj = Path(rel)
    parts = rel_path_obj.parts
    candidates = [
        Path(rel), CAPTURE_DIR / rel, SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel, SUBSET_DIR.parent.parent / rel,
    ]
    if parts and parts[0] == CAPTURE_DIR.name:
        candidates.append(CAPTURE_DIR.joinpath(*parts[1:]))
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    # Try the manifest path under each ancestor; original paths can be rooted
    # above the subset directory.
    for ancestor in (SUBSET_DIR, *SUBSET_DIR.parents):
        candidate = ancestor.joinpath(*parts)
        if candidate.is_file():
            return candidate.resolve()

    # The Colab export may flatten orig_fits/right/Keypoints to
    # keypoints_right/, and similarly for left.
    lower_parts = [part.lower() for part in parts]
    side = "right" if "right" in lower_parts else (
        "left" if "left" in lower_parts else None
    )
    hits = ASSET_INDEX.get(rel_path_obj.name.lower(), set())
    if side:
        side_dir = f"keypoints_{side}"
        side_hits = {
            path for path in hits
            if side_dir in [part.lower() for part in path.parts]
        }
        if len(side_hits) == 1:
            return next(iter(side_hits))
        if len(side_hits) > 1:
            exact = {p for p in side_hits if p.parent.name.lower() == side_dir}
            if len(exact) == 1:
                return next(iter(exact))

    if len(hits) == 1:
        return next(iter(hits))
    if len(hits) > 1:
        # For images or differently named exports, prefer a unique suffix match.
        wanted_suffix = "/".join(part.lower() for part in parts[-3:])
        suffix_hits = {
            p for p in hits
            if "/".join(p.as_posix().lower().split("/")[-3:]) == wanted_suffix
        }
        if len(suffix_hits) == 1:
            return next(iter(suffix_hits))
        raise FileNotFoundError(
            f"Asset basename is ambiguous: {rel}; matches: "
            f"{[str(p) for p in list(hits)[:6]]}"
        )
    raise FileNotFoundError(f"Could not resolve manifest asset: {rel}")


# --------------------------------- Geometry ----------------------------------

def crop_affine(uv: np.ndarray, margin: float = 1.4) -> np.ndarray:
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [IMAGE_W / side, 0.0, -left * IMAGE_W / side],
            [0.0, IMAGE_H / side, -top * IMAGE_H / side],
        ],
        dtype=np.float32,
    )


def camera_project(points: np.ndarray, focal: np.ndarray, principal: np.ndarray) -> np.ndarray:
    z = points[..., 2]
    if np.any(z <= 1e-6):
        raise ValueError("Predicted point has nonpositive camera-space depth.")
    return np.stack(
        (focal[0] * points[..., 0] / z + principal[0],
         focal[1] * points[..., 1] / z + principal[1]),
        axis=-1,
    ).astype(np.float32)


def channels_from_gray(gray_u8: np.ndarray) -> dict[str, np.ndarray]:
    gray = gray_u8.astype(np.float32) / 255.0
    raw = gray_u8.astype(np.float32)
    gx = cv2.Sobel(raw, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(raw, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    sobel = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return {
        "grayscale": gray[None].astype(np.float32),
        "sobel": sobel[None].astype(np.float32),
        "both": np.stack((gray, sobel)).astype(np.float32),
    }


def normalize_uv(uv: np.ndarray) -> np.ndarray:
    normalized = np.empty_like(uv, dtype=np.float32)
    normalized[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    normalized[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return normalized


def rotate_camera_xy(y3d: np.ndarray, angle_rad: float) -> np.ndarray:
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    out = y3d.copy()
    x, y = y3d[:, 0].copy(), y3d[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment_one(gray_u8: np.ndarray, uv: np.ndarray, y3d: np.ndarray, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle_rad = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    matrix[1, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_H)
    warped = cv2.warpAffine(
        gray_u8, matrix, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(warped.astype(np.float32) * brightness + offset, 0, 255).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle_rad)
    return channels_from_gray(warped), y3d_aug, normalize_uv(uv_aug)


# -------------------------- Model copied from Phase 0 ------------------------

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)


def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    batch = bone_vectors.shape[0]
    joints = torch.zeros((batch, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + bone_vectors[:, idx]
    return joints


class WebcamAlignedLCFTracker(nn.Module):
    """Phase 0 LCF tracker; only its first convolution is channel-configurable."""
    def __init__(self, canonical_lens: torch.Tensor, in_channels: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512), nn.ReLU(inplace=True), nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.proj2d_head = nn.Linear(512, 21 * 2)
        self.register_buffer("bone_lens", canonical_lens.clone())

    def forward(self, x: torch.Tensor):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3), dim=-1, eps=1e-7
        )
        latent_frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3), dim=-1, eps=1e-7
        )
        phalanx_cam_u = torch.bmm(
            latent_frame, phalanx_local_u.transpose(1, 2)
        ).transpose(1, 2)
        bone_dirs = torch.zeros((batch, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        for finger_idx, bone_offset in enumerate((4, 8, 12, 16)):
            phalanx_offset = 3 + finger_idx * 3
            bone_dirs[:, bone_offset] = meta_cam_u[:, finger_idx + 1]
            bone_dirs[:, bone_offset + 1:bone_offset + 4] = (
                phalanx_cam_u[:, phalanx_offset:phalanx_offset + 3]
            )
        bone_vectors = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vectors)
        proj2d = torch.tanh(self.proj2d_head(feat).reshape(batch, 21, 2))
        return joints, bone_dirs, proj2d


class MetricLossEvaluator:
    """Same Phase 0 position, bone-length, and bone-direction objective."""
    def __call__(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        l_pos = torch.norm(pred - target, dim=-1).mean()
        parent_ids = [p for p, _ in BONE_PAIRS]
        child_ids = [c for _, c in BONE_PAIRS]
        pred_bones = pred[:, child_ids] - pred[:, parent_ids]
        target_bones = target[:, child_ids] - target[:, parent_ids]
        pred_len = torch.norm(pred_bones, dim=-1)
        target_len = torch.norm(target_bones, dim=-1)
        l_len = F.l1_loss(pred_len, target_len)
        pred_u = pred_bones / (pred_len.unsqueeze(-1) + 1e-7)
        target_u = target_bones / (target_len.unsqueeze(-1) + 1e-7)
        cos_sim = torch.clamp((pred_u * target_u).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir


# -------------------------------- Data loader --------------------------------

def load_data():
    with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r.get("camera_id", CAMERA_ID) == CAMERA_ID]
    if not rows:
        raise RuntimeError(f"Manifest has no rows for camera {CAMERA_ID}.")

    with CAMERA_PATH.open("r", encoding="utf-8") as f:
        camera_root = json.load(f)
    camera = camera_root[CAMERA_ID]
    rotation = np.asarray(camera["R"], dtype=np.float32)
    translation = np.asarray(camera["t"], dtype=np.float32).reshape(3)
    focal = np.asarray(camera["focal"], dtype=np.float32).reshape(2)
    principal = np.asarray(camera["princpt"], dtype=np.float32).reshape(2)

    records = []
    image_size = None
    for index, row in enumerate(rows):
        kp_path = resolve_asset(row["orig_kp_right"])
        image_path = resolve_asset(row["image_path"])
        with kp_path.open("r", encoding="utf-8") as f:
            kp_world = np.asarray(json.load(f), dtype=np.float32)
        if kp_world.shape != (24, 3):
            raise ValueError(
                f"{row.get('frame_id', index)}: expected 24x3 right keypoints, "
                f"got {kp_world.shape}."
            )

        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")
        h, w = image.shape
        if image_size is None:
            image_size = (w, h)
        elif image_size != (w, h):
            raise ValueError("Image dimensions vary within the manifest.")

        camera_points = kp_world @ rotation.T + translation
        selected = camera_points[KEEP_21_INDICES]
        z = selected[:, 2]
        if np.any(z <= 1e-6):
            raise ValueError(f"Invalid camera depth in {row.get('frame_id', index)}")
        uv = np.stack(
            (focal[0] * selected[:, 0] / z + principal[0],
             focal[1] * selected[:, 1] / z + principal[1]),
            axis=1,
        ).astype(np.float32)
        affine = crop_affine(uv, margin=1.4)
        crop = cv2.warpAffine(
            image, affine, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )
        uv_crop = uv @ affine[:, :2].T + affine[:, 2]
        root_camera = selected[0].copy()
        root_relative = selected - root_camera[None, :]

        records.append({
            "frame_id": row.get("frame_id", str(index)),
            "segment": row["segment"],
            "crop_gray": crop,
            "uv_crop": uv_crop.astype(np.float32),
            "uv_image": uv,
            "root_camera": root_camera.astype(np.float32),
            "y3d": root_relative.astype(np.float32),
        })

    return records, image_size, focal, principal


# ------------------------------- Split/training ------------------------------

def build_split(records):
    segments = np.asarray(sorted({r["segment"] for r in records}), dtype=object)
    if len(segments) != TRAIN_SEGMENTS + VAL_SEGMENTS:
        raise ValueError(
            f"Expected 56 segments for the 44/12 split; found {len(segments)}."
        )
    split_rng = np.random.default_rng(SEED)
    split_rng.shuffle(segments)
    train_segments = set(segments[:TRAIN_SEGMENTS].tolist())
    val_segments = set(segments[TRAIN_SEGMENTS:].tolist())
    if train_segments & val_segments:
        raise AssertionError("Segment leakage in train/validation split.")
    train_idx = np.asarray(
        [i for i, r in enumerate(records) if r["segment"] in train_segments],
        dtype=np.int64,
    )
    val_idx = np.asarray(
        [i for i, r in enumerate(records) if r["segment"] in val_segments],
        dtype=np.int64,
    )
    return train_segments, val_segments, train_idx, val_idx


def make_base_features(records, mode: str, device: torch.device) -> torch.Tensor:
    return torch.from_numpy(np.stack([
        channels_from_gray(record["crop_gray"])[mode] for record in records
    ])).to(device)


def augment_mode(gray_u8, uv, y3d, rng, mode):
    channels, y3d_aug, y2d_aug = augment_one(gray_u8, uv, y3d, rng)
    return channels[mode], y3d_aug, y2d_aug


def train_one_mode(records, train_idx, y_all, y2d_all, mode, device):
    # Reset RNGs before each ablation so initialization, sample order, and
    # augmentation randomness are paired across the three input modes.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    x_all = make_base_features(records, mode, device)
    train_y = y_all[train_idx]
    canonical_lens = torch.from_numpy(np.stack([
        np.linalg.norm(
            train_y[:, child] - train_y[:, parent], axis=1
        ).mean()
        for parent, child in BONE_PAIRS
    ]).astype(np.float32))
    model = WebcamAlignedLCFTracker(
        canonical_lens, in_channels=x_all.shape[1]
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4
    )
    criterion = MetricLossEvaluator()
    aug_rng = np.random.default_rng(SEED + 1)
    started = time.perf_counter()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        model.train()
        order = np.random.permutation(train_idx)
        for start in range(0, len(order), BATCH_SIZE):
            batch_ids = order[start:start + BATCH_SIZE]
            samples = [
                augment_mode(
                    records[int(i)]["crop_gray"],
                    records[int(i)]["uv_crop"],
                    y_all[int(i)],
                    aug_rng,
                    mode,
                )
                for i in batch_ids
            ]
            bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(device)
            by3 = torch.from_numpy(np.stack([s[1] for s in samples])).to(device)
            by2 = torch.from_numpy(np.stack([s[2] for s in samples])).to(device)

            optimizer.zero_grad(set_to_none=True)
            pred3, _, pred2 = model(bx)
            loss = criterion(pred3, by3) + 15.0 * F.smooth_l1_loss(pred2, by2)
            loss.backward()
            optimizer.step()
        scheduler.step()
        print(f"  {mode}: epoch {epoch:02d}/{TRAIN_EPOCHS} complete")

    elapsed = time.perf_counter() - started
    model.eval()
    val_pred_batches = []
    with torch.inference_mode():
        for start in range(0, len(val_idx), 128):
            ids = val_idx[start:start + 128]
            pred, _, _ = model(x_all[ids])
            val_pred_batches.append(pred.cpu().numpy())
    predictions = np.concatenate(val_pred_batches, axis=0).astype(np.float32)
    return predictions, elapsed


# --------------------------------- Metrics ------------------------------------

def mean_joint_distance(pred, target):
    return np.linalg.norm(pred - target, axis=-1).mean(axis=0)


def component_mae(pred, target):
    return np.abs(pred - target).mean(axis=0)


def reprojection_metrics(pred_root_relative, target_root_relative, records,
                         val_idx, focal, principal):
    # Add the GT camera-space root to both prediction and label. This keeps the
    # root translation/depth known and evaluates the real calibrated projection.
    pred_camera = np.stack([
        pred + records[int(i)]["root_camera"][None, :]
        for i, pred in zip(val_idx, pred_root_relative)
    ])
    target_uv = np.stack([records[int(i)]["uv_image"] for i in val_idx])
    pred_uv = camera_project(pred_camera, focal, principal)
    pixel_error = np.linalg.norm(pred_uv - target_uv, axis=-1)
    return pixel_error.mean(axis=0)


def report_mode(mode, pred, records, val_idx, y_all, focal, principal):
    target = y_all[val_idx]
    xyz_mae = component_mae(pred, target)
    joint_mpjpe = mean_joint_distance(pred, target)
    reproj = reprojection_metrics(pred, target, records, val_idx, focal, principal)

    oracle_pred = pred.copy()
    oracle_pred[..., 2] = target[..., 2]
    oracle_mpjpe = mean_joint_distance(oracle_pred, target)

    # Component table: overall averages over all joints, plus each tip.
    print(f"\n[{mode.upper()}] AXIS COMPONENT MAE AND JOINT MPJPE (mm)")
    print(f"{'Point':<10}{'MPJPE':>10}{'|dx|':>10}{'|dy|':>10}{'|dz|':>10}")
    print("-" * 50)
    print(
        f"{'Overall':<10}{joint_mpjpe.mean():10.2f}"
        f"{xyz_mae[:, 0].mean():10.2f}{xyz_mae[:, 1].mean():10.2f}"
        f"{xyz_mae[:, 2].mean():10.2f}"
    )
    for name, j in zip(FINGER_NAMES, FINGERTIP_JOINTS):
        print(
            f"{name:<10}{joint_mpjpe[j]:10.2f}"
            f"{xyz_mae[j, 0]:10.2f}{xyz_mae[j, 1]:10.2f}"
            f"{xyz_mae[j, 2]:10.2f}"
        )

    print(f"\n[{mode.upper()}] ORACLE-DEPTH MPJPE (mm; replace predicted relative z with GT)")
    print(f"{'Point':<10}{'MPJPE':>10}")
    print("-" * 20)
    print(f"{'Overall':<10}{oracle_mpjpe.mean():10.2f}")
    for name, j in zip(FINGER_NAMES, FINGERTIP_JOINTS):
        print(f"{name:<10}{oracle_mpjpe[j]:10.2f}")

    print(f"\n[{mode.upper()}] REAL-CAMERA 2D REPROJECTION ERROR (pixels; GT root translation/depth)")
    print(f"{'Point':<10}{'Mean px':>10}")
    print("-" * 20)
    print(f"{'Overall':<10}{reproj.mean():10.2f}")
    for name, j in zip(FINGER_NAMES, FINGERTIP_JOINTS):
        print(f"{name:<10}{reproj[j]:10.2f}")

    segment_rows = []
    for segment in sorted({records[int(i)]["segment"] for i in val_idx}):
        local_positions = [
            pos for pos, i in enumerate(val_idx)
            if records[int(i)]["segment"] == segment
        ]
        segment_pred = pred[local_positions]
        segment_target = target[local_positions]
        segment_mpjpe = np.linalg.norm(
            segment_pred - segment_target, axis=-1
        ).mean()
        segment_rows.append((segment, len(local_positions), float(segment_mpjpe)))

    print(f"\n[{mode.upper()}] PER-SEGMENT MPJPE (mm; sorted by segment)")
    print(f"{'Segment':<58}{'Frames':>8}{'MPJPE':>10}")
    print("-" * 76)
    for segment, frames, value in segment_rows:
        print(f"{segment:<58}{frames:8d}{value:10.2f}")

    return float(joint_mpjpe.mean()), float(oracle_mpjpe.mean()), float(reproj.mean())


def main():
    wall_start = time.perf_counter()
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if DEVICE.type == "cpu":
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

    print(f"Active compute device: {DEVICE}")
    if DEVICE.type == "cuda":
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Manifest: {MANIFEST_PATH}")
    print(f"Camera:   {CAMERA_PATH} (camera {CAMERA_ID})")

    records, image_size, focal, principal = load_data()
    train_segments, val_segments, train_idx, val_idx = build_split(records)
    y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)

    print(
        f"Loaded {len(records):,} frames ({image_size[0]}x{image_size[1]}), "
        f"split {len(train_segments)}/{len(val_segments)} segments, "
        f"{len(train_idx):,}/{len(val_idx):,} frames."
    )
    print("Training three paired 20-epoch Phase 0 recipe runs.")

    summary = []
    for mode in MODES:
        print(f"\nTraining input mode: {mode}")
        pred, elapsed = train_one_mode(
            records, train_idx, y_all,
            np.stack([normalize_uv(r["uv_crop"]) for r in records]).astype(np.float32),
            mode, DEVICE,
        )
        metrics = report_mode(
            mode, pred, records, val_idx, y_all, focal, principal
        )
        summary.append((mode, *metrics, elapsed))

    print("\nINPUT ABLATION SUMMARY (validation)")
    print(f"{'Input':<12}{'MPJPE mm':>12}{'OracleZ mm':>12}{'2D px':>10}{'Train sec':>12}")
    print("-" * 58)
    for mode, mpjpe, oracle, reproj, elapsed in summary:
        print(f"{mode:<12}{mpjpe:12.2f}{oracle:12.2f}{reproj:10.2f}{elapsed:12.1f}")

    print("\nAssumptions:")
    print("- The phase0 KEEP_21_INDICES mapping selects the 21 right-hand joints.")
    print("- Camera-space labels, root-relative targets, and errors use millimeters.")
    print("- Axis components are mean absolute coordinate errors (|dx|, |dy|, |dz|); MPJPE is Euclidean 3D distance.")
    print("- Oracle depth replaces each predicted root-relative joint z with its GT root-relative z.")
    print("- Reprojection adds the GT camera-space root to predicted relative joints and uses the official camera intrinsics.")
    print("- No mirroring is applied; labels remain right-hand/chirality-consistent.")
    print(f"Total runtime: {time.perf_counter() - wall_start:.1f}s")


if __name__ == "__main__":
    main()


Active compute device: cpu
Manifest: /content/dataset/manifest.csv
Camera:   /content/dataset/cam_params.json (camera 400485)
Loaded 5,114 frames (667x1024), split 44/12 segments, 4,246/868 frames.
Training three paired 20-epoch Phase 0 recipe runs.

Training input mode: grayscale
  grayscale: epoch 01/20 complete
  grayscale: epoch 02/20 complete
  grayscale: epoch 03/20 complete
  grayscale: epoch 04/20 complete
  grayscale: epoch 05/20 complete
  grayscale: epoch 06/20 complete
  grayscale: epoch 07/20 complete
  grayscale: epoch 08/20 complete
  grayscale: epoch 09/20 complete
  grayscale: epoch 10/20 complete
  grayscale: epoch 11/20 complete
  grayscale: epoch 12/20 complete
  grayscale: epoch 13/20 complete
  grayscale: epoch 14/20 complete
  grayscale: epoch 15/20 complete
  grayscale: epoch 16/20 complete
  grayscale: epoch 17/20 complete
  grayscale: epoch 18/20 complete
  grayscale: epoch 19/20 complete
  grayscale: epoch 20/20 complete


NameError: name 'val_idx' is not defined

In [ ]:
# @title
"""Phase 2 diagnostics: axis errors, oracle depth, reprojection, and input ablation.

Single-file runner for Google Colab or the project checkout. It follows the
Phase 0 data/crop/split/training recipe and trains three fresh 20-epoch direct
models: grayscale only, Sobel only, and grayscale plus Sobel. It selects CUDA
automatically when available and otherwise runs on CPU. It does not download
data or write checkpoints; it prints diagnostic tables and stops.

The Phase 0 module forces CPU with CUDA_VISIBLE_DEVICES at import time, so this
script carries forward its loader/model/loss recipe locally while parameterizing
the first convolution for the requested one- and two-channel comparisons.
"""

from __future__ import annotations

import csv
import json
import os
import random
import time
from pathlib import Path

try:
    import cv2
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise SystemExit(
        "phase2_diagnose.py requires PyTorch, NumPy, and OpenCV (cv2). "
        "Install them in this runtime, then rerun."
    ) from exc


# ------------------------------- Configuration ------------------------------

SEED = 20251018
TRAIN_SEGMENTS = 44
VAL_SEGMENTS = 12
TRAIN_EPOCHS = 20
BATCH_SIZE = 64
LR = 1.2e-3
WEIGHT_DECAY = 1e-4
IMAGE_W, IMAGE_H = 128, 96  # phase0 uses W=128, H=96 (96x128 pixels).
CAMERA_ID = "400485"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
    12, 13, 14, 15, 16, 17, 19, 20, 21, 22,
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
MODES = ("grayscale", "sobel", "both")


# ----------------------------- Dataset discovery ----------------------------

def _find_manifest() -> Path:
    # In a pasted Colab cell, __file__ is not defined. In that case search
    # Colab's usual dataset location and the current working directory first.
    script_file = globals().get("__file__")
    if script_file:
        here = Path(script_file).resolve()
        project_root = here.parents[1] if len(here.parents) > 1 else here.parent
    else:
        project_root = Path.cwd()
    capture_id = (
        "m--20210701--1058--0000000--pilot--relightablehandsy--"
        "participant0--two-hands"
    )
    likely = [
        project_root / "datasets" / "reinterhand" / "captures" / capture_id
        / "derived" / "mugsy_subset" / "manifest.csv",
        Path("/content/dataset/manifest.csv"),
        Path.cwd() / "manifest.csv",
    ]
    for path in likely:
        if path.is_file():
            return path.resolve()

    roots = [Path("/content/dataset"), Path("/content"), Path.cwd()]
    for root in roots:
        if not root.is_dir():
            continue
        found = list(root.rglob("manifest.csv"))
        if found:
            return next(
                (p.resolve() for p in found if "mugsy_subset" in str(p).lower()),
                found[0].resolve(),
            )

    # If a Drive-mounted dataset is in use, mount/search only after local paths.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_root = Path("/content/drive/MyDrive")
        found = list(drive_root.rglob("manifest.csv"))
        if found:
            return next(
                (p.resolve() for p in found if "mugsy_subset" in str(p).lower()),
                found[0].resolve(),
            )
    except Exception as exc:
        print(f"Drive search unavailable: {exc}")

    raise FileNotFoundError(
        "manifest.csv not found. Put the dataset under /content/dataset or "
        "mount it in Google Drive."
    )


MANIFEST_PATH = _find_manifest()
SUBSET_DIR = MANIFEST_PATH.parent


def _find_camera_and_capture_root() -> tuple[Path, Path]:
    ancestors = (SUBSET_DIR, *SUBSET_DIR.parents)
    for parent in ancestors:
        candidate = parent / "Mugsy_cameras" / "cam_params.json"
        if candidate.is_file():
            return candidate.resolve(), parent.resolve()
        candidate = parent / "cam_params.json"
        if candidate.is_file():
            return candidate.resolve(), parent.resolve()

    for root in (SUBSET_DIR.parent, SUBSET_DIR.parent.parent, Path("/content")):
        if root.is_dir():
            found = list(root.rglob("cam_params.json"))
            if found:
                cam = next(
                    (p for p in found if p.parent.name.lower() == "mugsy_cameras"),
                    found[0],
                )
                cap = cam.parent.parent if cam.parent.name.lower() == "mugsy_cameras" else cam.parent
                return cam.resolve(), cap.resolve()

    raise FileNotFoundError("cam_params.json not found near the manifest.")


CAMERA_PATH, CAPTURE_DIR = _find_camera_and_capture_root()


def _build_asset_index() -> dict[str, set[Path]]:
    index: dict[str, set[Path]] = {}
    roots = dict.fromkeys((CAPTURE_DIR, SUBSET_DIR, SUBSET_DIR.parent))
    for root in roots:
        if not root.is_dir():
            continue
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in {
                ".json", ".png", ".jpg", ".jpeg",
            }:
                index.setdefault(path.name.lower(), set()).add(path.resolve())
    return index


ASSET_INDEX = _build_asset_index()


def resolve_asset(rel_path: str) -> Path:
    rel = str(rel_path).replace("\\", "/").strip()
    rel_path_obj = Path(rel)
    parts = rel_path_obj.parts
    candidates = [
        Path(rel), CAPTURE_DIR / rel, SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel, SUBSET_DIR.parent.parent / rel,
    ]
    if parts and parts[0] == CAPTURE_DIR.name:
        candidates.append(CAPTURE_DIR.joinpath(*parts[1:]))
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    # Try the manifest path under each ancestor; original paths can be rooted
    # above the subset directory.
    for ancestor in (SUBSET_DIR, *SUBSET_DIR.parents):
        candidate = ancestor.joinpath(*parts)
        if candidate.is_file():
            return candidate.resolve()

    # The Colab export may flatten orig_fits/right/Keypoints to
    # keypoints_right/, and similarly for left.
    lower_parts = [part.lower() for part in parts]
    side = "right" if "right" in lower_parts else (
        "left" if "left" in lower_parts else None
    )
    hits = ASSET_INDEX.get(rel_path_obj.name.lower(), set())
    if side:
        side_dir = f"keypoints_{side}"
        side_hits = {
            path for path in hits
            if side_dir in [part.lower() for part in path.parts]
        }
        if len(side_hits) == 1:
            return next(iter(side_hits))
        if len(side_hits) > 1:
            exact = {p for p in side_hits if p.parent.name.lower() == side_dir}
            if len(exact) == 1:
                return next(iter(exact))

    if len(hits) == 1:
        return next(iter(hits))
    if len(hits) > 1:
        # For images or differently named exports, prefer a unique suffix match.
        wanted_suffix = "/".join(part.lower() for part in parts[-3:])
        suffix_hits = {
            p for p in hits
            if "/".join(p.as_posix().lower().split("/")[-3:]) == wanted_suffix
        }
        if len(suffix_hits) == 1:
            return next(iter(suffix_hits))
        raise FileNotFoundError(
            f"Asset basename is ambiguous: {rel}; matches: "
            f"{[str(p) for p in list(hits)[:6]]}"
        )
    raise FileNotFoundError(f"Could not resolve manifest asset: {rel}")


# --------------------------------- Geometry ----------------------------------

def crop_affine(uv: np.ndarray, margin: float = 1.4) -> np.ndarray:
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [IMAGE_W / side, 0.0, -left * IMAGE_W / side],
            [0.0, IMAGE_H / side, -top * IMAGE_H / side],
        ],
        dtype=np.float32,
    )


def camera_project(points: np.ndarray, focal: np.ndarray, principal: np.ndarray) -> np.ndarray:
    z = points[..., 2]
    if np.any(z <= 1e-6):
        raise ValueError("Predicted point has nonpositive camera-space depth.")
    return np.stack(
        (focal[0] * points[..., 0] / z + principal[0],
         focal[1] * points[..., 1] / z + principal[1]),
        axis=-1,
    ).astype(np.float32)


def channels_from_gray(gray_u8: np.ndarray) -> dict[str, np.ndarray]:
    gray = gray_u8.astype(np.float32) / 255.0
    raw = gray_u8.astype(np.float32)
    gx = cv2.Sobel(raw, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(raw, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    sobel = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return {
        "grayscale": gray[None].astype(np.float32),
        "sobel": sobel[None].astype(np.float32),
        "both": np.stack((gray, sobel)).astype(np.float32),
    }


def normalize_uv(uv: np.ndarray) -> np.ndarray:
    normalized = np.empty_like(uv, dtype=np.float32)
    normalized[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    normalized[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return normalized


def rotate_camera_xy(y3d: np.ndarray, angle_rad: float) -> np.ndarray:
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    out = y3d.copy()
    x, y = y3d[:, 0].copy(), y3d[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment_one(gray_u8: np.ndarray, uv: np.ndarray, y3d: np.ndarray, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle_rad = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    matrix[1, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_H)
    warped = cv2.warpAffine(
        gray_u8, matrix, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(warped.astype(np.float32) * brightness + offset, 0, 255).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle_rad)
    return channels_from_gray(warped), y3d_aug, normalize_uv(uv_aug)


# -------------------------- Model copied from Phase 0 ------------------------

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)


def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    batch = bone_vectors.shape[0]
    joints = torch.zeros((batch, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + bone_vectors[:, idx]
    return joints


class WebcamAlignedLCFTracker(nn.Module):
    """Phase 0 LCF tracker; only its first convolution is channel-configurable."""
    def __init__(self, canonical_lens: torch.Tensor, in_channels: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512), nn.ReLU(inplace=True), nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.proj2d_head = nn.Linear(512, 21 * 2)
        self.register_buffer("bone_lens", canonical_lens.clone())

    def forward(self, x: torch.Tensor):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3), dim=-1, eps=1e-7
        )
        latent_frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3), dim=-1, eps=1e-7
        )
        phalanx_cam_u = torch.bmm(
            latent_frame, phalanx_local_u.transpose(1, 2)
        ).transpose(1, 2)
        bone_dirs = torch.zeros((batch, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        for finger_idx, bone_offset in enumerate((4, 8, 12, 16)):
            phalanx_offset = 3 + finger_idx * 3
            bone_dirs[:, bone_offset] = meta_cam_u[:, finger_idx + 1]
            bone_dirs[:, bone_offset + 1:bone_offset + 4] = (
                phalanx_cam_u[:, phalanx_offset:phalanx_offset + 3]
            )
        bone_vectors = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vectors)
        proj2d = torch.tanh(self.proj2d_head(feat).reshape(batch, 21, 2))
        return joints, bone_dirs, proj2d


class MetricLossEvaluator:
    """Same Phase 0 position, bone-length, and bone-direction objective."""
    def __call__(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        l_pos = torch.norm(pred - target, dim=-1).mean()
        parent_ids = [p for p, _ in BONE_PAIRS]
        child_ids = [c for _, c in BONE_PAIRS]
        pred_bones = pred[:, child_ids] - pred[:, parent_ids]
        target_bones = target[:, child_ids] - target[:, parent_ids]
        pred_len = torch.norm(pred_bones, dim=-1)
        target_len = torch.norm(target_bones, dim=-1)
        l_len = F.l1_loss(pred_len, target_len)
        pred_u = pred_bones / (pred_len.unsqueeze(-1) + 1e-7)
        target_u = target_bones / (target_len.unsqueeze(-1) + 1e-7)
        cos_sim = torch.clamp((pred_u * target_u).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir


# -------------------------------- Data loader --------------------------------

def load_data():
    with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r.get("camera_id", CAMERA_ID) == CAMERA_ID]
    if not rows:
        raise RuntimeError(f"Manifest has no rows for camera {CAMERA_ID}.")

    with CAMERA_PATH.open("r", encoding="utf-8") as f:
        camera_root = json.load(f)
    camera = camera_root[CAMERA_ID]
    rotation = np.asarray(camera["R"], dtype=np.float32)
    translation = np.asarray(camera["t"], dtype=np.float32).reshape(3)
    focal = np.asarray(camera["focal"], dtype=np.float32).reshape(2)
    principal = np.asarray(camera["princpt"], dtype=np.float32).reshape(2)

    records = []
    image_size = None
    for index, row in enumerate(rows):
        kp_path = resolve_asset(row["orig_kp_right"])
        image_path = resolve_asset(row["image_path"])
        with kp_path.open("r", encoding="utf-8") as f:
            kp_world = np.asarray(json.load(f), dtype=np.float32)
        if kp_world.shape != (24, 3):
            raise ValueError(
                f"{row.get('frame_id', index)}: expected 24x3 right keypoints, "
                f"got {kp_world.shape}."
            )

        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")
        h, w = image.shape
        if image_size is None:
            image_size = (w, h)
        elif image_size != (w, h):
            raise ValueError("Image dimensions vary within the manifest.")

        camera_points = kp_world @ rotation.T + translation
        selected = camera_points[KEEP_21_INDICES]
        z = selected[:, 2]
        if np.any(z <= 1e-6):
            raise ValueError(f"Invalid camera depth in {row.get('frame_id', index)}")
        uv = np.stack(
            (focal[0] * selected[:, 0] / z + principal[0],
             focal[1] * selected[:, 1] / z + principal[1]),
            axis=1,
        ).astype(np.float32)
        affine = crop_affine(uv, margin=1.4)
        crop = cv2.warpAffine(
            image, affine, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )
        uv_crop = uv @ affine[:, :2].T + affine[:, 2]
        root_camera = selected[0].copy()
        root_relative = selected - root_camera[None, :]

        records.append({
            "frame_id": row.get("frame_id", str(index)),
            "segment": row["segment"],
            "crop_gray": crop,
            "uv_crop": uv_crop.astype(np.float32),
            "uv_image": uv,
            "root_camera": root_camera.astype(np.float32),
            "y3d": root_relative.astype(np.float32),
        })

    return records, image_size, focal, principal


# ------------------------------- Split/training ------------------------------

def build_split(records):
    segments = np.asarray(sorted({r["segment"] for r in records}), dtype=object)
    if len(segments) != TRAIN_SEGMENTS + VAL_SEGMENTS:
        raise ValueError(
            f"Expected 56 segments for the 44/12 split; found {len(segments)}."
        )
    split_rng = np.random.default_rng(SEED)
    split_rng.shuffle(segments)
    train_segments = set(segments[:TRAIN_SEGMENTS].tolist())
    val_segments = set(segments[TRAIN_SEGMENTS:].tolist())
    if train_segments & val_segments:
        raise AssertionError("Segment leakage in train/validation split.")
    train_idx = np.asarray(
        [i for i, r in enumerate(records) if r["segment"] in train_segments],
        dtype=np.int64,
    )
    val_idx = np.asarray(
        [i for i, r in enumerate(records) if r["segment"] in val_segments],
        dtype=np.int64,
    )
    return train_segments, val_segments, train_idx, val_idx


def make_base_features(records, mode: str, device: torch.device) -> torch.Tensor:
    return torch.from_numpy(np.stack([
        channels_from_gray(record["crop_gray"])[mode] for record in records
    ])).to(device)


def augment_mode(gray_u8, uv, y3d, rng, mode):
    channels, y3d_aug, y2d_aug = augment_one(gray_u8, uv, y3d, rng)
    return channels[mode], y3d_aug, y2d_aug


def train_one_mode(records, train_idx, val_idx, y_all, y2d_all, mode, device):
    # Reset RNGs before each ablation so initialization, sample order, and
    # augmentation randomness are paired across the three input modes.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    x_all = make_base_features(records, mode, device)
    train_y = y_all[train_idx]
    canonical_lens = torch.from_numpy(np.stack([
        np.linalg.norm(
            train_y[:, child] - train_y[:, parent], axis=1
        ).mean()
        for parent, child in BONE_PAIRS
    ]).astype(np.float32))
    model = WebcamAlignedLCFTracker(
        canonical_lens, in_channels=x_all.shape[1]
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4
    )
    criterion = MetricLossEvaluator()
    aug_rng = np.random.default_rng(SEED + 1)
    started = time.perf_counter()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        model.train()
        order = np.random.permutation(train_idx)
        for start in range(0, len(order), BATCH_SIZE):
            batch_ids = order[start:start + BATCH_SIZE]
            samples = [
                augment_mode(
                    records[int(i)]["crop_gray"],
                    records[int(i)]["uv_crop"],
                    y_all[int(i)],
                    aug_rng,
                    mode,
                )
                for i in batch_ids
            ]
            bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(device)
            by3 = torch.from_numpy(np.stack([s[1] for s in samples])).to(device)
            by2 = torch.from_numpy(np.stack([s[2] for s in samples])).to(device)

            optimizer.zero_grad(set_to_none=True)
            pred3, _, pred2 = model(bx)
            loss = criterion(pred3, by3) + 15.0 * F.smooth_l1_loss(pred2, by2)
            loss.backward()
            optimizer.step()
        scheduler.step()
        print(f"  {mode}: epoch {epoch:02d}/{TRAIN_EPOCHS} complete")

    elapsed = time.perf_counter() - started
    model.eval()
    val_pred_batches = []
    with torch.inference_mode():
        for start in range(0, len(val_idx), 128):
            ids = val_idx[start:start + 128]
            pred, _, _ = model(x_all[ids])
            val_pred_batches.append(pred.cpu().numpy())
    predictions = np.concatenate(val_pred_batches, axis=0).astype(np.float32)
    return predictions, elapsed


# --------------------------------- Metrics ------------------------------------

def mean_joint_distance(pred, target):
    return np.linalg.norm(pred - target, axis=-1).mean(axis=0)


def component_mae(pred, target):
    return np.abs(pred - target).mean(axis=0)


def reprojection_metrics(pred_root_relative, target_root_relative, records,
                         val_idx, focal, principal):
    # Add the GT camera-space root to both prediction and label. This keeps the
    # root translation/depth known and evaluates the real calibrated projection.
    pred_camera = np.stack([
        pred + records[int(i)]["root_camera"][None, :]
        for i, pred in zip(val_idx, pred_root_relative)
    ])
    target_uv = np.stack([records[int(i)]["uv_image"] for i in val_idx])
    pred_uv = camera_project(pred_camera, focal, principal)
    pixel_error = np.linalg.norm(pred_uv - target_uv, axis=-1)
    return pixel_error.mean(axis=0)


def report_mode(mode, pred, records, val_idx, y_all, focal, principal):
    target = y_all[val_idx]
    xyz_mae = component_mae(pred, target)
    joint_mpjpe = mean_joint_distance(pred, target)
    reproj = reprojection_metrics(pred, target, records, val_idx, focal, principal)

    oracle_pred = pred.copy()
    oracle_pred[..., 2] = target[..., 2]
    oracle_mpjpe = mean_joint_distance(oracle_pred, target)

    # Component table: overall averages over all joints, plus each tip.
    print(f"\n[{mode.upper()}] AXIS COMPONENT MAE AND JOINT MPJPE (mm)")
    print(f"{'Point':<10}{'MPJPE':>10}{'|dx|':>10}{'|dy|':>10}{'|dz|':>10}")
    print("-" * 50)
    print(
        f"{'Overall':<10}{joint_mpjpe.mean():10.2f}"
        f"{xyz_mae[:, 0].mean():10.2f}{xyz_mae[:, 1].mean():10.2f}"
        f"{xyz_mae[:, 2].mean():10.2f}"
    )
    for name, j in zip(FINGER_NAMES, FINGERTIP_JOINTS):
        print(
            f"{name:<10}{joint_mpjpe[j]:10.2f}"
            f"{xyz_mae[j, 0]:10.2f}{xyz_mae[j, 1]:10.2f}"
            f"{xyz_mae[j, 2]:10.2f}"
        )

    print(f"\n[{mode.upper()}] ORACLE-DEPTH MPJPE (mm; replace predicted relative z with GT)")
    print(f"{'Point':<10}{'MPJPE':>10}")
    print("-" * 20)
    print(f"{'Overall':<10}{oracle_mpjpe.mean():10.2f}")
    for name, j in zip(FINGER_NAMES, FINGERTIP_JOINTS):
        print(f"{name:<10}{oracle_mpjpe[j]:10.2f}")

    print(f"\n[{mode.upper()}] REAL-CAMERA 2D REPROJECTION ERROR (pixels; GT root translation/depth)")
    print(f"{'Point':<10}{'Mean px':>10}")
    print("-" * 20)
    print(f"{'Overall':<10}{reproj.mean():10.2f}")
    for name, j in zip(FINGER_NAMES, FINGERTIP_JOINTS):
        print(f"{name:<10}{reproj[j]:10.2f}")

    segment_rows = []
    for segment in sorted({records[int(i)]["segment"] for i in val_idx}):
        local_positions = [
            pos for pos, i in enumerate(val_idx)
            if records[int(i)]["segment"] == segment
        ]
        segment_pred = pred[local_positions]
        segment_target = target[local_positions]
        segment_mpjpe = np.linalg.norm(
            segment_pred - segment_target, axis=-1
        ).mean()
        segment_rows.append((segment, len(local_positions), float(segment_mpjpe)))

    print(f"\n[{mode.upper()}] PER-SEGMENT MPJPE (mm; sorted by segment)")
    print(f"{'Segment':<58}{'Frames':>8}{'MPJPE':>10}")
    print("-" * 76)
    for segment, frames, value in segment_rows:
        print(f"{segment:<58}{frames:8d}{value:10.2f}")

    return float(joint_mpjpe.mean()), float(oracle_mpjpe.mean()), float(reproj.mean())


def main():
    wall_start = time.perf_counter()
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if DEVICE.type == "cpu":
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

    print(f"Active compute device: {DEVICE}")
    if DEVICE.type == "cuda":
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Manifest: {MANIFEST_PATH}")
    print(f"Camera:   {CAMERA_PATH} (camera {CAMERA_ID})")

    records, image_size, focal, principal = load_data()
    train_segments, val_segments, train_idx, val_idx = build_split(records)
    y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)

    print(
        f"Loaded {len(records):,} frames ({image_size[0]}x{image_size[1]}), "
        f"split {len(train_segments)}/{len(val_segments)} segments, "
        f"{len(train_idx):,}/{len(val_idx):,} frames."
    )
    print("Training three paired 20-epoch Phase 0 recipe runs.")

    summary = []
    for mode in MODES:
        print(f"\nTraining input mode: {mode}")
        pred, elapsed = train_one_mode(
            records, train_idx, val_idx, y_all,
            np.stack([normalize_uv(r["uv_crop"]) for r in records]).astype(np.float32),
            mode, DEVICE,
        )
        metrics = report_mode(
            mode, pred, records, val_idx, y_all, focal, principal
        )
        summary.append((mode, *metrics, elapsed))

    print("\nINPUT ABLATION SUMMARY (validation)")
    print(f"{'Input':<12}{'MPJPE mm':>12}{'OracleZ mm':>12}{'2D px':>10}{'Train sec':>12}")
    print("-" * 58)
    for mode, mpjpe, oracle, reproj, elapsed in summary:
        print(f"{mode:<12}{mpjpe:12.2f}{oracle:12.2f}{reproj:10.2f}{elapsed:12.1f}")

    print("\nAssumptions:")
    print("- The phase0 KEEP_21_INDICES mapping selects the 21 right-hand joints.")
    print("- Camera-space labels, root-relative targets, and errors use millimeters.")
    print("- Axis components are mean absolute coordinate errors (|dx|, |dy|, |dz|); MPJPE is Euclidean 3D distance.")
    print("- Oracle depth replaces each predicted root-relative joint z with its GT root-relative z.")
    print("- Reprojection adds the GT camera-space root to predicted relative joints and uses the official camera intrinsics.")
    print("- No mirroring is applied; labels remain right-hand/chirality-consistent.")
    print(f"Total runtime: {time.perf_counter() - wall_start:.1f}s")


if __name__ == "__main__":
    main()


Active compute device: cpu
Manifest: /content/dataset/manifest.csv
Camera:   /content/dataset/cam_params.json (camera 400485)
Loaded 5,114 frames (667x1024), split 44/12 segments, 4,246/868 frames.
Training three paired 20-epoch Phase 0 recipe runs.

Training input mode: grayscale
  grayscale: epoch 01/20 complete
  grayscale: epoch 02/20 complete
  grayscale: epoch 03/20 complete
  grayscale: epoch 04/20 complete
  grayscale: epoch 05/20 complete
  grayscale: epoch 06/20 complete
  grayscale: epoch 07/20 complete
  grayscale: epoch 08/20 complete
  grayscale: epoch 09/20 complete
  grayscale: epoch 10/20 complete
  grayscale: epoch 11/20 complete
  grayscale: epoch 12/20 complete
  grayscale: epoch 13/20 complete
  grayscale: epoch 14/20 complete
  grayscale: epoch 15/20 complete
  grayscale: epoch 16/20 complete
  grayscale: epoch 17/20 complete
  grayscale: epoch 18/20 complete
  grayscale: epoch 19/20 complete
  grayscale: epoch 20/20 complete

[GRAYSCALE] AXIS COMPONENT MAE AND JO

In [ ]:
# @title
# MediaPipe 1.1.1 Tasks API detection and chirality audit.
# Does not restart the runtime or replace the installed mediapipe package.

import os
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np

# --------------------------- Configuration ----------------------------------

MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError(
        f"MediaPipe {getattr(mp, '__version__', 'unknown')} does not expose "
        "mp.tasks.vision.HandLandmarker."
    )

if not MP_MODEL_PATH.is_file():
    print(f"Downloading MediaPipe hand model to {MP_MODEL_PATH} ...")
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

# In the legacy Solutions API, HAND_CONNECTIONS defines the 21-point hand
# skeleton. Tasks API does not expose that constant, so define the same edges.
MP_HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
]

# Assert the project joint ordering has the expected MediaPipe finger chains.
# Joint 0 is wrist; then thumb, index, middle, ring, pinky, four joints each.
EXPECTED_MP_BONES = {
    tuple(sorted(edge)) for edge in MP_HAND_CONNECTIONS
}
PROJECT_BONES = {
    tuple(sorted((int(a), int(b)))) for a, b in BONE_PAIRS
}
assert EXPECTED_MP_BONES == PROJECT_BONES, (
    "KEEP_21_INDICES/BONE_PAIRS do not match MediaPipe's 21-landmark "
    "wrist-thumb-index-middle-ring-pinky ordering."
)

# ------------------------------- Helpers ------------------------------------

def make_mp_crop(gray, uv):
    """Square crop around projected GT points, enlarged to at least 256 px."""
    uv = np.asarray(uv, dtype=np.float32)
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5

    side = max(
        float(hi[0] - lo[0]),
        float(hi[1] - lo[1]),
        16.0,
    ) * MP_CROP_MARGIN
    out_size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))

    left = center[0] - side * 0.5
    top = center[1] - side * 0.5
    scale = out_size / side

    affine = np.asarray(
        [
            [scale, 0.0, -left * scale],
            [0.0, scale, -top * scale],
        ],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        gray,
        affine,
        (out_size, out_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    gt_crop_uv = uv @ affine[:, :2].T + affine[:, 2]
    return crop, gt_crop_uv


def _result_hands(result, field_name):
    """Return a list for either populated or empty Tasks API result fields."""
    value = getattr(result, field_name, None)
    return [] if value is None else value


def extract_candidates(result, flipped, crop_size, gt_crop_uv):
    """Convert a Tasks result into scored, root-relative candidate poses."""
    candidates = []
    image_hands = _result_hands(result, "hand_landmarks")
    world_hands = _result_hands(result, "hand_world_landmarks")
    handedness_hands = _result_hands(result, "handedness")

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue

        xy = np.asarray(
            [[lm.x * crop_size, lm.y * crop_size] for lm in image_hand],
            dtype=np.float32,
        )

        # Return flipped image detections to original crop coordinates before
        # comparing them to the unflipped GT projection.
        if flipped:
            xy[:, 0] = (crop_size - 1) - xy[:, 0]

        pixel_error = np.linalg.norm(xy - gt_crop_uv, axis=-1)
        pixel_rmse = float(np.sqrt(np.mean(pixel_error ** 2)))

        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )

        # MediaPipe world landmarks are meters, centered at the hand's
        # geometric center. Convert to mm and make the wrist the root.
        world_root_mm = (world_m - world_m[0:1]) * 1000.0

        # Horizontal image mirroring reverses image chirality. Undo the
        # corresponding camera-space x reflection for the flipped candidate.
        if flipped:
            world_root_mm[:, 0] *= -1.0

        handedness = None
        if hand_i < len(handedness_hands):
            try:
                handedness = handedness_hands[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                try:
                    handedness = handedness_hands[hand_i][0].display_name
                except (AttributeError, IndexError, TypeError):
                    handedness = None

        candidates.append(
            {
                "score_px": pixel_rmse,
                "pose": world_root_mm.astype(np.float32),
                "flipped": bool(flipped),
                "handedness": handedness,
            }
        )

    return candidates


def create_hand_landmarker():
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(MP_MODEL_PATH)
        ),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)


# ------------------------ Run validation detections -------------------------

mp_predictions = [None] * len(val_idx)
flip_choices = {"normal": 0, "flipped": 0}
flip_counts_by_segment = {
    segment: {
        "normal": 0,
        "flipped": 0,
        "detected": 0,
        "frames": 0,
    }
    for segment in sorted(val_segments)
}
match_errors_px = []
handedness_counts = {}

with create_hand_landmarker() as hand_landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        rec = records[int(dataset_i)]

        # This section expects each record to retain the source grayscale
        # image under "gray", and projected right-hand GT points under
        # "uv_image", as in the earlier Phase 0/2 loaders.
        gray = rec.get("gray")
        if gray is None:
            raise KeyError(
                "records entries need a 'gray' source-image array for the "
                "MediaPipe crop. The earlier loader must retain it."
            )

        crop, gt_crop_uv = make_mp_crop(gray, rec["uv_image"])
        crop_size = crop.shape[0]
        candidates = []

        normal_rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        normal_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=np.ascontiguousarray(normal_rgb),
        )
        normal_result = hand_landmarker.detect(normal_image)
        candidates.extend(
            extract_candidates(
                normal_result,
                flipped=False,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        flipped_crop = cv2.flip(crop, 1)
        flipped_rgb = cv2.cvtColor(flipped_crop, cv2.COLOR_GRAY2RGB)
        flipped_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=np.ascontiguousarray(flipped_rgb),
        )
        flipped_result = hand_landmarker.detect(flipped_image)
        candidates.extend(
            extract_candidates(
                flipped_result,
                flipped=True,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        segment = rec["segment"]
        if segment not in flip_counts_by_segment:
            flip_counts_by_segment[segment] = {
                "normal": 0,
                "flipped": 0,
                "detected": 0,
                "frames": 0,
            }
        flip_counts_by_segment[segment]["frames"] += 1

        if candidates:
            best = min(candidates, key=lambda item: item["score_px"])
            mp_predictions[val_pos] = best["pose"]

            choice = "flipped" if best["flipped"] else "normal"
            flip_choices[choice] += 1
            flip_counts_by_segment[segment][choice] += 1
            flip_counts_by_segment[segment]["detected"] += 1
            match_errors_px.append(best["score_px"])

            label = best["handedness"] or "unknown"
            handedness_counts[label] = handedness_counts.get(label, 0) + 1

        if (val_pos + 1) % 100 == 0 or val_pos + 1 == len(val_idx):
            print(
                f"MediaPipe processed {val_pos + 1}/{len(val_idx)} "
                "validation frames"
            )

detected = np.asarray(
    [pose is not None for pose in mp_predictions],
    dtype=bool,
)
detection_rate = float(detected.mean()) if len(detected) else 0.0

print("\nMEDIAPIPE DETECTION / CHIRALITY AUDIT")
print(
    f"Detected: {int(detected.sum())}/{len(val_idx)} "
    f"({detection_rate:.2%})"
)
if match_errors_px:
    print(
        f"Best match chosen from normal crops: {flip_choices['normal']}; "
        f"horizontally flipped crops: {flip_choices['flipped']}; "
        f"mean 2D match RMSE: {np.mean(match_errors_px):.2f}px"
    )
else:
    print("No hand detections matched the validation frames.")
print(f"MediaPipe handedness labels: {handedness_counts}")

print("\nDETECTION RATE BY VALIDATION SEGMENT")
print(f"{'Segment':<32} {'Detected':>10} {'Frames':>8} {'Rate':>9}")
print("-" * 64)
for segment in sorted(flip_counts_by_segment):
    row = flip_counts_by_segment[segment]
    rate = row["detected"] / max(row["frames"], 1)
    print(
        f"{str(segment):<32} {row['detected']:>10} "
        f"{row['frames']:>8} {rate:>8.2%}"
    )

AssertionError: KEEP_21_INDICES/BONE_PAIRS do not match MediaPipe's 21-landmark wrist-thumb-index-middle-ring-pinky ordering.

In [ ]:
# @title
# MediaPipe 1.1.1 Tasks API detection and chirality audit.
# Keeps the current Colab runtime and installed MediaPipe version.

import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np

# --------------------------- Configuration ----------------------------------

MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError(
        f"MediaPipe {getattr(mp, '__version__', 'unknown')} "
        "does not expose mp.tasks.vision."
    )

if not MP_MODEL_PATH.is_file():
    print(f"Downloading MediaPipe hand model to {MP_MODEL_PATH} ...")
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

# MediaPipe landmark numbering is wrist=0, then four thumb landmarks,
# four index landmarks, four middle landmarks, four ring landmarks,
# and four pinky landmarks.
#
# HAND_CONNECTIONS is a drawing graph with palm cross-links, so it must not
# be compared directly with the project's finger-chain BONE_PAIRS.
EXPECTED_CHAIN_BONES = {
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
}
PROJECT_CHAIN_BONES = {
    tuple(sorted((int(parent), int(child))))
    for parent, child in BONE_PAIRS
}
assert EXPECTED_CHAIN_BONES == PROJECT_CHAIN_BONES, (
    "BONE_PAIRS does not match the assumed MediaPipe 21-landmark "
    "wrist/thumb/index/middle/ring/pinky chain ordering."
)

# ------------------------------- Helpers ------------------------------------

def make_mp_crop(gray, uv):
    """Square crop around projected GT points, enlarged to at least 256 px."""
    uv = np.asarray(uv, dtype=np.float32)
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5

    side = max(
        float(hi[0] - lo[0]),
        float(hi[1] - lo[1]),
        16.0,
    ) * MP_CROP_MARGIN
    out_size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))

    left = center[0] - side * 0.5
    top = center[1] - side * 0.5
    scale = out_size / side

    affine = np.asarray(
        [
            [scale, 0.0, -left * scale],
            [0.0, scale, -top * scale],
        ],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        gray,
        affine,
        (out_size, out_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    gt_crop_uv = uv @ affine[:, :2].T + affine[:, 2]
    return crop, gt_crop_uv


def _result_hands(result, field_name):
    value = getattr(result, field_name, None)
    return [] if value is None else value


def extract_candidates(result, flipped, crop_size, gt_crop_uv):
    """Convert Tasks results into scored, root-relative candidate poses."""
    candidates = []
    image_hands = _result_hands(result, "hand_landmarks")
    world_hands = _result_hands(result, "hand_world_landmarks")
    handedness_hands = _result_hands(result, "handedness")

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue

        xy = np.asarray(
            [[lm.x * crop_size, lm.y * crop_size] for lm in image_hand],
            dtype=np.float32,
        )

        # Undo image mirroring before comparing with the unflipped GT points.
        if flipped:
            xy[:, 0] = (crop_size - 1) - xy[:, 0]

        pixel_rmse = float(
            np.sqrt(np.mean(np.sum((xy - gt_crop_uv) ** 2, axis=-1)))
        )

        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )

        # MediaPipe world landmarks are in meters. Convert to mm and express
        # the prediction relative to its wrist.
        world_root_mm = (world_m - world_m[0:1]) * 1000.0

        # Approximate undoing of horizontal image mirroring in camera x.
        if flipped:
            world_root_mm[:, 0] *= -1.0

        handedness = None
        if hand_i < len(handedness_hands):
            try:
                handedness = handedness_hands[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                try:
                    handedness = handedness_hands[hand_i][0].display_name
                except (AttributeError, IndexError, TypeError):
                    handedness = None

        candidates.append(
            {
                "score_px": pixel_rmse,
                "pose": world_root_mm.astype(np.float32),
                "flipped": bool(flipped),
                "handedness": handedness,
            }
        )

    return candidates


def create_hand_landmarker():
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(MP_MODEL_PATH)
        ),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)


def get_record_gray(record):
    """Accept either a full grayscale frame or a stored grayscale crop."""
    gray = record.get("gray")
    if gray is None:
        gray = record.get("gray_image")
    if gray is None:
        raise KeyError(
            "Record has no source grayscale image. Store it as "
            "records[i]['gray'] in the data-loading section."
        )
    if gray.ndim != 2:
        raise ValueError("Expected a single-channel grayscale image.")
    return gray


# ------------------------ Run validation detections -------------------------

mp_predictions = [None] * len(val_idx)
flip_choices = {"normal": 0, "flipped": 0}
flip_counts_by_segment = {
    segment: {
        "normal": 0,
        "flipped": 0,
        "detected": 0,
        "frames": 0,
    }
    for segment in sorted(val_segments)
}
match_errors_px = []
handedness_counts = {}

with create_hand_landmarker() as hand_landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        rec = records[int(dataset_i)]
        gray = get_record_gray(rec)

        crop, gt_crop_uv = make_mp_crop(gray, rec["uv_image"])
        crop_size = crop.shape[0]
        candidates = []

        normal_rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        normal_result = hand_landmarker.detect(
            mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=np.ascontiguousarray(normal_rgb),
            )
        )
        candidates.extend(
            extract_candidates(
                normal_result,
                flipped=False,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        flipped_crop = cv2.flip(crop, 1)
        flipped_rgb = cv2.cvtColor(flipped_crop, cv2.COLOR_GRAY2RGB)
        flipped_result = hand_landmarker.detect(
            mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=np.ascontiguousarray(flipped_rgb),
            )
        )
        candidates.extend(
            extract_candidates(
                flipped_result,
                flipped=True,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        segment = rec["segment"]
        if segment not in flip_counts_by_segment:
            flip_counts_by_segment[segment] = {
                "normal": 0,
                "flipped": 0,
                "detected": 0,
                "frames": 0,
            }
        flip_counts_by_segment[segment]["frames"] += 1

        if candidates:
            best = min(candidates, key=lambda item: item["score_px"])
            mp_predictions[val_pos] = best["pose"]

            choice = "flipped" if best["flipped"] else "normal"
            flip_choices[choice] += 1
            flip_counts_by_segment[segment][choice] += 1
            flip_counts_by_segment[segment]["detected"] += 1
            match_errors_px.append(best["score_px"])

            label = best["handedness"] or "unknown"
            handedness_counts[label] = handedness_counts.get(label, 0) + 1

        if (val_pos + 1) % 100 == 0 or val_pos + 1 == len(val_idx):
            print(
                f"MediaPipe processed {val_pos + 1}/{len(val_idx)} "
                "validation frames"
            )

detected = np.asarray(
    [pose is not None for pose in mp_predictions],
    dtype=bool,
)
detection_rate = float(detected.mean()) if len(detected) else 0.0

print("\nMEDIAPIPE DETECTION / CHIRALITY AUDIT")
print(
    f"Detected: {int(detected.sum())}/{len(val_idx)} "
    f"({detection_rate:.2%})"
)
if match_errors_px:
    print(
        f"Best match from normal crops: {flip_choices['normal']}; "
        f"from flipped crops: {flip_choices['flipped']}; "
        f"mean 2D match RMSE: {np.mean(match_errors_px):.2f}px"
    )
else:
    print("No hand detections matched the validation frames.")
print(f"MediaPipe handedness labels: {handedness_counts}")

print("\nDETECTION RATE BY VALIDATION SEGMENT")
print(f"{'Segment':<32} {'Detected':>10} {'Frames':>8} {'Rate':>9}")
print("-" * 64)
for segment in sorted(flip_counts_by_segment):
    row = flip_counts_by_segment[segment]
    rate = row["detected"] / max(row["frames"], 1)
    print(
        f"{str(segment):<32} {row['detected']:>10} "
        f"{row['frames']:>8} {rate:>8.2%}"
    )

MediaPipe processed 100/868 validation frames
MediaPipe processed 200/868 validation frames
MediaPipe processed 300/868 validation frames
MediaPipe processed 400/868 validation frames
MediaPipe processed 500/868 validation frames
MediaPipe processed 600/868 validation frames
MediaPipe processed 700/868 validation frames
MediaPipe processed 800/868 validation frames
MediaPipe processed 868/868 validation frames

MEDIAPIPE DETECTION / CHIRALITY AUDIT
Detected: 464/868 (53.46%)
Best match from normal crops: 252; from flipped crops: 212; mean 2D match RMSE: 36.13px
MediaPipe handedness labels: {'Left': 185, 'Right': 279}

DETECTION RATE BY VALIDATION SEGMENT
Segment                            Detected   Frames      Rate
----------------------------------------------------------------
Rom0031_L_index_rubbing_back_of_R_hand         62      139   44.60%
Rom0040_L_hand_pulling_right_palm_skin        157      209   75.12%
T0004_right_fist_cover_left              22       44   50.00%
T0005_left_

In [ ]:
# @title
# =============================================================================
# END-TO-END COLAB: COLOR MEDIAPIPE + SOBEL DIRECT REGRESSOR
# Finds/loads dataset, creates the deterministic 44/12 segment split,
# reruns MediaPipe on original color frames, trains Sobel model for 20 epochs,
# and reports results on detected frames and >=90%-detection segments.
# =============================================================================

import csv
import json
import os
import random
import time
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------- Configuration ------------------------------

SEED = 20251018
CAMERA_ID = "400485"
TRAIN_SEGMENT_COUNT = 44
VAL_SEGMENT_COUNT = 12
IMAGE_W, IMAGE_H = 128, 96
MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256
TRAIN_CROP_MARGIN = 1.4
TRAIN_EPOCHS = 20
BATCH_SIZE = 64
LR = 1.2e-3
WEIGHT_DECAY = 1e-4

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

print(
    f"Active compute device: {DEVICE} "
    f"({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'})"
)
print(f"MediaPipe version: {getattr(mp, '__version__', 'unknown')}")

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError(
        "This cell needs MediaPipe with mp.tasks.vision.HandLandmarker. "
        "The current runtime does not expose that API."
    )

# ------------------------------- Locate data ---------------------------------

def _manifest_candidates(root):
    if not root.exists():
        return []
    try:
        return list(root.rglob("manifest.csv"))
    except Exception:
        return []


def _choose_manifest(hits):
    if not hits:
        return None
    return next(
        (path for path in hits if "mugsy_subset" in str(path).lower()),
        hits[0],
    )


MANIFEST_PATH = None
for search_root in (DATASET_DIR, Path("/content"), Path.cwd()):
    MANIFEST_PATH = _choose_manifest(_manifest_candidates(search_root))
    if MANIFEST_PATH is not None:
        break

if MANIFEST_PATH is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped or unavailable: {exc}")
    MANIFEST_PATH = _choose_manifest(_manifest_candidates(DRIVE_ROOT))

if MANIFEST_PATH is None:
    raise FileNotFoundError(
        "Could not find manifest.csv under /content or Google Drive. "
        "Set DATASET_DIR to the extracted dataset directory and rerun."
    )

SUBSET_DIR = MANIFEST_PATH.parent


def _find_camera_file(subset_dir):
    for parent in (subset_dir, *subset_dir.parents):
        for candidate in (
            parent / "Mugsy_cameras" / "cam_params.json",
            parent / "cam_params.json",
        ):
            if candidate.is_file():
                return candidate

    search_roots = [subset_dir.parent, subset_dir.parent.parent, Path("/content")]
    if DRIVE_ROOT.exists() and str(subset_dir).startswith(str(DRIVE_ROOT)):
        search_roots.append(DRIVE_ROOT)

    for root in search_roots:
        if root.exists():
            hits = list(root.rglob("cam_params.json"))
            if hits:
                return next(
                    (p for p in hits if p.parent.name.lower() == "mugsy_cameras"),
                    hits[0],
                )
    return None


CAMERA_PATH = _find_camera_file(SUBSET_DIR)
if CAMERA_PATH is None:
    raise FileNotFoundError(
        "Could not find cam_params.json near the manifest or in /content."
    )

# Capture root is the nearest ancestor containing the camera directory, or
# otherwise the manifest's parent.
CAPTURE_DIR = SUBSET_DIR
for parent in (SUBSET_DIR, *SUBSET_DIR.parents):
    if (
        (parent / "Mugsy_cameras" / "cam_params.json").is_file()
        or (parent / "cam_params.json").is_file()
        or (parent / "keypoints_right").is_dir()
    ):
        CAPTURE_DIR = parent
        break

print(f"Manifest: {MANIFEST_PATH}")
print(f"Capture root: {CAPTURE_DIR}")
print(f"Camera: {CAMERA_PATH}")

with MANIFEST_PATH.open(newline="", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))
rows = [
    row for row in manifest_rows
    if row.get("camera_id", CAMERA_ID) == CAMERA_ID
]
if not rows:
    raise RuntimeError(f"No manifest rows found for camera {CAMERA_ID}.")

# Build one deduplicated filename index. In flattened exports, keypoints and
# images may be in keypoints_right/ and images/ rather than manifest subfolders.
print("Indexing dataset files...")
ASSET_INDEX = {}
for root in dict.fromkeys(
    [CAPTURE_DIR, SUBSET_DIR, SUBSET_DIR.parent, SUBSET_DIR.parent.parent]
):
    if not root.exists():
        continue
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in {
            ".json", ".png", ".jpg", ".jpeg"
        }:
            ASSET_INDEX.setdefault(path.name.lower(), set()).add(path.resolve())


def resolve_asset(rel_path, kind):
    rel = str(rel_path).replace("\\", "/").strip()
    rel_path_obj = Path(rel)
    parts = rel_path_obj.parts
    basename = rel_path_obj.name.lower()

    # Literal and likely-root relative paths first.
    for candidate in (
        Path(rel),
        CAPTURE_DIR / rel,
        SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel,
        SUBSET_DIR.parent.parent / rel,
    ):
        if candidate.is_file():
            return candidate.resolve()

    hits = list(ASSET_INDEX.get(basename, set()))

    if kind == "keypoint":
        # orig_fits/right/... must resolve to keypoints_right, never the
        # same-basename keypoints_left file.
        path_lower = [part.lower() for part in parts]
        side = "right" if "right" in path_lower else (
            "left" if "left" in path_lower else None
        )
        if side:
            side_hits = [
                p for p in hits
                if f"keypoints_{side}" in str(p).replace("\\", "/").lower()
            ]
            if side_hits:
                hits = side_hits
    elif kind == "image":
        image_hits = [
            p for p in hits
            if any(
                marker in str(p).replace("\\", "/").lower()
                for marker in ("/images/", "/image/", "/rgb/")
            )
        ]
        if image_hits:
            hits = image_hits

    hits = sorted(set(hits))
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        # Prefer an exact expected side folder for keypoints.
        if kind == "keypoint":
            right_hits = [
                p for p in hits
                if "keypoints_right" in str(p).replace("\\", "/").lower()
            ]
            if len(right_hits) == 1:
                return right_hits[0]
        raise FileNotFoundError(
            f"Ambiguous {kind} asset {rel!r}: "
            + ", ".join(str(p) for p in hits[:5])
        )
    raise FileNotFoundError(f"Could not resolve {kind} asset: {rel}")


with CAMERA_PATH.open(encoding="utf-8") as f:
    camera_dict = json.load(f)
if CAMERA_ID not in camera_dict:
    raise KeyError(f"Camera {CAMERA_ID} is not in {CAMERA_PATH}.")
camera = camera_dict[CAMERA_ID]

CAM_R = np.asarray(camera["R"], dtype=np.float32).reshape(3, 3)
CAM_T_RAW = np.asarray(camera["t"], dtype=np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], dtype=np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], dtype=np.float32).reshape(2)

# Read keypoints first, then select a plausible translation scale. Mugsy world
# keypoints are expected in mm; camera t may be stored in mm or meters.
print(f"Reading {len(rows):,} keypoint files...")
raw_world = []
keypoint_paths = []
image_paths = []
valid_rows = []

for row_i, row in enumerate(rows):
    kp_path = resolve_asset(row["orig_kp_right"], "keypoint")
    img_path = resolve_asset(row["image_path"], "image")
    with kp_path.open(encoding="utf-8") as f:
        keypoints = np.asarray(json.load(f), dtype=np.float32)
    if keypoints.ndim != 2 or keypoints.shape[1] != 3:
        raise ValueError(
            f"Expected Nx3 keypoints in {kp_path}, got {keypoints.shape}"
        )
    if max(KEEP_21_INDICES) >= len(keypoints):
        raise ValueError(
            f"KEEP_21_INDICES exceeds keypoint count {len(keypoints)} "
            f"in {kp_path}"
        )
    raw_world.append(keypoints)
    keypoint_paths.append(kp_path)
    image_paths.append(img_path)
    valid_rows.append(row)

raw_world = np.stack(raw_world).astype(np.float32)

# Translation unit audit via fraction of selected points with plausible positive
# depth and image projections inside image bounds. Select among common scales.
first_image = cv2.imread(str(image_paths[0]), cv2.IMREAD_COLOR)
if first_image is None:
    raise FileNotFoundError(image_paths[0])
IMAGE_ORIG_H, IMAGE_ORIG_W = first_image.shape[:2]

def _projection_in_bounds_fraction(translation_scale):
    t = CAM_T_RAW * translation_scale
    sample_n = min(len(raw_world), 256)
    selected = raw_world[:sample_n, KEEP_21_INDICES]
    cam_pts = selected @ CAM_R.T + t[None, None, :]
    z = cam_pts[..., 2]
    valid_z = z > 1.0
    z_safe = np.maximum(z, 1.0)
    u = CAM_FOCAL[0] * cam_pts[..., 0] / z_safe + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * cam_pts[..., 1] / z_safe + CAM_PRINCPT[1]
    inside = (
        valid_z
        & (u >= 0) & (u < IMAGE_ORIG_W)
        & (v >= 0) & (v < IMAGE_ORIG_H)
    )
    return float(inside.mean())


translation_candidates = [0.001, 1.0, 1000.0]
translation_scores = {
    scale: _projection_in_bounds_fraction(scale)
    for scale in translation_candidates
}
CAM_T_SCALE = max(translation_scores, key=translation_scores.get)
CAM_T = CAM_T_RAW * CAM_T_SCALE
print(
    "Camera translation scale audit: "
    + ", ".join(
        f"{scale:g}x={score:.2%}"
        for scale, score in translation_scores.items()
    )
    + f"; selected {CAM_T_SCALE:g}x"
)


def project_camera(points_cam):
    z = points_cam[..., 2]
    z_safe = np.maximum(z, 1.0)
    u = CAM_FOCAL[0] * points_cam[..., 0] / z_safe + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * points_cam[..., 1] / z_safe + CAM_PRINCPT[1]
    return np.stack((u, v), axis=-1).astype(np.float32)


# --------------------------- Segment-wise split ------------------------------

segments = np.asarray(sorted({row["segment"] for row in valid_rows}), dtype=object)
if len(segments) != TRAIN_SEGMENT_COUNT + VAL_SEGMENT_COUNT:
    raise RuntimeError(
        f"Expected {TRAIN_SEGMENT_COUNT + VAL_SEGMENT_COUNT} segments, "
        f"found {len(segments)}."
    )

split_rng = np.random.default_rng(SEED)
shuffled_segments = segments.copy()
split_rng.shuffle(shuffled_segments)
train_segments = set(shuffled_segments[:TRAIN_SEGMENT_COUNT].tolist())
val_segments = set(shuffled_segments[TRAIN_SEGMENT_COUNT:].tolist())
assert not train_segments.intersection(val_segments)

train_idx = np.asarray(
    [i for i, row in enumerate(valid_rows) if row["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, row in enumerate(valid_rows) if row["segment"] in val_segments],
    dtype=np.int64,
)

# ---------------------------- Crop helpers ----------------------------------

def crop_affine(uv, image_w, image_h, margin, out_w, out_h):
    uv = np.asarray(uv, dtype=np.float32)
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        dtype=np.float32,
    )


def make_mp_crop(color_bgr, uv):
    affine = crop_affine(
        uv, IMAGE_ORIG_W, IMAGE_ORIG_H,
        MP_CROP_MARGIN, 1, 1,
    )
    # Determine square output size from the original projected box and margin.
    lo, hi = np.asarray(uv).min(axis=0), np.asarray(uv).max(axis=0)
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * MP_CROP_MARGIN
    size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))
    center = (lo + hi) * 0.5
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    scale = size / side
    affine = np.asarray(
        [[scale, 0.0, -left * scale], [0.0, scale, -top * scale]],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        color_bgr, affine, (size, size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    gt_crop_uv = np.asarray(uv, dtype=np.float32) @ affine[:, :2].T + affine[:, 2]
    return crop, gt_crop_uv


def make_train_crop(gray, uv):
    affine = crop_affine(
        uv, IMAGE_ORIG_W, IMAGE_ORIG_H,
        TRAIN_CROP_MARGIN, IMAGE_W, IMAGE_H,
    )
    crop = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    uv_crop = np.asarray(uv, dtype=np.float32) @ affine[:, :2].T + affine[:, 2]
    return crop, uv_crop


def make_sobel(gray_u8):
    gray = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return mag[None].astype(np.float32)


def normalize_crop_points(uv):
    uv = np.asarray(uv, dtype=np.float32)
    out = np.empty_like(uv)
    out[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    out[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return out


# --------------------- MediaPipe Tasks API initialization -------------------

if not MP_MODEL_PATH.is_file():
    print(f"Downloading MediaPipe hand model to {MP_MODEL_PATH} ...")
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

mp_options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MP_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp_options)

# ------------------------------- Load frames --------------------------------

print(
    f"Loading {len(valid_rows):,} frames; "
    f"{len(train_idx):,} train / {len(val_idx):,} val frames across "
    f"{len(train_segments)} / {len(val_segments)} segments."
)

records = []
mp_predictions = [None] * len(val_idx)
mp_handedness = ["miss"] * len(val_idx)
mp_match_rmse = np.full(len(val_idx), np.nan, dtype=np.float32)
val_position_by_dataset_index = {
    int(dataset_i): pos for pos, dataset_i in enumerate(val_idx)
}

projection_inside_count = 0
projection_point_count = 0

try:
    for i, (row, kp_world, image_path) in enumerate(
        zip(valid_rows, raw_world, image_paths)
    ):
        color_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if color_bgr is None:
            raise FileNotFoundError(f"Could not read image {image_path}")
        if color_bgr.shape[:2] != (IMAGE_ORIG_H, IMAGE_ORIG_W):
            raise ValueError(
                f"Image size changed at {image_path}: {color_bgr.shape[:2]}"
            )

        camera_pts = kp_world @ CAM_R.T + CAM_T
        selected = camera_pts[KEEP_21_INDICES]
        uv = project_camera(selected)
        root_relative = selected - selected[0:1]

        inside = (
            (selected[:, 2] > 1.0)
            & (uv[:, 0] >= 0) & (uv[:, 0] < IMAGE_ORIG_W)
            & (uv[:, 1] >= 0) & (uv[:, 1] < IMAGE_ORIG_H)
        )
        projection_inside_count += int(inside.sum())
        projection_point_count += len(inside)

        gray = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2GRAY)
        train_crop, train_uv_crop = make_train_crop(gray, uv)

        record = {
            "segment": row["segment"],
            "frame_id": row.get("frame_id", str(i)),
            "y3d": root_relative.astype(np.float32),
            "uv_image": uv.astype(np.float32),
            "crop_gray": train_crop,
            "uv_crop": train_uv_crop.astype(np.float32),
        }
        records.append(record)

        # Run MediaPipe only on validation frames, on original color (BGR->RGB).
        val_pos = val_position_by_dataset_index.get(i)
        if val_pos is not None:
            mp_crop_bgr, gt_crop_uv = make_mp_crop(color_bgr, uv)
            mp_rgb = cv2.cvtColor(mp_crop_bgr, cv2.COLOR_BGR2RGB)
            result = landmarker.detect(
                mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=np.ascontiguousarray(mp_rgb),
                )
            )

            image_hands = getattr(result, "hand_landmarks", None) or []
            world_hands = getattr(result, "hand_world_landmarks", None) or []
            handedness_hands = getattr(result, "handedness", None) or []
            candidates = []

            for hand_i, (image_hand, world_hand) in enumerate(
                zip(image_hands, world_hands)
            ):
                if len(image_hand) != 21 or len(world_hand) != 21:
                    continue

                pred_uv = np.asarray(
                    [
                        [lm.x * mp_rgb.shape[1], lm.y * mp_rgb.shape[0]]
                        for lm in image_hand
                    ],
                    dtype=np.float32,
                )
                match_rmse = float(
                    np.sqrt(
                        np.mean(np.sum((pred_uv - gt_crop_uv) ** 2, axis=-1))
                    )
                )

                world_m = np.asarray(
                    [[lm.x, lm.y, lm.z] for lm in world_hand],
                    dtype=np.float32,
                )
                # MediaPipe world landmarks are in meters; re-root at wrist.
                pose_mm = (world_m - world_m[0:1]) * 1000.0

                label = "unknown"
                if hand_i < len(handedness_hands):
                    try:
                        label = handedness_hands[hand_i][0].category_name
                    except (AttributeError, IndexError, TypeError):
                        pass

                candidates.append(
                    {
                        "score": match_rmse,
                        "pose": pose_mm.astype(np.float32),
                        "handedness": label,
                    }
                )

            if candidates:
                best = min(candidates, key=lambda item: item["score"])
                mp_predictions[val_pos] = best["pose"]
                mp_handedness[val_pos] = best["handedness"]
                mp_match_rmse[val_pos] = best["score"]

        if (i + 1) % 500 == 0 or i + 1 == len(valid_rows):
            print(f"Loaded and processed {i + 1}/{len(valid_rows)} frames")
finally:
    landmarker.close()

projection_fraction = (
    projection_inside_count / max(projection_point_count, 1)
)
print(
    f"Projected selected joints inside image: {projection_fraction:.2%}"
)

# Make sure the ordering matches the standard MediaPipe wrist/finger chains.
EXPECTED_CHAIN_BONES = {
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
}
PROJECT_CHAIN_BONES = {
    tuple(sorted(edge)) for edge in BONE_PAIRS
}
assert EXPECTED_CHAIN_BONES == PROJECT_CHAIN_BONES, (
    "BONE_PAIRS do not match the assumed MediaPipe wrist/thumb/index/"
    "middle/ring/pinky chain ordering."
)

# -------------------------- Verify split and arrays --------------------------

assert len(records) == len(valid_rows)
assert not train_segments.intersection(val_segments)
assert len(train_segments) == TRAIN_SEGMENT_COUNT
assert len(val_segments) == VAL_SEGMENT_COUNT

y_all = np.stack([record["y3d"] for record in records]).astype(np.float32)
y2d_all = np.stack(
    [normalize_crop_points(record["uv_crop"]) for record in records]
).astype(np.float32)
gray_crops = [record["crop_gray"] for record in records]

# Mean GT bone lengths from train segments only.
canonical_lens_np = np.asarray(
    [
        np.linalg.norm(
            y_all[train_idx, child] - y_all[train_idx, parent],
            axis=-1,
        ).mean()
        for parent, child in BONE_PAIRS
    ],
    dtype=np.float32,
)

# -------------------------- Sobel direct model ------------------------------

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-1)


def reconstruct_joints_from_bones(bone_vectors):
    batch = bone_vectors.shape[0]
    joints = torch.zeros(
        (batch, 21, 3),
        dtype=bone_vectors.dtype,
        device=bone_vectors.device,
    )
    for idx, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + bone_vectors[:, idx]
    return joints


class SobelDirectTracker(nn.Module):
    """Phase-0 WebcamAlignedLCFTracker with a one-channel Sobel input."""

    def __init__(self, canonical_lens):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer(
            "bone_lens",
            canonical_lens.clone(),
        )

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(
            torch.flatten(self.encoder(x), 1)
        )

        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3),
            dim=-1,
            eps=1e-7,
        )
        d6 = self.latent_frame_head(feat)
        frame = rotation_6d_to_matrix(d6)

        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1,
            eps=1e-7,
        )
        phalanx = torch.bmm(
            frame, phalanx.transpose(1, 2)
        ).transpose(1, 2)

        dirs = torch.zeros(
            (batch, 20, 3),
            dtype=x.dtype,
            device=x.device,
        )
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger_i, offset in enumerate((4, 8, 12, 16)):
            phalanx_offset = 3 + finger_i * 3
            dirs[:, offset] = meta[:, finger_i + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[
                :, phalanx_offset:phalanx_offset + 3
            ]

        bone_vecs = dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        uv = torch.tanh(
            self.proj2d_head(feat).reshape(batch, 21, 2)
        )
        return joints, dirs, uv


def rotate_camera_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x = points[:, 0].copy()
    y = points[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment_training_sample(gray, uv_crop, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    shift_x = float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    shift_y = float(rng.uniform(-0.10, 0.10) * IMAGE_H)

    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5),
        angle_deg,
        scale,
    ).astype(np.float32)
    matrix[0, 2] += shift_x
    matrix[1, 2] += shift_y

    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset,
        0,
        255,
    ).astype(np.uint8)

    uv_aug = uv_crop @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle)
    uv_norm = normalize_crop_points(uv_aug)
    return make_sobel(warped), y3d_aug.astype(np.float32), uv_norm


def direct_loss(pred3d, target3d, pred2d, target2d):
    l_pos = F.l1_loss(pred3d, target3d)

    pred_bones = (
        pred3d[:, [child for _, child in BONE_PAIRS]]
        - pred3d[:, [parent for parent, _ in BONE_PAIRS]]
    )
    target_bones = (
        target3d[:, [child for _, child in BONE_PAIRS]]
        - target3d[:, [parent for parent, _ in BONE_PAIRS]]
    )

    pred_len = torch.linalg.vector_norm(pred_bones, dim=-1)
    target_len = torch.linalg.vector_norm(target_bones, dim=-1)
    l_len = F.l1_loss(pred_len, target_len)

    pred_unit = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_unit = F.normalize(target_bones, dim=-1, eps=1e-7)
    cosine = torch.clamp((pred_unit * target_unit).sum(-1), -1.0, 1.0)
    l_dir = (1.0 - cosine).mean()

    l_2d = F.smooth_l1_loss(pred2d, target2d)
    return l_pos + 0.5 * l_len + 20.0 * l_dir + 15.0 * l_2d


sobel_model = SobelDirectTracker(
    torch.from_numpy(canonical_lens_np)
).to(DEVICE)

optimizer = torch.optim.AdamW(
    sobel_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TRAIN_EPOCHS,
    eta_min=1e-4,
)
augment_rng = np.random.default_rng(SEED + 1)

print(f"\nTraining Sobel direct regressor for {TRAIN_EPOCHS} epochs...")
for epoch in range(1, TRAIN_EPOCHS + 1):
    sobel_model.train()
    order = np.random.permutation(train_idx)

    for start in range(0, len(order), BATCH_SIZE):
        batch_ids = order[start:start + BATCH_SIZE]

        samples = [
            augment_training_sample(
                gray_crops[int(i)],
                records[int(i)]["uv_crop"],
                y_all[int(i)],
                augment_rng,
            )
            for i in batch_ids
        ]

        bx = torch.from_numpy(
            np.stack([sample[0] for sample in samples])
        ).to(DEVICE)
        by3d = torch.from_numpy(
            np.stack([sample[1] for sample in samples])
        ).to(DEVICE)
        by2d = torch.from_numpy(
            np.stack([sample[2] for sample in samples])
        ).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred3d, _, pred2d = sobel_model(bx)
        loss = direct_loss(pred3d, by3d, pred2d, by2d)
        loss.backward()
        optimizer.step()

    scheduler.step()
    print(f"Sobel direct epoch {epoch:02d}/{TRAIN_EPOCHS} complete")

# ------------------------------ Predictions ---------------------------------

sobel_model.eval()
sobel_predictions = [None] * len(val_idx)

with torch.inference_mode():
    for val_pos, dataset_i in enumerate(val_idx):
        gray = gray_crops[int(dataset_i)]
        inp = torch.from_numpy(make_sobel(gray)[None]).to(DEVICE)
        pred3d, _, _ = sobel_model(inp)
        sobel_predictions[val_pos] = pred3d[0].cpu().numpy().astype(np.float32)

# ------------------------ Metrics and bone scaling ---------------------------

def rescale_mp_bones_to_train_mean(pose):
    pose = np.asarray(pose, dtype=np.float32)
    out = np.zeros_like(pose)

    for bone_i, (parent, child) in enumerate(BONE_PAIRS):
        direction = pose[child] - pose[parent]
        length = float(np.linalg.norm(direction))
        if length < 1e-8:
            direction = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
        else:
            direction = direction / length
        out[child] = out[parent] + direction * canonical_lens_np[bone_i]

    return out


def component_metrics(pred, target):
    delta = np.asarray(pred) - np.asarray(target)
    mpjpe = float(np.linalg.norm(delta, axis=-1).mean())
    abs_xyz = np.abs(delta).mean(axis=(0, 1))
    return mpjpe, float(abs_xyz[0]), float(abs_xyz[1]), float(abs_xyz[2])


detected_mask = np.asarray(
    [pose is not None for pose in mp_predictions],
    dtype=bool,
)
detected_positions = np.flatnonzero(detected_mask)
if len(detected_positions) == 0:
    raise RuntimeError("MediaPipe detected no validation hands.")

mp_raw_predictions = [None] * len(val_idx)
mp_scaled_predictions = [None] * len(val_idx)
for pos in detected_positions:
    mp_raw_predictions[pos] = mp_predictions[pos]
    mp_scaled_predictions[pos] = rescale_mp_bones_to_train_mean(
        mp_predictions[pos]
    )

gt_val = np.stack([y_all[int(i)] for i in val_idx]).astype(np.float32)

# Detection and chirality per segment.
segment_stats = {}
for pos, dataset_i in enumerate(val_idx):
    segment = records[int(dataset_i)]["segment"]
    stats = segment_stats.setdefault(
        segment, {"frames": 0, "detected": 0}
    )
    stats["frames"] += 1
    stats["detected"] += int(detected_mask[pos])

print("\nCOLOR-FRAME MEDIAPIPE DETECTION")
print(
    f"Overall: {int(detected_mask.sum())}/{len(val_idx)} "
    f"({detected_mask.mean():.2%})"
)
print(f"{'Segment':<36} {'Detected':>9} {'Frames':>8} {'Rate':>9}")
print("-" * 66)
for segment in sorted(segment_stats, key=str):
    stats = segment_stats[segment]
    rate = stats["detected"] / max(stats["frames"], 1)
    print(
        f"{str(segment):<36} {stats['detected']:>9} "
        f"{stats['frames']:>8} {rate:>8.2%}"
    )

handedness_counts = {}
for pos in detected_positions:
    label = str(mp_handedness[pos])
    handedness_counts[label] = handedness_counts.get(label, 0) + 1
right_count = sum(
    count for label, count in handedness_counts.items()
    if label.strip().lower() == "right"
)
print("\nHANDEDNESS VS GT CHIRALITY")
print("GT chirality assumption: right hand (orig_kp_right)")
print(f"MediaPipe label counts: {handedness_counts}")
print(
    f"MediaPipe label 'Right': {right_count}/{len(detected_positions)} "
    f"({right_count / len(detected_positions):.2%})"
)

high_detection_segments = {
    segment
    for segment, stats in segment_stats.items()
    if stats["detected"] / max(stats["frames"], 1) >= 0.90
}
high_detection_positions = np.asarray(
    [
        pos for pos in detected_positions
        if records[int(val_idx[pos])]["segment"] in high_detection_segments
    ],
    dtype=np.int64,
)

def report_table(title, positions):
    print(f"\n{title}")
    print(
        f"{'Method':<25} {'MPJPE mm':>10} "
        f"{'|dx| mm':>10} {'|dy| mm':>10} {'|dz| mm':>10}"
    )
    print("-" * 70)

    methods = [
        ("MediaPipe raw", mp_raw_predictions),
        ("MediaPipe bone-scaled", mp_scaled_predictions),
        ("Sobel direct", sobel_predictions),
    ]
    for name, predictions in methods:
        pred = np.stack([predictions[int(pos)] for pos in positions])
        target = np.stack([gt_val[int(pos)] for pos in positions])
        metrics = component_metrics(pred, target)
        print(
            f"{name:<25} {metrics[0]:10.2f} "
            f"{metrics[1]:10.2f} {metrics[2]:10.2f} {metrics[3]:10.2f}"
        )


report_table("DETECTED VALIDATION FRAMES", detected_positions)

if len(high_detection_positions):
    print(
        "\nSegments meeting detection >=90%: "
        + ", ".join(map(str, sorted(high_detection_segments, key=str)))
    )
    report_table(
        "DETECTED FRAMES FROM >=90%-DETECTION SEGMENTS",
        high_detection_positions,
    )
else:
    print("\nNo validation segments reached the >=90% detection threshold.")

checkpoint_path = Path("/content/phase3_sobel_direct.pt")
torch.save(
    {
        "model_state_dict": {
            key: value.detach().cpu()
            for key, value in sobel_model.state_dict().items()
        },
        "canonical_bone_lengths_mm": canonical_lens_np,
        "seed": SEED,
        "train_segments": sorted(train_segments),
        "val_segments": sorted(val_segments),
        "camera_id": CAMERA_ID,
        "manifest_path": str(MANIFEST_PATH),
    },
    checkpoint_path,
)
print(f"\nSaved trained Sobel direct model: {checkpoint_path}")

print(
    "\nAssumptions: right-hand GT comes from orig_kp_right; world keypoints "
    "are millimeters; MediaPipe world landmarks are meters and are re-rooted "
    "at wrist joint 0; the camera translation scale is selected by in-image "
    "projection coverage; MediaPipe handedness labels are reported as returned "
    "for the unflipped color crop."
)

Active compute device: cpu (CPU)
MediaPipe version: 1.1.1
Manifest: /content/dataset/manifest.csv
Capture root: /content/dataset
Camera: /content/dataset/cam_params.json
Indexing dataset files...
Reading 5,114 keypoint files...
Camera translation scale audit: 0.001x=0.00%, 1x=99.87%, 1000x=0.00%; selected 1x
Loading 5,114 frames; 4,246 train / 868 val frames across 44 / 12 segments.
Loaded and processed 500/5114 frames
Loaded and processed 1000/5114 frames
Loaded and processed 1500/5114 frames
Loaded and processed 2000/5114 frames
Loaded and processed 2500/5114 frames
Loaded and processed 3000/5114 frames
Loaded and processed 3500/5114 frames
Loaded and processed 4000/5114 frames
Loaded and processed 4500/5114 frames
Loaded and processed 5000/5114 frames
Loaded and processed 5114/5114 frames
Projected selected joints inside image: 99.85%

Training Sobel direct regressor for 20 epochs...
Sobel direct epoch 01/20 complete
Sobel direct epoch 02/20 complete
Sobel direct epoch 03/20 complet

In [3]:
# @title
# =============================================================================
# STANDALONE PHASE 3 AUDIT
# - Loads dataset, camera, and segment split from scratch.
# - Trains a Sobel direct regressor; no checkpoint is required.
# - Runs MediaPipe on original and mirrored color crops, keeping all candidates.
# - Reports sign, 2D, candidate, and PA-MPJPE audits.
# - Saves the trained Sobel checkpoint to Google Drive.
# =============================================================================

# Install MediaPipe if this runtime does not have it.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("mediapipe") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "mediapipe"]
    )

import csv
import itertools
import json
import os
import random
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------- Settings -----------------------------------

SEED = 20251018
CAMERA_ID = "400485"
TRAIN_SEGMENT_COUNT, VAL_SEGMENT_COUNT = 44, 12
TRAIN_EPOCHS = 20
BATCH_SIZE = 64
LR, WEIGHT_DECAY = 1.2e-3, 1e-4
IMAGE_W, IMAGE_H = 128, 96
TRAIN_CROP_MARGIN = 1.4
MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
DRIVE_CHECKPOINT_PATH = DRIVE_ROOT / "phase3_sobel_direct.pt"

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
EXPECTED_CHAIN_BONES = {
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
}
assert {tuple(sorted(edge)) for edge in BONE_PAIRS} == EXPECTED_CHAIN_BONES

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# Mount Drive now so the model can be persisted.
try:
    from google.colab import drive
    if not DRIVE_ROOT.exists():
        drive.mount("/content/drive")
except Exception:
    pass

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError("MediaPipe must expose mp.tasks.vision.HandLandmarker.")

# ---------------------------- Find dataset ---------------------------------

def manifest_hits(root):
    if not root.exists():
        return []
    try:
        return list(root.rglob("manifest.csv"))
    except Exception:
        return []


def preferred_manifest(paths):
    if not paths:
        return None
    return next(
        (p for p in paths if "mugsy_subset" in str(p).lower()),
        paths[0],
    )


manifest_path = None
for root in (DATASET_DIR, Path("/content"), Path.cwd()):
    manifest_path = preferred_manifest(manifest_hits(root))
    if manifest_path:
        break

if manifest_path is None and DRIVE_ROOT.exists():
    manifest_path = preferred_manifest(manifest_hits(DRIVE_ROOT))
if manifest_path is None:
    raise FileNotFoundError(
        "manifest.csv was not found under /content or Google Drive."
    )

subset_dir = manifest_path.parent


def find_camera_file():
    for parent in (subset_dir, *subset_dir.parents):
        for candidate in (
            parent / "Mugsy_cameras" / "cam_params.json",
            parent / "cam_params.json",
        ):
            if candidate.is_file():
                return candidate

    for root in (subset_dir.parent, subset_dir.parent.parent, Path("/content")):
        if root.exists():
            found = list(root.rglob("cam_params.json"))
            if found:
                return next(
                    (p for p in found if p.parent.name.lower() == "mugsy_cameras"),
                    found[0],
                )
    if DRIVE_ROOT.exists():
        found = list(DRIVE_ROOT.rglob("cam_params.json"))
        if found:
            return found[0]
    return None


camera_path = find_camera_file()
if camera_path is None:
    raise FileNotFoundError("cam_params.json was not found.")

capture_dir = subset_dir
for parent in (subset_dir, *subset_dir.parents):
    if (
        (parent / "Mugsy_cameras" / "cam_params.json").is_file()
        or (parent / "cam_params.json").is_file()
        or (parent / "keypoints_right").is_dir()
    ):
        capture_dir = parent
        break

with manifest_path.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))
rows = [
    row for row in all_rows
    if row.get("camera_id", CAMERA_ID) == CAMERA_ID
]
if not rows:
    raise RuntimeError(f"No manifest rows for camera {CAMERA_ID}.")

# Index assets, deduplicating paths found through overlapping roots.
asset_index = {}
roots = list(dict.fromkeys(
    [capture_dir, subset_dir, subset_dir.parent, subset_dir.parent.parent]
))
for root in roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in {
                ".json", ".png", ".jpg", ".jpeg"
            }:
                asset_index.setdefault(path.name.lower(), set()).add(path.resolve())


def resolve_asset(relative_path, kind):
    rel = str(relative_path).replace("\\", "/").strip()
    rel_path = Path(rel)
    for candidate in (
        Path(rel),
        capture_dir / rel,
        subset_dir / rel,
        subset_dir.parent / rel,
        subset_dir.parent.parent / rel,
    ):
        if candidate.is_file():
            return candidate.resolve()

    hits = list(asset_index.get(rel_path.name.lower(), set()))
    if kind == "keypoint":
        parts = [p.lower() for p in rel_path.parts]
        side = "right" if "right" in parts else (
            "left" if "left" in parts else None
        )
        if side:
            side_hits = [
                p for p in hits
                if f"keypoints_{side}"
                in str(p).replace("\\", "/").lower()
            ]
            if side_hits:
                hits = side_hits
    elif kind == "image":
        image_hits = [
            p for p in hits
            if any(
                marker in str(p).replace("\\", "/").lower()
                for marker in ("/images/", "/image/", "/rgb/")
            )
        ]
        if image_hits:
            hits = image_hits

    hits = sorted(set(hits))
    if len(hits) == 1:
        return hits[0]
    if kind == "keypoint":
        right_hits = [
            p for p in hits
            if "keypoints_right" in str(p).replace("\\", "/").lower()
        ]
        if len(right_hits) == 1:
            return right_hits[0]
    raise FileNotFoundError(
        f"Cannot uniquely resolve {kind} asset {rel!r}; matches={hits[:5]}"
    )


# ---------------------------- Camera and split -----------------------------

with camera_path.open(encoding="utf-8") as f:
    camera_dict = json.load(f)
camera = camera_dict[CAMERA_ID]
CAM_R = np.asarray(camera["R"], np.float32).reshape(3, 3)
CAM_T_RAW = np.asarray(camera["t"], np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], np.float32).reshape(2)


def project(points):
    z = np.maximum(points[..., 2], 1.0)
    u = CAM_FOCAL[0] * points[..., 0] / z + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * points[..., 1] / z + CAM_PRINCPT[1]
    return np.stack((u, v), axis=-1).astype(np.float32)


segments = np.asarray(sorted({r["segment"] for r in rows}), dtype=object)
if len(segments) != TRAIN_SEGMENT_COUNT + VAL_SEGMENT_COUNT:
    raise RuntimeError(
        f"Expected 56 segments (44 train, 12 val); found {len(segments)}."
    )
split_rng = np.random.default_rng(SEED)
shuffled_segments = segments.copy()
split_rng.shuffle(shuffled_segments)
train_segments = set(shuffled_segments[:TRAIN_SEGMENT_COUNT].tolist())
val_segments = set(shuffled_segments[TRAIN_SEGMENT_COUNT:].tolist())
train_idx = np.asarray(
    [i for i, r in enumerate(rows) if r["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, r in enumerate(rows) if r["segment"] in val_segments],
    dtype=np.int64,
)

# Read labels and resolve images.
world_points = []
image_paths = []
for row in rows:
    kp_path = resolve_asset(row["orig_kp_right"], "keypoint")
    image_path = resolve_asset(row["image_path"], "image")
    with kp_path.open(encoding="utf-8") as f:
        points = np.asarray(json.load(f), dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError(f"Expected Nx3 keypoints in {kp_path}.")
    if max(KEEP_21_INDICES) >= len(points):
        raise ValueError(f"Too few keypoints in {kp_path}.")
    world_points.append(points)
    image_paths.append(image_path)
world_points = np.stack(world_points)

first_image = cv2.imread(str(image_paths[0]), cv2.IMREAD_COLOR)
if first_image is None:
    raise FileNotFoundError(image_paths[0])
IMAGE_ORIG_H, IMAGE_ORIG_W = first_image.shape[:2]


def camera_in_bounds_fraction(t_scale):
    points = world_points[:min(256, len(world_points)), KEEP_21_INDICES]
    cam = points @ CAM_R.T + CAM_T_RAW * t_scale
    uv = project(cam)
    inside = (
        (cam[..., 2] > 1.0)
        & (uv[..., 0] >= 0) & (uv[..., 0] < IMAGE_ORIG_W)
        & (uv[..., 1] >= 0) & (uv[..., 1] < IMAGE_ORIG_H)
    )
    return float(inside.mean())


T_SCALE = max((0.001, 1.0, 1000.0), key=camera_in_bounds_fraction)
CAM_T = CAM_T_RAW * T_SCALE

# ------------------------------- Crops --------------------------------------

def crop_affine(uv, out_w, out_h, margin):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        np.float32,
    )


def make_mp_crop(color_bgr, uv):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0)
    side *= MP_CROP_MARGIN
    size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    scale = size / side
    affine = np.asarray(
        [[scale, 0.0, -left * scale], [0.0, scale, -top * scale]],
        np.float32,
    )
    crop = cv2.warpAffine(
        color_bgr, affine, (size, size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    return crop, uv @ affine[:, :2].T + affine[:, 2]


def make_sobel(gray_u8):
    gray = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return mag[None].astype(np.float32)


def normalize_uv(uv):
    out = np.empty_like(uv, dtype=np.float32)
    out[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    out[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return out


# ---------------------------- Stage the dataset ----------------------------

records = []
for i, (row, world, image_path) in enumerate(
    zip(rows, world_points, image_paths)
):
    color = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if color is None:
        raise FileNotFoundError(image_path)
    if color.shape[:2] != (IMAGE_ORIG_H, IMAGE_ORIG_W):
        raise ValueError(f"Image dimensions differ at {image_path}.")

    cam = world @ CAM_R.T + CAM_T
    selected = cam[KEEP_21_INDICES]
    uv = project(selected)
    y3d = selected - selected[0:1]

    gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
    affine = crop_affine(uv, IMAGE_W, IMAGE_H, TRAIN_CROP_MARGIN)
    crop_gray = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    uv_crop = uv @ affine[:, :2].T + affine[:, 2]

    records.append(
        {
            "segment": row["segment"],
            "frame_id": row.get("frame_id", str(i)),
            "image_path": image_path,
            "uv_image": uv.astype(np.float32),
            "uv_crop": uv_crop.astype(np.float32),
            "crop_gray": crop_gray,
            "y3d": y3d.astype(np.float32),
        }
    )

y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)
y2d_all = np.stack([normalize_uv(r["uv_crop"]) for r in records])
canonical_lens_np = np.asarray(
    [
        np.linalg.norm(
            y_all[train_idx, child] - y_all[train_idx, parent],
            axis=-1,
        ).mean()
        for parent, child in BONE_PAIRS
    ],
    dtype=np.float32,
)

# -------------------------- Train Sobel model -------------------------------

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-1)


def reconstruct_joints_from_bones(vectors):
    joints = torch.zeros(
        (vectors.shape[0], 21, 3), dtype=vectors.dtype, device=vectors.device
    )
    for k, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + vectors[:, k]
    return joints


class SobelDirectTracker(nn.Module):
    def __init__(self, lengths):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer("bone_lens", lengths.clone())

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3),
            dim=-1, eps=1e-7,
        )
        frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1, eps=1e-7,
        )
        phalanx = torch.bmm(
            frame, phalanx.transpose(1, 2)
        ).transpose(1, 2)

        dirs = torch.zeros(
            (batch, 20, 3), dtype=x.dtype, device=x.device
        )
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger_i, offset in enumerate((4, 8, 12, 16)):
            start = 3 + finger_i * 3
            dirs[:, offset] = meta[:, finger_i + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[:, start:start + 3]

        joints = reconstruct_joints_from_bones(
            dirs * self.bone_lens.view(1, 20, 1)
        )
        uv = torch.tanh(
            self.proj2d_head(feat).reshape(batch, 21, 2)
        )
        return joints, dirs, uv


def rotate_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x, y = points[:, 0].copy(), points[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment(gray, uv, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    matrix[1, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_H)
    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset, 0, 255
    ).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    return make_sobel(warped), rotate_xy(y3d, angle), normalize_uv(uv_aug)


def direct_loss(pred3d, target3d, pred2d, target2d):
    pos = F.l1_loss(pred3d, target3d)
    pred_bones = (
        pred3d[:, [c for _, c in BONE_PAIRS]]
        - pred3d[:, [p for p, _ in BONE_PAIRS]]
    )
    target_bones = (
        target3d[:, [c for _, c in BONE_PAIRS]]
        - target3d[:, [p for p, _ in BONE_PAIRS]]
    )
    pred_len = torch.linalg.vector_norm(pred_bones, dim=-1)
    target_len = torch.linalg.vector_norm(target_bones, dim=-1)
    length = F.l1_loss(pred_len, target_len)
    pred_unit = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_unit = F.normalize(target_bones, dim=-1, eps=1e-7)
    direction = (
        1.0 - torch.clamp((pred_unit * target_unit).sum(-1), -1.0, 1.0)
    ).mean()
    reprojection = F.smooth_l1_loss(pred2d, target2d)
    return pos + 0.5 * length + 20.0 * direction + 15.0 * reprojection


sobel_model = SobelDirectTracker(
    torch.from_numpy(canonical_lens_np)
).to(DEVICE)
optimizer = torch.optim.AdamW(
    sobel_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4
)
augment_rng = np.random.default_rng(SEED + 1)

for epoch in range(TRAIN_EPOCHS):
    sobel_model.train()
    order = np.random.permutation(train_idx)
    for start in range(0, len(order), BATCH_SIZE):
        batch_ids = order[start:start + BATCH_SIZE]
        samples = [
            augment(
                records[int(i)]["crop_gray"],
                records[int(i)]["uv_crop"],
                y_all[int(i)],
                augment_rng,
            )
            for i in batch_ids
        ]
        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        by3 = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        by2 = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred3d, _, pred2d = sobel_model(bx)
        loss = direct_loss(pred3d, by3, pred2d, by2)
        loss.backward()
        optimizer.step()
    scheduler.step()

# Persist for future Colab sessions.
checkpoint_payload = {
    "model_state_dict": {
        k: v.detach().cpu() for k, v in sobel_model.state_dict().items()
    },
    "canonical_bone_lengths_mm": canonical_lens_np,
    "seed": SEED,
    "train_segments": sorted(train_segments),
    "val_segments": sorted(val_segments),
    "camera_id": CAMERA_ID,
    "manifest_path": str(manifest_path),
}
if DRIVE_ROOT.exists():
    DRIVE_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint_payload, DRIVE_CHECKPOINT_PATH)
    checkpoint_saved_to = DRIVE_CHECKPOINT_PATH
else:
    checkpoint_saved_to = Path("/content/phase3_sobel_direct.pt")
    torch.save(checkpoint_payload, checkpoint_saved_to)

sobel_model.eval()

# ---------------------- MediaPipe model and candidates ----------------------

if not MP_MODEL_PATH.is_file():
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

mp_options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MP_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)


def extract_candidates(result, crop_size, gt_crop_uv, mirrored):
    candidates = []
    image_hands = getattr(result, "hand_landmarks", None) or []
    world_hands = getattr(result, "hand_world_landmarks", None) or []
    handedness = getattr(result, "handedness", None) or []

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue
        xy = np.asarray(
            [[lm.x * crop_size, lm.y * crop_size] for lm in image_hand],
            dtype=np.float32,
        )
        if mirrored:
            xy[:, 0] = crop_size - 1 - xy[:, 0]
        pixel_error = float(np.linalg.norm(xy - gt_crop_uv, axis=-1).mean())

        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )
        pose = (world_m - world_m[0:1]) * 1000.0
        if mirrored:
            pose[:, 0] *= -1.0

        label = "unknown"
        if hand_i < len(handedness):
            try:
                label = handedness[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                pass
        candidates.append(
            {
                "pose": pose.astype(np.float32),
                "xy": xy,
                "pixel_error": pixel_error,
                "mirrored_input": bool(mirrored),
                "hand_index": hand_i,
                "handedness": label,
            }
        )
    return candidates


all_candidates = [[] for _ in val_idx]
normal_2d_errors, best_2d_errors = [], []

with mp.tasks.vision.HandLandmarker.create_from_options(mp_options) as landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        record = records[int(dataset_i)]
        color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
        crop_bgr, gt_crop_uv = make_mp_crop(color, record["uv_image"])
        crop_size = crop_bgr.shape[0]
        found = []

        for mirrored in (False, True):
            view_bgr = cv2.flip(crop_bgr, 1) if mirrored else crop_bgr
            view_rgb = cv2.cvtColor(view_bgr, cv2.COLOR_BGR2RGB)
            result = landmarker.detect(
                mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=np.ascontiguousarray(view_rgb),
                )
            )
            found.extend(
                extract_candidates(result, crop_size, gt_crop_uv, mirrored)
            )

        all_candidates[val_pos] = found
        normal = [c for c in found if not c["mirrored_input"]]
        if normal:
            normal_2d_errors.append(
                min(normal, key=lambda c: c["pixel_error"])["pixel_error"]
            )
        if found:
            best_2d_errors.append(
                min(found, key=lambda c: c["pixel_error"])["pixel_error"]
            )

detected_positions = np.asarray(
    [i for i, c in enumerate(all_candidates) if c],
    dtype=np.int64,
)
if not len(detected_positions):
    raise RuntimeError("MediaPipe detected no validation hands.")

nearest_candidates = [
    min(c, key=lambda candidate: candidate["pixel_error"]) if c else None
    for c in all_candidates
]

# Sobel inference on validation crops.
sobel_predictions = [None] * len(val_idx)
with torch.inference_mode():
    for pos, dataset_i in enumerate(val_idx):
        x = torch.from_numpy(
            make_sobel(records[int(dataset_i)]["crop_gray"])[None]
        ).to(DEVICE)
        joints, _, _ = sobel_model(x)
        sobel_predictions[pos] = joints[0].cpu().numpy().astype(np.float32)

gt_val = [records[int(i)]["y3d"] for i in val_idx]


def mpjpe(pred, target):
    return float(np.linalg.norm(pred - target, axis=-1).mean())


def rescale_bones(pose):
    result = np.zeros_like(pose, dtype=np.float32)
    lengths = canonical_lens_np
    for bone_i, (parent, child) in enumerate(BONE_PAIRS):
        direction = pose[child] - pose[parent]
        norm = float(np.linalg.norm(direction))
        direction = (
            np.array([1.0, 0.0, 0.0], np.float32)
            if norm < 1e-8 else direction / norm
        )
        result[child] = result[parent] + direction * lengths[bone_i]
    return result


def pa_mpjpe_one(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    x = pred - pred.mean(axis=0, keepdims=True)
    y = target - target.mean(axis=0, keepdims=True)
    u, singular, vt = np.linalg.svd(x.T @ y)
    correction = np.eye(3)
    correction[-1, -1] = np.linalg.det(u @ vt)
    rotation = u @ correction @ vt
    denom = np.sum(x * x)
    if denom < 1e-12:
        return float("nan")
    scale = np.sum(singular * np.diag(correction)) / denom
    aligned = scale * (x @ rotation) + target.mean(axis=0, keepdims=True)
    return float(np.linalg.norm(aligned - target, axis=-1).mean())


mp_raw = [None] * len(val_idx)
mp_scaled = [None] * len(val_idx)
for pos in detected_positions:
    pose = nearest_candidates[int(pos)]["pose"]
    mp_raw[int(pos)] = pose
    mp_scaled[int(pos)] = rescale_bones(pose)

# ------------------------------- Tables -------------------------------------

print("FRAME AUDIT — 2D-NEAREST CANDIDATE")
print(f"{'XYZ signs':<12} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 34)
for signs in itertools.product((1.0, -1.0), repeat=3):
    errors = [
        mpjpe(
            nearest_candidates[int(pos)]["pose"]
            * np.asarray(signs, np.float32),
            gt_val[int(pos)],
        )
        for pos in detected_positions
    ]
    label = "".join("+" if s > 0 else "-" for s in signs)
    print(f"{label:<12} {len(errors):>8} {np.mean(errors):10.2f}")

print("\n2D CHECK — MEAN JOINT DISTANCE")
print(f"{'Candidate pool':<36} {'Frames':>8} {'Error px':>10}")
print("-" * 58)
print(
    f"{'Original crop only':<36} {len(normal_2d_errors):>8} "
    f"{np.mean(normal_2d_errors):10.2f}"
)
print(
    f"{'Best original/mirrored per frame':<36} {len(best_2d_errors):>8} "
    f"{np.mean(best_2d_errors):10.2f}"
)

nearest_errors, oracle_errors = [], []
different = 0
for pos in detected_positions:
    pos = int(pos)
    candidates = all_candidates[pos]
    nearest = min(candidates, key=lambda c: c["pixel_error"])
    oracle = min(candidates, key=lambda c: mpjpe(c["pose"], gt_val[pos]))
    nearest_errors.append(mpjpe(nearest["pose"], gt_val[pos]))
    oracle_errors.append(mpjpe(oracle["pose"], gt_val[pos]))
    different += int(
        (nearest["mirrored_input"], nearest["hand_index"])
        != (oracle["mirrored_input"], oracle["hand_index"])
    )

print("\nCANDIDATE CHECK")
print(f"{'Selection':<28} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 50)
print(f"{'2D-nearest':<28} {len(nearest_errors):>8} {np.mean(nearest_errors):10.2f}")
print(f"{'3D oracle':<28} {len(oracle_errors):>8} {np.mean(oracle_errors):10.2f}")
print(
    f"Different candidate identity: {different}/{len(detected_positions)} "
    f"({different / len(detected_positions):.2%})"
)

print("\nPA-MPJPE — PER-FRAME ROTATION + SCALE")
print(f"{'Method':<26} {'Frames':>8} {'PA-MPJPE mm':>13}")
print("-" * 52)
for name, predictions in (
    ("MediaPipe raw", mp_raw),
    ("MediaPipe bone-scaled", mp_scaled),
    ("Sobel direct", sobel_predictions),
):
    values = [
        pa_mpjpe_one(predictions[int(pos)], gt_val[int(pos)])
        for pos in detected_positions
    ]
    print(f"{name:<26} {len(values):>8} {np.nanmean(values):13.2f}")

# -------------------------------- Overlays -----------------------------------

def draw_overlay(candidate, crop_bgr, gt_uv, path):
    canvas = crop_bgr.copy()
    gt_points = np.rint(gt_uv).astype(int)
    mp_points = np.rint(candidate["xy"]).astype(int)
    for parent, child in BONE_PAIRS:
        cv2.line(canvas, tuple(gt_points[parent]), tuple(gt_points[child]),
                 (0, 255, 0), 2, cv2.LINE_AA)
        cv2.line(canvas, tuple(mp_points[parent]), tuple(mp_points[child]),
                 (0, 0, 255), 2, cv2.LINE_AA)
    for point in gt_points:
        cv2.circle(canvas, tuple(point), 4, (0, 255, 0), -1, cv2.LINE_AA)
    for point in mp_points:
        cv2.circle(canvas, tuple(point), 4, (255, 0, 0), -1, cv2.LINE_AA)
    if not cv2.imwrite(str(path), canvas):
        raise IOError(f"Could not save {path}")


overlay_dir = Path("/content/phase3_audit_overlays")
overlay_dir.mkdir(parents=True, exist_ok=True)
saved = []

for segment_name in ("T0011", "T0020"):
    positions = [
        int(pos)
        for pos, dataset_i in enumerate(val_idx)
        if str(records[int(dataset_i)]["segment"]) == segment_name
        and all_candidates[int(pos)]
    ]
    if len(positions) < 2:
        raise RuntimeError(
            f"{segment_name} needs two detected validation frames; "
            f"found {len(positions)}."
        )

    for rank, pos in enumerate(positions[:2], start=1):
        dataset_i = int(val_idx[pos])
        record = records[dataset_i]
        color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
        crop, gt_uv = make_mp_crop(color, record["uv_image"])
        frame_id = str(record.get("frame_id", pos))
        safe_id = "".join(
            ch if ch.isalnum() or ch in "-_" else "_" for ch in frame_id
        )
        path = overlay_dir / f"{segment_name}_{rank}_{safe_id}.png"
        draw_overlay(nearest_candidates[pos], crop, gt_uv, path)
        saved.append(path)

print("\nOVERLAYS")
for path in saved:
    print(path)

print(f"\nSobel checkpoint saved to Google Drive: {DRIVE_CHECKPOINT_PATH}")
print(
    "\nAssumptions: right-hand GT is orig_kp_right; 3D keypoints are mm; "
    "MediaPipe world landmarks are meters and are wrist-rooted. Mirrored "
    "candidate x is reflected back before scoring. The 3D oracle uses GT and "
    "is an upper-bound diagnostic; PA-MPJPE fits proper rotation and uniform "
    "scale per frame. Camera translation units are selected by image coverage."
)

KeyboardInterrupt: 

In [7]:
# =============================================================================
# STANDALONE COLAB RUNNER — PHASE 3 AUDIT
# - Searches only the dataset/capture directory, never / or all of /content.
# - Loads the right-hand data, camera and deterministic 44/12 segment split.
# - Trains the Sobel direct regressor, with per-epoch resumable state on Drive.
# - Runs MediaPipe on normal and mirrored ORIGINAL COLOR crops.
# - Keeps every MediaPipe candidate, creates audits and four overlays.
# - Reports RAM/VRAM and emits regular progress output to avoid idle timeouts.
# =============================================================================

# Install MediaPipe in this runtime if it is absent.
import importlib.util
import subprocess
import sys
import os
import time
import csv
import json
import random
import itertools
import urllib.request
from pathlib import Path

if importlib.util.find_spec("mediapipe") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "mediapipe"]
    )

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------- Settings ----------------------------------

SEED = 20251018
CAMERA_ID = "400485"
N_TRAIN_SEGMENTS = 44
N_VAL_SEGMENTS = 12
N_EPOCHS = 20
LR = 1.2e-3
WEIGHT_DECAY = 1e-4
IMAGE_W, IMAGE_H = 128, 96
TRAIN_CROP_MARGIN = 1.4
MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
RESUME_PATH = DRIVE_ROOT / "phase3_sobel_direct_resume.pt"
FINAL_MODEL_PATH = DRIVE_ROOT / "phase3_sobel_direct.pt"
OVERLAY_DIR = Path("/content/phase3_audit_overlays")

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# Use smaller batches on devices with less free VRAM.
if DEVICE.type == "cuda":
    free_vram_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
    BATCH_SIZE = 64 if free_vram_gb >= 8 else 32 if free_vram_gb >= 4 else 16
else:
    BATCH_SIZE = 16

# -------------------------------- Resources ---------------------------------

try:
    import psutil
except ImportError:
    psutil = None


def memory_status(stage):
    ram = "RAM unavailable"
    if psutil is not None:
        proc = psutil.Process(os.getpid())
        vm = psutil.virtual_memory()
        ram = (
            f"process={proc.memory_info().rss / (1024**3):.2f} GiB, "
            f"available={vm.available / (1024**3):.2f} GiB"
        )

    if DEVICE.type == "cuda":
        free_b, total_b = torch.cuda.mem_get_info()
        gpu = (
            f"{torch.cuda.get_device_name(0)}, "
            f"allocated={torch.cuda.memory_allocated() / (1024**3):.2f} GiB, "
            f"reserved={torch.cuda.memory_reserved() / (1024**3):.2f} GiB, "
            f"free={free_b / (1024**3):.2f}/{total_b / (1024**3):.2f} GiB"
        )
    else:
        gpu = f"CPU, {torch.get_num_threads()} threads"

    print(f"[{stage}] {gpu}; RAM {ram}", flush=True)


print(
    f"Device: {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""),
    flush=True,
)
memory_status("start")

# ---------------------------- Google Drive ---------------------------------

try:
    from google.colab import drive
    if not DRIVE_ROOT.exists():
        print("Mounting Google Drive for resumable model saves...", flush=True)
        drive.mount("/content/drive")
except Exception as exc:
    print(f"Drive mount unavailable: {exc}", flush=True)

DRIVE_AVAILABLE = DRIVE_ROOT.exists()
if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError("MediaPipe mp.tasks.vision.HandLandmarker is required.")

if not MP_MODEL_PATH.exists():
    print("Downloading MediaPipe hand model...", flush=True)
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

# ----------------------- Bounded manifest discovery ------------------------

# Check the known dataset path first. Avoid rglob('/content') or rglob('/').
def find_manifest():
    direct = DATASET_DIR / "manifest.csv"
    if direct.is_file():
        return direct

    if DATASET_DIR.exists():
        print(f"Searching only inside {DATASET_DIR} for manifest.csv...", flush=True)
        matches = list(DATASET_DIR.rglob("manifest.csv"))
        if matches:
            return next(
                (p for p in matches if "mugsy_subset" in str(p).lower()),
                matches[0],
            )

    if DRIVE_AVAILABLE:
        # Drive search occurs only if the local dataset path had no manifest.
        print(f"Searching Google Drive for manifest.csv...", flush=True)
        matches = list(DRIVE_ROOT.rglob("manifest.csv"))
        if matches:
            return next(
                (p for p in matches if "mugsy_subset" in str(p).lower()),
                matches[0],
            )
    return None


MANIFEST_PATH = find_manifest()
if MANIFEST_PATH is None:
    raise FileNotFoundError(
        "manifest.csv not found. Put the dataset under /content/dataset "
        "or mount Drive and set DATASET_DIR to its extracted folder."
    )

SUBSET_DIR = MANIFEST_PATH.parent


def find_camera_path():
    # Walk upward only from the manifest. No broad recursive camera search.
    for parent in (SUBSET_DIR, *SUBSET_DIR.parents):
        for candidate in (
            parent / "Mugsy_cameras" / "cam_params.json",
            parent / "cam_params.json",
        ):
            if candidate.is_file():
                return candidate
        # Do not climb above /content or a mounted Drive root.
        if parent == Path("/content") or parent == DRIVE_ROOT:
            break
    return None


CAMERA_PATH = find_camera_path()
if CAMERA_PATH is None:
    raise FileNotFoundError(
        "cam_params.json was not found beside the manifest or in its "
        "Mugsy_cameras directory."
    )

CAPTURE_DIR = SUBSET_DIR
for parent in (SUBSET_DIR, *SUBSET_DIR.parents):
    if (
        (parent / "keypoints_right").is_dir()
        or (parent / "Mugsy_cameras" / "cam_params.json").is_file()
        or (parent / "cam_params.json").is_file()
    ):
        CAPTURE_DIR = parent
        break
    if parent == Path("/content") or parent == DRIVE_ROOT:
        break

print(f"Manifest: {MANIFEST_PATH}", flush=True)
print(f"Capture directory: {CAPTURE_DIR}", flush=True)
print(f"Camera file: {CAMERA_PATH}", flush=True)

# --------------------- Index only the capture directory ---------------------

# The earlier timeout risk was indexing CAPTURE_DIR plus its ancestors, which
# could make a parent path equal to / and scan the entire runtime.
ASSET_INDEX = {}
allowed_suffixes = {".json", ".png", ".jpg", ".jpeg"}
print(f"Indexing assets under {CAPTURE_DIR} only...", flush=True)

for count, path in enumerate(CAPTURE_DIR.rglob("*"), start=1):
    if path.is_file() and path.suffix.lower() in allowed_suffixes:
        ASSET_INDEX.setdefault(path.name.lower(), []).append(path.resolve())
    if count % 5000 == 0:
        print(f"  scanned {count:,} paths", flush=True)

print(
    f"Indexed {sum(len(v) for v in ASSET_INDEX.values()):,} paths.",
    flush=True,
)


def resolve_asset(relative_path, kind):
    rel = str(relative_path).replace("\\", "/").strip()
    rel_path = Path(rel)

    # Fast direct paths first.
    for candidate in (
        Path(rel),
        CAPTURE_DIR / rel,
        SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel,
    ):
        if candidate.is_file():
            return candidate.resolve()

    hits = list(dict.fromkeys(ASSET_INDEX.get(rel_path.name.lower(), [])))

    if kind == "keypoint":
        parts = [p.lower() for p in rel_path.parts]
        side = "right" if "right" in parts else (
            "left" if "left" in parts else None
        )
        if side:
            side_hits = [
                p for p in hits
                if f"keypoints_{side}"
                in str(p).replace("\\", "/").lower()
            ]
            if side_hits:
                hits = side_hits

    elif kind == "image":
        image_hits = [
            p for p in hits
            if any(
                token in str(p).replace("\\", "/").lower()
                for token in ("/images/", "/image/", "/rgb/")
            )
        ]
        if image_hits:
            hits = image_hits

    if len(hits) == 1:
        return hits[0]

    if kind == "keypoint":
        right_hits = [
            p for p in hits
            if "keypoints_right" in str(p).replace("\\", "/").lower()
        ]
        if len(right_hits) == 1:
            return right_hits[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve {kind} asset {rel!r}; "
        f"matches={[str(p) for p in hits[:5]]}"
    )

# ----------------------------- Read metadata --------------------------------

with MANIFEST_PATH.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))
rows = [
    row for row in all_rows
    if row.get("camera_id", CAMERA_ID) == CAMERA_ID
]
if not rows:
    raise RuntimeError(f"No manifest rows for camera {CAMERA_ID}.")

with CAMERA_PATH.open(encoding="utf-8") as f:
    camera_dict = json.load(f)
if CAMERA_ID not in camera_dict:
    raise KeyError(f"Camera {CAMERA_ID} absent from {CAMERA_PATH}.")

camera = camera_dict[CAMERA_ID]
CAM_R = np.asarray(camera["R"], dtype=np.float32).reshape(3, 3)
CAM_T_RAW = np.asarray(camera["t"], dtype=np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], dtype=np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], dtype=np.float32).reshape(2)

segments = np.asarray(sorted({row["segment"] for row in rows}), dtype=object)
if len(segments) != 56:
    raise RuntimeError(f"Expected 56 segments; found {len(segments)}.")

split_rng = np.random.default_rng(SEED)
shuffled_segments = segments.copy()
split_rng.shuffle(shuffled_segments)
train_segments = set(shuffled_segments[:44].tolist())
val_segments = set(shuffled_segments[44:].tolist())
train_idx = np.asarray(
    [i for i, row in enumerate(rows) if row["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, row in enumerate(rows) if row["segment"] in val_segments],
    dtype=np.int64,
)

# Resolve all paths once, then load the relatively small keypoint JSONs.
print(f"Resolving {len(rows):,} manifest assets...", flush=True)
keypoint_paths, image_paths = [], []
for i, row in enumerate(rows):
    keypoint_paths.append(resolve_asset(row["orig_kp_right"], "keypoint"))
    image_paths.append(resolve_asset(row["image_path"], "image"))
    if (i + 1) % 1000 == 0:
        print(f"  resolved {i + 1:,}/{len(rows):,}", flush=True)

world_points = []
for i, path in enumerate(keypoint_paths):
    with path.open(encoding="utf-8") as f:
        points = np.asarray(json.load(f), dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError(f"Expected Nx3 points in {path}; got {points.shape}.")
    if len(points) <= max(KEEP_21_INDICES):
        raise ValueError(f"Too few keypoints in {path}.")
    world_points.append(points)
    if (i + 1) % 1000 == 0:
        print(f"  loaded keypoints {i + 1:,}/{len(rows):,}", flush=True)
world_points = np.stack(world_points).astype(np.float32)

first_image = cv2.imread(str(image_paths[0]), cv2.IMREAD_COLOR)
if first_image is None:
    raise FileNotFoundError(image_paths[0])
IMAGE_ORIG_H, IMAGE_ORIG_W = first_image.shape[:2]


def project(points):
    z = np.maximum(points[..., 2], 1.0)
    u = CAM_FOCAL[0] * points[..., 0] / z + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * points[..., 1] / z + CAM_PRINCPT[1]
    return np.stack((u, v), axis=-1).astype(np.float32)


def in_frame_fraction(t_scale):
    sample = world_points[:min(len(world_points), 256), KEEP_21_INDICES]
    camera_sample = sample @ CAM_R.T + CAM_T_RAW * t_scale
    uv = project(camera_sample)
    inside = (
        (camera_sample[..., 2] > 1.0)
        & (uv[..., 0] >= 0) & (uv[..., 0] < IMAGE_ORIG_W)
        & (uv[..., 1] >= 0) & (uv[..., 1] < IMAGE_ORIG_H)
    )
    return float(inside.mean())


T_SCALE_OPTIONS = (0.001, 1.0, 1000.0)
CAM_T_SCALE = max(T_SCALE_OPTIONS, key=in_frame_fraction)
CAM_T = CAM_T_RAW * CAM_T_SCALE
print(
    f"Camera translation scale: {CAM_T_SCALE:g}; "
    f"projection in-frame fraction={in_frame_fraction(CAM_T_SCALE):.2%}",
    flush=True,
)

# -------------------------- Crop and Sobel helpers --------------------------

def affine_for_crop(uv, out_w, out_h, margin):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        dtype=np.float32,
    )


def make_mp_crop(color_bgr, uv):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0)
    side *= MP_CROP_MARGIN
    size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    scale = size / side
    affine = np.asarray(
        [[scale, 0.0, -left * scale], [0.0, scale, -top * scale]],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        color_bgr, affine, (size, size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    return crop, uv @ affine[:, :2].T + affine[:, 2]


def normalize_uv(uv):
    result = np.empty_like(uv, dtype=np.float32)
    result[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    result[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return result


def sobel_channel(gray_u8):
    gray = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return mag[None].astype(np.float32)


# --------------------------- Stage small crops ------------------------------

print("Staging grayscale crops and labels (full color frames are not cached)...",
      flush=True)
records = []
for i, (row, world, image_path) in enumerate(
    zip(rows, world_points, image_paths)
):
    color = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if color is None:
        raise FileNotFoundError(image_path)
    if color.shape[:2] != (IMAGE_ORIG_H, IMAGE_ORIG_W):
        raise ValueError(f"Unexpected image dimensions at {image_path}.")

    camera_points = world @ CAM_R.T + CAM_T
    selected = camera_points[KEEP_21_INDICES]
    uv = project(selected)
    y3d = selected - selected[0:1]

    gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
    del color

    affine = affine_for_crop(uv, IMAGE_W, IMAGE_H, TRAIN_CROP_MARGIN)
    crop_gray = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    crop_uv = uv @ affine[:, :2].T + affine[:, 2]

    records.append(
        {
            "segment": row["segment"],
            "frame_id": row.get("frame_id", str(i)),
            "image_path": image_path,
            "uv_image": uv.astype(np.float32),
            "uv_crop": crop_uv.astype(np.float32),
            "crop_gray": crop_gray,
            "y3d": y3d.astype(np.float32),
        }
    )

    if (i + 1) % 500 == 0 or i + 1 == len(rows):
        print(f"  staged {i + 1:,}/{len(rows):,} frames", flush=True)

y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)
y2d_all = np.stack([normalize_uv(r["uv_crop"]) for r in records])
memory_status("after data staging")

# -------------------------- Sobel model -------------------------------------

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-1)


def reconstruct_joints(vectors):
    joints = torch.zeros(
        (vectors.shape[0], 21, 3), dtype=vectors.dtype, device=vectors.device
    )
    for k, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + vectors[:, k]
    return joints


class SobelDirectTracker(nn.Module):
    def __init__(self, lengths):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer("bone_lens", lengths.clone())

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3),
            dim=-1,
            eps=1e-7,
        )
        frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1,
            eps=1e-7,
        )
        phalanx = torch.bmm(
            frame, phalanx.transpose(1, 2)
        ).transpose(1, 2)

        dirs = torch.zeros(
            (batch, 20, 3), dtype=x.dtype, device=x.device
        )
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger_i, offset in enumerate((4, 8, 12, 16)):
            start = 3 + finger_i * 3
            dirs[:, offset] = meta[:, finger_i + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[
                :, start:start + 3
            ]

        joints = reconstruct_joints(
            dirs * self.bone_lens.view(1, 20, 1)
        )
        uv = torch.tanh(
            self.proj2d_head(feat).reshape(batch, 21, 2)
        )
        return joints, dirs, uv


def rotate_camera_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x, y = points[:, 0].copy(), points[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment_sample(gray, uv, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    matrix[1, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_H)

    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset, 0, 255
    ).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    return (
        sobel_channel(warped),
        rotate_camera_xy(y3d, angle).astype(np.float32),
        normalize_uv(uv_aug),
    )


def direct_loss(pred3d, target3d, pred2d, target2d):
    pos = F.l1_loss(pred3d, target3d)
    pred_bones = (
        pred3d[:, [c for _, c in BONE_PAIRS]]
        - pred3d[:, [p for p, _ in BONE_PAIRS]]
    )
    target_bones = (
        target3d[:, [c for _, c in BONE_PAIRS]]
        - target3d[:, [p for p, _ in BONE_PAIRS]]
    )
    pred_len = torch.linalg.vector_norm(pred_bones, dim=-1)
    target_len = torch.linalg.vector_norm(target_bones, dim=-1)
    length = F.l1_loss(pred_len, target_len)
    pred_unit = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_unit = F.normalize(target_bones, dim=-1, eps=1e-7)
    direction = (
        1.0 - torch.clamp((pred_unit * target_unit).sum(-1), -1.0, 1.0)
    ).mean()
    reprojection = F.smooth_l1_loss(pred2d, target2d)
    return pos + 0.5 * length + 20.0 * direction + 15.0 * reprojection


canonical_lengths = np.asarray(
    [
        np.linalg.norm(
            y_all[train_idx, child] - y_all[train_idx, parent],
            axis=-1,
        ).mean()
        for parent, child in BONE_PAIRS
    ],
    dtype=np.float32,
)
model = SobelDirectTracker(
    torch.from_numpy(canonical_lengths)
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-4
)
rng = np.random.default_rng(SEED + 1)
start_epoch = 0

# Resume automatically if a valid prior interrupted-run state is on Drive.
if DRIVE_AVAILABLE and RESUME_PATH.is_file():
    try:
        try:
            resume = torch.load(
                RESUME_PATH, map_location=DEVICE, weights_only=False
            )
        except TypeError:
            resume = torch.load(RESUME_PATH, map_location=DEVICE)

        same_run = (
            resume.get("seed") == SEED
            and resume.get("camera_id") == CAMERA_ID
            and resume.get("train_segments") == sorted(train_segments)
            and resume.get("val_segments") == sorted(val_segments)
        )
        if same_run:
            model.load_state_dict(resume["model_state_dict"])
            optimizer.load_state_dict(resume["optimizer_state_dict"])
            scheduler.load_state_dict(resume["scheduler_state_dict"])
            start_epoch = int(resume["completed_epoch"])
            print(
                f"Resuming Sobel training at epoch {start_epoch + 1}/"
                f"{N_EPOCHS}.",
                flush=True,
            )
        else:
            print(
                "Ignoring incompatible resume state; starting training fresh.",
                flush=True,
            )
    except Exception as exc:
        print(
            f"Could not load resume state ({exc}); starting fresh.",
            flush=True,
        )

# ----------------------------- Train with logs ------------------------------

print(
    f"Training Sobel direct model: {N_EPOCHS} epochs, "
    f"{len(train_idx):,} train frames, batch {BATCH_SIZE}.",
    flush=True,
)

for epoch in range(start_epoch, N_EPOCHS):
    epoch_start = time.perf_counter()
    model.train()
    order = np.random.permutation(train_idx)
    losses = []
    n_batches = (len(order) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_number, start in enumerate(
        range(0, len(order), BATCH_SIZE), start=1
    ):
        ids = order[start:start + BATCH_SIZE]
        samples = [
            augment_sample(
                records[int(i)]["crop_gray"],
                records[int(i)]["uv_crop"],
                y_all[int(i)],
                rng,
            )
            for i in ids
        ]
        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        by3d = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        by2d = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred3d, _, pred2d = model(bx)
        loss = direct_loss(pred3d, by3d, pred2d, by2d)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

        # Heartbeat output prevents long silent intervals on slow CPUs.
        if batch_number % 25 == 0 or batch_number == n_batches:
            print(
                f"  epoch {epoch + 1:02d}/{N_EPOCHS}, "
                f"batch {batch_number}/{n_batches}, "
                f"loss={np.mean(losses):.3f}",
                flush=True,
            )

    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    print(
        f"Epoch {epoch + 1:02d}/{N_EPOCHS} complete; "
        f"mean loss={np.mean(losses):.4f}; "
        f"{time.perf_counter() - epoch_start:.1f}s",
        flush=True,
    )

    # Save resume state after each epoch. An interrupted run can resume.
    if DRIVE_AVAILABLE:
        temporary_path = RESUME_PATH.with_suffix(".tmp")
        state = {
            "model_state_dict": {
                k: v.detach().cpu() for k, v in model.state_dict().items()
            },
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "completed_epoch": epoch + 1,
            "seed": SEED,
            "camera_id": CAMERA_ID,
            "train_segments": sorted(train_segments),
            "val_segments": sorted(val_segments),
        }
        torch.save(state, temporary_path)
        os.replace(temporary_path, RESUME_PATH)
        print("  resumable state saved to Drive", flush=True)

model.eval()
memory_status("after training")

# Save an inference-ready checkpoint to Drive (or local /content fallback).
final_checkpoint = {
    "model_state_dict": {
        k: v.detach().cpu() for k, v in model.state_dict().items()
    },
    "canonical_bone_lengths_mm": canonical_lengths,
    "seed": SEED,
    "train_segments": sorted(train_segments),
    "val_segments": sorted(val_segments),
    "camera_id": CAMERA_ID,
    "manifest_path": str(MANIFEST_PATH),
}
if DRIVE_AVAILABLE:
    FINAL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(final_checkpoint, FINAL_MODEL_PATH)
    saved_checkpoint_path = FINAL_MODEL_PATH
else:
    saved_checkpoint_path = Path("/content/phase3_sobel_direct.pt")
    torch.save(final_checkpoint, saved_checkpoint_path)

# ------------------------- MediaPipe candidate pass -------------------------

mp_options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MP_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)


def result_candidates(result, size, gt_uv, mirrored):
    candidates = []
    image_hands = getattr(result, "hand_landmarks", None) or []
    world_hands = getattr(result, "hand_world_landmarks", None) or []
    handedness = getattr(result, "handedness", None) or []

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue

        xy = np.asarray(
            [[lm.x * size, lm.y * size] for lm in image_hand],
            dtype=np.float32,
        )
        if mirrored:
            xy[:, 0] = size - 1 - xy[:, 0]

        pixel_error = float(np.linalg.norm(xy - gt_uv, axis=-1).mean())
        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )
        pose = (world_m - world_m[0:1]) * 1000.0
        if mirrored:
            pose[:, 0] *= -1.0

        label = "unknown"
        if hand_i < len(handedness):
            try:
                label = handedness[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                pass

        candidates.append(
            {
                "pose": pose.astype(np.float32),
                "xy": xy,
                "pixel_error": pixel_error,
                "mirrored_input": bool(mirrored),
                "hand_index": hand_i,
                "handedness": label,
            }
        )
    return candidates


all_candidates = [[] for _ in val_idx]
normal_2d_errors, best_2d_errors = [], []
val_pos_by_dataset_index = {
    int(dataset_i): pos for pos, dataset_i in enumerate(val_idx)
}

print(
    f"Running MediaPipe on {len(val_idx):,} validation frames "
    "(normal + mirrored color crops)...",
    flush=True,
)

with mp.tasks.vision.HandLandmarker.create_from_options(mp_options) as landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        record = records[int(dataset_i)]
        color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
        if color is None:
            raise FileNotFoundError(record["image_path"])

        crop, gt_crop_uv = make_mp_crop(color, record["uv_image"])
        crop_size = crop.shape[0]
        candidates = []

        for mirrored in (False, True):
            view = cv2.flip(crop, 1) if mirrored else crop
            rgb = cv2.cvtColor(view, cv2.COLOR_BGR2RGB)
            result = landmarker.detect(
                mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=np.ascontiguousarray(rgb),
                )
            )
            candidates.extend(
                result_candidates(
                    result, crop_size, gt_crop_uv, mirrored
                )
            )

        all_candidates[val_pos] = candidates
        normal = [c for c in candidates if not c["mirrored_input"]]
        if normal:
            normal_2d_errors.append(
                min(normal, key=lambda c: c["pixel_error"])["pixel_error"]
            )
        if candidates:
            best_2d_errors.append(
                min(candidates, key=lambda c: c["pixel_error"])["pixel_error"]
            )

        if (val_pos + 1) % 50 == 0 or val_pos + 1 == len(val_idx):
            print(
                f"  MediaPipe {val_pos + 1}/{len(val_idx)} frames",
                flush=True,
            )

detected_positions = np.asarray(
    [i for i, candidates in enumerate(all_candidates) if candidates],
    dtype=np.int64,
)
if not len(detected_positions):
    raise RuntimeError("MediaPipe found no hands on validation frames.")

nearest_candidates = [
    min(c, key=lambda item: item["pixel_error"]) if c else None
    for c in all_candidates
]

# ------------------------- Batched Sobel inference ---------------------------

sobel_predictions = [None] * len(val_idx)
print("Running batched Sobel inference...", flush=True)

with torch.inference_mode():
    for start in range(0, len(val_idx), 128):
        pos_batch = list(range(start, min(start + 128, len(val_idx))))
        x_np = np.stack([
            sobel_channel(records[int(val_idx[pos])]["crop_gray"])
            for pos in pos_batch
        ])
        x = torch.from_numpy(x_np).to(DEVICE)
        joints, _, _ = model(x)
        pred = joints.cpu().numpy().astype(np.float32)
        for j, pos in enumerate(pos_batch):
            sobel_predictions[pos] = pred[j]

gt_val = [records[int(i)]["y3d"] for i in val_idx]


def mpjpe(pred, target):
    return float(np.linalg.norm(pred - target, axis=-1).mean())


def rescale_bones(pose):
    result = np.zeros_like(pose, dtype=np.float32)
    for bone_i, (parent, child) in enumerate(BONE_PAIRS):
        direction = pose[child] - pose[parent]
        norm = float(np.linalg.norm(direction))
        if norm < 1e-8:
            direction = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
        else:
            direction = direction / norm
        result[child] = result[parent] + direction * canonical_lengths[bone_i]
    return result


def pa_mpjpe_one(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    x = pred - pred.mean(axis=0, keepdims=True)
    y = target - target.mean(axis=0, keepdims=True)

    u, singular, vt = np.linalg.svd(x.T @ y)
    correction = np.eye(3)
    correction[-1, -1] = np.linalg.det(u @ vt)
    rotation = u @ correction @ vt
    denominator = np.sum(x * x)
    if denominator < 1e-12:
        return float("nan")
    scale = np.sum(singular * np.diag(correction)) / denominator
    aligned = scale * (x @ rotation) + target.mean(axis=0, keepdims=True)
    return float(np.linalg.norm(aligned - target, axis=-1).mean())


mp_raw = [None] * len(val_idx)
mp_scaled = [None] * len(val_idx)
for pos in detected_positions:
    pose = nearest_candidates[int(pos)]["pose"]
    mp_raw[int(pos)] = pose
    mp_scaled[int(pos)] = rescale_bones(pose)

# ---------------------------- Audit tables ----------------------------------

print("\nFRAME AUDIT — 2D-NEAREST CANDIDATE")
print(f"{'XYZ signs':<12} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 34)
for signs in itertools.product((1.0, -1.0), repeat=3):
    errors = [
        mpjpe(
            nearest_candidates[int(pos)]["pose"]
            * np.asarray(signs, dtype=np.float32),
            gt_val[int(pos)],
        )
        for pos in detected_positions
    ]
    label = "".join("+" if s > 0 else "-" for s in signs)
    print(f"{label:<12} {len(errors):>8} {np.mean(errors):10.2f}")

print("\n2D CHECK — MEAN LANDMARK DISTANCE")
print(f"{'Candidate pool':<36} {'Frames':>8} {'Error px':>10}")
print("-" * 58)
print(
    f"{'Original crop only':<36} {len(normal_2d_errors):>8} "
    f"{np.mean(normal_2d_errors):10.2f}"
)
print(
    f"{'Best original/mirrored per frame':<36} {len(best_2d_errors):>8} "
    f"{np.mean(best_2d_errors):10.2f}"
)

nearest_errors, oracle_errors = [], []
different_candidates = 0
for pos in detected_positions:
    pos = int(pos)
    candidates = all_candidates[pos]
    nearest = min(candidates, key=lambda c: c["pixel_error"])
    oracle = min(candidates, key=lambda c: mpjpe(c["pose"], gt_val[pos]))
    nearest_errors.append(mpjpe(nearest["pose"], gt_val[pos]))
    oracle_errors.append(mpjpe(oracle["pose"], gt_val[pos]))
    different_candidates += int(
        (nearest["mirrored_input"], nearest["hand_index"])
        != (oracle["mirrored_input"], oracle["hand_index"])
    )

print("\nCANDIDATE CHECK")
print(f"{'Selection':<28} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 50)
print(
    f"{'2D-nearest':<28} {len(nearest_errors):>8} "
    f"{np.mean(nearest_errors):10.2f}"
)
print(
    f"{'3D oracle':<28} {len(oracle_errors):>8} "
    f"{np.mean(oracle_errors):10.2f}"
)
print(
    f"Different candidate identity: {different_candidates}/"
    f"{len(detected_positions)} "
    f"({different_candidates / len(detected_positions):.2%})"
)

print("\nPA-MPJPE — PER-FRAME ROTATION + SCALE")
print(f"{'Method':<26} {'Frames':>8} {'PA-MPJPE mm':>13}")
print("-" * 52)
for name, predictions in (
    ("MediaPipe raw", mp_raw),
    ("MediaPipe bone-scaled", mp_scaled),
    ("Sobel direct", sobel_predictions),
):
    errors = [
        pa_mpjpe_one(predictions[int(pos)], gt_val[int(pos)])
        for pos in detected_positions
    ]
    print(
        f"{name:<26} {len(errors):>8} "
        f"{np.nanmean(errors):13.2f}"
    )

# ------------------------- Detection by segment ----------------------------

segment_stats = {}
for val_pos, dataset_i in enumerate(val_idx):
    segment = records[int(dataset_i)]["segment"]
    entry = segment_stats.setdefault(segment, {"frames": 0, "detected": 0})
    entry["frames"] += 1
    entry["detected"] += int(bool(all_candidates[val_pos]))

print("\nDETECTION BY VALIDATION SEGMENT")
print(f"{'Segment':<34} {'Detected':>9} {'Frames':>8} {'Rate':>9}")
print("-" * 64)
for segment in sorted(segment_stats, key=str):
    entry = segment_stats[segment]
    rate = entry["detected"] / max(entry["frames"], 1)
    print(
        f"{str(segment):<34} {entry['detected']:>9} "
        f"{entry['frames']:>8} {rate:>8.2%}"
    )

# -------------------- Four overlays across all records ----------------------

def draw_overlay(candidate, crop, gt_uv, path):
    canvas = crop.copy()
    gt_pts = np.rint(gt_uv).astype(int)
    mp_pts = np.rint(candidate["xy"]).astype(int)

    for parent, child in BONE_PAIRS:
        cv2.line(canvas, tuple(gt_pts[parent]), tuple(gt_pts[child]),
                 (0, 255, 0), 2, cv2.LINE_AA)
        cv2.line(canvas, tuple(mp_pts[parent]), tuple(mp_pts[child]),
                 (0, 0, 255), 2, cv2.LINE_AA)
    for point in gt_pts:
        cv2.circle(canvas, tuple(point), 4, (0, 255, 0), -1, cv2.LINE_AA)
    for point in mp_pts:
        cv2.circle(canvas, tuple(point), 4, (255, 0, 0), -1, cv2.LINE_AA)

    cv2.putText(
        canvas,
        f"GT green; MP red/blue; mirrored={candidate['mirrored_input']}",
        (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
        (255, 255, 255), 1, cv2.LINE_AA,
    )
    if not cv2.imwrite(str(path), canvas):
        raise IOError(f"Could not save overlay {path}")


OVERLAY_DIR.mkdir(parents=True, exist_ok=True)
overlay_paths = []
print("\nSaving overlays for T0011 and T0020...", flush=True)

# A constant-time map avoids scanning all validation indices for every record.
val_position_by_index = {
    int(dataset_i): pos for pos, dataset_i in enumerate(val_idx)
}

# Keep one detector open; use cached val candidates where possible, and run
# detection only for requested target-segment frames that were in train.
with mp.tasks.vision.HandLandmarker.create_from_options(mp_options) as landmarker:
    for segment_name in ("T0011", "T0020"):
        saved_count = 0

        for record_i, record in enumerate(records):
            if str(record["segment"]) != segment_name:
                continue

            color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
            if color is None:
                continue
            crop, gt_crop_uv = make_mp_crop(color, record["uv_image"])

            val_pos = val_position_by_index.get(record_i)
            if val_pos is not None and all_candidates[val_pos]:
                candidate = min(
                    all_candidates[val_pos],
                    key=lambda c: c["pixel_error"],
                )
            else:
                size = crop.shape[0]
                found = []
                for mirrored in (False, True):
                    view = cv2.flip(crop, 1) if mirrored else crop
                    rgb = cv2.cvtColor(view, cv2.COLOR_BGR2RGB)
                    result = landmarker.detect(
                        mp.Image(
                            image_format=mp.ImageFormat.SRGB,
                            data=np.ascontiguousarray(rgb),
                        )
                    )
                    found.extend(
                        result_candidates(
                            result, size, gt_crop_uv, mirrored
                        )
                    )
                if not found:
                    continue
                candidate = min(found, key=lambda c: c["pixel_error"])

            saved_count += 1
            frame_id = str(record.get("frame_id", record_i))
            safe_id = "".join(
                c if c.isalnum() or c in "-_" else "_"
                for c in frame_id
            )
            path = OVERLAY_DIR / f"{segment_name}_{saved_count}_{safe_id}.png"
            draw_overlay(candidate, crop, gt_crop_uv, path)
            overlay_paths.append(path)

            if saved_count == 2:
                break

        if saved_count < 2:
            print(
                f"Warning: only {saved_count} detected overlay frame(s) "
                f"found for {segment_name}.",
                flush=True,
            )

print("\nOVERLAYS")
for path in overlay_paths:
    print(path)

memory_status("finished")
print(f"Saved model checkpoint: {saved_checkpoint_path}")
print(
    "Assumptions: right-hand GT is orig_kp_right; source 3D points are mm; "
    "MediaPipe world coordinates are meters and are re-rooted at wrist 0. "
    "Mirrored-input 3D x is reflected before scoring. The 3D oracle uses GT "
    "and is an upper-bound diagnostic. PA-MPJPE uses a proper rotation and "
    "one uniform scale per frame. T0011/T0020 overlays search all records, "
    "not only validation records."
)

Device: cpu
[start] CPU, 2 threads; RAM process=1.06 GiB, available=10.24 GiB
Manifest: /content/dataset/manifest.csv
Capture directory: /content/dataset
Camera file: /content/dataset/cam_params.json
Indexing assets under /content/dataset only...
  scanned 5,000 paths
  scanned 10,000 paths
  scanned 15,000 paths
  scanned 20,000 paths
  scanned 25,000 paths
Indexed 25,571 paths.
Resolving 5,114 manifest assets...
  resolved 1,000/5,114
  resolved 2,000/5,114
  resolved 3,000/5,114
  resolved 4,000/5,114
  resolved 5,000/5,114
  loaded keypoints 1,000/5,114
  loaded keypoints 2,000/5,114
  loaded keypoints 3,000/5,114
  loaded keypoints 4,000/5,114
  loaded keypoints 5,000/5,114
Camera translation scale: 1; projection in-frame fraction=99.87%
Staging grayscale crops and labels (full color frames are not cached)...
  staged 500/5,114 frames
  staged 1,000/5,114 frames
  staged 1,500/5,114 frames
  staged 2,000/5,114 frames
  staged 2,500/5,114 frames
  staged 3,000/5,114 frames
  staged